In [14]:
!pip install colab-xterm
%load_ext colabxterm
%xterm

Launching Xterm...

<IPython.core.display.Javascript object>

In [43]:
import os
# Importaciones modernas: eliminamos por completo langchain.chains
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from IPython.display import Markdown, display

# =====================================================================
# 1. TU BASE DE CONOCIMIENTO OPTIMIZADA (Aquí pegas tus guías Markdown)
# =====================================================================
CONOCIMIENTO_FACTORIO = r"""
### Base de conocimiento técnica de EngiAI: Factorio 2.x y Space Age
Corpus enciclopédico organizado por mecánicas. Se conservan las cifras, recetas, condiciones, nombres técnicos y fuentes presentes en las guías de origen...
### Base de conocimiento técnica de EngiAI: Factorio 2.x y Space Age

Corpus enciclopédico organizado por mecánicas. Se conservan las cifras, recetas, condiciones, nombres técnicos y fuentes presentes en las guías de origen. Las preguntas cortas se integran en los capítulos temáticos; se excluyen saludos y consultas ajenas a Factorio.

### Guía técnica de Factorio: Minería y extracción de recursos

**Versión de referencia:** *Factorio 2.x*, incluyendo las mecánicas de la expansión *Space Age*, identificadas por separado cuando corresponda.

La minería y la extracción de recursos constituyen la base de la automatización en *Factorio*. Prácticamente todas las cadenas de producción comienzan con la obtención de materias primas, que posteriormente se transforman en materiales, componentes, combustibles, productos químicos y tecnologías.

Esta guía explica cómo encontrar y extraer recursos, qué edificios intervienen, cuáles son los insumos necesarios y cómo diseñar sistemas de minería eficientes y escalables.

---

#### 1. Fundamentos de la minería y la extracción

##### 1.1. ¿Qué es la minería?

La minería es el proceso mediante el cual el jugador obtiene materias primas del terreno para utilizarlas en la fabricación de objetos y en el desarrollo de la fábrica.

Existen dos métodos principales:

- **Minería manual:** el jugador extrae determinados recursos directamente del entorno mediante las acciones disponibles del personaje. Es útil durante los primeros minutos de una partida y para recoger cantidades pequeñas de materiales.
- **Minería automatizada:** los taladros y las estructuras de extracción trabajan continuamente, siempre que dispongan de los recursos necesarios para funcionar y de una salida para su producción.

La automatización es el método fundamental para mantener una producción industrial constante. La minería manual no puede satisfacer las necesidades de una fábrica avanzada sin una cantidad considerable de intervención del jugador.

La mecánica está documentada en la [Factorio Wiki: Mining](https://wiki.factorio.com/Mining).

##### 1.2. Yacimientos de recursos

Los yacimientos son zonas del mapa donde se encuentran concentradas materias primas. Se distinguen visualmente por el color y la textura del terreno.

Los principales yacimientos de *Factorio* en el juego base son:

| Recurso | Apariencia característica | Utilidad principal |
|---|---|---|
| Mineral de hierro | Gris azulado | Producción de placas de hierro y acero. |
| Mineral de cobre | Naranja rojizo | Fabricación de placas de cobre, cables y componentes electrónicos. |
| Carbón | Negro | Combustible y materia prima para procesos industriales. |
| Piedra | Gris claro | Fabricación de ladrillos de piedra, hornos y otros componentes. |
| Mineral de uranio | Verde brillante | Producción de materiales para energía nuclear y munición especializada. |
| Petróleo crudo | Manchas oscuras en el terreno | Producción de derivados del petróleo mediante procesamiento químico. |

Los yacimientos de minerales sólidos normalmente contienen cantidades limitadas de recursos. Cuando se agotan las casillas de mineral accesibles a un taladro, este deja de extraerlos.

El petróleo funciona de manera diferente: sus campos disminuyen gradualmente su rendimiento, pero, con la configuración normal del mapa, no suelen agotarse por completo.


##### 1.3. ¿Cómo se desarrolla el proceso de extracción?

El procedimiento habitual para obtener y utilizar materias primas consta de cinco etapas.

1. **Localizar el recurso:** explorar el mapa e identificar los yacimientos que necesita la fábrica.
2. **Instalar la estructura de extracción:** colocar el taladro sobre una zona que contenga recursos compatibles con él.
3. **Suministrar energía o combustible:** proporcionar electricidad o combustible, según el tipo de máquina.
4. **Transportar la producción:** recoger los materiales mediante cintas transportadoras, cofres, insertores o alimentación directa a otras máquinas.
5. **Procesar las materias primas:** convertir los recursos en productos utilizables, como las placas de hierro obtenidas mediante fundición.

Por ejemplo, una cadena de producción de hierro puede organizarse así:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción de hierro | Yacimiento de hierro | Taladros mineros eléctricos | Mineral de hierro | — |
| Transporte del mineral | Mineral de hierro | Cintas transportadoras | Mineral transportado | — |
| Fundición | Mineral de hierro | Hornos | Placas de hierro | — |
| Fabricación de componentes | Placas de hierro | Fabricación de componentes | Componentes | — |

El objetivo consiste en mantener un flujo continuo de materias primas, evitando que los edificios posteriores se queden sin insumos.

---

#### 2. Edificios y estructuras utilizados en la minería

##### 2.1. Taladro minero a combustión

El **taladro minero a combustión** (*Burner mining drill*) es la primera máquina de extracción disponible en una partida normal.


**Características principales:**

- Tamaño físico: 2 × 2 casillas.
- Área de extracción: 2 × 2 casillas.
- Velocidad de minería: 0,25.
- Consumo energético nominal: 150 kW, suministrados mediante combustible.
- Requiere combustible para funcionar.
- Produce contaminación durante su funcionamiento.

**Función:** extraer minerales sin depender de una red eléctrica.

Es especialmente útil durante las primeras etapas de la partida, cuando el jugador todavía no dispone de una infraestructura eléctrica estable.

Puede funcionar con combustibles compatibles, como carbón o madera. Al colocarlo sobre un depósito de carbón, es posible diseñar sistemas que utilicen parte del carbón extraído para mantener los taladros alimentados.

Sin embargo, presenta dos limitaciones importantes: su velocidad es reducida y genera una cantidad relativamente elevada de contaminación por unidad de mineral extraída.

##### 2.2. Taladro minero eléctrico

El **taladro minero eléctrico** (*Electric mining drill*) es la estructura estándar de extracción automatizada del juego base.


**Características principales:**

- Tamaño físico: 3 × 3 casillas.
- Área de extracción: 5 × 5 casillas.
- Velocidad de minería: 0,5.
- Consumo eléctrico nominal: 90 kW.
- Dispone de tres ranuras para módulos en su configuración básica.
- Puede extraer mineral de hierro, mineral de cobre, carbón, piedra y uranio.

El área de extracción es mayor que la superficie que ocupa la máquina. Esto permite que el taladro aproveche varias casillas del yacimiento sin necesidad de cubrirlas individualmente con máquinas diferentes.

A diferencia del modelo a combustión, necesita conectarse a una red eléctrica mediante postes y disponer de una generación eléctrica suficiente.

**Ventajas principales:**

- Mayor velocidad de extracción.
- Funcionamiento automatizado sin reposición de combustible.
- Integración sencilla con cintas transportadoras.
- Compatibilidad con módulos de mejora.
- Posibilidad de suministrar directamente mineral a cofres, hornos y otras máquinas.

Un aspecto importante es que la máquina extrae el recurso y lo deposita en su casilla de salida. Por lo tanto, es posible colocar una cinta transportadora directamente delante del taladro, sin necesidad de instalar un insertor entre ambos.

##### 2.3. Gran taladro minero (*Big mining drill*)

El gran taladro minero es una estructura avanzada exclusiva de la expansión *Space Age*.


Sus características principales son:

- Tamaño físico: 5 × 5 casillas.
- Área de extracción: 13 × 13 casillas.
- Velocidad de minería: 2,5.
- Consumo eléctrico nominal: 300 kW.
- Cuatro ranuras para módulos.
- Capacidad para extraer minerales más difíciles, como el tungsteno.
- Capacidad para trabajar con fluidos cuando el recurso lo requiere.
- Agotamiento reducido del yacimiento en comparación con los taladros convencionales.

Su característica más importante es que utiliza un **desgaste de recursos del 50 % en condiciones básicas**. En términos generales, esto significa que el recurso bajo su área se consume con menor frecuencia, lo que prolonga la vida útil del yacimiento.

El gran taladro minero resulta especialmente útil en las etapas avanzadas de la fábrica, cuando la demanda de materias primas es elevada y se busca extraer grandes cantidades con menos edificios.

Para obtener más detalles, consulta la [Factorio Wiki: Big mining drill](https://wiki.factorio.com/Big_mining_drill).

##### 2.4. Bomba extractora (*Pumpjack*)

La bomba extractora es la estructura encargada de obtener fluidos de los campos de recursos.


En el juego base, su función principal es extraer petróleo crudo.

**Características principales:**

- Tamaño físico: 3 × 3 casillas.
- Consumo eléctrico nominal: 90 kW.
- Necesita una conexión eléctrica.
- Se instala directamente sobre una casilla de un campo de fluido.
- El rendimiento depende de la riqueza del campo.
- Admite módulos de mejora.

En condiciones normales, el rendimiento inicial se calcula multiplicando el porcentaje mostrado en el campo petrolífero por 10 para obtener las unidades de fluido producidas por segundo. Por ejemplo, un campo con un rendimiento del 80 % produce inicialmente aproximadamente 8 unidades de petróleo crudo por segundo.

El rendimiento disminuye a medida que se extrae petróleo. No obstante, los campos petrolíferos estándar mantienen una producción mínima, por lo que pueden seguir utilizándose incluso después de una explotación prolongada.

El petróleo obtenido se transporta mediante tuberías, depósitos, barriles o vagones de fluidos, y posteriormente puede procesarse en una refinería.

Consulta la [Factorio Wiki: Pumpjack](https://wiki.factorio.com/Pumpjack).

##### 2.5. Bomba de aguas profundas (*Offshore pump*)


Esta estructura extrae fluidos de cuerpos de líquido. En Nauvis, su uso principal es obtener agua.

**Características principales:**

- Se coloca junto a un cuerpo de agua.
- No necesita electricidad ni combustible para funcionar.
- Produce nominalmente 1200 unidades de fluido por segundo en su configuración básica.
- La fuente de fluido es inagotable en las condiciones normales del juego.
- Debe conectarse a tuberías u otros edificios compatibles con fluidos.

El agua es fundamental para producir vapor, generar electricidad mediante máquinas de vapor y fabricar numerosos productos químicos.

En *Space Age*, las bombas de este tipo también pueden extraer otros fluidos de determinados entornos planetarios. Por ejemplo, según el planeta, pueden obtener lava, petróleo pesado o solución amoniacal.

Más información: [Factorio Wiki: Offshore pump](https://wiki.factorio.com/Offshore_pump).

##### 2.6. Estructuras de apoyo

La extracción automática no termina con la colocación de un taladro. Es necesario transportar, almacenar y, en muchos casos, procesar los recursos obtenidos.

| Estructura | Función dentro del sistema de minería |
|---|---|
| **Cintas transportadoras** | Desplazan minerales desde los taladros hasta otras instalaciones. |
| **Insertores** | Transfieren objetos entre cintas, cofres, hornos y máquinas. |
| **Cofres** | Almacenan materias primas y pueden utilizarse como reservas temporales. |
| **Hornos** | Transforman minerales como el hierro y el cobre en placas metálicas. |
| **Postes eléctricos** | Proporcionan electricidad a los taladros eléctricos y las bombas extractoras. |
| **Tuberías** | Transportan petróleo, agua, ácido sulfúrico y otros fluidos. |
| **Depósitos de fluidos** | Almacenan líquidos y permiten disponer de reservas para procesos posteriores. |
| **Refinerías de petróleo** | Procesan el petróleo crudo para obtener derivados. |
| **Centrífugas** | Procesan el mineral de uranio para obtener materiales nucleares. |
| **Balizas** | Distribuyen efectos de módulos a máquinas cercanas cuando la configuración lo permite. |

La selección de estas estructuras depende de la escala de la explotación. Una pequeña instalación puede funcionar con un solo taladro y un horno, mientras que una explotación avanzada puede necesitar decenas de taladros, varias líneas de cintas, sistemas de filtrado y reservas de materiales.

---

#### 3. Objetos, insumos y recursos obtenidos

##### 3.1. Insumos necesarios para el funcionamiento

Los insumos necesarios para ejecutar la minería dependen del tipo de estructura empleada.

| Insumo | Utilizado por | Propósito |
|---|---|---|
| **Combustible** | Taladro a combustión | Proporcionar la energía necesaria para extraer minerales. |
| **Electricidad** | Taladro eléctrico, gran taladro y bomba extractora | Mantener la extracción automatizada. |
| **Ácido sulfúrico** | Taladros eléctricos sobre uranio | Permitir la extracción del mineral de uranio. |
| **Tuberías** | Redes de fluidos | Transportar líquidos hasta las máquinas que los utilizan. |
| **Cintas transportadoras** | Sistemas de minería automatizados | Transportar minerales sin intervención manual. |
| **Postes eléctricos** | Redes eléctricas | Distribuir energía a las máquinas. |
| **Hornos** | Procesamiento de minerales | Transformar minerales sólidos en materiales procesados. |

Es importante distinguir entre los **insumos de construcción** y los **insumos operativos**. Los primeros son los materiales necesarios para fabricar el edificio; los segundos son los recursos que la instalación consume mientras trabaja.

Por ejemplo, un taladro eléctrico necesita materiales para ser fabricado, pero, una vez instalado, funciona con electricidad. En cambio, el taladro a combustión necesita seguir recibiendo combustible.

##### 3.2. Recursos sólidos y sus aplicaciones

Los recursos extraídos no tienen todos la misma función. Algunos se utilizan directamente en recetas, mientras que otros deben procesarse antes de poder incorporarse a una cadena de fabricación.

| Recurso extraído | Producto obtenido inicialmente | Aplicaciones principales |
|---|---|---|
| **Mineral de hierro** | Mineral de hierro | Fabricación de placas de hierro mediante fundición. |
| **Mineral de cobre** | Mineral de cobre | Producción de placas de cobre. |
| **Carbón** | Carbón | Combustible, producción química y fabricación de productos industriales. |
| **Piedra** | Piedra | Ladrillos de piedra, hornos, muros y otros objetos. |
| **Mineral de uranio** | Mineral de uranio | Procesamiento en centrífugas para obtener uranio-235 y uranio-238. |
| **Chatarra (*Scrap*)** | Chatarra, en *Space Age* | Reciclaje para recuperar distintos materiales. |
| **Calcita (*Calcite*)** | Calcita, en *Space Age* | Procesos industriales y producción avanzada. |
| **Mineral de tungsteno** | Mineral de tungsteno, en *Space Age* | Producción de placas de tungsteno y materiales avanzados. |

Los tres últimos recursos pertenecen a mecánicas de la expansión *Space Age*. Su disponibilidad depende del planeta y de las estructuras de extracción compatibles.

##### 3.3. Recursos fluidos

Los fluidos requieren estructuras y sistemas de transporte diferentes a los utilizados para los minerales sólidos.

| Fluido | Estructura de extracción | Uso principal |
|---|---|---|
| **Agua** | Bomba de aguas profundas | Generación de vapor, producción química y otras recetas. |
| **Petróleo crudo** | Bomba extractora | Procesamiento en refinerías para obtener derivados del petróleo. |
| **Ácido sulfúrico** | Producción química en el juego base; extracción en determinados entornos de *Space Age* | Extracción de uranio y fabricación de otros productos. |
| **Otros fluidos planetarios** | Estructuras compatibles con el entorno correspondiente | Recetas especiales de la expansión. |

El transporte de fluidos se realiza mediante tuberías, depósitos y otros edificios que admiten líquidos. A diferencia de los minerales, los fluidos no se transportan como objetos normales en una cinta transportadora.

##### 3.4. El caso especial del uranio

El mineral de uranio tiene una particularidad importante: **no puede extraerse con el taladro a combustión ni mediante minería manual normal**.

Para extraerlo se necesita un taladro minero eléctrico con conexión de ácido sulfúrico. El ácido se consume durante la extracción y debe suministrarse de forma continua.

La relación básica es de **10 unidades de ácido sulfúrico por cada 10 unidades de mineral de uranio extraídas**, antes de considerar los efectos de productividad.

El mineral obtenido se transporta hasta una centrífuga, donde se procesa para producir uranio-235 y uranio-238. Estos materiales permiten desarrollar sistemas de energía nuclear y fabricar determinados productos especializados.

Los taladros eléctricos colocados de forma adyacente sobre un yacimiento de uranio pueden compartir el suministro de ácido sulfúrico cuando sus conectores están correctamente alineados.

Consulta la [Factorio Wiki: Uranium ore](https://wiki.factorio.com/Uranium_ore).

---

#### 4. Velocidad de extracción, productividad y agotamiento

##### 4.1. Velocidad de minería

La velocidad de extracción es uno de los principales factores que determinan la producción de una instalación.

La relación básica es:

\[
\text{Producción} \approx \frac{\text{Velocidad de minería}}{\text{Tiempo de extracción}}
\]

El tiempo de extracción representa, entre otras cosas, la dificultad del recurso para ser minado. Cuanto mayor sea este valor, más tiempo necesita la máquina para extraerlo.

La producción efectiva también depende del rendimiento de las casillas del yacimiento, de los módulos, de los modificadores tecnológicos y de otras características de la máquina.

La siguiente tabla muestra las tasas nominales de extracción indicadas por la [Factorio Wiki: Mining](https://wiki.factorio.com/Mining).

| Recurso | Taladro a combustión | Taladro eléctrico | Gran taladro minero |
|---|---:|---:|---:|
| Hierro | 0,25/s | 0,5/s | 2,5/s |
| Cobre | 0,25/s | 0,5/s | 2,5/s |
| Carbón | 0,25/s | 0,5/s | 2,5/s |
| Piedra | 0,25/s | 0,5/s | 2,5/s |
| Uranio | No disponible | 0,25/s | 1,25/s |
| Calcita | 0,25/s | 0,5/s | 2,5/s |
| Chatarra | 0,5/s | 1/s | 5/s |
| Tungsteno | No disponible | No disponible | 0,5/s |

*Las cifras representan tasas nominales de referencia en objetos por segundo, sin mejoras adicionales. Los recursos exclusivos de *Space Age* requieren sus entornos correspondientes; las cifras no deben interpretarse como una garantía de producción independiente de las condiciones del yacimiento.*

##### 4.2. Agotamiento de los yacimientos

Los taladros mineros consumen los recursos contenidos en las casillas que explotan. Por tanto, la extracción continua de un yacimiento finito terminará agotándolo.

La duración de un yacimiento depende, principalmente, de:

- La cantidad inicial de mineral disponible.
- El número de taladros instalados.
- La velocidad de extracción total.
- Las mejoras de productividad.
- Las características de agotamiento de los taladros utilizados.

Un error frecuente es interpretar que una instalación funciona correctamente solo porque continúa produciendo. Es posible que un conjunto de taladros esté generando mineral a buen ritmo y, al mismo tiempo, agotando el yacimiento demasiado rápido para sostener la demanda futura.

Por eso, es recomendable vigilar los recursos restantes y preparar nuevas explotaciones antes de que la reserva actual deje de satisfacer las necesidades de la fábrica.

##### 4.3. Productividad minera

La tecnología de **productividad minera** incrementa la cantidad de recursos que se obtienen por cada unidad extraída del yacimiento.

En el juego base, cada nivel de esta tecnología añade un 10 % de productividad a los taladros y a las bombas extractoras. La bonificación permite obtener materiales adicionales sin que estos productos extra tengan que consumirse de la reserva del yacimiento.

Esto presenta dos ventajas:

1. Aumenta la cantidad de recursos producidos durante el mismo periodo de tiempo.
2. Alarga la cantidad total de productos que se pueden obtener de un yacimiento.

A diferencia de los módulos de productividad, la investigación de productividad minera no introduce la penalización de velocidad asociada a esos módulos. Por ello, constituye una mejora especialmente útil para fábricas con una gran demanda de materias primas.

Fuente: [Factorio Wiki: Mining productivity](https://wiki.factorio.com/Mining_productivity).

##### 4.4. Módulos y mejoras

Los módulos pueden utilizarse para modificar el rendimiento de las estructuras compatibles.

- **Módulos de velocidad:** aumentan la velocidad de trabajo, aunque pueden elevar el consumo energético.
- **Módulos de productividad:** permiten obtener productos adicionales, pero pueden reducir la velocidad de la máquina.
- **Módulos de eficiencia:** reducen el consumo energético y ayudan a controlar la demanda eléctrica.
- **Módulos de calidad:** forman parte de las mecánicas de calidad de *Factorio 2.x* y pueden mejorar determinadas características de los edificios.

La elección debe responder a la necesidad concreta de la instalación. Una mina destinada a alimentar una fundición que se queda sin minerales puede beneficiarse de una mayor velocidad de extracción; una mina que dispone de suficiente rendimiento pero necesita aprovechar mejor un depósito limitado puede beneficiarse de una estrategia centrada en la productividad.

---

#### 5. Diseño de una instalación minera automatizada

##### 5.1. Organización de los taladros

Una instalación minera eficiente debe disponer los taladros de forma que aprovechen el yacimiento y depositen los recursos en una salida accesible.

Las principales recomendaciones son:

- Cubrir la mayor cantidad posible de mineral útil con los taladros disponibles.
- Evitar que las salidas queden bloqueadas por edificios mal colocados.
- Orientar los taladros hacia las cintas transportadoras o los edificios receptores.
- Dejar espacio suficiente para añadir nuevas máquinas cuando la producción aumente.
- Distribuir las conexiones eléctricas de forma que todas las máquinas reciban energía.

En el caso del taladro eléctrico, recuerda que el edificio ocupa una superficie de 3 × 3 casillas, pero explota un área de 5 × 5. Esta diferencia es importante al planificar el espacio y evitar colocar estructuras innecesarias entre los taladros.

##### 5.2. Transporte del mineral

El mineral debe abandonar la zona de extracción para llegar a las máquinas que lo procesarán.


Existen tres configuraciones frecuentes:

**Extracción hacia cintas transportadoras.** Es la solución habitual cuando se necesita transportar grandes cantidades de minerales a largas distancias. Permite combinar producción procedente de varios taladros en una misma línea.

**Extracción directa hacia hornos.** Un taladro puede alimentar directamente un horno si la disposición de ambos edificios lo permite. Esto reduce el número de insertores y simplifica algunas instalaciones pequeñas.

**Extracción hacia cofres.** Resulta útil cuando se necesita almacenar temporalmente la producción o separar la zona minera del sistema de transporte principal.

Para instalaciones grandes, las cintas transportadoras suelen proporcionar el equilibrio más conveniente entre simplicidad, capacidad y facilidad de ampliación.

##### 5.3. Ejemplo: una mina de hierro

Una instalación sencilla puede organizarse de la siguiente forma:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción en paralelo | Yacimiento de hierro | Tres taladros eléctricos | Mineral de hierro | El esquema representa tres taladros en paralelo. |
| Transporte hacia la fundición | Mineral de hierro | Cinta transportadora | Mineral de hierro transportado | — |
| Procesamiento | Mineral de hierro transportado | Hornos | Placas de hierro | — |
| Montaje | Placas de hierro | Línea de montaje | Componentes | — |

Este esquema representa la organización lógica del sistema, no una distribución exacta de casillas. En una construcción real, deben respetarse la orientación de las salidas, la superficie que ocupa cada edificio y la posición de los postes eléctricos.

La instalación puede ampliarse agregando más taladros mientras exista mineral disponible y la infraestructura posterior tenga capacidad suficiente para procesar la producción adicional.

##### 5.4. Extracción de petróleo

La extracción de petróleo necesita una organización distinta a la minería sólida.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción de petróleo | Campo petrolífero | Bomba extractora (*Pumpjack*) | Petróleo crudo | — |
| Transporte del crudo | Petróleo crudo | Tuberías | Petróleo crudo transportado | — |
| Almacenamiento intermedio | Petróleo crudo transportado | Depósito opcional | Reserva de petróleo crudo | El depósito es opcional en el esquema. |
| Refinación | Petróleo crudo | Refinería de petróleo | Productos de refinación | Las salidas dependen de la receta seleccionada. |
| Rama de refinación | Productos de la refinería | Refinería de petróleo | Gasóleo pesado | Etiqueta conservada del esquema original. |
| Rama de refinación | Productos de la refinería | Refinería de petróleo | Petróleo ligero | Etiqueta conservada del esquema original. |
| Rama de refinación | Productos de la refinería | Refinería de petróleo | Petróleo pesado\* | Etiqueta conservada tal como aparece en el esquema original. |

*El esquema muestra la separación conceptual de productos de la refinación; las salidas disponibles dependen de la receta seleccionada en la refinería.*

Los depósitos pueden ayudar a gestionar las reservas de fluidos, aunque no sustituyen la necesidad de mantener una demanda y una capacidad de procesamiento adecuadas.

También conviene instalar tuberías de manera que no interfieran con las vías de transporte de minerales ni dificulten la ampliación posterior de la fábrica.

##### 5.5. Producción de uranio

El sistema de extracción de uranio incorpora un suministro químico adicional.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Producción del insumo químico | Materias primas para la receta correspondiente | Producción de ácido sulfúrico | Ácido sulfúrico | — |
| Transporte del ácido | Ácido sulfúrico | Tuberías de ácido | Ácido suministrado a la mina | — |
| Extracción de uranio | Yacimiento de uranio y ácido sulfúrico | Taladros eléctricos sobre uranio | Mineral de uranio | — |
| Transporte del mineral | Mineral de uranio | Sistema de transporte | Mineral de uranio transportado | — |
| Procesamiento | Mineral de uranio | Centrífugas | Uranio-235 y uranio-238 | — |

El suministro constante de ácido es fundamental. Si se interrumpe, los taladros afectados dejan de extraer uranio hasta recuperar las condiciones necesarias.

---

#### 6. Estrategias y consejos de optimización

##### 6.1. Priorizar los recursos esenciales

Al principio de la partida, el hierro y el cobre suelen constituir las prioridades principales, ya que intervienen en numerosos edificios, componentes y productos intermedios.

El carbón es especialmente importante para la producción inicial de energía y para diversos procesos industriales. La piedra también resulta necesaria para fabricar hornos y otros componentes.

Una buena práctica es disponer de explotaciones separadas para los recursos esenciales, en lugar de depender de una sola reserva que abastezca todas las necesidades de la fábrica.

##### 6.2. Preparar la ampliación antes de necesitarla

No esperes a que una mina deje de producir para comenzar a buscar otro yacimiento.

Cuando el consumo aumenta, es recomendable:

- Inspeccionar las reservas disponibles en el mapa.
- Identificar un nuevo yacimiento del mismo recurso.
- Preparar los taladros, las cintas y las conexiones eléctricas necesarias.
- Instalar la nueva explotación antes de que la antigua resulte insuficiente.

Esta estrategia reduce las interrupciones y permite mantener el flujo de materias primas mientras se construyen nuevas instalaciones.

##### 6.3. Evitar cuellos de botella

La producción de una mina depende de algo más que la cantidad de taladros instalados.

Una explotación puede producir más mineral del que la cinta transportadora puede desplazar. De la misma forma, una mina puede alimentar una línea de hornos que no tenga capacidad para procesar todos los recursos que recibe.

Para prevenir estos problemas, comprueba periódicamente:

- Si los taladros funcionan de manera continua.
- Si las cintas transportadoras se encuentran saturadas.
- Si los hornos consumen todo el mineral que reciben.
- Si las máquinas posteriores disponen de suficiente capacidad.
- Si la red eléctrica mantiene la potencia necesaria.

**Aumentar el número de taladros no mejora necesariamente la producción final si el siguiente eslabón de la cadena permanece limitado.**

##### 6.4. Gestionar la electricidad

Los taladros eléctricos necesitan una red que pueda suministrar su demanda energética.

Cuando una fábrica crece, la suma del consumo de los taladros puede contribuir a que la producción eléctrica sea insuficiente para el conjunto de edificios.

Es recomendable disponer de una capacidad de generación que permita absorber los incrementos de consumo, especialmente cuando se añaden nuevas minas, bombas extractoras o líneas de procesamiento.

También es conveniente comprobar que los postes eléctricos alcanzan todas las máquinas. Una instalación que parece completa puede contener taladros sin suministro por una conexión omitida.

##### 6.5. Reducir el impacto de la contaminación

Las estructuras mineras producen contaminación durante su funcionamiento, aunque la intensidad depende del tipo de edificio y de sus condiciones de trabajo.

La contaminación puede propagarse por el entorno y contribuir a que los enemigos reaccionen contra la fábrica. Por ello, las explotaciones alejadas de la base deben planificarse teniendo en cuenta tanto la producción como su protección.

Algunas medidas útiles son:

- Construir las minas con espacio para instalar defensas.
- Mantener accesibles los alrededores para facilitar las reparaciones.
- Evitar instalar estructuras sin una utilidad real.
- Utilizar mejoras de eficiencia cuando resulte conveniente.
- Vigilar las zonas de expansión de la contaminación.

La protección de las minas es especialmente importante cuando los recursos esenciales se encuentran lejos de la base principal.

##### 6.6. Utilizar módulos con un objetivo concreto

Es conveniente elegir los módulos en función del resultado buscado.

Si el problema es la falta de mineral, los módulos de velocidad o la investigación de productividad pueden ser alternativas apropiadas. Si el problema es el consumo energético, los módulos de eficiencia pueden ayudar a controlar la demanda. Si la instalación utiliza un recurso difícil de conseguir, conviene estudiar si la productividad permitirá aprovechar mejor cada unidad extraída.

Antes de introducir módulos, comprueba que la red eléctrica puede asumir los cambios de consumo y que las máquinas posteriores podrán aprovechar la nueva producción.

##### 6.7. Aprovechar la extracción directa

No todas las máquinas necesitan una cinta transportadora entre ellas.

Un taladro que alimenta directamente un horno puede reducir el número de insertores, ahorrar espacio y simplificar una instalación pequeña.

Sin embargo, la alimentación directa limita la flexibilidad del sistema. Si varios hornos necesitan recibir recursos de distintos taladros o la producción debe dividirse entre varias líneas, las cintas transportadoras suelen ofrecer una solución más adaptable.

---

#### 7. Errores comunes y cómo solucionarlos

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| El taladro no produce recursos. | El área de extracción está agotada, el recurso no es compatible o la máquina carece de energía. | Inspeccionar el yacimiento, comprobar el tipo de taladro y revisar el suministro. |
| El taladro funciona, pero no salen minerales. | La casilla de salida está bloqueada. | Liberar la salida o reorganizar la cinta, el cofre o el edificio receptor. |
| Los taladros eléctricos se detienen. | Suministro eléctrico insuficiente o conexión incorrecta. | Revisar los postes y ampliar la capacidad de generación. |
| La mina de uranio deja de producir. | Falta ácido sulfúrico o existe un problema de salida. | Comprobar las tuberías y el transporte del mineral. |
| La producción final no aumenta al añadir taladros. | La cinta transportadora o las máquinas posteriores no pueden procesar la producción adicional. | Ampliar o reorganizar los sistemas de transporte y procesamiento. |
| La mina de petróleo produce cada vez menos. | El campo ha perdido parte de su rendimiento. | Incorporar módulos de velocidad, buscar campos adicionales o adaptar la demanda. |
| Los taladros dejan de funcionar al agotar el yacimiento. | La reserva disponible ha disminuido hasta agotarse. | Buscar otros yacimientos o mejorar la productividad para futuras explotaciones. |

##### 7.1. Interpretar los indicadores de los taladros eléctricos

Los indicadores luminosos del taladro eléctrico ayudan a detectar problemas de funcionamiento.

- **Verde:** la máquina funciona normalmente.
- **Amarillo:** existe un problema que impide la extracción o la salida normal de los recursos. Puede deberse a una salida bloqueada, falta de energía, desactivación por circuito o ausencia de fluido necesario.
- **Rojo:** el taladro ha agotado los recursos que podía extraer o presenta una condición de extracción no disponible, como la falta del fluido requerido.

Estos indicadores permiten localizar problemas sin necesidad de reconstruir una instalación completa. La [Factorio Wiki: Electric mining drill](https://wiki.factorio.com/Electric_mining_drill) proporciona más detalles sobre sus estados.

---

#### 8. Diferencias de extracción en *Space Age*

La expansión *Space Age* amplía la mecánica de extracción mediante nuevos recursos, entornos planetarios y edificios.

| Elemento | Particularidad |
|---|---|
| **Gran taladro minero** | Amplía el área de extracción, aumenta la velocidad y puede extraer recursos más difíciles, como el tungsteno. |
| **Calcita** | Se obtiene en entornos compatibles y se utiliza en procesos industriales avanzados. |
| **Chatarra** | Se extrae en Fulgora y puede reciclarse para recuperar materiales. |
| **Tungsteno** | Se obtiene en Vulcanus y requiere el gran taladro minero. |
| **Ácido sulfúrico y otros fluidos planetarios** | Algunos entornos permiten extraer fluidos directamente en lugar de producirlos mediante las recetas habituales del juego base. |
| **Bombas de fluidos adaptadas al planeta** | Permiten obtener los fluidos disponibles en determinados entornos, como lava o soluciones especiales. |

Las nuevas mecánicas mantienen los principios fundamentales de la minería del juego base: identificar el recurso, colocar una estructura compatible, proporcionar los insumos necesarios y transportar el producto.

Sin embargo, obligan a planificar las explotaciones teniendo en cuenta las condiciones particulares de cada planeta. Además, el gran taladro minero y las recetas propias de la expansión abren posibilidades para obtener recursos más difíciles y mejorar la eficiencia de las instalaciones avanzadas.

---

#### 9. Lista de comprobación para una explotación minera

Antes de considerar terminada una instalación, conviene verificar los siguientes puntos.

- [ ] El yacimiento contiene suficiente cantidad del recurso requerido.
- [ ] Los taladros son compatibles con el recurso que se desea extraer.
- [ ] Las áreas de extracción cubren correctamente el yacimiento.
- [ ] Todas las máquinas tienen combustible o suministro eléctrico suficiente.
- [ ] Los taladros tienen una salida libre para depositar sus productos.
- [ ] Las cintas transportadoras y los sistemas de fluidos tienen capacidad suficiente.
- [ ] Las máquinas de procesamiento pueden aprovechar el mineral extraído.
- [ ] La instalación dispone de espacio para crecer.
- [ ] Se controla el agotamiento de las reservas y se preparan nuevas explotaciones.
- [ ] Las defensas y la gestión de la contaminación son adecuadas para la ubicación de la mina.

Una instalación que cumple estos puntos tiene una base sólida para mantener un flujo de recursos estable y servir como punto de partida para cadenas de producción más complejas.

#### 10. Fuentes de referencia

Para consultar las estadísticas actualizadas, las recetas y las mecánicas particulares de cada estructura, estas páginas de la *Factorio Wiki* son las referencias técnicas más útiles:

- [Mining — Mecánica general de minería](https://wiki.factorio.com/Mining)
- [Burner mining drill — Taladro a combustión](https://wiki.factorio.com/Burner_mining_drill)
- [Electric mining drill — Taladro eléctrico](https://wiki.factorio.com/Electric_mining_drill)
- [Big mining drill — Gran taladro minero](https://wiki.factorio.com/Big_mining_drill)
- [Pumpjack — Bomba extractora](https://wiki.factorio.com/Pumpjack)
- [Offshore pump — Bomba de aguas profundas](https://wiki.factorio.com/Offshore_pump)
- [Uranium ore — Mineral de uranio](https://wiki.factorio.com/Uranium_ore)
- [Mining productivity — Productividad minera](https://wiki.factorio.com/Mining_productivity)

---

### Guía técnica de Factorio: Logística por cintas transportadoras

**Versión de referencia:** *Factorio 2.x*, con las mecánicas exclusivas de la expansión *Space Age* identificadas por separado. Las cifras de capacidad corresponden a cintas de calidad normal y sin tecnologías que aumenten el apilamiento sobre cintas.

Las cintas transportadoras son uno de los sistemas básicos de logística de *Factorio*. Permiten mover objetos automáticamente entre minas, hornos, máquinas de montaje, cofres y estaciones de carga, sin consumir electricidad durante el transporte. Una red bien diseñada mantiene un suministro constante de ingredientes, evita interrupciones y permite ampliar la fábrica por módulos.

Esta guía explica cómo funcionan las cintas, qué estructuras las complementan, cómo calcular su capacidad, cómo distribuir y equilibrar objetos, y qué errores suelen reducir el rendimiento.

---

#### 1. Fundamentos de las cintas transportadoras

##### 1.1. ¿Qué es una cinta transportadora?

Una **cinta transportadora** (*Transport belt*) es una estructura logística que desplaza objetos sólidos a lo largo de una ruta. Cada segmento ocupa una casilla y orienta el movimiento en una dirección. Los segmentos pueden conectarse para formar trayectos rectos, curvas, ramificaciones y redes extensas.

Las cintas no necesitan electricidad ni combustible. Una vez colocadas, funcionan continuamente siempre que haya objetos que transportar y una ruta de salida disponible.

Sus funciones principales son:

- Trasladar materias primas desde los taladros mineros hasta las zonas de procesamiento.
- Llevar placas, engranajes, circuitos y otros componentes hacia las máquinas de montaje.
- Distribuir ingredientes a varias líneas de producción.
- Retirar productos terminados para almacenarlos o enviarlos a otras áreas.
- Transportar combustible y objetos necesarios para mantener en marcha otras estructuras.

La logística por cintas es especialmente adecuada para flujos repetitivos y de gran volumen. A diferencia de los robots logísticos, no requiere energía para mover cada objeto ni depende de una flota de unidades móviles.

##### 1.2. Cómo circulan los objetos

Cada cinta tiene **dos carriles paralelos**, uno a cada lado de la dirección de movimiento. Los carriles pueden transportar objetos distintos o el mismo objeto. Cada carril se mueve de forma independiente, por lo que una cinta puede llevar dos tipos de material sin mezclarlos dentro de un mismo carril.

Una cinta recta completamente comprimida puede mantener hasta **4 objetos por casilla en cada carril**, es decir, 8 objetos por casilla si ambos carriles están llenos. En la práctica, la cantidad que llega al final depende de la velocidad, de los huecos entre objetos y de si las máquinas que cargan o descargan la cinta pueden mantener el ritmo.

Los objetos pueden incorporarse a una cinta desde un taladro minero, un inserter u otra cinta que se conecte lateralmente. Los insertores también pueden retirar objetos de un carril específico dependiendo de su posición y orientación.

##### 1.3. Flujo básico de una fábrica

Un sistema de transporte habitual se organiza así:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción | Yacimiento de mineral | Taladros mineros | Mineral extraído | — |
| Transporte de mineral | Mineral extraído | Cinta de mineral | Mineral transportado | — |
| Fundición | Mineral transportado | Hornos de fundición | Placas | — |
| Transporte de placas | Placas | Cinta de placas | Placas transportadas | — |
| Ensamblado | Placas y otros ingredientes requeridos | Máquinas de montaje | Productos terminados | — |
| Transporte del producto | Productos terminados | Cinta de productos terminados | Productos transportados | — |
| Almacenamiento o siguiente proceso | Productos transportados | Almacén o siguiente etapa de producción | Material disponible para la etapa siguiente | — |

Este esquema representa el flujo lógico, no una plantilla de construcción. La colocación real debe tener en cuenta la dirección de las cintas, la orientación de las máquinas, los espacios para insertores y los cruces con otras líneas.

##### 1.4. Cintas frente a otros sistemas logísticos

Las cintas son adecuadas para transportar grandes flujos de objetos por rutas predecibles. Los trenes son más útiles para conectar grandes distancias y mover grandes cantidades entre estaciones. Los robots logísticos ofrecen flexibilidad y pueden llevar objetos a distintos destinos, pero consumen electricidad y dependen de la red logística.

En una fábrica avanzada, los sistemas pueden coexistir: las cintas distribuyen objetos dentro de una zona industrial, los trenes conectan minas y fábricas distantes, y los robots cubren tareas puntuales o transportes flexibles.

---

#### 2. Tipos de cintas transportadoras

En el juego base existen tres niveles principales. *Space Age* agrega un cuarto nivel: la cinta turbo.

##### 2.1. Cinta transportadora básica (amarilla)


La **cinta transportadora básica** es el primer nivel y está disponible desde el comienzo de la partida.

**Características principales:**

- Capacidad por carril: 7,5 objetos por segundo.
- No requiere investigación para desbloquearse.
- Es económica y fácil de fabricar.

Es suficiente para líneas pequeñas, para transportar productos con baja demanda o para construir las primeras redes de producción. Puede mantenerse útil durante toda la partida en rutas secundarias que no necesitan más capacidad.

Más información: [Factorio Wiki: Transport belt](https://wiki.factorio.com/Transport_belt).

##### 2.2. Cinta transportadora rápida (roja)


La **cinta rápida** mueve los objetos al doble de velocidad que una cinta básica.

**Características principales:**

- Se desbloquea mediante la tecnología *Logistics 2*.

Es una mejora apropiada cuando una línea básica se satura o cuando varias máquinas comparten una ruta y necesitan recibir ingredientes con mayor frecuencia. No es necesario reemplazar toda la fábrica al desbloquearla: suele ser suficiente mejorar los tramos que se han convertido en cuellos de botella.

Más información: [Factorio Wiki: Fast transport belt](https://wiki.factorio.com/Fast_transport_belt).

##### 2.3. Cinta transportadora exprés (azul)


La **cinta exprés** es el nivel más rápido del juego base. Se desplaza a una velocidad tres veces superior a la cinta básica.

**Características principales:**

- Se desbloquea mediante la tecnología *Logistics 3*.
- Su receta requiere lubricante, además de componentes sólidos.

Es útil en líneas industriales de gran demanda, especialmente cuando transportan grandes cantidades de placas, circuitos u otros ingredientes. Como su fabricación requiere más recursos que las cintas anteriores, conviene concentrarla en rutas que realmente necesiten esa capacidad.

Más información: [Factorio Wiki: Express transport belt](https://wiki.factorio.com/Express_transport_belt).

##### 2.4. Cinta transportadora turbo (verde) — *Space Age*


La **cinta turbo** es exclusiva de *Space Age* y constituye el nivel de mayor velocidad entre las cintas transportadoras estándar.

**Características principales:**

- Se desbloquea mediante la tecnología *Turbo transport belt*.
- Solo se puede fabricar en Vulcanus.

Es adecuada para redes de gran capacidad, aunque su velocidad no elimina por sí sola los cuellos de botella en insertores, divisores, máquinas de producción o cintas conectadas más lentas. Para obtener una mejora real, el resto de la ruta debe poder abastecerla y descargarla al ritmo requerido.

Más información: [Factorio Wiki: Turbo transport belt](https://wiki.factorio.com/Turbo_transport_belt).

##### 2.5. Comparativa de capacidad

| Tipo de cinta | Color | Velocidad (casillas/s) | Capacidad total (objetos/s) | Capacidad por carril (objetos/s) | Desbloqueo |
|---|---|---:|---:|---:|---|
| Básica | Amarillo | 1,875 | 15 | 7,5 | Disponible al inicio |
| Rápida | Rojo | 3,75 | 30 | 15 | *Logistics 2* |
| Exprés | Azul | 5,625 | 45 | 22,5 | *Logistics 3* |
| Turbo | Verde | 7,5 | 60 | 30 | Tecnología *Turbo transport belt*, *Space Age* |

Las capacidades de la tabla corresponden a cintas completamente comprimidas, sin tecnologías de apilamiento sobre cintas. Los huecos entre objetos y las limitaciones de entrada o salida pueden reducir el flujo real.

---

#### 3. Estructuras auxiliares de la logística por cintas

##### 3.1. Cinta transportadora subterránea (*Underground belt*)


Las **cintas subterráneas** permiten cruzar otras rutas, edificios o zonas ocupadas sin interrumpir el transporte. Cada tramo está formado por una entrada y una salida orientadas correctamente; los objetos desaparecen bajo tierra y reaparecen en la salida correspondiente.

**Usos principales:**

- Cruzar por debajo de otra cinta sin mezclar las líneas.
- Pasar por debajo de edificios o estructuras que ocupan el nivel de superficie.
- Reducir cruces complicados y conservar rutas de transporte paralelas.
- Mantener compacto el diseño de una fundición o línea de montaje.

La distancia máxima entre extremos depende del nivel:

| Tipo | Distancia subterránea máxima de referencia |
|---|---:|
| Básica | 4 casillas |
| Rápida | 6 casillas |
| Exprés | 8 casillas |
| Turbo (*Space Age*) | 10 casillas |

Las cintas subterráneas conservan la velocidad de su nivel correspondiente. Para evitar conexiones incorrectas, comprueba la orientación de las entradas y salidas y asegúrate de que las dos piezas forman un par válido.

Más información: [Factorio Wiki: Underground belt](https://wiki.factorio.com/Underground_belt).

##### 3.2. Divisor (*Splitter*)


El **divisor** distribuye objetos entre cintas o combina dos líneas en una. Sus dos entradas y dos salidas le permiten dividir un flujo, unir dos flujos o repartir de forma equilibrada los objetos que llegan.

Funciones principales:

- Dividir una cinta en dos salidas.
- Unir dos cintas en una línea de salida.
- Equilibrar el flujo entre dos cintas.
- Aplicar una prioridad a una entrada o una salida.
- Filtrar un tipo de objeto para que se dirija a una salida seleccionada.

Los divisores pueden ser de nivel básico, rápido, exprés o turbo, y el nivel debe coincidir con el ritmo que se necesita mantener en la ruta. Un divisor lento puede limitar una línea rápida si se coloca en medio de ella.

Desde Factorio 2.0.67, los divisores también pueden conectarse a la red de circuitos, lo que permite configurar prioridades o filtros mediante señales bajo las condiciones compatibles.

Más información: [Factorio Wiki: Splitter](https://wiki.factorio.com/Splitter).

##### 3.3. Insertores (*Inserters*)


Los **insertores** transfieren objetos entre cintas, cofres y edificios. No son cintas propiamente dichas, pero son esenciales para cargar y descargar la mayoría de las máquinas.

Pueden utilizarse para:

- Tomar mineral de una cinta e introducirlo en un horno.
- Retirar placas de los hornos y colocarlas sobre otra cinta.
- Cargar combustible en calderas u hornos que lo necesiten.
- Transferir productos entre cofres y cintas.
- Alimentar máquinas de montaje con ingredientes específicos.

El rendimiento de un insertor depende del tipo, de la distancia que debe cubrir, de los objetos que manipula y de su configuración. Las cintas rápidas no solucionan un cuello de botella causado por un insertor que no puede cargar o descargar suficientes objetos por segundo.

Más información: [Factorio Wiki: Inserter](https://wiki.factorio.com/Inserter).

##### 3.4. Cofres y contenedores


Los cofres pueden actuar como almacenamiento intermedio entre diferentes etapas de una red. Los insertores permiten depositar objetos en los cofres o extraerlos para devolverlos a una cinta.

Son útiles para acumular productos, separar una zona de producción de otra o almacenar materiales antes de cargarlos en trenes. Sin embargo, introducir un cofre en una línea no aumenta automáticamente el rendimiento: puede generar una nueva limitación si los insertores son demasiado lentos o si el cofre llega a llenarse.

##### 3.5. Postes, red de circuitos y señales

Las cintas no necesitan electricidad para moverse, pero los edificios que las alimentan o descargan suelen consumir energía. Los postes eléctricos mantienen en funcionamiento los insertores eléctricos y las máquinas de producción.

La **red de circuitos** puede ampliar el control logístico mediante condiciones. En Factorio 2.x, las cintas pueden configurarse para enviar señales de contenido y habilitarse o deshabilitarse bajo condiciones compatibles. Los divisores pueden ajustar filtros y prioridades mediante la red de circuitos en las versiones que admiten estas opciones.

Estos mecanismos son opcionales. Para la mayoría de las fábricas iniciales, la orientación de las cintas, los divisores y los insertores es suficiente.

---

#### 4. Carriles y distribución de objetos

##### 4.1. Los dos carriles de una cinta

Una cinta tiene dos carriles independientes. Esta característica permite transportar dos ingredientes diferentes por la misma ruta, siempre que se organice correctamente su entrada.

Por ejemplo, una cinta puede llevar placas de hierro por un carril y placas de cobre por el otro. Para lograrlo, es habitual introducir los materiales desde lados opuestos o diseñar los insertores para que dejen cada material en el carril correspondiente.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Transporte por carril superior | Hierro | Cinta transportadora | Hierro en el carril superior | El sentido indicado es hacia la derecha. |
| Transporte por carril inferior | Cobre | Cinta transportadora | Cobre en el carril inferior | El sentido indicado es hacia la derecha; los dos carriles permiten llevar materiales diferentes. |

Esta disposición es útil cuando dos componentes deben llegar juntos a máquinas de montaje cercanas. Sin embargo, si los consumidores necesitan cantidades muy diferentes de cada ingrediente, uno de los carriles puede agotarse antes que el otro.

##### 4.2. Cargar un solo carril

Cuando los objetos entran lateralmente, es posible que ocupen únicamente uno de los dos carriles. El resultado es una cinta que parece llena, pero solo utiliza la mitad de su capacidad.

Esto puede ser intencional, por ejemplo, si el otro carril está reservado para otro material. Si se necesita transportar un único recurso y se pretende maximizar el flujo, conviene distribuir objetos en ambos carriles.

Una técnica habitual consiste en introducir el recurso en el carril vacío desde un lado o utilizar un divisor para reorganizar las líneas. También se pueden colocar insertores frente a la cinta con una configuración que coloque los objetos en el lado deseado.

##### 4.3. Alimentación lateral (*Sideloading*)

La alimentación lateral consiste en conectar una cinta a otra desde un lado, haciendo que los objetos se incorporen a la ruta principal. Puede servir para llenar un carril concreto, combinar la producción de varios taladros o alimentar una línea común.

Debe usarse con cuidado: la entrada lateral no siempre distribuye el flujo entre ambos carriles, y una de las líneas puede bloquearse si la cinta principal está llena. Cuando sea importante controlar con precisión la proporción entre entradas, un divisor ofrece opciones más claras de distribución.

##### 4.4. Mezcla de tipos de objetos

Una cinta puede transportar varios tipos de objetos, pero mezclar materiales sin un plan suele complicar la alimentación de máquinas. Si una receta necesita exclusivamente placas de hierro, los objetos de cobre que pasen por la misma línea no ayudarán a esa máquina y pueden ocupar espacio útil.

Las cintas mixtas tienen sentido cuando se combina una selección pequeña y conocida de objetos y existe un mecanismo de clasificación en el destino. Para grandes fábricas suele ser más fácil mantener cintas dedicadas a recursos o grupos de materiales.

---

#### 5. Divisores, prioridades y filtros

##### 5.1. Dividir y combinar flujos

Un divisor distribuye el flujo de entrada entre las salidas disponibles. Si ambas salidas pueden aceptar objetos, el comportamiento normal busca repartirlos de manera equilibrada según las condiciones de las líneas. Si una salida se bloquea, los objetos pueden continuar por la otra, siempre que haya capacidad.

Al unir dos cintas, el divisor mezcla los flujos de entrada en sus salidas. El resultado depende de cuánto material aporta cada entrada y del espacio disponible en las salidas.

##### 5.2. Prioridad de entrada

La prioridad de entrada permite favorecer una de las entradas frente a la otra. Esto es útil cuando un recurso principal debe utilizarse antes que una fuente secundaria.

Ejemplo: una fábrica recibe mineral desde una cinta principal y otra de respaldo. La entrada prioritaria se consume primero y la secundaria aporta recursos cuando la principal no cubre toda la demanda.

##### 5.3. Prioridad de salida

La prioridad de salida favorece una ruta antes que la otra. Puede utilizarse para suministrar primero una línea de producción prioritaria y dirigir el excedente hacia un almacén o una cadena secundaria.

Las prioridades deben diseñarse con cuidado. Si la salida prioritaria siempre puede aceptar objetos, la ruta secundaria podría recibir poco o ningún material. Esto es útil si se desea dar preferencia a una línea, pero no si se espera repartir el flujo por igual.

##### 5.4. Filtrado por tipo de objeto

El filtro de un divisor permite dirigir un tipo de objeto a una de las salidas. Es útil para separar materiales que comparten una cinta, como placas, engranajes o circuitos, siempre que se conozcan los productos que circulan por ella.

Una línea de clasificación puede tener esta forma:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada de objetos variados | Objetos variados | Cinta transportadora o cofre | Mezcla de objetos | — |
| Clasificación | Mezcla de objetos | Divisor con filtro | Objetos separados por salida | El divisor está configurado con un filtro. |
| Salida filtrada | Placas de hierro dentro de la mezcla | Divisor con filtro | Placas de hierro | — |
| Salida de otros objetos | Objetos restantes de la mezcla | Divisor con filtro | Otros objetos hacia otra ruta | — |

Si se utiliza una red de circuitos para establecer el filtro, comprueba las condiciones configuradas y las señales recibidas. Una condición incorrecta puede dejar el filtro sin efecto o enviar los objetos a una salida no deseada.

---

#### 6. Capacidad y rendimiento de las cintas

##### 6.1. Velocidad frente a capacidad

La velocidad describe cuánto avanza el flujo por segundo. La capacidad o **rendimiento** (*throughput*) indica cuántos objetos pueden pasar por una sección de cinta en un intervalo de tiempo.

La capacidad máxima depende de la velocidad de la cinta y de la densidad de los objetos. La capacidad total se obtiene utilizando ambos carriles; los valores nominales por nivel están centralizados en la tabla comparativa de tipos de cinta.

##### 6.2. Capacidad por minuto

| Tipo | Objetos/s (ambos carriles) | Objetos/min (ambos carriles) |
|---|---:|---:|
| Básica | 15 | 900 |
| Rápida | 30 | 1800 |
| Exprés | 45 | 2700 |
| Turbo | 60 | 3600 |

Estas cifras describen la capacidad teórica máxima de una cinta completamente comprimida. No son una garantía de que una ruta completa alcance ese rendimiento: un divisor, inserter, giro, tramo de cinta lento o consumidor puede reducir el flujo final.

##### 6.3. Ejemplo de cálculo

Supongamos que una línea de producción necesita transportar 20 placas por segundo.

- Una cinta básica no alcanza la capacidad requerida: su capacidad es inferior a la de los niveles superiores.
- Una cinta rápida admite hasta 30 objetos/s y puede cubrir la demanda si ambos carriles están bien abastecidos.
- Una cinta exprés tiene margen adicional, pero puede resultar innecesaria si el resto del sistema solo necesita 20 objetos/s.

Por lo tanto, conviene seleccionar el nivel más económico que satisfaga la demanda prevista y cambiarlo cuando la línea se convierta en un cuello de botella.

##### 6.4. Carriles vacíos y compresión

Una cinta completamente comprimida tiene los objetos colocados sin huecos útiles entre ellos. Si una cinta contiene objetos en un solo carril, su capacidad máxima efectiva se reduce a la mitad.

También puede suceder que una cinta se encuentre parcialmente vacía porque la máquina de origen produce poco o porque el recurso se consume antes de llegar al final. En estos casos, aumentar la velocidad de la cinta no crea objetos adicionales: solo aumenta el límite de transporte disponible.

##### 6.5. Cintas rápidas y tramos lentos

Cuando una ruta contiene segmentos de distintos niveles, el rendimiento máximo de la ruta está condicionado por los tramos que no pueden mover el flujo al ritmo requerido. Por ejemplo, una cinta exprés seguida de una cinta básica no puede mantener una salida continua superior a la capacidad de la cinta básica.

Las mejoras deben aplicarse a los tramos que limitan la producción, no necesariamente a toda la red. En fábricas grandes es común reservar cintas exprés o turbo para los corredores principales y utilizar niveles inferiores en rutas secundarias.

##### 6.6. Apilamiento sobre cintas en Factorio 2.x y *Space Age*

En las mecánicas actuales, algunas tecnologías y edificios permiten **apilar objetos encima de las cintas**, aumentando la cantidad transportada sin cambiar la velocidad de movimiento de la cinta. La investigación de apilamiento de objetos transportados y las tecnologías de capacidad de cinta de *Space Age* permiten elevar progresivamente la capacidad por sección.

Estas mejoras no se aplican automáticamente a todos los objetos de cualquier forma: la posibilidad de crear pilas depende de la tecnología y de las estructuras compatibles, como insertor apilador, gran taladro minero y reciclador en las situaciones correspondientes. Los insertores pueden retirar los objetos de las pilas en el flujo de producción.

Para comparar la capacidad de las cintas en su estado básico, utiliza la tabla anterior. Si se emplean tecnologías de apilamiento, consulta la interfaz del juego y la [Factorio Wiki: Transport belt capacity](https://wiki.factorio.com/Transport_belt_capacity_(research)) para calcular el rendimiento de la configuración concreta.

---

#### 7. Diseño de redes logísticas

##### 7.1. Mantener rutas legibles

Una fábrica resulta más fácil de ampliar cuando las líneas siguen direcciones claras y los cruces son intencionales. Las cintas subterráneas permiten cruzar por debajo de otras rutas sin mezclar materiales; los divisores sirven para distribuir o reunir flujos de forma controlada.

Recomendaciones:

- Agrupa rutas por recurso o por zona de producción.
- Reserva corredores principales para los flujos de gran volumen.
- Evita hacer cambios de dirección innecesarios.
- Deja espacio para insertar divisores, cintas subterráneas e insertores.
- Mantén visibles las rutas de entrada y salida de las máquinas.
- Usa planos (*blueprints*) cuando el mismo diseño vaya a repetirse.

##### 7.2. Alimentación de fundiciones

Una línea habitual de fundición utiliza una cinta para llevar mineral a los hornos y otra para retirar las placas. Los insertores toman el mineral de la cinta de entrada y lo depositan en los hornos; otros insertores retiran las placas y las colocan en la cinta de salida.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada de mineral | Cinta de mineral | Cinta de entrada de la línea de hornos | Mineral distribuido | — |
| Procesamiento en paralelo | Mineral de la cinta de entrada | Tres hornos | Placas procesadas | El diagrama muestra tres hornos en paralelo. |
| Retirada de placas | Placas producidas por los hornos | Cinta de placas | Placas transportadas | La flecha del esquema indica que esta cinta circula en sentido opuesto a la de mineral. |

La tabla es esquemática. Al construir, ajusta la posición de los hornos e insertores para que la entrada y salida de cada máquina coincidan con los carriles correctos. Comprueba también que el flujo de mineral alcance para mantener todos los hornos ocupados.

##### 7.3. Alimentar máquinas de montaje

Las máquinas de montaje suelen necesitar más de un ingrediente. Una solución común consiste en llevar dos materiales por carriles distintos de una misma cinta, o acercar varias cintas a los laterales de la máquina.

Antes de elegir el diseño, revisa cuántos ingredientes necesita la receta y dónde se encuentran las entradas disponibles. No todas las máquinas admiten todos los ingredientes desde cualquier lado, así que conviene comprobar la orientación de los insertores y la posición de los puertos.

##### 7.4. Diseñar cruces con cintas subterráneas

Los cruces a nivel pueden combinar o bloquear rutas. Si dos cintas deben cruzarse sin intercambiar objetos, utiliza una cinta subterránea para una de las direcciones. La distancia máxima depende del nivel de cinta y la orientación debe formar una entrada y salida válidas.

El uso de cintas subterráneas reduce la superficie ocupada, pero requiere más piezas por cruce. En rutas muy largas, alterna tramos visibles y subterráneos para mantener un diseño claro y fácil de reparar.

##### 7.5. Utilizar divisores para distribuir recursos

Un divisor puede repartir mineral o placas entre dos líneas de producción. Si las dos líneas consumen cantidades similares, el reparto equilibrado puede ayudar a abastecerlas. Cuando una línea es más importante que otra, la prioridad de salida permite reservarle el suministro preferente.

Sin embargo, los divisores no corrigen todos los desequilibrios. Si la entrada aporta menos material que la demanda conjunta, seguirá habiendo escasez. Primero hay que comprobar la producción total y después decidir cómo distribuirla.

##### 7.6. Alimentación de combustible

Las cintas también transportan carbón y otros combustibles compatibles hacia calderas, hornos y determinadas máquinas. En estas rutas, es importante que los insertores puedan retirar combustible con la frecuencia necesaria y que el transporte no se interrumpa por falta de suministro.

Para una central eléctrica o una fundición de gran tamaño, una ruta de combustible puede justificar un corredor dedicado. De este modo se reduce el riesgo de que otras líneas consuman o bloqueen la capacidad destinada a la energía o al procesamiento de minerales.

---

#### 8. Estrategias y consejos de optimización

##### 8.1. Identificar primero el cuello de botella

Antes de cambiar todas las cintas por un nivel superior, observa dónde se acumulan los objetos y qué máquinas se quedan sin insumos.

- Si la cinta se encuentra saturada y el consumidor necesita más material, puede hacer falta una cinta más rápida o una ruta adicional.
- Si la cinta está vacía, es probable que la máquina que produce el material no esté generando suficiente.
- Si se acumulan objetos antes de una máquina, comprueba si la entrada de la máquina puede aceptar el flujo.
- Si solo se usa un carril, decide si el otro debe reservarse para otro material o llenarse para aumentar la capacidad.

##### 8.2. No sobredimensionar todas las rutas

Una cinta turbo no ofrece beneficios si la máquina de origen produce muy poco o si el destino solo consume unos pocos objetos por segundo. Es mejor utilizar el nivel adecuado en cada tramo y reservar las cintas más rápidas para zonas con demanda elevada.

##### 8.3. Mantener las líneas de suministro comprimidas

Para aprovechar al máximo una cinta, la entrada debe suministrar objetos con suficiente regularidad. Un flujo continuo desde varios taladros, hornos o insertores ayuda a mantener una cinta comprimida.

No obstante, una cinta saturada no es necesariamente una señal de problema: si todos los consumidores reciben lo que necesitan, la saturación puede indicar que se está aprovechando la capacidad disponible. Lo importante es detectar si las máquinas posteriores están quedándose sin materiales.

##### 8.4. Equilibrar carriles cuando sea necesario

Un desequilibrio de carriles puede reducir la cantidad de material que una línea entrega a ciertos consumidores, especialmente si los insertores solo toman objetos desde el lado que tiene recursos. Según el diseño, puede ser útil introducir objetos en ambos carriles o emplear un mecanismo de balanceo.

No hace falta equilibrar todas las cintas por defecto. Si todos los consumidores reciben materiales suficientes y el otro carril no tiene un uso previsto, el sistema puede funcionar perfectamente sin una estructura adicional.

##### 8.5. Evitar mezclas difíciles de clasificar

Las cintas con varios materiales pueden ahorrar espacio, pero también complicar las recetas y crear situaciones en las que el material incorrecto ocupa un lugar necesario para otro recurso. Usa divisores con filtros solo cuando tengas una razón concreta para compartir la ruta y puedas controlar la selección de objetos.

##### 8.6. Aprovechar insertores apiladores en etapas avanzadas

Los insertores apiladores y las tecnologías de capacidad de cinta pueden elevar el rendimiento de los corredores de alto volumen. Antes de incorporarlos, comprueba que las máquinas de los extremos pueden generar y consumir objetos al mismo ritmo.

Una cinta con capacidad ampliada sigue pudiendo bloquearse si el siguiente tramo, divisor o consumidor no acepta el volumen entrante.

##### 8.7. Automatizar las mejoras con planos

Cuando una solución de transporte funciona bien, conviértela en un plano. Los planos ayudan a repetir distancias, orientar correctamente las cintas subterráneas y mantener una disposición consistente de divisores e insertores.

Es aconsejable incluir en los planos espacio para postes eléctricos, acceso de construcción y posibles ampliaciones. Un diseño ligeramente más amplio suele ser más sencillo de mantener que una red excesivamente compacta y difícil de modificar.

---

#### 9. Errores frecuentes y cómo solucionarlos

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| La cinta no transporta objetos. | Está orientada en la dirección equivocada, desconectada o no recibe objetos. | Comprueba las flechas de dirección y el tramo que aporta los recursos. |
| La cinta parece llena, pero una máquina no recibe suficiente material. | El recurso circula por el carril contrario al que puede tomar el insertor, o el flujo total es insuficiente. | Reorganiza el carril de entrada, cambia la posición del insertor o aumenta el suministro. |
| Un tramo de cinta exprés no aumenta el rendimiento. | Otro tramo, divisor o insertor tiene menor capacidad. | Identifica y mejora el cuello de botella real. |
| Los objetos se mezclan con materiales incorrectos. | Se han combinado varios recursos sin filtros o separación adecuada. | Separa las líneas o añade divisores con filtro. |
| La cinta subterránea no se conecta. | La entrada y la salida están mal orientadas, son incompatibles o exceden la distancia máxima. | Corrige la orientación y usa un nivel con más alcance si hace falta. |
| Un divisor envía todos los objetos a una ruta. | Existe prioridad de entrada/salida, una salida bloqueada o un ajuste de filtro. | Revisa sus parámetros y comprueba la capacidad de ambas salidas. |
| Una de las dos líneas recibe menos material. | Los flujos de entrada son diferentes o existe un cuello de botella después del divisor. | Comprueba el suministro de entrada, las prioridades y el consumo de cada rama. |
| El combustible no llega a los hornos o calderas. | La cinta está vacía, bloqueada o los insertores no alcanzan el ritmo necesario. | Revisa la producción de combustible y aumenta la capacidad de alimentación. |
| Una cinta se bloquea al final. | El cofre o la máquina de destino está lleno, o no puede aceptar el tipo de objeto. | Libera la salida o corrige el destino de los objetos. |
| La cinta básica se congestiona al ampliar la fábrica. | El volumen transportado supera la capacidad máxima de la cinta. | Sustituye el tramo crítico por uno más rápido o crea una ruta paralela. |

---

#### 10. Particularidades de *Space Age*

La expansión *Space Age* amplía la logística por cintas con cintas turbo, investigación de capacidad de cinta y mecánicas de apilamiento.

##### 10.1. Cintas turbo

Las cintas turbo tienen una capacidad base de 60 objetos/s, el doble que una cinta rápida, un tercio más que una exprés y cuatro veces la capacidad de una cinta básica. Solo se fabrican en Vulcanus, por lo que es necesario acceder a ese planeta y desarrollar su cadena de producción para obtenerlas.

##### 10.2. Apilamiento de objetos

La capacidad de transporte puede incrementarse mediante tecnologías que permiten apilar objetos sobre la cinta. Al aumentar el número de objetos transportados verticalmente, el rendimiento puede superar los límites de la cinta normal sin aumentar la velocidad horizontal.

Las tecnologías de capacidad de cinta amplían progresivamente el apilamiento permitido. Para aprovecharlo, hay que utilizar las estructuras compatibles que pueden crear esas pilas y verificar que el conjunto de la ruta mantiene el flujo esperado.

##### 10.3. Grandes taladros y recicladores

En *Space Age*, ciertos edificios, como los grandes taladros mineros y los recicladores, pueden contribuir a generar pilas de objetos sobre cintas. Estas mecánicas permiten construir cadenas logísticas más densas, pero requieren planificar las transiciones entre cintas apiladas y las máquinas que consumen los objetos.

##### 10.4. Entornos planetarios

Los corredores logísticos deben adaptarse al terreno y a los recursos propios de cada planeta. Las cintas subterráneas no pueden transportar objetos por debajo de lava o del vacío espacial. En esos lugares hay que rodear el obstáculo o diseñar la ruta por otra superficie válida.

---

#### 11. Lista de comprobación de una red de cintas

- [ ] Todas las cintas están orientadas correctamente y forman rutas continuas.
- [ ] La capacidad de cada tramo es suficiente para el flujo previsto.
- [ ] Los carriles contienen los objetos que necesitan las máquinas de destino.
- [ ] Las entradas y salidas de los insertores coinciden con la disposición de la cinta y del edificio.
- [ ] Las cintas subterráneas forman pares correctos y respetan la distancia máxima del nivel utilizado.
- [ ] Los divisores están orientados correctamente y sus filtros o prioridades son intencionales.
- [ ] Las cintas de combustible pueden abastecer todas las máquinas conectadas.
- [ ] No hay cofres, máquinas o destinos llenos que bloqueen la salida de productos.
- [ ] Los tramos lentos no limitan las rutas de alta capacidad.
- [ ] Las rutas están organizadas por recurso o por zona para facilitar el diagnóstico.
- [ ] Hay espacio para ampliar las cintas y añadir nuevos consumidores.
- [ ] Si se utilizan pilas sobre cintas, todos los edificios implicados son compatibles con esa configuración.

---

#### 12. Fuentes de referencia

Estas páginas de la *Factorio Wiki* permiten consultar estadísticas, recetas, mecánicas de los carriles y cambios introducidos por versiones recientes:

- [Belt transport system — Sistema de cintas transportadoras](https://wiki.factorio.com/Transport_network)
- [Transport belt — Cinta transportadora básica](https://wiki.factorio.com/Transport_belt)
- [Fast transport belt — Cinta rápida](https://wiki.factorio.com/Fast_transport_belt)
- [Express transport belt — Cinta exprés](https://wiki.factorio.com/Express_transport_belt)
- [Turbo transport belt — Cinta turbo de *Space Age*](https://wiki.factorio.com/Turbo_transport_belt)
- [Underground belt — Cinta subterránea](https://wiki.factorio.com/Underground_belt)
- [Fast underground belt — Cinta subterránea rápida](https://wiki.factorio.com/Fast_underground_belt)
- [Splitter — Divisor](https://wiki.factorio.com/Splitter)
- [Transport belts/Physics — Física y capacidad de las cintas](https://wiki.factorio.com/Transport_belts/Physics)
- [Inserter — Insertores](https://wiki.factorio.com/Inserter)
- [Circuit network — Red de circuitos](https://wiki.factorio.com/Circuit_network)
- [Transport belt capacity — Capacidad de cintas](https://wiki.factorio.com/Transport_belt_capacity_(research))


---

### Guía técnica de Factorio: Funcionamiento y tipos de insertores

**Versión de referencia:** *Factorio 2.x*. Las características exclusivas de *Space Age* se identifican por separado. Las cifras de consumo corresponden a edificios de calidad normal y pueden variar con la calidad del inserter y otros modificadores.

Los **insertores** (*inserters*) son brazos mecánicos que transfieren objetos a distancias cortas. Son fundamentales para automatizar la fábrica: extraen materiales de cintas transportadoras, cofres, hornos y máquinas, y los colocan en otros edificios, contenedores o cintas. Sin ellos, muchas cadenas de fabricación necesitarían intervención manual.

Esta guía explica su funcionamiento, los distintos modelos disponibles, sus filtros y controles, los insumos necesarios para fabricarlos y las estrategias para utilizarlos con eficiencia.

---

#### 1. Fundamentos del funcionamiento de los insertores

##### 1.1. ¿Qué hace un inserter?

Un inserter toma objetos del punto situado detrás de su base y los deposita en el punto situado delante. Al colocarlo, su orientación determina de dónde recoge los materiales y hacia dónde los entrega.

Se utiliza para conectar, por ejemplo:

- Una cinta transportadora con una máquina de montaje.
- Un horno con la cinta que retira sus productos.
- Un cofre con una cinta transportadora.
- Una cinta con otra cinta colocada en una dirección distinta.
- Un vagón de carga con cintas o cofres.
- Un edificio de producción con otro edificio que necesita sus productos.

Los insertores pueden recoger objetos de cintas, del suelo y de entidades con inventario, como cofres, hornos, máquinas de montaje y vehículos. También pueden depositar objetos en esos destinos, siempre que el objeto sea válido para el inventario receptor.

##### 1.2. Ciclo de movimiento

El funcionamiento básico de un inserter se puede dividir en cuatro etapas:

1. **Búsqueda y recogida:** comprueba si hay un objeto que pueda recoger y si tiene espacio para hacerlo.
2. **Giro:** mueve el brazo hacia el destino.
3. **Entrega:** deposita los objetos en el destino, si este tiene espacio y admite esos objetos.
4. **Retorno:** vuelve al punto de recogida y comienza otro ciclo.

La velocidad real no depende únicamente de la velocidad de giro. También influyen la distancia de extensión del brazo, el tipo de destino, la cantidad de objetos que puede llevar en la mano, la disponibilidad de materiales y los espacios libres en una cinta.

##### 1.3. Reglas importantes

- Un inserter no recogerá un objeto que el edificio de destino no pueda aceptar.
- Si el inventario de destino está lleno, puede quedarse esperando con el brazo cargado.
- Si no llegan objetos al punto de recogida, no puede mantener su ritmo de operación.
- Si el inserter eléctrico no tiene suficiente energía, funcionará más lentamente.
- No puede colocar más de un objeto en la misma casilla del suelo al mismo tiempo.
- La capacidad máxima de su mano depende del tipo de inserter y de las tecnologías investigadas.
- Un filtro puede restringir los tipos de objetos que puede recoger o depositar.

Para conocer los pormenores de las mecánicas, consulta la [Factorio Wiki: Inserters](https://wiki.factorio.com/Inserters).

##### 1.4. Insertores y cintas transportadoras

Al transferir objetos desde una cinta, el inserter debe recoger los objetos que pasan por la posición de alcance. Al depositarlos en una cinta, necesita encontrar un espacio libre. Si la cinta está completamente llena, puede quedarse esperando hasta que se abra un hueco.

Este comportamiento es importante al diseñar líneas rápidas: aumentar la velocidad del inserter no solucionará un problema si la cinta de salida está saturada o la cinta de entrada no transporta suficientes materiales.

---

#### 2. Tipos de insertores

En el juego base de *Factorio 2.x* hay cinco tipos principales: inserter a combustión, inserter normal, inserter de brazo largo, inserter rápido e inserter a granel. La expansión *Space Age* añade el inserter de apilamiento.

##### 2.1. Inserter a combustión (*Burner inserter*)


El **inserter a combustión** es el modelo inicial. Consume combustible en lugar de electricidad, por lo que puede seguir trabajando incluso durante una interrupción del suministro eléctrico.

**Características principales:**

- Tamaño: 1 × 1 casilla.
- Alimentación: combustible compatible, por ejemplo carbón o madera.
- Es el modelo más lento de los insertores comunes.
- No consume combustible cuando está inactivo.
- Puede reponer su propio combustible cuando recoge combustible que pasa por su zona de recogida.
- No produce contaminación directamente mientras mueve objetos, aunque la producción y el transporte del combustible pueden generarla indirectamente.

Es útil al comienzo de la partida para cargar carbón en calderas o para alimentar máquinas cuando aún no se ha construido una red eléctrica fiable. También sirve para mantener una parte de la central de vapor operativa durante ciertos apagones.

Su desventaja es la velocidad relativamente baja. Para una fábrica grande, los insertores eléctricos suelen ser más convenientes.

Más información: [Factorio Wiki: Burner inserter](https://wiki.factorio.com/Burner_inserter).

##### 2.2. Inserter normal (*Inserter*)


El **inserter normal**, reconocido por su color amarillo, es el modelo eléctrico básico. Se utiliza en la mayor parte de las primeras cadenas automatizadas.

**Características principales:**

- Tamaño: 1 × 1 casilla.
- Consumo eléctrico máximo de referencia: 15,1 kW mientras trabaja.
- Consumo en espera: 400 W.
- Velocidad intermedia entre el modelo a combustión y los modelos especializados.
- Admite filtros en *Factorio 2.x*.
- La capacidad de su mano puede aumentar mediante la investigación de bonificación de capacidad de los insertores.

Suele ser suficiente para alimentar una máquina de montaje o retirar productos de una máquina que no necesite un ritmo muy elevado. Es un buen punto de partida porque es económico y consume relativamente poca electricidad frente a los modelos más rápidos.

Más información: [Factorio Wiki: Inserter](https://wiki.factorio.com/Inserter).

##### 2.3. Inserter de brazo largo (*Long-handed inserter*)


El **inserter de brazo largo** puede recoger y depositar objetos a una distancia de dos casillas desde su posición, en lugar del alcance habitual de una casilla. Se utiliza cuando los edificios o las cintas no están inmediatamente adyacentes.

**Características principales:**

- Tamaño: 1 × 1 casilla.
- Alcance: hasta dos casillas desde la base para las posiciones correspondientes de recogida y entrega.
- Consumo eléctrico máximo de referencia: 21,4 kW mientras trabaja.
- Consumo en espera: 400 W.
- Velocidad de giro inferior a la de los insertores rápido, a granel y de apilamiento.

Es especialmente útil para:

- Alimentar una máquina desde una cinta que está al otro lado de una fila de hornos.
- Retirar placas de un horno y colocarlas en una cinta más alejada.
- Crear diseños compactos con diferentes líneas de materiales.
- Cruzar determinadas separaciones de terreno que no bloqueen su alcance.

Su alcance adicional no implica que sea el modelo más rápido. Si la distancia no es necesaria, un inserter normal o rápido suele ser una opción más sencilla.

Más información: [Factorio Wiki: Long-handed inserter](https://wiki.factorio.com/Long-handed_inserter).

##### 2.4. Inserter rápido (*Fast inserter*)


El **inserter rápido** es un modelo eléctrico que gira considerablemente más rápido que el inserter normal. Es apropiado para líneas que necesitan transferir materiales a un ritmo elevado.

**Características principales:**

- Tamaño: 1 × 1 casilla.
- Consumo eléctrico máximo de referencia: 59,3 kW mientras trabaja.
- Consumo en espera: 500 W.
- Velocidad de giro aproximadamente tres veces mayor que la del inserter normal en la comparación nominal de giro.
- Admite filtros y control mediante redes cuando las tecnologías correspondientes están disponibles.

Usos habituales:

- Cargar y descargar hornos en líneas de producción intensivas.
- Alimentar máquinas de montaje rápidas.
- Cargar cofres o vagones con un ritmo elevado.
- Transferir materiales desde cintas rápidas o expresas, donde un inserter normal puede tener dificultades en ciertas curvas o posiciones.

Su consumo máximo es mayor que el del inserter normal. No conviene utilizarlo en todas partes automáticamente: debe reservarse para los lugares donde la capacidad de transferencia adicional sea útil.

Más información: [Factorio Wiki: Fast inserter](https://wiki.factorio.com/Fast_inserter).

##### 2.5. Inserter a granel (*Bulk inserter*)


El **inserter a granel** puede transportar varios objetos en cada ciclo. Tiene una velocidad de giro similar a la del inserter rápido, pero su mano puede llevar más objetos, lo que aumenta considerablemente el caudal en transferencias adecuadas.

**Características principales:**

- Tamaño: 1 × 1 casilla.
- Consumo eléctrico máximo de referencia: 169 kW mientras trabaja.
- Consumo en espera: 1 kW.
- Velocidad de giro similar a la del inserter rápido.
- La capacidad máxima de su mano aumenta con la tecnología de bonificación de capacidad de insertores.
- En el juego base, la capacidad llega hasta 12 objetos por ciclo con todas las mejoras indicadas por la wiki, sujeto a la capacidad de apilamiento del objeto.

Es especialmente útil en:

- Carga y descarga de vagones de carga.
- Transferencias entre cofres con gran volumen de materiales.
- Alimentación de máquinas que consumen muchos ingredientes por segundo.
- Estaciones ferroviarias que necesitan mover grandes cantidades de objetos en poco tiempo.

Cuando recoge o deja objetos sobre una cinta transportadora, cada objeto requiere un tratamiento individual del movimiento; por eso su caudal real sobre cintas es inferior al de una transferencia entre cofres y depende de los huecos libres de la cinta.

**Diferencia importante:** en la versión 2.0 se cambió el nombre del antiguo inserter de apilamiento a *Bulk inserter* (inserter a granel). No debe confundirse con el nuevo inserter de apilamiento de *Space Age*.

Más información: [Factorio Wiki: Bulk inserter](https://wiki.factorio.com/Bulk_inserter).

##### 2.6. Inserter de apilamiento (*Stack inserter*) — *Space Age*


El **inserter de apilamiento** es una estructura exclusiva de *Space Age*. Puede manipular pilas grandes y colocar objetos en capas sobre las cintas, elevando la cantidad de materiales transportados sin necesidad de aumentar proporcionalmente el número de cintas.

**Características principales:**

- Tamaño: 1 × 1 casilla.
- Consumo eléctrico máximo de referencia: 337 kW mientras trabaja.
- Consumo en espera: 1 kW.
- Velocidad de giro similar a la del inserter a granel.
- Tiene una capacidad de mano mayor que la del inserter a granel: comienza con 6 objetos y puede llegar a 16 con la investigación de capacidad completa, según las reglas de la expansión.
- Al introducir objetos en una cinta, suele esperar hasta completar la pila máxima disponible antes de girar.

La capacidad de apilar objetos en una cinta puede aumentar notablemente el caudal de una línea. Sin embargo, el inserter puede esperar más tiempo cuando no llegan suficientes objetos para completar su mano. Esto es importante para los objetos perecederos de *Space Age*: esperar a completar una pila puede reducir su frescura o permitir que se estropeen.

No es un reemplazo universal del inserter a granel. Para artículos perecederos o flujos irregulares, el inserter a granel puede resultar más adecuado porque no está sujeto al mismo comportamiento de espera para completar pilas.

Más información: [Factorio Wiki: Stack inserter](https://wiki.factorio.com/Stack_inserter).

---

#### 3. Comparación de tipos

| Tipo | Fuente de energía | Velocidad relativa | Alcance | Capacidad y usos más característicos |
|---|---|---|---|---|
| Inserter a combustión | Combustible | Baja | Normal | Respaldo y automatización temprana sin red eléctrica. |
| Inserter normal | Electricidad | Media-baja | Normal | Transferencias generales y líneas de consumo moderado. |
| Inserter de brazo largo | Electricidad | Media | Largo | Alcanzar cintas y edificios más alejados. |
| Inserter rápido | Electricidad | Alta | Normal | Producción rápida y alimentación intensiva. |
| Inserter a granel | Electricidad | Giro rápido y capacidad grande | Normal | Transferencias voluminosas, cofres y trenes. |
| Inserter de apilamiento (*Space Age*) | Electricidad | Giro rápido y gran capacidad | Normal | Apilar objetos sobre cintas y maximizar el caudal. |

Las velocidades efectivas dependen de la posición de recogida, la distancia de giro, el contenido de la cinta, la cantidad de objetos que puede llevar y el estado de la energía. La tabla es una guía de elección, no una promesa de caudal exacto para cualquier configuración.

---

#### 4. Filtros, orientación y configuración avanzada

##### 4.1. Orientación del inserter

La orientación establece el punto de recogida y el punto de entrega. Un inserter mal orientado puede dejar de funcionar aunque se encuentre al lado de la cinta o de la máquina correctas.

Al colocarlo, comprueba:

- Que la base está en la casilla correcta.
- Que el punto de recogida coincide con la cinta, el cofre o el edificio de origen.
- Que el punto de entrega coincide con la salida o el inventario de destino.
- Que el destino admite el objeto que se está transfiriendo.

En versiones recientes, los insertores pueden voltearse en condiciones compatibles para facilitar la modificación de planos (*blueprints*). El volteo es útil al copiar una línea simétrica, pero conviene revisar siempre la dirección de transferencia después de editar el plano.

##### 4.2. Filtros de objetos

Desde *Factorio 2.0.7*, todos los tipos de insertores pueden utilizar filtros. En versiones anteriores existían modelos especiales de inserter filtrador, pero en 2.x el filtrado se incorpora en la interfaz del inserter.

Un filtro permite restringir los objetos válidos. Por ejemplo, un inserter conectado a una cinta con varios recursos puede configurarse para recoger únicamente placas de hierro y dejar pasar el resto.

Los filtros se utilizan para:

- Clasificar objetos al vaciar un cofre o vagón.
- Separar productos en cintas mixtas.
- Impedir que una máquina reciba ingredientes que no utiliza.
- Extraer materiales específicos de inventarios compartidos.
- Controlar el tipo de objeto que un inserter de apilamiento puede agrupar.

La interfaz permite definir hasta cinco tipos filtrados por inserter. Cuando la configuración está conectada a una red de circuitos, también puede elegirse una lista de filtros a partir de señales, con las limitaciones descritas en la wiki.

##### 4.3. Redes de circuitos

Los insertores pueden conectarse a una **red de circuitos** mediante cables rojos o verdes y controlarse con condiciones. Según la configuración, permiten:

- **Habilitar o deshabilitar:** el inserter funciona cuando la condición configurada es verdadera.
- **Leer el contenido de la mano:** envía a la red una señal que representa los objetos transportados. Puede emitir un pulso breve o mantener la señal mientras sostiene objetos.
- **Establecer el tamaño de la mano:** ajusta cuántos objetos intenta mover, con un límite mínimo de 1 y un máximo determinado por la capacidad investigada del inserter.
- **Configurar filtros desde señales:** selecciona los tipos permitidos a partir de las señales de la red.

Un valor de tamaño de mano igual a cero o negativo no desactiva el inserter; ajusta su tamaño de mano a 1. Para activar o desactivar el movimiento, utiliza el ajuste específico de habilitación.

Las condiciones de circuito son útiles para limitar la carga de un cofre, detener una extracción cuando hay suficientes materiales o coordinar líneas que comparten recursos.

##### 4.4. Red logística

Cuando se ha investigado y construido una red logística, determinados insertores pueden controlarse mediante las condiciones del sistema logístico. Esto permite automatizar transferencias en función de la cantidad de un objeto disponible en la red, en vez de limitarse únicamente al contenido de una cinta o un cofre concreto.

No hay que confundir el control por red logística con el control por red de circuitos: el primero utiliza las existencias y condiciones de la red de robots logísticos; el segundo utiliza señales transmitidas por cables y dispositivos de circuito.

---

#### 5. Objetos e insumos necesarios

##### 5.1. Materiales para fabricar insertores

Los ingredientes exactos dependen del tipo y de la versión. Las recetas normales de referencia son las siguientes:

| Inserter | Ingredientes habituales | Observaciones |
|---|---|---|
| Inserter a combustión | 1 placa de hierro y 1 engranaje de hierro | Disponible desde el inicio; no necesita componentes electrónicos. |
| Inserter normal | 1 circuito electrónico, 1 engranaje de hierro y 1 placa de hierro | Desbloqueado en la progresión de electrónica. |
| Inserter de brazo largo | Inserter normal, componentes mecánicos adicionales según la receta | Se fabrica a partir de la base de inserter eléctrico. |
| Inserter rápido | Inserter normal, circuitos electrónicos y engranajes adicionales | Requiere la tecnología correspondiente. |
| Inserter a granel | Inserter rápido, engranajes y circuitos en grandes cantidades | Necesita investigación avanzada y una cantidad mayor de materiales. |
| Inserter de apilamiento (*Space Age*) | Inserter a granel, unidad de procesamiento, fibra de carbono y gelatina | La receta utiliza recursos propios de la expansión. |

Las recetas se pueden modificar entre versiones y expansiones. Para evitar errores al construir una cadena de fabricación, verifica las cantidades en la interfaz de recetas del juego o en la página individual enlazada al final de esta guía.

##### 5.2. Insumos operativos

| Insumo | Tipos que lo requieren | Función |
|---|---|---|
| Combustible | Inserter a combustión | Aporta la energía para mover el brazo. |
| Electricidad | Insertores eléctricos | Permite girar, extender y retraer el brazo. |
| Cables de circuito | Insertores conectados a una red de circuitos | Transmiten señales de control y lectura. |
| Señales de circuito | Configuración avanzada | Establecen condiciones, filtros o tamaños de mano. |
| Señales logísticas | Red logística | Permiten habilitar o deshabilitar insertores según el estado de la red. |
| Investigación de capacidad de insertores | Todos los modelos compatibles | Aumenta la cantidad de objetos que pueden transportar por ciclo. |

##### 5.3. Objetos que puede mover

Los insertores transportan la mayoría de los objetos sólidos del juego, siempre que el destino sea compatible. Algunos ejemplos:

| Categoría | Ejemplos | Uso frecuente |
|---|---|---|
| Minerales y recursos | Mineral de hierro, cobre, carbón, piedra | Alimentación de hornos y máquinas. |
| Materiales procesados | Placas de hierro, cobre y acero, engranajes | Fabricación de componentes. |
| Componentes electrónicos | Circuitos electrónicos y avanzados | Alimentación de líneas electrónicas. |
| Combustibles | Carbón, madera, combustible sólido | Calderas, hornos y vehículos compatibles. |
| Productos químicos sólidos | Plástico, azufre, baterías | Cadenas químicas y de fabricación. |
| Paquetes de ciencia | Paquetes de ciencia de distintos colores | Alimentación de laboratorios. |
| Productos de *Space Age* | Objetos y recursos planetarios compatibles | Producción avanzada y logística planetaria. |

Los fluidos no se transportan directamente con insertores; deben circular por tuberías, bombas u otros elementos del sistema de fluidos. Los insertores sí pueden mover recipientes u objetos sólidos relacionados con fluidos cuando la receta y el inventario admiten esos objetos.

---

#### 6. Rendimiento, capacidad y velocidad

##### 6.1. Velocidad de giro

La velocidad de giro se mide como un ritmo de rotación. Como referencia para calidad normal, las velocidades publicadas en la wiki equivalen aproximadamente a los siguientes ciclos de giro completos:

| Tipo | Tiempo orientativo por giro completo | Consideraciones |
|---|---:|---|
| Inserter a combustión | 1,27 s | Es el más lento de los modelos comunes. |
| Inserter normal | 1,17 s | Adecuado para transferencias generales. |
| Inserter de brazo largo | 0,80 s | El mayor alcance cambia la trayectoria del brazo. |
| Inserter rápido | 0,40 s | Gira rápidamente, pero requiere más potencia. |
| Inserter a granel | 0,40 s | Velocidad de giro similar al rápido, pero con mayor mano. |
| Inserter de apilamiento | 0,40 s | Velocidad de giro similar al a granel, con capacidad mayor. |

Estos tiempos no equivalen necesariamente a una transferencia de ida y vuelta con objetos. Los ciclos reales varían según la recogida, la entrega, el alcance, la disponibilidad de productos y la geometría del destino. Son útiles para comparar modelos, no para calcular el caudal exacto sin más datos.

##### 6.2. Bonificación de capacidad de los insertores

La tecnología de **bonificación de capacidad de insertores** aumenta la cantidad de objetos que un inserter puede transportar por ciclo. Esto puede aumentar la producción sin añadir más brazos mecánicos.

La capacidad máxima depende del tipo:

- Los insertores comunes pueden aumentar su tamaño de mano mediante investigación.
- El inserter a granel puede transportar más objetos por ciclo que un inserter común; con el nivel máximo citado por la wiki puede alcanzar 12 objetos.
- El inserter de apilamiento de *Space Age* comienza con una capacidad mayor y puede llegar a 16 objetos por ciclo con las mejoras correspondientes.

La cantidad real puede quedar limitada por el tamaño de pila del objeto, la cantidad disponible en el origen y las reglas de apilamiento del destino.

##### 6.3. Caudal sobre cintas frente a cofres

Un inserter suele transferir objetos entre cofres con mayor eficiencia que cuando recoge o entrega en una cinta. En los cofres puede mover una pila en una sola operación de entrega, mientras que sobre una cinta tiene que respetar los espacios disponibles y las reglas de apilamiento.

Cuando un inserter descarga sobre una cinta llena, se queda esperando a que aparezca espacio. Por esta razón, el caudal de una línea depende tanto de la capacidad del inserter como de la velocidad de la cinta y de la ocupación de sus carriles.

##### 6.4. Consumo energético

Los insertores eléctricos consumen energía al moverse y tienen un pequeño consumo en espera. Como referencia de calidad normal:

| Tipo | Consumo máximo durante el trabajo | Consumo en espera |
|---|---:|---:|
| Inserter normal | 15,1 kW | 0,4 kW |
| Inserter de brazo largo | 21,4 kW | 0,4 kW |
| Inserter rápido | 59,3 kW | 0,5 kW |
| Inserter a granel | 169 kW | 1 kW |
| Inserter de apilamiento | 337 kW | 1 kW |

El valor máximo no implica que ese consumo se mantenga continuamente. El gasto depende de cuánto se mueva el inserter y de su comportamiento operativo. Cuando una fábrica utiliza cientos de insertores de alta capacidad, el consumo agregado puede ser importante.

---

#### 7. Diseños y aplicaciones prácticas

##### 7.1. Alimentar una máquina de montaje

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada de ingredientes | Ingredientes sobre una cinta | Inserter de entrada | Ingredientes transferidos | — |
| Fabricación | Ingredientes transferidos | Máquina de montaje | Producto fabricado | — |
| Retirada del producto | Producto fabricado | Inserter de salida | Producto transferido | — |
| Transporte del producto | Producto transferido | Cinta del producto final | Producto final transportado | — |

Utiliza insertores de entrada para suministrar los ingredientes que necesita la receta y otro inserter para retirar el producto. Si la máquina trabaja despacio, un inserter normal suele ser suficiente. En líneas rápidas, comprueba si el tiempo de transferencia del inserter está limitando la producción.

##### 7.2. Línea de hornos

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Alimentación de mineral | Cinta de mineral | Inserters | Mineral para los hornos | — |
| Alimentación de combustible | Cinta de combustible | Inserters | Combustible para los hornos | — |
| Retirada de placas | Hornos | Inserters | Placas sobre la cinta de salida | — |
| Transporte de placas | Placas | Cinta de placas | Placas transportadas | — |

Los insertores de entrada colocan el mineral y, si corresponde, el combustible. Los insertores de salida retiran las placas terminadas. La posición exacta depende de la orientación del horno y de la receta; revisa que los materiales de entrada no compitan por una misma casilla de destino.

##### 7.3. Cargar y descargar un vagón de carga

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada para carga | Cinta de entrada | Insertores a granel | Objetos transferidos al sistema de carga | — |
| Carga del tren | Objetos de entrada | Insertores a granel y vagón de carga | Vagón cargado | — |
| Descarga del tren | Objetos del vagón de carga | Insertores a granel | Objetos descargados | — |
| Transporte de salida | Objetos descargados | Cintas de salida | Objetos transportados | — |

Los insertores a granel son útiles porque pueden mover varios objetos por ciclo. Para una estación ferroviaria de alto rendimiento, utiliza suficientes insertores por vagón y asegura que las cintas de entrada y salida tengan capacidad para absorber el caudal total.

El equilibrio de una estación depende de los tiempos de carga y descarga, el número de insertores, el tipo de cinta y el espacio disponible en el inventario del vagón.

##### 7.4. Clasificar materiales con filtros

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada de objetos variados | Objetos variados | Cinta o cofre | Mezcla de objetos | — |
| Filtrado | Mezcla de objetos | Inserter con filtro | Objeto elegido | El filtro limita qué objeto puede mover el inserter. |
| Transporte posterior | Objeto elegido | Ruta de salida | Objeto elegido en la ruta correspondiente | — |

En *Factorio 2.x* ya no es necesario fabricar modelos separados de inserter filtrador para realizar una clasificación sencilla: se configura el filtro en el inserter que corresponda. Una configuración de filtro mal definida puede dejar objetos en la cinta o impedir que un cofre se vacíe por completo; revisa todas las categorías de objetos que pasan por el sistema.

##### 7.5. Apilar objetos en cintas — *Space Age*

El inserter de apilamiento se utiliza cuando el objetivo es aumentar el número de objetos que circulan sobre una cinta mediante capas. Para aprovecharlo, la línea debe estar preparada para manejar las pilas y las tecnologías de apilamiento correspondientes.

No todas las líneas necesitan apilamiento: en una fábrica de baja producción, la complejidad y el consumo adicional pueden no justificarse. En líneas de gran volumen, puede reducir la cantidad de cintas necesarias para transportar el mismo caudal.

---

#### 8. Estrategias y consejos de optimización

##### 8.1. Elige el inserter según el cuello de botella

- **Normal:** transferencia general a velocidades moderadas.
- **De brazo largo:** cuando se necesita más alcance, no como sustituto universal del normal.
- **Rápido:** cuando los materiales deben moverse con mucha frecuencia.
- **A granel:** cuando la prioridad es transferir muchas unidades por ciclo.
- **De apilamiento:** cuando se necesita una gran capacidad y apilar objetos en cintas, en *Space Age*.
- **A combustión:** para el inicio o como respaldo específico cuando interesa no depender de la red eléctrica.

Antes de sustituir todos los insertores por modelos rápidos, revisa si la máquina receptora realmente consume materiales más deprisa. Si la salida está llena o la cinta de entrada no tiene suficientes objetos, un inserter más caro no aumentará la producción real.

##### 8.2. No saturar las cintas

Un inserter puede estar preparado para descargar, pero no hacerlo porque no encuentra espacio libre en la cinta. Para corregirlo, amplía la capacidad de transporte, utiliza una ruta alternativa o incrementa la velocidad de la cinta cuando sea necesario.

La saturación puede detectarse si un inserter permanece con objetos en la mano mientras la máquina anterior sigue produciendo. También puede ocurrir que una cinta de entrada se vacíe de manera intermitente, dejando a los insertores sin material durante parte de su ciclo.

##### 8.3. Usar filtros en los puntos correctos

Los filtros convienen cuando una misma fuente contiene varios tipos de objetos o cuando se necesita impedir que un destino reciba un material incorrecto. En una cadena con cintas separadas por tipo de recurso, no siempre hacen falta filtros: puede ser más sencillo mantener cada producto en su propia cinta.

Utiliza filtros especialmente en estaciones ferroviarias, cofres compartidos y sistemas de clasificación. Si una red de circuitos altera los filtros, revisa que las señales enviadas correspondan a los objetos que realmente se quieren permitir.

##### 8.4. Evitar tamaños de mano innecesarios

Una mano grande aumenta la cantidad de objetos transferidos por ciclo, pero puede provocar esperas o transferir más unidades de las que se necesitan en ese momento. En máquinas con inventarios pequeños, configura el tamaño de mano si las transferencias grandes provocan sobreabastecimiento.

En *Space Age*, evalúa cuidadosamente el uso de insertores de apilamiento para objetos perecederos. La espera para completar una pila puede perjudicar la frescura del producto.

##### 8.5. Planificar el consumo eléctrico

Los insertores rápidos, a granel y de apilamiento pueden consumir mucha más potencia durante el movimiento que los modelos básicos. En una fábrica extensa, conviene valorar el consumo acumulado y evitar colocar modelos de alta capacidad donde no aporten una ventaja.

Si la fábrica tiene problemas de electricidad, puede ser útil utilizar insertores normales en las transferencias lentas y reservar los modelos de mayor consumo para las áreas de producción intensiva.

##### 8.6. Diseñar planos repetibles

Los planos (*blueprints*) facilitan copiar filas de hornos, máquinas o estaciones de carga. Al construir un plano, verifica la orientación de todos los insertores, sus filtros, las condiciones de circuito y los tamaños de mano. Un diseño que funciona con una orientación incorrecta puede ser difícil de corregir cuando se copia cientos de veces.

---

#### 9. Errores comunes y soluciones

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| El inserter no recoge objetos. | Está mal orientado, no hay objetos en el punto de recogida o el filtro los rechaza. | Verifica la orientación, la cinta o inventario de origen y los filtros. |
| Recoge objetos, pero no los entrega. | La máquina o el cofre de destino está lleno o no acepta ese tipo de objeto. | Libera espacio o revisa la receta y las restricciones del inventario. |
| La máquina recibe materiales demasiado despacio. | El inserter es demasiado lento o el origen no suministra suficiente flujo. | Comprueba la cinta y cambia a un inserter rápido si el brazo es el cuello de botella. |
| La cinta de salida se llena. | La línea posterior no consume lo bastante rápido o la cinta tiene capacidad insuficiente. | Aumenta el transporte o el consumo posterior y evita que se bloquee el destino. |
| El inserter eléctrico se detiene o funciona lentamente. | La red eléctrica es insuficiente o la conexión está ausente. | Revisa los postes y la producción eléctrica. |
| El inserter a combustión deja de moverse. | Se agotó su combustible. | Suministra combustible y comprueba que el diseño puede reabastecerlo. |
| El filtro no recoge el objeto esperado. | El filtro está mal seleccionado o la señal de circuito no coincide. | Revisa la configuración de filtros y la lógica de señales. |
| La carga ferroviaria es lenta. | Hay pocos insertores, poca capacidad en las cintas o una configuración de mano ineficiente. | Añade capacidad donde sea necesario y comprueba el caudal por vagón. |
| El inserter de apilamiento espera demasiado. | No llegan suficientes objetos para completar la pila o hay objetos perecederos. | Ajusta el flujo, configura filtros cuando corresponda o usa un inserter a granel para artículos perecederos. |
| El sistema de circuitos no desactiva el inserter. | Se configuró el tamaño de mano en lugar de la condición de habilitación. | Usa la opción de habilitar/deshabilitar y comprueba la señal de entrada. |

---

#### 10. Lista de comprobación para una instalación automatizada

- [ ] El inserter está orientado correctamente entre el origen y el destino.
- [ ] El destino admite los objetos que recibe.
- [ ] La fuente puede suministrar suficientes objetos para la tasa requerida.
- [ ] La cinta de salida tiene espacio disponible.
- [ ] El tipo de inserter corresponde a la distancia y al caudal necesario.
- [ ] Los insertores eléctricos reciben energía suficiente.
- [ ] Los insertores a combustión tienen combustible y una forma de reabastecerse.
- [ ] Los filtros permiten los objetos correctos y no bloquean la producción.
- [ ] El tamaño de mano es apropiado para la receta y el destino.
- [ ] Las condiciones de circuito o de red logística se han probado.
- [ ] En *Space Age*, se consideró el efecto de la espera sobre los objetos perecederos al usar insertores de apilamiento.
- [ ] Los planos mantienen orientación, filtros y cableado coherentes al copiarse.

---

#### 11. Fuentes de referencia

Las siguientes páginas de la *Factorio Wiki* contienen las estadísticas, recetas, tecnologías y explicaciones detalladas de las mecánicas:

- [Inserters — Mecánicas generales y rendimiento](https://wiki.factorio.com/Inserters)
- [Burner inserter — Inserter a combustión](https://wiki.factorio.com/Burner_inserter)
- [Inserter — Inserter normal](https://wiki.factorio.com/Inserter)
- [Long-handed inserter — Inserter de brazo largo](https://wiki.factorio.com/Long-handed_inserter)
- [Fast inserter — Inserter rápido](https://wiki.factorio.com/Fast_inserter)
- [Bulk inserter — Inserter a granel](https://wiki.factorio.com/Bulk_inserter)
- [Stack inserter — Inserter de apilamiento de *Space Age*](https://wiki.factorio.com/Stack_inserter)
- [Inserter capacity bonus — Bonificación de capacidad de insertores](https://wiki.factorio.com/Inserter_capacity_bonus_(research))
- [Circuit network — Red de circuitos](https://wiki.factorio.com/Circuit_network)
- [Logistic network — Red logística](https://wiki.factorio.com/Logistic_network)


---

### Guía técnica de Factorio: Fundición y refinamiento de materiales

**Alcance:** mecánicas del juego base de *Factorio 2.x*, con las funciones exclusivas de la expansión *Space Age* identificadas por separado.

La **fundición y el refinamiento** convierten las materias primas extraídas del mapa en materiales que sirven para fabricar edificios, componentes, maquinaria, productos químicos y tecnologías. Aunque suelen tratarse como una misma etapa de producción, en *Factorio* hay dos procesos distintos:

- **Fundición:** transforma minerales y otros materiales sólidos en productos como placas de hierro, placas de cobre, acero y ladrillos de piedra.
- **Refinamiento:** procesa petróleo crudo y otros insumos para obtener fluidos industriales, como gas de petróleo, petróleo ligero y petróleo pesado, que luego se emplean en recetas químicas.

En *Space Age*, las **fundiciones** (*Foundries*) amplían la metalurgia al producir metales fundidos y moldearlos directamente en placas y otros productos.

---

#### 1. Fundamentos de la fundición y el refinamiento

##### 1.1. ¿Qué es la fundición?

La fundición es el proceso mediante el cual una máquina compatible aplica una receta de fundición a una materia prima sólida. El ejemplo más común consiste en convertir mineral de hierro en placas de hierro.

La fundición es indispensable porque la mayoría de las líneas de fabricación no utilizan directamente el mineral extraído. Necesitan **placas metálicas** u otros materiales procesados como insumos.

Ejemplos de recetas básicas:

| Receta | Insumos | Producto | Tiempo base de receta |
|---|---|---|---:|
| Placa de hierro | 1 mineral de hierro | 1 placa de hierro | 3,2 s |
| Placa de cobre | 1 mineral de cobre | 1 placa de cobre | 3,2 s |
| Ladrillo de piedra | 2 piedras | 1 ladrillo de piedra | 3,2 s |
| Placa de acero | 5 placas de hierro | 1 placa de acero | 16 s |

Los tiempos indicados son los de la receta, antes de aplicar la velocidad propia de la máquina, los módulos u otros modificadores. Por eso dos hornos diferentes pueden completar la misma receta en distinto tiempo.

Las placas de hierro y cobre son materiales centrales desde el comienzo de la partida. El hierro aparece en una gran cantidad de recetas de construcción; el cobre es esencial para cables y circuitos. El acero se vuelve cada vez más importante a medida que se desbloquean edificios y componentes avanzados.

##### 1.2. ¿Qué es el refinamiento?

En el contexto de *Factorio*, el refinamiento de petróleo es el proceso industrial que convierte petróleo crudo —o carbón, en recetas específicas— en productos derivados. La refinería de petróleo procesa fluidos y puede generar diferentes fracciones según la receta seleccionada.

Los principales productos son:

- **Gas de petróleo:** se utiliza, entre otras cosas, para producir plástico y azufre.
- **Petróleo ligero:** puede transformarse en combustible sólido y convertirse en gas de petróleo mediante craqueo.
- **Petróleo pesado:** sirve para fabricar lubricante y puede transformarse en petróleo ligero.
- **Combustible sólido:** se puede producir a partir de determinados derivados del petróleo y utilizarse como combustible.

El refinamiento no finaliza necesariamente en la refinería. Las plantas químicas convierten los fluidos en productos que se consumen en otras líneas de producción.

##### 1.3. Cómo encajan ambos procesos en la fábrica

Un flujo típico de materiales sólidos es:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción | Yacimiento de mineral | Taladros mineros | Mineral | — |
| Transporte | Mineral | Cintas transportadoras | Mineral transportado | — |
| Fundición o proceso especializado | Mineral transportado | Hornos o fundición | Placas metálicas o ladrillos | — |
| Fabricación | Placas metálicas o ladrillos | Línea de fabricación | Componentes | — |

Un flujo típico de petróleo es:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción de crudo | Campo petrolífero | Bombas extractoras | Petróleo crudo | — |
| Transporte de crudo | Petróleo crudo | Tuberías | Petróleo crudo transportado | — |
| Refinación | Petróleo crudo y agua si la receta seleccionada la necesita | Refinería de petróleo | Petróleo pesado, petróleo ligero y gas de petróleo | Las tres ramas se muestran en el esquema; la receta seleccionada determina las salidas. |
| Uso del petróleo pesado | Petróleo pesado | Procesamiento de lubricante o craqueo | Lubricante o productos de craqueo | — |
| Uso del petróleo ligero | Petróleo ligero | Producción de combustible sólido o craqueo | Combustible sólido o productos de craqueo | — |
| Uso del gas de petróleo | Gas de petróleo | Plantas químicas u otras recetas | Plástico, azufre u otros productos | — |

La importancia de estos sistemas está en que una fábrica no puede crecer si sus líneas de montaje no reciben materiales procesados de forma estable. Una buena fundición evita la falta de placas; un sistema de refinamiento bien diseñado evita que las cadenas químicas se detengan.

---

#### 2. Edificios y estructuras de fundición

##### 2.1. Horno de piedra (*Stone furnace*)


El **horno de piedra** es la primera estructura de fundición disponible. Es económico, funciona con combustible y resulta suficiente para iniciar las primeras líneas de placas de hierro, placas de cobre y ladrillos de piedra.

**Características principales:**

- Tamaño: 2 × 2 casillas.
- Velocidad de fabricación: 1.
- Consumo nominal: 90 kW de energía térmica, aportada por combustible.
- Contaminación propia: 2 unidades por minuto, según los valores de referencia de la wiki.
- No requiere conexión eléctrica.
- No dispone de ranuras para módulos.

**Función:** transformar minerales y otros ingredientes compatibles mediante recetas de fundición.

Durante las primeras etapas es habitual colocar una hilera de hornos junto a una cinta con mineral y otra cinta o sistema de insertores para retirar las placas terminadas. El horno puede recibir combustible manualmente o mediante una cinta e insertores.

Su principal ventaja es el bajo costo de construcción. La desventaja es su velocidad de fabricación de 1, la más baja de los tres hornos comunes.

##### 2.2. Horno de acero (*Steel furnace*)


El **horno de acero** es la segunda categoría de horno. A pesar de su nombre, no funde exclusivamente acero: puede realizar las recetas de fundición disponibles para los hornos comunes.

**Características principales:**

- Tamaño: 2 × 2 casillas.
- Velocidad de fabricación: 2.
- Consumo nominal: 90 kW térmicos, mediante combustible.
- Contaminación propia: 4 unidades por minuto, según los valores de referencia de la wiki.
- No requiere electricidad.
- No admite módulos.

Con una velocidad de 2, produce el doble por segundo que un horno de piedra para las recetas compatibles. Aunque su consumo de combustible por unidad de tiempo es similar, realiza más trabajo durante ese tiempo, por lo que aprovecha mejor el combustible por producto fabricado.

Puede reemplazar directamente un horno de piedra en el mismo lugar. Esto facilita mejorar una fundición existente sin tener que rediseñar toda la línea.

##### 2.3. Horno eléctrico (*Electric furnace*)


El **horno eléctrico** es el tercer nivel de horno del juego base. Utiliza electricidad en vez de combustible, por lo que elimina la necesidad de transportar carbón u otro combustible hasta cada horno.

**Características principales:**

- Tamaño: 3 × 3 casillas.
- Velocidad de fabricación: 2.
- Consumo eléctrico nominal: 180 kW cuando está trabajando.
- Consumo eléctrico en espera: aproximadamente 6 kW.
- Contaminación propia: 1 unidad por minuto.
- Dispone de 2 ranuras para módulos.
- Necesita una red eléctrica que pueda suministrar su consumo.

Su velocidad base es igual a la del horno de acero, pero ocupa más espacio y exige energía eléctrica. A cambio, no necesita combustible, produce menos contaminación directamente y puede aprovechar módulos de velocidad, eficiencia y otros efectos compatibles.

No siempre es una mejora automática si la red eléctrica es débil. Si la generación eléctrica depende de calderas y máquinas de vapor, hay que considerar también la contaminación y el costo energético de abastecer una gran cantidad de hornos eléctricos.

En las plataformas espaciales de *Space Age*, los dispositivos que necesitan combustible no funcionan de la forma habitual, por lo que el horno eléctrico es importante para la fundición que se realice en el espacio.

##### 2.4. Fundición (*Foundry*) — *Space Age*


La **fundición** es un edificio metalúrgico avanzado exclusivo de *Space Age*. Se fabrica en Vulcanus y puede transportarse a otros planetas mediante una plataforma espacial.

**Características principales:**

- Tamaño: 5 × 5 casillas.
- Velocidad de fabricación base: 4.
- Consumo eléctrico nominal: 2,5 MW.
- Dispone de 4 ranuras para módulos.
- Tiene una bonificación de productividad integrada del 50 % para sus recetas aplicables.
- Puede trabajar con fluidos metálicos fundidos y recetas metalúrgicas especializadas.

La fundición amplía la metalurgia de dos formas:

1. **Produce metal fundido:** puede convertir minerales en hierro o cobre fundidos, y en Vulcanus también puede obtener estos metales a partir de lava usando calcita.
2. **Moldea el metal fundido:** convierte fluidos metálicos directamente en placas y en otros productos, en lugar de obligar a realizar todas las etapas mediante hornos y máquinas de montaje separados.

El hierro y el cobre fundidos son fluidos. No se pueden guardar en barriles ni transportar entre planetas como una carga líquida convencional; el proceso de fundición y moldeo debe planificarse dentro de la infraestructura donde se manejen.

La fundición utiliza mucha electricidad, por lo que conviene comprobar la capacidad de generación y el diseño de las tuberías antes de instalarla en grandes cantidades.

Más información: [Factorio Wiki: Foundry](https://wiki.factorio.com/Foundry).

##### 2.5. Estructuras auxiliares de la fundición

| Estructura | Función |
|---|---|
| **Taladros mineros** | Extraen hierro, cobre, piedra y otros recursos que deben procesarse. |
| **Cintas transportadoras** | Llevan los minerales a los hornos y retiran los productos. |
| **Cintas subterráneas y divisores** | Permiten cruzar líneas y distribuir minerales o placas entre distintas rutas. |
| **Insertores** | Introducen mineral y combustible, o retiran los productos de los edificios. |
| **Cofres** | Almacenan mineral, combustible, placas y otros materiales. |
| **Postes eléctricos** | Suministran electricidad a hornos eléctricos, fundiciones e insertores eléctricos. |
| **Módulos** | Modifican la velocidad, productividad o consumo de edificios compatibles. |
| **Balizas** | Transmiten los efectos de sus módulos a edificios cercanos que puedan recibirlos. |
| **Tuberías y bombas** | Transportan y controlan el flujo de fluidos metálicos en las recetas de la fundición. |

---

#### 3. Edificios y estructuras de refinamiento

##### 3.1. Refinería de petróleo (*Oil refinery*)


La **refinería de petróleo** procesa petróleo crudo o carbón mediante recetas específicas para producir fluidos derivados.

**Características principales:**

- Tamaño: 5 × 5 casillas.
- Consumo eléctrico nominal: 420 kW cuando trabaja.
- Velocidad de fabricación base: 1.
- Dispone de 3 ranuras para módulos.
- Requiere electricidad.
- Hay que seleccionar una receta antes de que comience a trabajar.

En el juego base moderno hay varias recetas de refinamiento:

| Proceso | Insumos principales | Productos | Requisito |
|---|---|---|---|
| Procesamiento básico del petróleo | Petróleo crudo | Gas de petróleo | Tecnología de procesamiento del petróleo |
| Procesamiento avanzado del petróleo | Petróleo crudo y agua | Petróleo pesado, petróleo ligero y gas de petróleo | Procesamiento avanzado del petróleo |
| Licuefacción del carbón | Carbón, petróleo pesado y vapor | Petróleo pesado, petróleo ligero y gas de petróleo | Tecnología de licuefacción del carbón |

En la receta básica actual, el producto principal es el gas de petróleo. El procesamiento avanzado permite obtener las tres fracciones del petróleo, pero exige agua y una infraestructura de tuberías más compleja.

**Punto crítico:** si uno de los productos de salida no puede evacuarse porque los tanques o las tuberías están llenos, la refinería puede detenerse, incluso aunque todavía tenga petróleo crudo y energía.

Más información: [Factorio Wiki: Oil refinery](https://wiki.factorio.com/Oil_refinery).

##### 3.2. Planta química (*Chemical plant*)


La **planta química** realiza numerosas recetas basadas en fluidos. Es el edificio que suele recibir los productos de la refinería para transformarlos en materiales que necesita la fábrica.

**Características principales:**

- Tamaño: 3 × 3 casillas.
- Consumo eléctrico nominal: 210 kW mientras trabaja.
- Velocidad de fabricación base: 1.
- Dispone de 3 ranuras para módulos.
- Necesita electricidad y, según la receta, sólidos y fluidos de entrada.

Entre sus aplicaciones habituales se encuentran:

- Fabricar **plástico** a partir de carbón y gas de petróleo.
- Fabricar **azufre** a partir de gas de petróleo y agua.
- Producir **ácido sulfúrico**.
- Convertir petróleo pesado en petróleo ligero mediante craqueo.
- Convertir petróleo ligero en gas de petróleo mediante craqueo.
- Fabricar **lubricante**, **combustible sólido**, baterías y otros productos químicos.

La planta química cuenta con conexiones de fluidos para las recetas correspondientes. Al construirla, conviene activar la visualización de información adicional para comprobar qué tuberías corresponden a cada entrada y salida de la receta seleccionada.

Más información: [Factorio Wiki: Chemical plant](https://wiki.factorio.com/Chemical_plant).

##### 3.3. Tuberías, bombas y depósitos

Las tuberías son esenciales para mover petróleo crudo, agua y productos refinados entre bombas, refinerías, plantas químicas y depósitos.

- **Tuberías:** conectan edificios y transportan fluidos.
- **Bombas:** permiten controlar la dirección del flujo y mover fluidos entre segmentos o vehículos de carga de fluidos.
- **Depósitos de almacenamiento:** guardan cantidades grandes de fluidos y permiten observar visualmente cuánto queda almacenado.
- **Vagones cisterna y estaciones compatibles:** sirven para transportar fluidos por ferrocarril en redes de gran tamaño.

Los fluidos no se transportan por cintas normales. Una línea de refinamiento necesita una red de tuberías coherente con las conexiones de cada edificio.

---

#### 4. Insumos, materiales procesados y usos

##### 4.1. Insumos de la fundición

| Insumo | Se utiliza en | Función |
|---|---|---|
| **Mineral de hierro** | Hornos comunes o fundición | Se convierte en placas de hierro o en hierro fundido mediante recetas compatibles. |
| **Mineral de cobre** | Hornos comunes o fundición | Se convierte en placas de cobre o cobre fundido. |
| **Piedra** | Hornos comunes | Se transforma en ladrillos de piedra, utilizados para edificios y otras recetas. |
| **Placas de hierro** | Hornos comunes o fundición | Se convierten en placas de acero; en *Space Age* también hay recetas de moldeo directas desde metal fundido. |
| **Combustible** | Hornos de piedra y de acero | Aporta energía térmica para completar las recetas. |
| **Electricidad** | Horno eléctrico y fundición | Alimenta las máquinas sin necesidad de combustible sólido. |
| **Calcita** | Fundición de *Space Age* | Se utiliza en ciertas recetas para generar metal fundido, incluso a partir de lava en Vulcanus. |
| **Fluidos metálicos fundidos** | Fundición de *Space Age* | Se moldean en placas y productos metalúrgicos compatibles. |

Es importante distinguir los insumos para **fabricar** un edificio de los insumos que consume durante su uso. Por ejemplo, el horno eléctrico necesita placas de acero, ladrillos y circuitos para construirse, pero durante su operación consume electricidad.

##### 4.2. Productos de la fundición

| Producto | Cómo se obtiene | Aplicaciones principales |
|---|---|---|
| **Placa de hierro** | Fundir mineral de hierro o moldear hierro fundido | Cintas, insertores, máquinas, estructuras, engranajes y muchos componentes. |
| **Placa de cobre** | Fundir mineral de cobre o moldear cobre fundido | Cables, circuitos y componentes electrónicos. |
| **Placa de acero** | Fundir 5 placas de hierro o usar una receta de moldeo compatible | Edificios robustos, vehículos, estructuras y maquinaria avanzada. |
| **Ladrillo de piedra** | Fundir 2 unidades de piedra | Hornos, muros y diversas estructuras. |
| **Hierro fundido** | Recetas de metalurgia de *Space Age* | Moldeo de placas de hierro y otros productos que admiten esa receta. |
| **Cobre fundido** | Recetas de metalurgia de *Space Age* | Moldeo de placas de cobre y otros productos compatibles. |
| **Placa de litio** | Fundición del recurso de litio en las condiciones compatibles de *Space Age* | Cadenas avanzadas que requieren materiales de Aquilo. |

No todos los recursos se procesan en hornos. Por ejemplo, el mineral de uranio se procesa en una centrífuga y no mediante una receta estándar de fundición. Para decidir qué edificio utilizar, hay que comprobar la receta concreta del producto.

##### 4.3. Insumos del refinamiento

| Insumo | Uso | Observaciones |
|---|---|---|
| **Petróleo crudo** | Refinería de petróleo | Se extrae de campos petrolíferos con bombas extractoras. |
| **Agua** | Procesamiento avanzado y recetas químicas | Se suministra mediante tuberías y bombas compatibles. |
| **Carbón** | Plástico y licuefacción del carbón | Puede actuar como material químico o como combustible según la receta. |
| **Petróleo pesado** | Lubricante y craqueo | La refinería debe poder evacuarlo para evitar bloqueos. |
| **Petróleo ligero** | Combustible sólido y craqueo | Se puede convertir en gas de petróleo cuando la fábrica lo necesita. |
| **Gas de petróleo** | Plástico, azufre y otras recetas químicas | Una demanda alta de plástico puede convertirlo en un recurso limitante. |

##### 4.4. Productos refinados y sus aplicaciones

| Producto | Edificio de producción habitual | Usos |
|---|---|---|
| **Gas de petróleo** | Refinería o planta química con la receta correspondiente | Plástico, azufre y combustible sólido, entre otros. |
| **Petróleo ligero** | Refinería | Combustible sólido y producción de gas de petróleo mediante craqueo. |
| **Petróleo pesado** | Refinería | Lubricante y conversión en petróleo ligero. |
| **Lubricante** | Planta química | Componentes mecánicos y otras recetas industriales. |
| **Plástico** | Planta química | Circuitos avanzados y estructuras de baja densidad. |
| **Azufre** | Planta química | Ácido sulfúrico, explosivos y otras recetas. |
| **Ácido sulfúrico** | Planta química | Baterías, extracción de uranio y cadenas químicas. |
| **Combustible sólido** | Planta química | Fuente de energía para máquinas compatibles y otros procesos. |

---

#### 5. Tasas de producción y planificación de capacidad

##### 5.1. Velocidad base de los hornos

La velocidad de fabricación indica cuántas veces más rápido trabaja una máquina frente a una velocidad de referencia. Para las recetas comunes, el horno de piedra tiene velocidad 1 y los hornos de acero y eléctricos tienen velocidad 2.

| Receta | Tiempo de receta | Horno de piedra | Horno de acero o eléctrico |
|---|---:|---:|---:|
| Placa de hierro | 3,2 s | 0,3125 placas/s | 0,625 placas/s |
| Placa de cobre | 3,2 s | 0,3125 placas/s | 0,625 placas/s |
| Ladrillo de piedra | 3,2 s | 0,3125 ladrillos/s | 0,625 ladrillos/s |
| Placa de acero | 16 s | 0,0625 placas/s | 0,125 placas/s |

Son tasas nominales por horno, sin módulos ni otros modificadores. El resultado real de una línea también depende de que se suministren suficientes ingredientes y de que se retiren los productos con regularidad.

##### 5.2. Calcular cuántos hornos hacen falta

Para estimar la cantidad de hornos, divide la producción requerida por la producción por horno.

Por ejemplo, si una línea debe producir 10 placas de hierro por segundo usando hornos de acero o eléctricos:

\[
\text{Cantidad de hornos} = \frac{10}{0{,}625} = 16
\]

Por lo tanto, se necesitan 16 hornos funcionando a su tasa nominal para obtener 10 placas por segundo. Este cálculo no incluye bonificaciones de módulos, balizas ni productividad.

También hay que considerar el flujo de mineral y el ancho de banda de las cintas. Si el transporte no puede abastecer a los hornos, añadir máquinas no aumentará la producción real.

##### 5.3. Productividad y módulos

Los módulos pueden modificar el rendimiento de las máquinas compatibles.

- **Módulos de velocidad:** aumentan la velocidad de fabricación, normalmente a costa de un mayor consumo energético.
- **Módulos de productividad:** generan productos adicionales en recetas compatibles, pero suelen reducir la velocidad de fabricación.
- **Módulos de eficiencia:** reducen el consumo energético, lo que resulta útil cuando muchas máquinas compiten por una red eléctrica limitada.
- **Módulos de calidad:** en *Factorio 2.x*, influyen en la probabilidad de obtener resultados de mayor calidad en edificios y recetas compatibles.

Los hornos eléctricos admiten dos módulos y las fundiciones cuatro. Las refinerías y plantas químicas también admiten módulos, aunque la selección debe hacerse según el objetivo de producción y las limitaciones de energía.

##### 5.4. Equilibrar la producción de petróleo

El refinamiento avanzado genera varias salidas. Si la fábrica consume mucho gas de petróleo pero casi nada de petróleo pesado, este último puede acumularse y detener la refinería.

Para evitarlo, puede emplearse una combinación de:

- Craqueo de petróleo pesado en petróleo ligero.
- Craqueo de petróleo ligero en gas de petróleo.
- Producción de lubricante cuando hay demanda de petróleo pesado.
- Depósitos como reserva intermedia, sin depender exclusivamente de ellos para solucionar un desequilibrio permanente.

La mejor distribución depende de las recetas que la fábrica consume. Conviene dimensionar el refinamiento de acuerdo con la demanda real de plástico, azufre, lubricante y combustible.

---

#### 6. Ejemplos de diseño

##### 6.1. Línea básica de fundición de hierro


Una línea sencilla puede organizarse así:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Alimentación de hornos | Cinta de mineral de hierro | Insertores | Mineral transferido a hornos de acero o eléctricos | — |
| Fundición | Mineral de hierro | Hornos de acero o eléctricos | Placas de hierro | — |
| Retirada del producto | Hornos | Insertores de salida | Placas de hierro transferidas | — |
| Transporte del producto | Placas de hierro | Cinta de placas de hierro | Placas de hierro transportadas | — |

Para que funcione correctamente, debe haber un suministro de mineral, espacio para el combustible si se utilizan hornos de combustión, energía si se utilizan hornos eléctricos y una cinta que pueda retirar las placas.

En una fábrica pequeña, los hornos pueden colocarse en filas a ambos lados de una cinta central. En diseños mayores conviene utilizar planos (*blueprints*) para mantener una distribución repetible.

##### 6.2. Producción de acero

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Fundición de hierro | Mineral de hierro | Horno de fundición | Hierro procesado en placas de hierro | — |
| Producción de acero | Placas de hierro | Horno con receta de acero | Placas de acero | — |

La receta estándar de acero consume **5 placas de hierro para producir 1 placa de acero en 16 segundos de receta**. La fabricación de acero es más lenta por unidad de producto que la de placas de hierro, así que una fábrica que necesite mucho acero tendrá que dedicarle suficiente capacidad.

El acero debe planificarse como una línea propia, especialmente cuando empieza a utilizarse en edificios, maquinaria y componentes avanzados.

##### 6.3. Refinamiento de petróleo


Antes de conectar todas las tuberías, selecciona la receta de refinamiento y revisa las posiciones de sus entradas y salidas. La distribución de conexiones depende de la orientación de la máquina.

Un diseño fiable separa las salidas de los fluidos y dirige cada una hacia los consumidores apropiados. Los depósitos ayudan a amortiguar picos de demanda, pero no sustituyen el uso de los fluidos acumulados.

##### 6.4. Fundición y moldeo con *Space Age*

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada a la fundición | Mineral de hierro y calcita | Fundición (*Foundry*) | Material preparado para la receta de fundición | La receta de fundición seleccionada determina el producto. |
| Fundición | Material preparado para la receta | Fundición (*Foundry*) | Hierro fundido | — |
| Moldeo | Hierro fundido | Receta de moldeo en fundición | Placas o componentes | — |

En Vulcanus, las fundiciones también pueden usar lava y calcita para obtener metales fundidos. El moldeo permite crear directamente determinados productos a partir del fluido metálico, con mejoras de rendimiento respecto a algunas cadenas tradicionales.

Sin embargo, una fundición necesita una gran cantidad de electricidad y una distribución de fluidos bien planteada. Los planos deben dejar espacio para tuberías, postes, accesos y cintas de entrada y salida.

---

#### 7. Estrategias y consejos de optimización

##### 7.1. Elegir el horno correcto para cada etapa

- **Inicio de partida:** el horno de piedra tiene bajo costo y permite empezar a producir placas rápidamente.
- **Expansión intermedia:** el horno de acero duplica la velocidad base sin exigir una red eléctrica para cada horno.
- **Etapa avanzada:** el horno eléctrico simplifica la distribución de combustible y admite módulos. Resulta especialmente útil si la electricidad es abundante.
- **Metalurgia avanzada de *Space Age*:** la fundición puede mejorar el rendimiento y producir algunos materiales de forma más directa, pero exige mucha electricidad y una infraestructura metalúrgica específica.

No es obligatorio reemplazar todos los hornos de inmediato. La decisión debe tener en cuenta la capacidad total de producción, el espacio disponible, el combustible y la electricidad.

##### 7.2. Separar las líneas por tipo de producto

Mantener rutas claras para hierro, cobre, acero y ladrillos de piedra facilita detectar problemas. Las cintas mixtas pueden resultar útiles, pero también complican la distribución si no se diseñan con divisores y filtros apropiados.

Las líneas separadas permiten medir mejor la producción y ampliar únicamente la parte que se ha convertido en un cuello de botella.

##### 7.3. Evitar bloqueos en las cintas

Una fundición puede detenerse cuando la salida está llena. Asegúrate de que las placas terminadas puedan abandonar los hornos y que no exista un cofre lleno o una cinta bloqueada aguas abajo.

En los diseños compactos, comprueba la orientación de los insertores y deja suficiente espacio para que las cintas subterráneas crucen las líneas sin impedir el flujo de productos.

##### 7.4. Planificar combustible y electricidad

Los hornos de piedra y acero necesitan combustible. Si se alimentan mediante cintas, el flujo de carbón debe ser suficiente para toda la línea. Los hornos eléctricos, las refinerías, las plantas químicas y las fundiciones requieren capacidad eléctrica adecuada.

Cuando una fábrica comienza a quedarse sin potencia, el consumo de las máquinas puede aumentar los tiempos de fabricación en distintas áreas. La ampliación de una fundición debe acompañarse de una revisión de la generación y distribución eléctrica.

##### 7.5. Diseñar refinerías con salidas claras

Cada salida de fluido debe tener un consumidor o una ruta de almacenamiento. Evita conectar por accidente dos productos diferentes a la misma red. La interfaz de los edificios y la visualización de información adicional ayudan a identificar los conectores.

Si una refinería se detiene, comprueba en este orden:

1. Que la receta correcta está seleccionada.
2. Que llega petróleo crudo y, si corresponde, agua u otros insumos.
3. Que la red eléctrica funciona.
4. Que las tuberías de salida están conectadas al fluido correcto.
5. Que ningún producto de salida está bloqueado por falta de espacio.

##### 7.6. Utilizar el craqueo para equilibrar la demanda

El craqueo es útil cuando la proporción de productos del petróleo no coincide con lo que consume la fábrica. Por ejemplo, si se acumula petróleo pesado, una planta química puede convertir parte de él en petróleo ligero. Si se acumula petróleo ligero mientras falta gas de petróleo, puede craquearse a gas.

No conviene craquear todo automáticamente: el petróleo pesado puede ser necesario para lubricante y el petróleo ligero suele ser una buena fuente de combustible sólido. La configuración debe responder a la demanda de la fábrica, no solamente al objetivo de vaciar un depósito.

##### 7.7. Aprovechar módulos y balizas cuando estén disponibles

Los módulos de velocidad pueden aumentar la producción de una línea compacta, mientras que los de productividad pueden ahorrar recursos en recetas compatibles. Las balizas permiten aplicar efectos a varias máquinas cercanas.

Antes de emplearlos, comprueba el aumento de consumo de energía y el equilibrio de las cintas y tuberías. Una máquina más rápida solo mejora la producción final si sus insumos llegan a tiempo y sus productos pueden retirarse.

---

#### 8. Errores comunes y soluciones

| Problema | Causa probable | Solución |
|---|---|---|
| El horno no funde nada. | No recibe el insumo correcto o se quedó sin combustible/energía. | Revisa la receta, la entrada, el combustible o la red eléctrica. |
| Se acumulan minerales y faltan placas. | Hay pocos hornos o la salida de productos está bloqueada. | Aumenta capacidad de fundición o libera las cintas de salida. |
| Los hornos eléctricos se ralentizan. | Suministro eléctrico insuficiente. | Mejora la generación y revisa las conexiones eléctricas. |
| Una línea de acero no produce suficiente. | La receta consume 5 placas de hierro por cada placa de acero y tarda más. | Añade capacidad dedicada para acero y asegura la entrada de placas. |
| La refinería no arranca. | No hay receta seleccionada, falta un insumo o no tiene electricidad. | Revisa receta, fluidos de entrada y potencia. |
| La refinería se detiene después de un tiempo. | Algún producto de salida está bloqueado. | Libera la tubería, consume el fluido acumulado o utiliza craqueo. |
| La planta química no produce plástico. | Falta gas de petróleo o carbón, o la salida de plástico está bloqueada. | Revisa ambos insumos y el transporte del producto. |
| Se acumula petróleo pesado. | Falta consumo de lubricante y no hay craqueo suficiente. | Añade producción de lubricante o craqueo a petróleo ligero. |
| El petróleo ligero se acumula mientras falta gas de petróleo. | No existe suficiente capacidad de craqueo. | Añade plantas químicas para craquear petróleo ligero. |
| Una fundición de *Space Age* no produce. | Faltan minerales, calcita, electricidad o una conexión de fluido necesaria. | Revisa la receta, los insumos, la energía y las tuberías. |

---

#### 9. Diferencias específicas de *Space Age*

La expansión *Space Age* incorpora cambios importantes a la metalurgia y al procesamiento de materiales.

- **Fundición avanzada:** permite producir hierro y cobre fundidos y moldearlos en placas y otros productos compatibles.
- **Vulcanus:** ofrece recetas que aprovechan la lava y la calcita como parte de una cadena metalúrgica diferente de la tradicional.
- **Gran fundición de 5 × 5:** tiene una velocidad base de 4, cuatro ranuras para módulos y productividad integrada del 50 % en recetas aplicables.
- **Líneas de materiales planetarios:** algunos recursos y productos, como la calcita o los materiales de litio, solo están disponibles en entornos específicos.
- **Plataformas espaciales:** ciertos edificios que dependen de combustible no funcionan en el espacio de la forma habitual, por lo que el horno eléctrico es relevante para la fundición en plataformas.

Estas mecánicas amplían las opciones de producción, pero no eliminan la necesidad de equilibrar entradas y salidas. La energía, la disponibilidad de recursos, las recetas seleccionadas y la capacidad de transporte siguen determinando el rendimiento final.

---

#### 10. Lista de comprobación de una instalación

Antes de dar por terminada una línea de fundición o refinamiento, comprueba los siguientes puntos:

- [ ] La máquina es compatible con la receta seleccionada.
- [ ] Llegan todos los insumos sólidos y fluidos necesarios.
- [ ] Los hornos de combustión reciben combustible suficiente.
- [ ] Los edificios eléctricos tienen una conexión funcional y suficiente potencia.
- [ ] Las cintas pueden transportar todos los materiales de entrada y salida.
- [ ] Las tuberías conectan los fluidos correctos con las entradas y salidas adecuadas.
- [ ] Ningún cofre, cinta, tubería o depósito lleno bloquea la producción.
- [ ] La capacidad de fundición corresponde a la demanda real de la fábrica.
- [ ] La refinería tiene una salida o un consumidor para cada producto generado.
- [ ] La disposición deja espacio para ampliar la instalación y acceder a las máquinas.
- [ ] En *Space Age*, hay suficiente electricidad y están disponibles los insumos planetarios requeridos.

---

#### 11. Fuentes de referencia

Las siguientes páginas de la *Factorio Wiki* ofrecen estadísticas, recetas y detalles técnicos para ampliar o verificar la información de esta guía:

- [Stone furnace — Horno de piedra](https://wiki.factorio.com/Stone_furnace)
- [Steel furnace — Horno de acero](https://wiki.factorio.com/Steel_furnace)
- [Electric furnace — Horno eléctrico](https://wiki.factorio.com/Electric_furnace)
- [Foundry — Fundición de *Space Age*](https://wiki.factorio.com/Foundry)
- [Iron plate — Placa de hierro](https://wiki.factorio.com/Iron_plate)
- [Oil refinery — Refinería de petróleo](https://wiki.factorio.com/Oil_refinery)
- [Oil processing — Procesamiento del petróleo](https://wiki.factorio.com/Light_oil_cracking)
- [Chemical plant — Planta química](https://wiki.factorio.com/Chemical_plant)
- [Fluid system — Sistema de fluidos](https://wiki.factorio.com/Fluid_system)


---

### Guía técnica de Factorio: Ensamblado y automatización de producción

**Versión de referencia:** *Factorio 2.x*. Las estructuras y mecánicas exclusivas de la expansión *Space Age* están identificadas por separado. Las cifras indicadas corresponden a edificios de calidad normal y sin bonificaciones de módulos, salvo que se especifique lo contrario.

El ensamblado y la automatización de producción constituyen el núcleo de una fábrica de *Factorio*. La minería proporciona materias primas y la fundición las transforma en materiales utilizables, pero son las máquinas de producción las que convierten esos materiales en componentes, edificios, munición, circuitos, ciencia y maquinaria avanzada.

Esta guía explica cómo funciona el ensamblado, cuáles son las máquinas y estructuras que intervienen, cómo alimentar las recetas automáticamente y qué técnicas permiten crear líneas de producción estables, eficientes y fáciles de ampliar.

---

#### 1. Fundamentos del ensamblado y la automatización

##### 1.1. ¿Qué es el ensamblado?

El **ensamblado** es el proceso mediante el cual una máquina recibe los ingredientes de una receta y los transforma en uno o más productos. Por ejemplo, una máquina puede combinar placas de hierro para producir engranajes, o combinar placas de hierro, placas de cobre y circuitos para fabricar un inserter.

La automatización consiste en mantener ese proceso funcionando sin que el jugador tenga que introducir manualmente cada ingrediente ni recoger cada producto. Para ello, se conectan las máquinas con cintas transportadoras, insertadores, tuberías, cofres, redes logísticas y, cuando es necesario, circuitos de control.

Una cadena automatizada suele seguir este flujo:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Suministro de materias primas | Materias primas | Cintas transportadoras u otros sistemas de transporte | Ingredientes entregados a producción | Flujo general integrado con el esquema de alimentación por insertores. |
| Entrada de ingredientes | Ingredientes transportados | Inserter de entrada | Ingredientes dentro de la máquina | — |
| Fabricación según receta | Ingredientes requeridos por la receta | Máquina con la receta seleccionada | Producto fabricado | — |
| Retirada del producto | Producto fabricado | Inserter de salida | Producto extraído de la máquina | — |
| Destino del producto | Producto extraído de la máquina | Almacenamiento, otra máquina o línea de montaje | Producto disponible para la siguiente etapa | — |

##### 1.2. Diferencia entre fabricación manual y automatizada

La fabricación manual permite crear objetos desde el inventario del personaje cuando se dispone de los ingredientes. La máquina de ensamblaje, en cambio, trabaja con una receta seleccionada y necesita recibir de manera independiente todos los ingredientes que consume.

Las máquinas **no encadenan automáticamente todas las etapas de una receta**. Si una receta requiere circuitos electrónicos y esos circuitos necesitan cables de cobre, hay que producir los cables y circuitos en etapas previas o suministrarlos desde otras líneas.

Por ejemplo, una línea automatizada de insertores puede tener esta estructura:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Fabricación de cables | Placas de cobre | Proceso de fabricación de cables de cobre | Cables de cobre | — |
| Fabricación de circuitos | Cables de cobre y placas de hierro | Máquina de circuitos | Circuitos electrónicos | — |
| Fabricación de engranajes | Placas de hierro | No especificado en el esquema original | Engranajes | — |
| Fabricación de insertores | Circuitos electrónicos y engranajes | Máquina de insertores | Insertores | — |

La disposición exacta cambia según la receta y la escala, pero el principio es el mismo: **cada etapa de fabricación necesita sus propios ingredientes y capacidad de producción**.

##### 1.3. ¿Por qué es importante en la progresión?

El ensamblado automatizado permite producir en grandes cantidades objetos que serían lentos o poco prácticos de fabricar manualmente. También es necesario para mantener suministros constantes de edificios, materiales intermedios y paquetes de ciencia.

Sus principales ventajas son:

- **Producción continua:** las máquinas siguen trabajando mientras reciban ingredientes y tengan una salida disponible.
- **Escalabilidad:** se puede aumentar la capacidad añadiendo máquinas o replicando diseños.
- **Especialización:** cada línea puede dedicarse a una receta o una familia de productos.
- **Regularidad:** los flujos de ingredientes y productos pueden medirse y optimizarse.
- **Integración:** las líneas pueden alimentar automáticamente la construcción, la investigación, la logística y la defensa de la fábrica.

---

#### 2. Máquinas de ensamblaje del juego base

##### 2.1. Máquina de ensamblaje 1 (*Assembling machine 1*)


La **máquina de ensamblaje 1** es la primera máquina de ensamblaje disponible tras investigar la tecnología *Automatización*. Permite abandonar la fabricación manual de muchos objetos sencillos y empezar a construir líneas de producción permanentes.

**Características principales:**

- Tamaño: 3 × 3 casillas.
- Velocidad de fabricación: 0,5.
- Consumo eléctrico nominal: 75 kW mientras trabaja.
- Consumo en espera: aproximadamente 2,5 kW.
- Ranuras para módulos: ninguna.
- No puede realizar recetas que requieran entradas de fluidos.
- Es adecuada para muchas recetas sencillas con pocos tipos de ingredientes.

La velocidad de fabricación de 0,5 significa que completa una receta a la mitad de la velocidad de referencia de una máquina con velocidad 1, antes de considerar modificadores. No significa que todas las recetas tarden exactamente el doble que a mano: el tiempo base depende de cada receta.

La máquina de ensamblaje 1 es una buena opción para automatizar componentes básicos, pero tiene limitaciones. Algunas recetas avanzadas requieren una máquina de nivel superior por la cantidad de ingredientes o porque necesitan fluidos.

Fuente: [Factorio Wiki: Assembling machine 1](https://wiki.factorio.com/Assembling_machine_1).

##### 2.2. Máquina de ensamblaje 2 (*Assembling machine 2*)


La **máquina de ensamblaje 2** amplía las posibilidades de producción y es el primer modelo que admite módulos y recetas con ingredientes fluidos.

**Características principales:**

- Tamaño: 3 × 3 casillas.
- Velocidad de fabricación: 0,75.
- Consumo eléctrico nominal: 150 kW mientras trabaja.
- Consumo en espera: aproximadamente 5 kW.
- Ranuras para módulos: 2.
- Puede realizar recetas que requieren fluidos.
- Admite más tipos de ingredientes por receta que la máquina de ensamblaje 1.

Sus conexiones para fluidos se activan cuando se selecciona una receta que requiere un líquido. La posición de las conexiones puede cambiarse al rotar la máquina antes de colocarla o mediante la rotación disponible en el juego.

Esta máquina es apropiada para una gran parte de las líneas de producción intermedias, incluidas algunas que necesitan plástico, ácido sulfúrico u otros fluidos. También permite introducir módulos para mejorar la velocidad, la eficiencia o la productividad en recetas compatibles.

Fuente: [Factorio Wiki: Assembling machine 2](https://wiki.factorio.com/Assembling_machine_2).

##### 2.3. Máquina de ensamblaje 3 (*Assembling machine 3*)


La **máquina de ensamblaje 3** es el nivel superior de la línea tradicional de ensambladoras del juego base. Se caracteriza por su velocidad, sus cuatro espacios para módulos y la capacidad para trabajar con recetas complejas.

**Características principales:**

- Tamaño: 3 × 3 casillas.
- Velocidad de fabricación: 1,25.
- Consumo eléctrico nominal: 375 kW mientras trabaja.
- Consumo en espera: aproximadamente 12,5 kW.
- Ranuras para módulos: 4.
- Puede usar fluidos en recetas compatibles.
- Puede realizar las recetas que pertenecen a su categoría, incluidas recetas con muchos tipos de ingredientes.

Su velocidad base es más del doble que la de una máquina de ensamblaje 1. Además, los cuatro espacios para módulos permiten modificar de forma importante su rendimiento. No obstante, su consumo eléctrico nominal es mayor, por lo que sustituir muchas máquinas de nivel 1 por máquinas de nivel 3 puede exigir una ampliación de la red eléctrica.

Fuente: [Factorio Wiki: Assembling machine 3](https://wiki.factorio.com/Assembling_machine_3).

##### 2.4. Comparación de las máquinas de ensamblaje

| Característica | Ensamblaje 1 | Ensamblaje 2 | Ensamblaje 3 |
|---|---:|---:|---:|
| Tamaño | 3 × 3 | 3 × 3 | 3 × 3 |
| Velocidad de fabricación | 0,5 | 0,75 | 1,25 |
| Consumo nominal | 75 kW | 150 kW | 375 kW |
| Ranuras para módulos | 0 | 2 | 4 |
| Recetas con fluidos | No | Sí | Sí |
| Uso habitual | Componentes sencillos y primera automatización | Producción intermedia y recetas con fluidos | Alta producción y líneas avanzadas |

Las velocidades y consumos corresponden a edificios de calidad normal sin módulos. Los consumos se refieren al funcionamiento; las máquinas también tienen un consumo en espera.

##### 2.5. Cómo elegir el nivel adecuado

No es necesario usar siempre la máquina más avanzada. El nivel correcto depende de los ingredientes disponibles, la producción deseada, el espacio y la energía.

- Utiliza **máquinas de ensamblaje 1** para automatizaciones tempranas o productos de baja demanda.
- Utiliza **máquinas de ensamblaje 2** cuando la receta necesite fluidos, más flexibilidad o módulos básicos.
- Utiliza **máquinas de ensamblaje 3** para líneas que requieran mayor velocidad, más ranuras de módulos o una densidad de producción elevada.

Si una línea produce lo suficiente, la sustitución por máquinas más caras puede no aportar un beneficio proporcional. Es preferible identificar primero el cuello de botella y mejorar la máquina o etapa que limita el flujo total.

---

#### 3. Edificios especializados para producción

No todas las recetas se realizan en máquinas de ensamblaje convencionales. Algunas familias de productos utilizan edificios especializados.

##### 3.1. Planta química (*Chemical plant*)


La **planta química** procesa recetas químicas que utilizan fluidos y, en algunos casos, ingredientes sólidos. Es necesaria para muchas cadenas relacionadas con petróleo, plástico, azufre, ácido sulfúrico y otros productos químicos.

**Características principales:**

- Tamaño: 3 × 3 casillas.
- Velocidad de fabricación: 1.
- Consumo eléctrico nominal: 210 kW.
- Ranuras para módulos: 3.
- Tiene conexiones de entrada y salida para fluidos.
- Puede necesitar ingredientes sólidos suministrados por insertores o cintas.

Las posiciones de las conexiones de fluidos son fijas según la orientación de la máquina; algunas se pueden cambiar girando la construcción. Conviene revisar la receta seleccionada antes de diseñar las tuberías.

Fuente: [Factorio Wiki: Chemical plant](https://wiki.factorio.com/Chemical_plant).

##### 3.2. Refinería de petróleo (*Oil refinery*)


La **refinería de petróleo** procesa petróleo crudo o carbón mediante recetas específicas para producir fluidos derivados. Las recetas avanzadas pueden producir petróleo pesado, petróleo ligero y gas de petróleo en una misma operación.

**Características principales:**

- Tamaño: 5 × 5 casillas.
- Velocidad de fabricación: 1.
- Consumo eléctrico nominal: 420 kW.
- Ranuras para módulos: 3.
- Utiliza tuberías para recibir insumos y evacuar productos.
- Una receta debe estar seleccionada y sus salidas deben poder recibir los fluidos producidos.

La refinería es una parte especializada del sistema de producción: no sustituye a las máquinas de ensamblaje, sino que aporta fluidos que otras máquinas consumen posteriormente.

Fuente: [Factorio Wiki: Oil refinery](https://wiki.factorio.com/Oil_refinery).

##### 3.3. Fundición (*Foundry*) — *Space Age*


La **fundición** es un edificio metalúrgico avanzado de *Space Age*. Puede fabricar metales fundidos y moldear metales en placas y otros productos compatibles, con una bonificación de productividad integrada en muchas de sus recetas.

**Características principales:**

- Tamaño: 5 × 5 casillas.
- Velocidad de fabricación: 4.
- Consumo eléctrico nominal: 2,5 MW.
- Ranuras para módulos: 4.
- Bonificación de productividad integrada: +50 % en las recetas aplicables.
- Necesita una infraestructura apropiada para los fluidos metálicos y demás ingredientes.

En líneas de producción metálica, puede sustituir varias etapas de una cadena convencional. Aun así, exige una potencia eléctrica elevada y una distribución de fluidos bien organizada.

Fuente: [Factorio Wiki: Foundry](https://wiki.factorio.com/Foundry).

##### 3.4. Planta electromagnética (*Electromagnetic plant*) — *Space Age*


La **planta electromagnética**, obtenida en la progresión de Fulgora, está especializada en productos electrónicos y electromagnéticos.

**Características principales:**

- Tamaño: 4 × 4 casillas.
- Velocidad de fabricación: 2.
- Consumo eléctrico nominal: 2 MW.
- Ranuras para módulos: 5.
- Bonificación de productividad integrada: +50 % en las recetas aplicables.

Es especialmente relevante para líneas de circuitos y componentes eléctricos avanzados. Su productividad integrada puede reducir el consumo de materiales por producto final, mientras que sus numerosas ranuras para módulos permiten especializar su rendimiento.

Fuente: [Factorio Wiki: Electromagnetic plant](https://wiki.factorio.com/Electromagnetic_plant).

##### 3.5. Cámara biológica (*Biochamber*) — *Space Age*


La **cámara biológica** es una estructura de Gleba que combina funciones de ensamblado y procesamiento químico para recetas biológicas.

**Características principales:**

- Tamaño: 3 × 3 casillas.
- Velocidad de fabricación: 2.
- Utiliza nutrientes como fuente de energía en lugar de electricidad convencional.
- Ranuras para módulos: 4.
- Bonificación de productividad integrada: +50 % en las recetas aplicables.
- Puede absorber contaminación en vez de producirla, según su funcionamiento específico.

La cámara biológica necesita un suministro continuo de nutrientes. A diferencia de una ensambladora eléctrica, una falta de nutrientes interrumpe el trabajo; por tanto, su cadena de alimentación debe planificarse junto con las recetas biológicas.

Fuente: [Factorio Wiki: Biochamber](https://wiki.factorio.com/Bio_chamber).

##### 3.6. Planta criogénica (*Cryogenic plant*) — *Space Age*


La **planta criogénica** es un edificio especializado de Aquilo para procesos de baja temperatura y recetas avanzadas.

**Características principales:**

- Tamaño: 5 × 5 casillas.
- Velocidad de fabricación: 2.
- Consumo eléctrico nominal: 1,5 MW.
- Ranuras para módulos: 8.
- No tiene la bonificación de productividad integrada de las plantas electromagnéticas, biológicas o fundiciones.

Su mayor ventaja es la cantidad de ranuras para módulos y el conjunto de recetas que puede realizar. La instalación requiere suficiente potencia eléctrica y un diseño cuidadoso de tuberías y suministros.

Fuente: [Factorio Wiki: Cryogenic plant](https://wiki.factorio.com/Cryogenic_plant).

##### 3.7. Laboratorio (*Lab*)


El **laboratorio** no fabrica objetos convencionales: consume paquetes de ciencia para investigar tecnologías. Forma parte de la automatización general porque una fábrica suele producir paquetes de ciencia en máquinas de ensamblaje y llevarlos a una o varias líneas de laboratorios mediante insertores.

Para investigar de forma continua, hay que automatizar tanto la fabricación de los paquetes de ciencia como su entrega. Si la cadena de paquetes se detiene o no hay laboratorios suficientes, la investigación pierde velocidad.

Fuente: [Factorio Wiki: Lab](https://wiki.factorio.com/Lab).

---

#### 4. Estructuras de apoyo para automatizar la producción

##### 4.1. Insertores (*Inserters*)


Los **insertores** trasladan objetos entre cintas, cofres y máquinas. Son esenciales para que los ingredientes entren en las máquinas de ensamblaje y los productos terminados salgan de ellas.

Los tipos principales incluyen:

- **Insertor básico:** apropiado para líneas simples y distancias cortas.
- **Insertor rápido:** mueve objetos a mayor velocidad y resulta útil en líneas con más caudal.
- **Insertor apilador:** transporta grandes cantidades de objetos por movimiento, según el nivel de investigación y las condiciones.
- **Insertor filtrado:** permite limitar los objetos que recoge, en los modelos compatibles.
- **Insertor a combustión:** puede operar sin depender de la red eléctrica, algo útil en sistemas de respaldo.

El tipo de insertor debe elegirse de acuerdo con el caudal de objetos necesario. Un insertor lento puede limitar una máquina de alta velocidad, mientras que colocar insertores demasiado rápidos en todos los puntos puede aumentar innecesariamente el costo y el consumo.

Consulta la [Factorio Wiki: Inserters](https://wiki.factorio.com/Inserters).

##### 4.2. Cintas transportadoras


Las cintas transportadoras suministran ingredientes y evacuan productos. Pueden llevar objetos en dos carriles, lo que permite distribuir varios ingredientes por una misma cinta si el diseño de la línea lo requiere.

Las cintas de mayor velocidad transportan más objetos por segundo, pero no necesariamente solucionan un cuello de botella si el problema está en la cantidad de ingredientes, la velocidad de los insertores o la capacidad de las máquinas.

Las **cintas subterráneas** sirven para cruzar otras líneas sin mezclarlas, mientras que los **divisores** distribuyen el flujo entre dos salidas y pueden balancear cantidades o filtrar productos según la configuración.

Consulta la [Factorio Wiki: Transport belts](https://wiki.factorio.com/Transport_belts).

##### 4.3. Cofres y almacenamiento

Los cofres pueden actuar como depósitos intermedios de ingredientes, productos terminados o reservas logísticas. Son útiles cuando la producción y el consumo no ocurren al mismo ritmo, pero no sustituyen a una capacidad de fabricación adecuada.

Un cofre puede llenarse por completo y bloquear una máquina si esta no puede evacuar sus productos. Para evitarlo, utiliza una salida continua, límites de inventario o circuitos que regulen la producción cuando el almacenamiento alcanza la cantidad deseada.

##### 4.4. Tuberías y conexiones de fluidos

Algunas recetas de las máquinas de ensamblaje 2 y 3 requieren fluidos. En esos casos se necesita una red de tuberías correctamente conectada a las entradas de la máquina.

Antes de diseñar una línea de recetas con fluidos:

- Selecciona primero la receta de la máquina.
- Comprueba dónde aparecen las conexiones de entrada y salida.
- Asegúrate de que cada tubo transporta el fluido correcto.
- Evita conectar redes de fluidos incompatibles.
- Mantén las tuberías accesibles para poder corregir o ampliar el diseño.

---

#### 5. Ingredientes, productos y recetas

##### 5.1. Ingredientes habituales

Las máquinas de ensamblaje consumen productos fabricados en etapas anteriores. Entre los ingredientes más comunes se encuentran:

| Ingrediente | Procedencia típica | Ejemplos de uso |
|---|---|---|
| Placas de hierro | Fundición de mineral de hierro | Engranajes, cintas, insertores y máquinas. |
| Placas de cobre | Fundición de mineral de cobre | Cables y circuitos electrónicos. |
| Engranajes de hierro | Máquina de ensamblaje | Insertores, maquinaria y componentes mecánicos. |
| Cables de cobre | Máquina de ensamblaje | Circuitos, componentes eléctricos y electrónica. |
| Circuitos electrónicos | Máquina de ensamblaje | Insertores, ensambladoras, módulos y sistemas eléctricos. |
| Plástico | Planta química | Circuitos avanzados y componentes tecnológicos. |
| Acero | Fundición o receta de acero | Edificios y productos avanzados. |
| Petróleo y derivados | Refinería y plantas químicas | Plástico, azufre, baterías y otras cadenas químicas. |
| Fluidos industriales | Refinerías, plantas químicas o extracción | Recetas con fluidos y procesos avanzados. |

La lista no es exhaustiva: las recetas disponibles aumentan con la investigación y, en *Space Age*, con el acceso a nuevos planetas y edificios especializados.

##### 5.2. Productos típicos

| Producto | Uso dentro de la fábrica |
|---|---|
| Engranajes de hierro | Componentes mecánicos y recetas intermedias. |
| Cables de cobre | Circuitos y productos eléctricos. |
| Circuitos electrónicos | Automatización, módulos, insertores y tecnología. |
| Insertores | Movimiento automático de objetos. |
| Cintas transportadoras | Transporte de ingredientes y productos. |
| Máquinas de ensamblaje | Ampliación de la capacidad de fabricación. |
| Paquetes de ciencia | Investigación tecnológica. |
| Módulos | Mejora de máquinas y edificios compatibles. |
| Productos químicos y componentes avanzados | Cadenas de producción de etapas posteriores. |

Una fábrica bien organizada suele separar los **ingredientes comunes** de los productos finales. Por ejemplo, una línea central de circuitos electrónicos puede alimentar varias fábricas de componentes en lugar de fabricar circuitos de forma aislada dentro de cada línea.

##### 5.3. Cómo seleccionar una receta

Al abrir una máquina de ensamblaje se selecciona la receta que va a fabricar. La interfaz indica los ingredientes, la cantidad consumida y el producto resultante. La máquina necesita recibir cada ingrediente y disponer de espacio en su salida para continuar trabajando.

Al cambiar una receta, revisa también las cintas, insertores y tuberías conectadas. Una máquina configurada para producir un objeto diferente puede dejar de consumir ciertos ingredientes, mientras que el nuevo producto puede necesitar recursos que todavía no llegan a la instalación.

---

#### 6. Velocidad, capacidad y productividad

##### 6.1. Velocidad de fabricación

La **velocidad de fabricación** determina cuántas veces más rápido realiza una receta una máquina con respecto a la velocidad de referencia. El tiempo base de cada receta es independiente de la máquina; la velocidad efectiva altera el tiempo necesario para completarla.

Una fórmula básica para estimar la producción es:

\[
\text{Unidades por segundo} = \frac{\text{Velocidad efectiva de fabricación}}{\text{Tiempo base de la receta}} \times \text{Cantidad producida por operación}
\]

Si una receta tarda 1 segundo y la máquina tiene velocidad 0,75, producirá 0,75 unidades por segundo si la receta entrega una unidad por operación y no hay otros modificadores. Una máquina con velocidad 1,25 produciría 1,25 unidades por segundo en condiciones equivalentes.

La tasa real puede reducirse si faltan ingredientes, la salida está bloqueada, la máquina no tiene energía suficiente o existe otra limitación en la línea.

##### 6.2. Tiempo de receta y velocidad efectiva

No todas las recetas tienen la misma duración. Algunas operaciones simples son muy rápidas; otros productos, como las placas de acero o los componentes avanzados, tienen tiempos base mayores.

Por ello, no se debe comparar únicamente la cantidad de máquinas. Una sola máquina de alta velocidad puede no ser suficiente para una receta larga, mientras que varias máquinas de velocidad moderada pueden producir un caudal elevado cuando la receta es breve.

##### 6.3. Módulos de velocidad


Los **módulos de velocidad** incrementan la rapidez de fabricación de los edificios compatibles. Los niveles superiores suelen ofrecer un aumento mayor, pero también elevan el consumo eléctrico y pueden reducir la probabilidad de producir objetos de mayor calidad en las mecánicas de calidad.

Son útiles cuando se necesita más producción en el mismo espacio o cuando una receta está limitando la capacidad del resto de la fábrica.

##### 6.4. Módulos de productividad


Los **módulos de productividad** generan productos adicionales en las recetas compatibles sin consumir ingredientes adicionales por esos productos extra. La barra de productividad de la máquina se llena durante el trabajo; al completarse, se genera producción adicional.

En muchas recetas, su ventaja principal es ahorrar materias primas a largo plazo. Sin embargo, suelen reducir la velocidad de fabricación y aumentar el consumo energético. Deben evaluarse según el costo de los ingredientes y la importancia de la receta.

##### 6.5. Módulos de eficiencia


Los **módulos de eficiencia** reducen el consumo de energía de edificios compatibles. Esto puede reducir también la contaminación vinculada al consumo de energía de la máquina. La reducción total del consumo tiene un límite mínimo, por lo que los módulos no pueden llevar la potencia a cero.

Son especialmente útiles cuando una fábrica tiene una red eléctrica limitada o cuando el consumo energético es un costo importante de una línea de producción.

##### 6.6. Módulos de calidad — *Factorio 2.x*


Los **módulos de calidad** aumentan la probabilidad de producir objetos de mayor calidad en recetas y edificios compatibles. Los objetos de calidad pueden tener estadísticas mejoradas, por ejemplo, más velocidad de fabricación en ciertas máquinas o mayor capacidad en algunos edificios.

La calidad introduce una decisión distinta a la de aumentar solo la velocidad o la productividad. Cuando una línea busca objetos de alta calidad, conviene priorizar las recetas y máquinas adecuadas y evitar combinaciones de módulos que reduzcan ese objetivo.

Consulta la [Factorio Wiki: Module](https://wiki.factorio.com/Module) y [Quality](https://wiki.factorio.com/Quality).

##### 6.7. Balizas (*Beacons*)


Las **balizas** transmiten efectos de módulos a edificios compatibles cercanos. Esto permite aumentar la velocidad o mejorar la eficiencia de varias máquinas sin instalar todos los módulos directamente dentro de cada una.

Características importantes:

- La baliza tiene su propio consumo eléctrico.
- Solo transmiten los efectos de los módulos permitidos por la mecánica del juego, normalmente velocidad y eficiencia.
- Los edificios necesitan ser compatibles con módulos para recibir los efectos.
- La contribución efectiva depende de la baliza, la calidad de sus módulos y la cantidad de balizas que afectan al edificio.
- La superposición de muchas balizas tiene rendimientos decrecientes en la transmisión de efectos.

Las balizas son más útiles en líneas densas que contienen muchas máquinas trabajando de forma continua. En una máquina que trabaja de manera esporádica, una baliza encendida permanentemente puede gastar energía sin obtener suficiente beneficio.

Fuente: [Factorio Wiki: Beacon](https://wiki.factorio.com/Beacon).

---

#### 7. Diseño de una línea de producción automatizada

##### 7.1. Flujo básico de ingredientes

Una máquina debe tener una entrada fiable de ingredientes y una salida disponible para los productos terminados. El diseño más sencillo consiste en llevar las materias primas con cintas e insertarlas en la máquina, y después retirar el producto con otro insertor.

Es posible colocar varios ingredientes en la misma cinta aprovechando sus dos carriles, utilizar cintas separadas o suministrar algunos materiales por insertores desde cofres. La elección depende de la cantidad de ingredientes y del caudal que necesite la máquina.

##### 7.2. Alimentación directa (*Direct insertion*)

En algunos diseños, una máquina puede suministrar directamente productos a otra sin una cinta intermedia. Por ejemplo, una ensambladora que fabrica engranajes puede entregar directamente a una máquina que consume esos engranajes.

Ventajas:

- Reduce el espacio ocupado por cintas.
- Evita transportar productos innecesariamente.
- Puede simplificar líneas pequeñas y recetas de alto consumo.

Limitaciones:

- Une de forma más rígida las dos etapas.
- Puede dificultar distribuir el mismo producto a varios consumidores.
- El flujo depende de la velocidad de los insertores y de la disposición de las máquinas.

La alimentación directa es conveniente cuando una máquina produce principalmente para una sola máquina consumidora. Si varios edificios necesitan el mismo material, una cinta compartida suele resultar más flexible.

##### 7.3. Producción por módulos o bloques

Una fábrica escalable puede dividirse en **bloques de producción**. Cada bloque recibe ingredientes definidos y entrega productos específicos. Por ejemplo, un bloque fabrica circuitos electrónicos, otro produce engranajes y un tercero monta insertores.

Este enfoque facilita:

- Medir el rendimiento de cada bloque.
- Añadir capacidad sin reconstruir toda la fábrica.
- Compartir productos intermedios entre varias líneas.
- Usar planos (*blueprints*) para repetir configuraciones probadas.
- Detectar con rapidez qué etapa está limitando la producción.

##### 7.4. Producir paquetes de ciencia

La investigación requiere paquetes de ciencia en cantidades regulares. Para automatizarla, crea líneas específicas que fabrican cada tipo de paquete y los conducen a los laboratorios. Los insertores pueden alimentar los laboratorios y retirar los paquetes vacíos cuando la receta o el funcionamiento lo requieran.

Asegúrate de que las líneas de ingredientes no se queden sin materiales. Si la producción de un solo tipo de paquete se detiene, esa carencia puede convertirse en el límite de la investigación.

##### 7.5. Ejemplo: línea automatizada de circuitos electrónicos

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Fabricación de cables | Placas de cobre | Máquina de cables de cobre | Cables de cobre | — |
| Fabricación de circuitos | Cables de cobre y placas de hierro | Máquina de circuitos | Circuitos electrónicos | — |
| Distribución | Circuitos electrónicos | Cinta de distribución | Circuitos electrónicos transportados | — |
| Fabricación de insertores | Circuitos electrónicos distribuidos y otros insumos de la receta | Máquinas de insertores | Insertores | — |
| Fabricación de módulos | Circuitos electrónicos distribuidos y otros insumos de la receta | Máquinas de módulos | Módulos | — |

El ejemplo representa la lógica de producción. La cantidad de máquinas necesarias depende del tiempo de cada receta y de la demanda de circuitos. Si la máquina de circuitos trabaja más lentamente que la línea de cables, el cable sobrante se acumulará; esto no significa necesariamente que haya un error, pero puede indicar que la capacidad de los bloques no está equilibrada.

##### 7.6. Automatizar con redes logísticas

En etapas posteriores, los robots logísticos pueden mover ingredientes y productos entre cofres compatibles dentro de su red. Esto permite una producción basada en cofres logísticos en lugar de cintas para cada movimiento.

La logística con robots es flexible, pero depende de la cantidad de robots, la capacidad de carga, la red de roboportales y la energía disponible. Las cintas suelen ser eficientes para transportar flujos altos y constantes; los robots resultan convenientes para entregar muchos tipos de materiales a destinos variables o para bloques independientes.

##### 7.7. Control mediante circuitos

Los circuitos permiten condicionar la actividad de ciertas máquinas, insertores y elementos de transporte de acuerdo con señales. En versiones actuales, las máquinas de ensamblaje pueden configurarse mediante redes de circuitos para modificar la receta bajo condiciones compatibles.

Entre sus usos están:

- Evitar producir más de lo que puede almacenarse.
- Activar una línea cuando el inventario cae por debajo de una cantidad.
- Controlar la entrega de ingredientes.
- Compartir señales de inventario entre bloques.
- Diseñar fábricas que cambien de producto según la demanda.

Los circuitos aumentan la flexibilidad, pero también la complejidad del diseño. Para una primera línea, es preferible automatizar el flujo básico y añadir control cuando exista una necesidad concreta.

---

#### 8. Cálculo de capacidad y equilibrio de producción

##### 8.1. Calcular la tasa requerida

Antes de añadir máquinas, determina cuántos productos por segundo necesita la etapa siguiente. Después estima la producción de cada máquina con la velocidad efectiva y el tiempo de receta.

data:

\[
R = \frac{S}{T} \times O
\]

Donde:

- \(R\) es la producción nominal en unidades por segundo.
- \(S\) es la velocidad efectiva de fabricación.
- \(T\) es el tiempo base de la receta en segundos.
- \(O\) es la cantidad de productos obtenida por operación.

Si se usan módulos de productividad, hay que incluir el producto adicional esperado. Si la máquina no está abastecida continuamente, la producción efectiva será menor que el resultado teórico.

##### 8.2. Ejemplo de cálculo

Supón una receta que produce una unidad cada 0,5 segundos. Una máquina de ensamblaje 2 tiene velocidad 0,75, por lo que su producción teórica es:

\[
R = \frac{0{,}75}{0{,}5} = 1{,}5\text{ unidades/s}
\]

Si la demanda de la etapa siguiente es 6 unidades por segundo, se necesitarían al menos:

\[
N = \frac{6}{1{,}5} = 4\text{ máquinas}
\]

El cálculo supone suministro continuo de ingredientes y una salida sin bloqueos. Si las máquinas se paran con frecuencia, añadir más máquinas no resolverá el problema hasta corregir la causa.

##### 8.3. Identificar un cuello de botella

Un **cuello de botella** es la etapa que limita la producción total de una cadena. Puede deberse a:

- Una máquina con capacidad insuficiente.
- Un insertor demasiado lento.
- Una cinta que no puede transportar todo el caudal.
- Un ingrediente que llega en cantidad insuficiente.
- Una salida o un cofre llenos.
- Una red eléctrica que no suministra potencia adecuada.
- Una receta previa que produce menos de lo que consume la siguiente etapa.

La forma más útil de detectar el cuello de botella consiste en observar dónde se acumulan los productos y qué máquinas están paradas por falta de ingredientes o por salidas bloqueadas.

##### 8.4. Evitar la sobreproducción

No todos los productos necesitan una producción ilimitada. Si se fabrica un objeto que se acumula en cofres sin utilizarse, la máquina y los ingredientes podrían estar mejor dedicados a otra receta.

Puedes limitar la cantidad almacenada, usar señales de circuito o hacer que los insertores extraigan objetos solo cuando se cumplan ciertas condiciones. En sistemas sencillos, una salida permanente hacia el siguiente proceso suele ser suficiente.

---

#### 9. Estrategias de optimización

##### 9.1. Automatizar primero los componentes de uso frecuente

Empieza por los productos que se necesitan en muchas recetas: engranajes, cables, circuitos, cintas, insertores y máquinas de ensamblaje. Al disponer de una fuente automática de estos componentes, resulta mucho más fácil ampliar la fábrica.

##### 9.2. Mantener rutas claras para los ingredientes

Usa cintas separadas cuando las recetas sean complejas o cuando el caudal sea alto. Si dos ingredientes comparten una cinta, asigna carriles y posiciones de insertores de forma consistente. Evita que un cambio pequeño termine mezclando el flujo o impidiendo que un material llegue a su máquina.

##### 9.3. Equilibrar entradas y salidas

La máquina debería recibir ingredientes a la misma velocidad o a una velocidad algo mayor que su consumo esperado, y sus productos deben poder retirarse al mismo ritmo que se fabrican. Un inventario pequeño dentro de la máquina amortigua variaciones, pero no puede compensar una falta constante de recursos.

##### 9.4. Usar módulos según el objetivo

- Prioriza **velocidad** cuando se necesiten más productos por segundo y haya energía suficiente.
- Prioriza **productividad** en recetas compatibles cuando los ingredientes son caros o escasos.
- Prioriza **eficiencia** cuando el consumo energético limita el crecimiento de la fábrica.
- Prioriza **calidad** cuando la línea esté diseñada para producir artículos de calidad superior.
- Usa **balizas** cuando muchas máquinas compatibles estén agrupadas y el beneficio supere el costo energético y material.

No es necesario maximizar todos los atributos. Una máquina más rápida pero mucho más costosa en electricidad puede no ser la opción correcta en una red limitada.

##### 9.5. Construir líneas fáciles de ampliar

Utiliza espacios repetibles y rutas de transporte claras. Deja espacio para postes, insertores, tuberías y futuras cintas. Los diseños compactos pueden ahorrar terreno, pero si complican las conexiones y las reparaciones, pueden ser menos eficientes durante el crecimiento de la fábrica.

##### 9.6. Aprovechar planos (*blueprints*)

Cuando una línea funcione correctamente, guárdala como plano. Esto permite replicar bloques completos sin volver a colocar cada máquina manualmente y ayuda a mantener consistentes las proporciones de entradas, salidas y conexiones eléctricas.

Antes de convertir una línea en plano, verifica que no incluya configuraciones accidentales, cofres llenos, recetas incorrectas ni cables que no deban repetirse.

##### 9.7. Tener en cuenta la contaminación

Las máquinas eléctricas pueden producir contaminación durante su funcionamiento. La contaminación está relacionada con el consumo energético y con las características de cada edificio. Los módulos que aumentan el consumo pueden incrementar indirectamente la contaminación; los módulos de eficiencia pueden reducirla al disminuir el consumo.

Cuando la fábrica está cerca de nidos enemigos, conviene tener en cuenta la ubicación de las líneas y mantener defensas adecuadas. Una línea de producción muy eficiente en materiales puede aun así requerir más capacidad eléctrica y más protección si se amplía sin planificación.

---

#### 10. Errores comunes y soluciones

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| La máquina no empieza a fabricar. | No tiene receta, le faltan ingredientes o no recibe electricidad. | Comprueba la receta, los suministros y la conexión eléctrica. |
| La máquina no acepta una receta avanzada. | El nivel de la máquina no admite la receta o necesita fluidos. | Usa una máquina de nivel superior o el edificio especializado requerido. |
| La máquina de ensamblaje 1 no recibe un fluido. | No admite recetas con fluidos. | Sustitúyela por una máquina de ensamblaje 2 o 3 compatible. |
| Los productos se acumulan en la salida. | El insertor, la cinta o el cofre no pueden retirarlos. | Libera la salida, aumenta el caudal o limita la producción. |
| Una línea se queda sin ingredientes con frecuencia. | La producción de la etapa anterior no es suficiente. | Amplía la máquina anterior o revisa el flujo de transporte. |
| Las cintas están llenas, pero la máquina no trabaja. | La cinta puede contener los ingredientes equivocados, o la máquina no puede acceder al carril correcto. | Revisa el diseño de carriles, los filtros y la posición de los insertores. |
| La máquina produce más lento de lo esperado. | Velocidad base baja, receta larga, falta de energía o ingredientes insuficientes. | Calcula la tasa nominal y observa cuándo se detiene la máquina. |
| Los módulos de velocidad no mejoran el producto final. | Otro eslabón de la cadena limita la producción o los ingredientes no llegan. | Mejora primero el cuello de botella real. |
| La red eléctrica se queda corta al ampliar la línea. | Se añadieron edificios de mayor consumo o balizas. | Amplía la generación o usa módulos de eficiencia según convenga. |
| Una receta con fluidos no funciona. | Las tuberías están mal conectadas o el fluido de entrada no es correcto. | Revisa los conectores, la orientación y la receta seleccionada. |
| La investigación se detiene. | Faltan paquetes de ciencia o laboratorios disponibles. | Comprueba cada línea de paquetes y la alimentación de los laboratorios. |
| Se acumulan demasiados productos sin uso. | La producción supera la demanda o no hay una salida útil. | Limita el inventario, cambia la receta o reutiliza el producto en otra línea. |

---

#### 11. Particularidades de *Space Age*

La expansión *Space Age* incorpora edificios y cadenas especializados que amplían las posibilidades del ensamblado tradicional:

- **Fundición:** produce metales fundidos y puede moldear productos metálicos con una productividad integrada del 50 % en recetas aplicables.
- **Planta electromagnética:** está especializada en electrónica y electromagnetismo, tiene cinco ranuras para módulos y productividad integrada.
- **Cámara biológica:** realiza recetas biológicas, utiliza nutrientes como fuente de energía y dispone de productividad integrada.
- **Planta criogénica:** permite recetas avanzadas en Aquilo y cuenta con ocho ranuras para módulos.
- **Producción planetaria:** algunos ingredientes, fluidos y recetas están ligados a planetas concretos. Por eso, la elección del edificio y la logística entre planetas se convierten en parte del diseño de producción.

Estas estructuras no reemplazan a las máquinas de ensamblaje convencionales en todas las recetas. Cada una está especializada en determinados productos o tipos de proceso. Antes de planificar una línea avanzada, comprueba qué edificio puede ejecutar la receta, qué ingredientes requiere y si el proceso depende de un recurso planetario específico.

---

#### 12. Lista de comprobación de una línea automatizada

- [ ] La máquina seleccionada admite la receta deseada.
- [ ] La receta está configurada correctamente.
- [ ] Todos los ingredientes llegan de forma continua.
- [ ] Los fluidos, cuando se necesitan, están conectados a la entrada correcta.
- [ ] Los insertores tienen velocidad suficiente para la demanda.
- [ ] Las cintas transportadoras pueden mover todos los ingredientes y productos.
- [ ] La salida de la máquina está libre y tiene un destino útil.
- [ ] La red eléctrica puede suministrar la potencia requerida.
- [ ] Los módulos instalados corresponden al objetivo de la línea.
- [ ] La línea tiene capacidad suficiente para abastecer la siguiente etapa.
- [ ] La producción no se acumula innecesariamente en cofres llenos.
- [ ] El diseño deja espacio para mantenimiento y ampliación.
- [ ] Se puede guardar la configuración como plano cuando esté probada.

---

#### 13. Fuentes de referencia

Las siguientes páginas de la *Factorio Wiki* contienen estadísticas, recetas y explicaciones adicionales sobre los edificios y sistemas descritos en esta guía:

- [Crafting — Fabricación y ensamblado](https://wiki.factorio.com/Crafting)
- [Assembling machine 1 — Máquina de ensamblaje 1](https://wiki.factorio.com/Assembling_machine_1)
- [Assembling machine 2 — Máquina de ensamblaje 2](https://wiki.factorio.com/Assembling_machine_2)
- [Assembling machine 3 — Máquina de ensamblaje 3](https://wiki.factorio.com/Assembling_machine_3)
- [Chemical plant — Planta química](https://wiki.factorio.com/Chemical_plant)
- [Oil refinery — Refinería de petróleo](https://wiki.factorio.com/Oil_refinery)
- [Foundry — Fundición de *Space Age*](https://wiki.factorio.com/Foundry)
- [Electromagnetic plant — Planta electromagnética](https://wiki.factorio.com/Electromagnetic_plant)
- [Biochamber — Cámara biológica](https://wiki.factorio.com/Bio_chamber)
- [Cryogenic plant — Planta criogénica](https://wiki.factorio.com/Cryogenic_plant)
- [Inserters — Insertores](https://wiki.factorio.com/Inserters)
- [Transport belts — Cintas transportadoras](https://wiki.factorio.com/Transport_belts)
- [Module — Módulos](https://wiki.factorio.com/Module)
- [Beacon — Baliza](https://wiki.factorio.com/Beacon)
- [Quality — Calidad](https://wiki.factorio.com/Quality)
- [Lab — Laboratorio](https://wiki.factorio.com/Lab)


---

### Guía técnica de Factorio: Generación de energía (calderas, solar y nuclear)

**Versión de referencia:** *Factorio 2.x*. Las mecánicas exclusivas de la expansión *Space Age* se señalan por separado. Las cifras corresponden a edificios de calidad normal y sin bonificaciones de módulos, salvo que se indique lo contrario.

La generación de energía es una de las infraestructuras más importantes de *Factorio*: alimenta taladros eléctricos, insertores, cintas y máquinas de montaje, además de permitir automatizar cadenas de producción cada vez más complejas. A medida que la fábrica crece, también aumenta la demanda energética, por lo que es necesario combinar fuentes de electricidad, almacenamiento y una distribución fiable.

Esta guía cubre tres sistemas principales:

- **Energía a vapor con calderas:** quema combustible para calentar agua, producir vapor y accionar máquinas de vapor. Es la opción inicial y fácil de ampliar.
- **Energía solar:** convierte la luz del día en electricidad sin consumir combustible ni generar contaminación. Para cubrir la noche en un planeta, suele combinarse con acumuladores.
- **Energía nuclear:** utiliza celdas de combustible de uranio para producir calor, que se transforma en vapor de alta temperatura y finalmente en electricidad mediante turbinas.

---

#### 1. Fundamentos de la generación eléctrica

##### 1.1. ¿Cómo funciona la red eléctrica?

La red eléctrica conecta generadores y consumidores mediante postes eléctricos y sus áreas de cobertura. Los edificios conectados a la misma red comparten la energía producida por las fuentes que pueden abastecerla.

Conviene distinguir tres conceptos:

- **Potencia (kW o MW):** ritmo al que se produce o consume energía en un momento dado.
- **Energía (kJ, MJ o GJ):** cantidad acumulada o consumida durante un intervalo.
- **Almacenamiento:** energía guardada para utilizarla cuando la generación no alcance la demanda.

Un generador que puede producir 10 MW no garantiza que la fábrica reciba 10 MW en todo momento: la potencia efectiva dependerá de la disponibilidad de combustible, agua, vapor, calor y capacidad de distribución.

La pantalla de electricidad permite comprobar la producción, el consumo, la capacidad disponible y, cuando corresponde, la carga de los acumuladores. Si la demanda se acerca o supera la producción disponible, las máquinas eléctricas pueden funcionar más lentamente.

##### 1.2. Prioridad de las fuentes

En una red con varias fuentes, el juego utiliza la generación disponible para satisfacer la demanda. En términos generales, los paneles solares cubren primero la demanda que pueden satisfacer; los generadores de vapor aportan la energía restante; y los acumuladores se descargan cuando la generación activa ya no alcanza. Cuando sobra energía, los acumuladores pueden cargarse.

Por eso, una red híbrida de paneles solares, acumuladores y generadores de respaldo puede reducir el consumo de combustible durante el día sin perder capacidad de suministro por la noche.

##### 1.3. Progresión habitual

La progresión energética suele seguir esta secuencia:

1. **Inicio:** calderas alimentadas con carbón y máquinas de vapor.
2. **Expansión:** más grupos de vapor o una red solar con acumuladores, según los recursos y el espacio disponible.
3. **Etapa avanzada:** energía nuclear cuando la fábrica ya puede extraer y procesar uranio, fabricar celdas de combustible y manejar una infraestructura térmica más compleja.

No es obligatorio abandonar una fuente al desbloquear otra. Las fuentes pueden coexistir y complementarse: por ejemplo, la energía solar reduce el uso de carbón, mientras que una central nuclear suministra una base estable de potencia.

---

#### 2. Generación con calderas y máquinas de vapor

##### 2.1. Bomba de aguas profundas (*Offshore pump*)


La **bomba de aguas profundas** extrae agua de un lago u océano. No necesita electricidad ni combustible para funcionar y debe colocarse en una orilla compatible.

**Características principales:**

- Tamaño: 2 × 1 casillas.
- Producción nominal: 1200 unidades de fluido por segundo.
- No consume electricidad ni combustible.
- Debe conectarse a tuberías, calderas u otros edificios que acepten fluidos.

En un sistema de vapor, la bomba suministra el agua que las calderas convierten en vapor. Una bomba puede suministrar agua suficiente para hasta 200 calderas normales a plena capacidad, siempre que el diseño de tuberías y el flujo de fluidos no introduzcan otras limitaciones.

Consulta la [Factorio Wiki: Offshore pump](https://wiki.factorio.com/Offshore_pump).

##### 2.2. Caldera (*Boiler*)


La **caldera** quema combustible y utiliza el calor para convertir agua en vapor a 165 °C. Este vapor alimenta las máquinas de vapor.

**Características principales, calidad normal:**

- Tamaño: 2 × 3 casillas.
- Consumo térmico de combustible: 1,8 MW cuando trabaja a plena capacidad.
- Producción: 60 unidades de vapor por segundo a 165 °C.
- Consumo de agua: 6 unidades por segundo.
- Genera contaminación durante su funcionamiento.
- Puede usar combustibles compatibles, como madera, carbón o combustible sólido.

La caldera solo consume combustible cuando necesita producir vapor. La tasa de consumo depende del valor energético del combustible utilizado.

**Importante:** las calderas no producen electricidad directamente. Transforman energía química del combustible en energía térmica almacenada en el vapor; las máquinas de vapor convierten después esa energía en electricidad.

La [Factorio Wiki: Boiler](https://wiki.factorio.com/Boiler) detalla las recetas, los combustibles y las estadísticas del edificio.

##### 2.3. Máquina de vapor (*Steam engine*)


La **máquina de vapor** consume vapor y genera electricidad. Es el generador clásico del inicio de la partida.

**Características principales, calidad normal:**

- Tamaño: 3 × 5 casillas.
- Potencia máxima: 900 kW (0,9 MW).
- Consumo máximo: 30 unidades de vapor por segundo.
- Temperatura máxima del vapor aprovechado: 165 °C.

El vapor más caliente que 165 °C no hace que una máquina de vapor común produzca más electricidad. Para aprovechar el vapor a 500 °C generado por una central nuclear se utilizan turbinas de vapor.

##### 2.4. Proporciones recomendadas para una central de vapor

Para edificios de calidad normal y a plena capacidad, las proporciones básicas son:

| Estructura | Cantidad | Motivo |
|---|---:|---|
| Bomba de aguas profundas | 1 | Suministra hasta 1200 unidades de agua por segundo. |
| Calderas | 200 | Consumen 6 de agua/s cada una; en conjunto, 1200 de agua/s. |
| Máquinas de vapor | 400 | Cada caldera produce 60 de vapor/s y cada máquina consume 30 de vapor/s. |

Estas proporciones son teóricas y suponen que el agua y el vapor pueden circular correctamente. En diseños reales conviene comprobar las conexiones y evitar tramos de tubería innecesariamente largos o estrechos.

Para un bloque pequeño, la relación más sencilla es **1 caldera por cada 2 máquinas de vapor**. Cada caldera aporta hasta 1,8 MW de energía térmica, suficiente para dos máquinas de vapor de 0,9 MW cada una.

##### 2.5. Combustibles y suministro

La fuente de combustible debe mantener la central funcionando sin interrupciones. El carbón es común al principio, pero también se pueden utilizar otros combustibles compatibles.

Consejos prácticos:

- Asegúrate de que los insertores suministren combustible suficiente a todas las calderas.
- Usa cintas con capacidad suficiente para el número de calderas.
- Evita depender de insertores eléctricos como único método para volver a arrancar una red que se ha quedado completamente sin energía. Un insertor a combustión puede ayudar a reiniciar el suministro en centrales que usan combustible sólido.
- Mantén el agua accesible y revisa que la tubería de entrada esté conectada a la caldera.
- Comprueba que las máquinas de vapor estén conectadas a la misma red eléctrica que los edificios consumidores.

##### 2.6. Esquema de una central de vapor


| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Captación de agua | Lago u océano | Bomba de aguas profundas | Agua | La bomba de aguas profundas no necesita electricidad ni combustible, según la guía. |
| Transporte de agua | Agua | Tuberías | Agua suministrada a las calderas | — |
| Alimentación de combustible | Combustible | Cinta de combustible | Combustible suministrado a las calderas | — |
| Calentamiento del agua | Agua y combustible | Calderas | Vapor a 165 °C | — |
| Generación eléctrica | Vapor a 165 °C | Máquinas de vapor | Electricidad | — |
| Distribución eléctrica | Electricidad | Postes eléctricos | Electricidad disponible en la fábrica | — |

La tabla representa el flujo lógico, no una plantilla exacta de colocación. Al construir, utiliza la vista de conexiones de fluidos para comprobar que el agua entra por la conexión correcta y que la salida de vapor alimenta los generadores.

---

#### 3. Generación solar

##### 3.1. Panel solar (*Solar panel*)


El **panel solar** produce electricidad a partir de la luz del día. No consume combustible y no genera contaminación durante su funcionamiento.

**Características principales, calidad normal:**

- Tamaño: 3 × 3 casillas.
- Potencia máxima durante el día en Nauvis: 60 kW.
- Producción media durante el ciclo completo día/noche en Nauvis: 42 kW por panel.
- No funciona durante la noche en superficies planetarias normales.
- La producción aumenta o disminuye gradualmente durante el amanecer y el atardecer.

Un panel solar produce 60 kW en el periodo de máxima iluminación, pero no mantiene esa potencia durante el ciclo completo. Por este motivo, para dimensionar un sistema que cubra la demanda durante día y noche es necesario tener en cuenta la producción media y el almacenamiento.

En *Space Age*, la producción solar depende también de las condiciones de cada planeta. Las plataformas espaciales no tienen un ciclo normal de día y noche, por lo que sus paneles pueden producir electricidad de forma constante, con potencia ajustada a su ubicación o trayecto.

Más información: [Factorio Wiki: Solar panel](https://wiki.factorio.com/Solar_panel).

##### 3.2. Acumulador (*Accumulator*)


El **acumulador** almacena electricidad cuando la producción disponible supera la demanda y la devuelve a la red cuando los generadores no pueden cubrirla.

**Características principales, calidad normal:**

- Tamaño: 2 × 2 casillas.
- Capacidad de almacenamiento: 5 MJ.
- Potencia máxima de carga: 300 kW.
- Potencia máxima de descarga: 300 kW.
- No produce energía por sí mismo; solo almacena energía generada en otro lugar.

Un acumulador puede entregar hasta 300 kW, pero solo durante el tiempo que le permita su energía almacenada. La capacidad total y la potencia máxima son límites diferentes: una instalación puede tener energía suficiente para muchas horas, pero no necesariamente entregar toda esa energía al instante si la potencia de descarga agregada es insuficiente.

Consulta la [Factorio Wiki: Accumulator](https://wiki.factorio.com/Accumulator).

##### 3.3. Proporción solar y almacenamiento

En Nauvis, la proporción de referencia para cubrir una demanda estable durante todo el ciclo día/noche es aproximadamente:

- **23,8 paneles solares por cada 1 MW de demanda continua.**
- **0,84 acumuladores por cada panel solar**, es decir, una relación aproximada de 21 acumuladores por 25 paneles.

Una regla sencilla que se aproxima bastante a la proporción óptima es **24 paneles solares y 20 acumuladores por cada 1 MW de demanda continua**. Para una fábrica que necesita 10 MW de forma constante, eso equivale aproximadamente a 240 paneles y 200 acumuladores.

La cifra real puede variar con el tipo de superficie, la calidad de los edificios y los cambios de la expansión. Los paneles solares proporcionan energía gratuita durante la operación, pero requieren una inversión inicial en materiales y espacio.

##### 3.4. Consejos de diseño solar

- Construye campos amplios de paneles con una separación que permita conectar los postes eléctricos a toda la instalación.
- Coloca acumuladores en la misma red eléctrica y deja capacidad suficiente para atravesar toda la noche.
- Observa el nivel de carga de los acumuladores antes de ampliar el consumo de la fábrica.
- No confundas la potencia máxima diurna con la producción media del ciclo completo.
- Aprovecha zonas abiertas, lejos de las áreas congestionadas de la fábrica.
- Si el objetivo es evitar contaminación, la energía solar es especialmente útil para reemplazar parte de la generación basada en carbón.

##### 3.5. Esquema de un sistema solar


| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Generación solar | Luz solar | Paneles solares | Electricidad | — |
| Distribución | Electricidad generada | Red eléctrica | Electricidad para la fábrica | — |
| Almacenamiento durante excedentes | Electricidad excedente | Acumuladores | Energía almacenada | Los acumuladores cargan durante el día en el esquema. |
| Descarga durante déficit | Energía almacenada | Acumuladores conectados a la red | Electricidad suministrada cuando falta generación | El diagrama muestra que los acumuladores se cargan durante el día y descargan cuando falta generación. |

Los acumuladores no necesitan cableado especial además de la conexión a la red eléctrica. La propia red gestiona la carga y descarga de acuerdo con la disponibilidad energética y la demanda.

---

#### 4. Generación nuclear

##### 4.1. ¿Cómo funciona una central nuclear?

La generación nuclear aprovecha la energía liberada al consumir **celdas de combustible de uranio** dentro de un reactor. El reactor no produce electricidad directamente: genera calor. Ese calor se transmite mediante tuberías térmicas a los intercambiadores de calor, que convierten agua en vapor a 500 °C. Las turbinas consumen este vapor y lo transforman en electricidad.

La cadena de producción completa es:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Procesamiento del mineral | Mineral de uranio | Procesamiento de uranio | Uranio-235 y uranio-238 | — |
| Fabricación de combustible | Uranio-235 y uranio-238 | Producción de celdas de combustible de uranio | Celdas de combustible de uranio | — |
| Generación de calor | Celdas de combustible de uranio | Reactor nuclear | Calor | — |
| Transporte térmico | Calor | Tuberías térmicas | Calor transportado | — |
| Transferencia de calor al agua | Calor y agua de la bomba de aguas profundas | Intercambiadores de calor | Vapor a 500 °C | El diagrama complementario muestra el suministro de agua al intercambiador. |
| Generación eléctrica | Vapor a 500 °C | Turbinas de vapor | Electricidad en la red eléctrica | El esquema complementario termina en la red eléctrica. |

La energía nuclear permite producir mucha electricidad en poco espacio y con una demanda relativamente baja de combustible en comparación con las centrales convencionales de carbón. A cambio, requiere investigación, procesamiento de uranio, gestión del combustible y una instalación térmica diseñada correctamente.

##### 4.2. Reactor nuclear (*Nuclear reactor*)


El **reactor nuclear** consume celdas de combustible para generar calor. Puede alcanzar una temperatura máxima de 1000 °C.

**Características principales, calidad normal:**

- Tamaño: 5 × 5 casillas.
- Potencia térmica nominal: 40 MW por reactor sin bonificación por vecinos.
- Duración de una celda de combustible: 200 segundos de funcionamiento.
- Temperatura máxima: 1000 °C.
- Necesita celdas de combustible de uranio para operar.

A diferencia de una caldera, el reactor continúa consumiendo una celda de combustible aunque no haya suficiente demanda de calor. Si alcanza la temperatura máxima, la energía térmica adicional que no pueda almacenarse o transferirse se desperdicia. Por ello, el diseño debe equilibrar reactores, intercambiadores, tuberías térmicas y turbinas.

##### 4.3. Bonificación por adyacencia

Los reactores nucleares reciben una bonificación térmica cuando están colocados junto a otros reactores. Cada reactor adyacente que comparte un lado aporta una bonificación del 100 % a la producción térmica del reactor correspondiente.

Por ejemplo, en una disposición de cuatro reactores en un cuadrado de 2 × 2, cada reactor tiene dos vecinos y produce 120 MW en lugar de 40 MW, para un total teórico de 480 MW térmicos.

La bonificación solo depende de la adyacencia efectiva entre reactores; colocarlos cerca, pero sin compartir un lado, no produce el mismo efecto. Los diseños compactos pueden ser mucho más eficientes en combustible, aunque exigen mayor capacidad de conversión y una planificación térmica más cuidadosa.

##### 4.4. Tubería térmica (*Heat pipe*)


Las **tuberías térmicas** transportan calor desde el reactor hasta los intercambiadores de calor. A diferencia de las tuberías de fluidos, trasladan energía térmica y no agua ni vapor.

- Tamaño: 1 × 1 casilla.
- Temperatura máxima: 1000 °C.
- Conectan reactores con intercambiadores de calor y otros segmentos térmicos compatibles.
- Pueden utilizarse para distribuir calor a lo largo de una instalación.

El calor se transmite a través de los segmentos conectados y la eficiencia práctica depende de la disposición y las temperaturas. Evita colocar intercambiadores demasiado lejos del reactor sin considerar el trazado térmico.

##### 4.5. Intercambiador de calor (*Heat exchanger*)


El **intercambiador de calor** utiliza energía térmica procedente de un reactor y la transfiere al agua para producir vapor a 500 °C.

**Características principales, calidad normal:**

- Tamaño: 2 × 3 casillas.
- Consumo térmico nominal: 10 MW.
- Producción: aproximadamente 103 unidades de vapor por segundo a 500 °C.
- Consumo de agua: aproximadamente 10,3 unidades por segundo a plena producción.
- No convierte el calor en electricidad directamente: necesita turbinas de vapor para completar el proceso.

Un reactor de 40 MW sin bonificación de adyacencia puede suministrar cuatro intercambiadores de calor funcionando a plena capacidad. Por cada 100 % de bonificación térmica adicional, se necesitan cuatro intercambiadores adicionales para convertir ese calor en vapor.

##### 4.6. Turbina de vapor (*Steam turbine*)


La **turbina de vapor** convierte vapor a alta temperatura en electricidad.

**Características principales, calidad normal:**

- Tamaño: 3 × 5 casillas.
- Potencia máxima con vapor a 500 °C: aproximadamente 5,82 MW.
- Consumo máximo de vapor a esa temperatura: 60 unidades por segundo.
- Puede consumir vapor a 165 °C, pero en ese caso produce hasta 1,8 MW, equivalente a dos máquinas de vapor comunes.

Las turbinas de vapor están diseñadas especialmente para centrales nucleares. Para convertir 40 MW térmicos, un bloque puede necesitar cuatro intercambiadores y aproximadamente siete turbinas, que compartirán el vapor producido y no necesariamente trabajarán todas al 100 % de su capacidad máxima simultáneamente.

##### 4.7. Proporciones y dimensionamiento nuclear

Las proporciones exactas dependen de la bonificación por adyacencia y de la distribución física de los edificios. Como referencia de escala, la wiki ofrece una relación aproximada de **2 bombas de agua, 233 intercambiadores de calor y 400 turbinas** para una instalación nuclear de gran tamaño que utiliza ese conjunto de equipos a su capacidad prevista.

Para un único reactor de 40 MW sin vecinos, la planificación inicial puede partir de:

- 1 reactor nuclear.
- 4 intercambiadores de calor.
- Aproximadamente 7 turbinas de vapor.
- 1 bomba de aguas profundas para suministrar agua, siempre que las conexiones y el caudal sean suficientes.
- Tuberías térmicas que conecten el reactor con los intercambiadores.
- Tuberías de agua para alimentar los intercambiadores.
- Insertores y cintas para suministrar las celdas de combustible y retirar las celdas gastadas.

Estas cifras son una base de diseño, no una plantilla universal. Una central con varios reactores adyacentes necesita más intercambiadores, turbinas, agua y capacidad de transporte.

##### 4.8. Procesamiento y combustible nuclear


La **centrífuga** procesa mineral de uranio y permite obtener uranio-235 y uranio-238 mediante la receta de procesamiento de uranio. El uranio-235 es relativamente raro; por eso, la producción de celdas de combustible depende de una cadena de extracción y procesamiento que debe mantenerse de manera constante.

Las **celdas de combustible de uranio** se fabrican con uranio-235, uranio-238 y otros componentes de la receta. Cada celda tiene un valor energético de 8 GJ y se consume en un reactor durante 200 segundos. Al terminar, produce una celda de combustible de uranio agotada, que puede procesarse en una centrífuga para recuperar parte del uranio-238.


La cadena incluye:

1. Extraer mineral de uranio con taladros eléctricos y suministro de ácido sulfúrico.
2. Procesar el mineral en una centrífuga.
3. Fabricar celdas de combustible con los isótopos obtenidos y los insumos necesarios.
4. Introducir las celdas en el reactor con insertores.
5. Retirar las celdas agotadas y, si se desea, reprocesarlas para recuperar uranio-238.

Más información: [Factorio Wiki: Uranium fuel cell](https://wiki.factorio.com/Uranium_fuel_cell) y [Factorio Wiki: Nuclear reactor](https://wiki.factorio.com/Nuclear_reactor).

##### 4.9. Esquema de una central nuclear


La central debe disponer de espacio para reactores, intercambiadores, turbinas, tuberías, insertores y logística de celdas. Es aconsejable construirla en módulos para poder ampliarla sin tener que reorganizar toda la instalación.

---

#### 5. Objetos e insumos necesarios

##### 5.1. Generación con calderas

| Objeto o insumo | Función |
|---|---|
| Agua | Se calienta en las calderas para producir vapor. |
| Carbón, madera u otro combustible compatible | Aporta energía térmica a la caldera. |
| Bomba de aguas profundas | Extrae agua de una costa y no necesita energía. |
| Caldera | Convierte agua en vapor a 165 °C. |
| Máquina de vapor | Consume vapor y produce electricidad. |
| Insertores | Introducen combustible y permiten retirar materiales si hace falta. |
| Cintas transportadoras | Llevan combustible a las calderas. |
| Postes eléctricos | Conectan generadores y consumidores a la red. |

##### 5.2. Generación solar

| Objeto | Función |
|---|---|
| Panel solar | Genera energía durante el periodo de iluminación. |
| Acumulador | Guarda energía para cubrir parte de la demanda cuando la generación solar baja. |
| Postes eléctricos | Conectan los paneles y los acumuladores con la red. |
| Espacio despejado | Permite colocar grandes campos solares y ampliar la instalación. |

No hay insumos de operación para los paneles: una vez construidos y conectados, generan energía de acuerdo con la iluminación y las condiciones locales.

##### 5.3. Generación nuclear

| Objeto o insumo | Función |
|---|---|
| Mineral de uranio | Materia prima que se procesa para obtener isótopos de uranio. |
| Ácido sulfúrico | Necesario para extraer mineral de uranio con taladros eléctricos. |
| Centrífuga | Procesa mineral de uranio y puede reprocesar celdas agotadas. |
| Uranio-235 y uranio-238 | Insumos para fabricar celdas de combustible. |
| Celdas de combustible de uranio | Combustible del reactor nuclear. |
| Reactor nuclear | Convierte la energía de las celdas en calor. |
| Tuberías térmicas | Transportan calor hacia los intercambiadores. |
| Agua | Insumo de los intercambiadores de calor. |
| Intercambiadores de calor | Convierten calor en vapor a 500 °C. |
| Turbinas de vapor | Convierten el vapor caliente en electricidad. |
| Insertores y cintas | Alimentan el reactor y retiran las celdas agotadas. |
| Bombas y tuberías de agua | Suministran agua a los intercambiadores. |

---

#### 6. Comparación entre los sistemas

| Característica | Calderas y máquinas de vapor | Solar con acumuladores | Nuclear |
|---|---|---|---|
| Etapa habitual | Inicio y expansión temprana | Intermedia y avanzada | Avanzada |
| Recurso de operación | Combustible y agua | Ninguno | Celdas de combustible de uranio y agua para los intercambiadores |
| Contaminación directa | Sí, por combustión | No | No equivalente a la combustión convencional; la cadena industrial y logística sí tiene impacto operativo |
| Producción durante la noche | Sí, mientras haya combustible y agua | Solo con acumuladores cargados | Sí, mientras haya combustible y capacidad térmica |
| Espacio requerido | Moderado | Grande para campos de paneles y acumuladores | Compacto en potencia, pero necesita espacio técnico y logística compleja |
| Complejidad | Baja | Media: exige calcular almacenamiento | Alta: necesita procesamiento de uranio y diseño térmico |
| Mantenimiento logístico | Reposición de combustible | Ampliación de paneles y almacenamiento | Suministro de celdas y retirada de celdas agotadas |
| Ventaja principal | Fácil de construir y ampliar | Energía sin combustible ni contaminación operativa | Gran potencia en un área relativamente pequeña |
| Limitación principal | Consumo de combustible y contaminación | Producción variable según la iluminación | Coste inicial, complejidad y posible desperdicio de calor |

---

#### 7. Estrategias y consejos de optimización

##### 7.1. Vigilar la pantalla de electricidad

Revisa la gráfica de producción y consumo cuando amplíes la fábrica. Si la barra de producción no cubre la demanda, añade capacidad o identifica consumidores innecesarios. Si las máquinas funcionan lentamente, la falta de energía puede ser la causa, aunque también conviene comprobar si hay cuellos de botella de insumos.

##### 7.2. Dimensionar según la demanda real

No construyas generadores únicamente por intuición. Calcula la potencia aproximada de las máquinas que esperas conectar y agrega un margen para futuras ampliaciones.

- Para vapor: planifica las relaciones caldera/máquina de vapor y revisa el suministro de agua y combustible.
- Para solar: usa la producción media del ciclo, no solo los 60 kW máximos diurnos por panel.
- Para nuclear: calcula la potencia térmica total del reactor, los intercambiadores que pueden absorberla y las turbinas que consumirán el vapor producido.

##### 7.3. Evitar quedarse sin energía por un fallo de combustible

Una central de vapor puede apagarse si el combustible deja de llegar. Si eso provoca un apagón, los insertores eléctricos pueden dejar de funcionar y no reponer el combustible automáticamente. Para sistemas importantes, reserva una fuente de reinicio independiente o utiliza al menos un insertor a combustión en una parte apropiada de la instalación.

##### 7.4. Aprovechar la energía solar como fuente prioritaria

Los paneles solares pueden reducir el trabajo que debe realizar una central de vapor durante el día. Si hay acumuladores suficientes, parte de la energía solar sobrante se guarda para la noche.

La solar es especialmente útil cuando la fábrica aumenta su consumo de electricidad, pero no conviene instalar paneles sin almacenamiento y asumir que darán potencia constante durante todo el ciclo.

##### 7.5. Controlar el calor nuclear

Los reactores consumen combustible aun cuando no hay suficiente demanda térmica. Si el reactor se mantiene cerca de 1000 °C y los intercambiadores no pueden extraer más calor, parte de la energía del siguiente combustible se desperdiciará.

Para optimizar una central nuclear:

- Instala suficientes intercambiadores para absorber la potencia térmica prevista.
- Añade turbinas suficientes para consumir el vapor producido.
- Mantén una ruta de agua fiable hacia los intercambiadores.
- Considera almacenar vapor en depósitos cuando se quiera guardar temporalmente energía térmica para los momentos de mayor consumo.
- Usa la red de circuitos para vigilar la temperatura del reactor y limitar la inserción de combustible cuando corresponda, siempre que la configuración asegure que no se agote el calor necesario.
- Aprovecha la bonificación por adyacencia cuando el objetivo sea mejorar la eficiencia por celda de combustible.

##### 7.6. No confundir almacenamiento con potencia

Un acumulador almacena 5 MJ y puede entregar como máximo 300 kW. Aunque haya muchos acumuladores, la potencia total entregable está limitada por la suma de las tasas de descarga de todos ellos.

De manera similar, un depósito con vapor caliente almacena energía, pero hace falta una cantidad suficiente de máquinas de vapor o turbinas para convertir esa energía almacenada en electricidad al ritmo requerido.

##### 7.7. Separar y proteger las instalaciones

Las centrales con carbón pueden atraer enemigos debido a la contaminación. Conviene proteger las zonas que contienen calderas, minas de carbón, bombas y líneas de combustible. Las instalaciones solares generan menos presión medioambiental, mientras que las centrales nucleares necesitan una zona técnica despejada para crecer y facilitar el mantenimiento.

---

#### 8. Errores comunes y soluciones

| Problema | Causa posible | Solución recomendada |
|---|---|---|
| Las máquinas eléctricas trabajan lentamente. | La demanda supera la capacidad de generación disponible. | Amplía la generación o reduce temporalmente consumidores prescindibles. |
| Las máquinas de vapor no producen electricidad. | No llega vapor, no hay agua o la caldera se quedó sin combustible. | Revisa la bomba, las conexiones de fluidos, el combustible y la salida de vapor. |
| Se apaga la central al quedarse sin electricidad. | Los insertores eléctricos dejaron de alimentar las calderas. | Usa un método de reinicio independiente, como un insertor a combustión en una parte de la instalación. |
| Hay paneles solares, pero falta energía de noche. | No hay suficientes acumuladores o paneles para cubrir el ciclo completo. | Aumenta ambos según la demanda media y revisa el nivel de carga. |
| Los acumuladores se vacían antes del amanecer. | La capacidad almacenada o la potencia de descarga es insuficiente. | Añade almacenamiento, paneles o generación de respaldo. |
| El reactor está muy caliente y no aumenta la producción eléctrica. | No hay suficientes intercambiadores, vapor o turbinas para usar el calor. | Amplía el circuito térmico y de vapor. |
| El reactor consume celdas, pero se desperdicia energía. | La demanda térmica no aprovecha la generación del reactor, especialmente cerca de 1000 °C. | Ajusta la cantidad de combustible y la capacidad de conversión. |
| Las turbinas no alcanzan su potencia máxima. | El vapor es insuficiente, está a una temperatura menor o la red térmica no puede mantener el caudal. | Comprueba que reciben vapor a 500 °C y amplía los intercambiadores o tuberías según haga falta. |
| Los intercambiadores no producen vapor. | No alcanzaron 500 °C o no reciben agua. | Verifica las conexiones térmicas, la temperatura y el suministro de agua. |
| Una central nuclear deja de funcionar tras agotar el combustible disponible. | No se fabrican o transportan nuevas celdas de combustible. | Revisa la extracción y procesamiento del uranio, la fabricación y los insertores. |

---

#### 9. Particularidades de *Space Age*

La expansión *Space Age* introduce variaciones a las mecánicas de energía:

- **Paneles solares planetarios:** su producción puede ser mayor o menor según el planeta. En Vulcanus, por ejemplo, la producción solar es muy superior a la de Nauvis; por ello, la proporción óptima de acumuladores y paneles cambia.
- **Plataformas espaciales:** no tienen un ciclo normal de día y noche, por lo que los paneles no requieren el mismo tipo de almacenamiento diario que en una superficie planetaria. La potencia cambia con la ubicación o el trayecto.
- **Torre de calentamiento (*Heating tower*):** en Gleba se desbloquea una fuente térmica que quema combustibles y puede trabajar con intercambiadores de calor y turbinas. No es una central nuclear, pero utiliza parte de la misma infraestructura térmica.
- **Fuentes planetarias especiales:** en Fulgora, los pararrayos y colectores de rayos permiten aprovechar la actividad eléctrica atmosférica; la energía de fusión también forma parte de la progresión avanzada de la expansión.

Estas opciones no reemplazan la necesidad de dimensionar correctamente la red. Cada entorno exige evaluar la generación disponible, el almacenamiento, el espacio, los insumos y la potencia que necesita la instalación.

---

#### 10. Lista de comprobación de una instalación energética

- [ ] La red eléctrica conecta todos los generadores con los consumidores necesarios.
- [ ] La producción prevista cubre la demanda máxima o dispone de almacenamiento suficiente.
- [ ] Las calderas reciben agua y combustible de forma constante.
- [ ] Las bombas de agua están correctamente colocadas en una orilla compatible.
- [ ] Las máquinas de vapor reciben vapor a la temperatura adecuada.
- [ ] Los paneles solares y acumuladores son suficientes para la demanda diurna y nocturna.
- [ ] La red de acumuladores tiene suficiente capacidad y potencia de descarga.
- [ ] Los reactores disponen de celdas de combustible y los insertores pueden reponerlas.
- [ ] Los intercambiadores reciben calor y agua, y producen vapor a 500 °C.
- [ ] Hay suficientes turbinas para aprovechar el vapor de los intercambiadores.
- [ ] La central nuclear dispone de espacio y tuberías térmicas para futuras ampliaciones.
- [ ] Se revisa regularmente la gráfica de producción y consumo para anticipar cuellos de botella.

---

#### 11. Fuentes de referencia

Las páginas de la *Factorio Wiki* permiten comprobar las estadísticas, recetas e interfaces de cada edificio:

- [Power production — Producción de energía](https://wiki.factorio.com/Power_production)
- [Electric system — Sistema eléctrico](https://wiki.factorio.com/Electric_system)
- [Offshore pump — Bomba de aguas profundas](https://wiki.factorio.com/Offshore_pump)
- [Boiler — Caldera](https://wiki.factorio.com/Boiler)
- [Steam engine — Máquina de vapor](https://wiki.factorio.com/Steam_engine)
- [Solar panel — Panel solar](https://wiki.factorio.com/Solar_panel)
- [Accumulator — Acumulador](https://wiki.factorio.com/Accumulator)
- [Nuclear reactor — Reactor nuclear](https://wiki.factorio.com/Nuclear_reactor)
- [Heat pipe — Tubería térmica](https://wiki.factorio.com/Heat_pipe)
- [Heat exchanger — Intercambiador de calor](https://wiki.factorio.com/Heat_exchanger)
- [Steam turbine — Turbina de vapor](https://wiki.factorio.com/Steam_turbine)
- [Uranium fuel cell — Celda de combustible de uranio](https://wiki.factorio.com/Uranium_fuel_cell)


---

### Guía técnica de Factorio: Procesamiento de fluidos y refinería de petróleo

**Versión de referencia:** *Factorio 2.x*. Las mecánicas exclusivas de la expansión *Space Age* se identifican por separado. Los valores corresponden a edificios de calidad normal y sin módulos, salvo que se indique lo contrario.

El procesamiento de fluidos transforma materias primas como el petróleo crudo en productos industriales utilizados en cadenas químicas, energía y fabricación avanzada. A diferencia de los objetos sólidos, los fluidos se transportan principalmente mediante tuberías, depósitos, bombas y vagones cisterna.

Esta guía explica la extracción de petróleo, el funcionamiento de la refinería, las recetas de procesamiento, los sistemas de tuberías, las plantas químicas, el equilibrio de subproductos y las técnicas para diseñar una instalación fiable.

---

#### 1. Fundamentos del procesamiento de fluidos

##### 1.1. ¿Qué se considera un fluido?

En *Factorio*, un **fluido** es un recurso que se almacena y mueve dentro de edificios y estructuras compatibles, en lugar de apilarse como un objeto convencional en el inventario. El agua, el petróleo crudo, el petróleo pesado, el petróleo ligero, el gas de petróleo, el lubricante y el ácido sulfúrico son ejemplos de fluidos.

Las diferencias más importantes frente a la logística de objetos sólidos son:

- Las cintas transportadoras no pueden transportar fluidos directamente. Para moverlos mediante cintas es posible envasarlos en barriles con las recetas correspondientes.
- Los insertores no levantan fluidos de una tubería. Los edificios con entradas y salidas de fluido se conectan mediante sus puntos de conexión y las redes de tuberías.
- Una red conectada de tuberías, bombas y depósitos forma un segmento de fluido. En *Factorio 2.0*, el sistema se simplificó: el fluido de un segmento está disponible a lo largo del segmento, en lugar de depender de una simulación clásica de flujo entre cada tubería.
- Cada segmento debe contener un solo tipo de fluido. No conviene conectar por error tuberías de productos distintos: una mezcla incorrecta puede eliminar fluido y detener la producción.
- Los edificios que procesan fluidos deben tener una receta seleccionada y sus entradas/salidas deben coincidir con la receta. El modo de información adicional permite comprobar los conectores.

La red de fluidos es fundamental porque una cadena de refinamiento puede dejar de producir aunque la refinería tenga petróleo crudo y electricidad: basta con que una salida se encuentre llena o que falte uno de los fluidos requeridos.

##### 1.2. Ciclo de producción del petróleo

Una cadena básica se compone de las siguientes etapas:

1. Localizar un campo petrolífero en el mapa.
2. Colocar una bomba extractora (*Pumpjack*) sobre una casilla del campo.
3. Conectar la extracción con la refinería mediante tuberías.
4. Seleccionar una receta de procesamiento en la refinería.
5. Transportar los fluidos resultantes hacia tanques, plantas químicas u otros consumidores.
6. Convertir los fluidos en materiales utilizables, como plástico, azufre, lubricante o combustible sólido.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción de petróleo | Campo petrolífero | Bombas extractoras | Petróleo crudo | — |
| Transporte | Petróleo crudo | Tuberías de petróleo crudo | Petróleo crudo transportado | — |
| Refinación | Petróleo crudo | Refinería de petróleo | Petróleo pesado, petróleo ligero y gas de petróleo | Las salidas dependen de la receta seleccionada. |
| Almacenamiento de petróleo pesado | Petróleo pesado | Depósito de petróleo pesado | Reserva de petróleo pesado | — |
| Producción de lubricante | Petróleo pesado | Planta de lubricante | Lubricante | El esquema muestra una reserva posterior de lubricante. |
| Craqueo | Petróleo pesado | Planta de craqueo | Productos de craqueo | — |
| Almacenamiento de petróleo ligero | Petróleo ligero | Depósito de petróleo ligero | Reserva de petróleo ligero | — |
| Craqueo de petróleo ligero | Petróleo ligero | Planta de craqueo | Gas de petróleo | — |
| Almacenamiento de gas | Gas de petróleo | Depósito de gas de petróleo | Reserva de gas de petróleo | — |
| Producción química | Gas de petróleo | Plantas químicas | Plástico y azufre | El diagrama también indica otras recetas posibles. |

##### 1.3. Importancia en la progresión

El procesamiento del petróleo amplía la fábrica más allá de las recetas mecánicas básicas. El gas de petróleo se emplea en plástico y azufre; el azufre es necesario para fabricar ácido sulfúrico y otros productos químicos; y el lubricante interviene en componentes de máquinas de mayor rendimiento, como los motores eléctricos y las cintas transportadoras rápidas.

A medida que la demanda crece, el reto deja de ser simplemente extraer petróleo y pasa a ser **equilibrar los productos derivados**. La refinación avanzada genera varias salidas a la vez, y la instalación debe ser capaz de consumirlas, convertirlas o almacenarlas.

---

#### 2. Extracción de petróleo crudo

##### 2.1. Bomba extractora (*Pumpjack*)


La **bomba extractora** se coloca directamente sobre una casilla de un campo petrolífero. Cada casilla de recurso admite una bomba extractora y muestra su rendimiento mediante un porcentaje.

**Características de referencia:**

- Tamaño: 3 × 3 casillas.
- Consumo eléctrico: 90 kW mientras trabaja.
- Velocidad de minería base: 1.
- Ranuras para módulos: 2.
- Produce petróleo crudo en el juego base; en *Space Age* también puede extraer determinados fluidos de recursos planetarios especiales.
- Necesita estar conectada a una red eléctrica.

El rendimiento se calcula a partir del porcentaje del campo: cada punto porcentual equivale aproximadamente a 0,1 unidades de fluido por segundo. Por ejemplo, una casilla con un rendimiento del 80 % produce inicialmente alrededor de 8 unidades de petróleo crudo por segundo, antes de mejoras por módulos u otras bonificaciones.

El rendimiento de un campo petrolífero disminuye gradualmente con la extracción, pero en condiciones normales no cae por debajo del 20 % del rendimiento inicial. Esto significa que el yacimiento puede seguir funcionando durante mucho tiempo, aunque su producción disminuya.

**Consejos de colocación:**

- Coloca una bomba sobre cada casilla petrolífera aprovechable.
- Comprueba el rendimiento individual antes de invertir en tuberías largas.
- Usa módulos de velocidad cuando necesites aumentar la extracción, especialmente en campos con rendimiento bajo.
- Deja una ruta para reunir las salidas de varias bombas sin crear conexiones confusas.
- Protege las instalaciones alejadas de la base y sus postes eléctricos.

Más información: [Factorio Wiki: Pumpjack](https://wiki.factorio.com/Pumpjack).

##### 2.2. Campo petrolífero y rendimiento

Un campo petrolífero está formado por varias casillas de recurso. Cada una tiene su propio rendimiento y puede alimentar una bomba extractora independiente. La extracción reduce ese porcentaje a lo largo del tiempo, aunque normalmente queda una producción mínima.

El rendimiento visible puede cambiar por los módulos instalados en la bomba extractora. En consecuencia, conviene diferenciar entre:

- **Rendimiento del yacimiento:** la producción base indicada por la casilla de petróleo.
- **Producción efectiva de la bomba:** la cantidad real extraída después de aplicar módulos y modificadores.
- **Producción total del campo:** la suma de la producción efectiva de todas las bombas conectadas.

No hace falta que cada bomba llegue directamente a una refinería distinta. Varias bombas pueden compartir una red de tuberías que abastezca un grupo de refinerías, siempre que se mantenga el flujo requerido y no se mezclen fluidos diferentes.

---

#### 3. Edificios y estructuras principales

##### 3.1. Refinería de petróleo (*Oil refinery*)


La **refinería de petróleo** es la estructura que procesa petróleo crudo o determinados insumos alternativos para obtener productos de refinamiento.

**Características de referencia:**

- Tamaño: 5 × 5 casillas.
- Consumo eléctrico: 420 kW mientras trabaja; consumo en espera aproximado de 14 kW.
- Velocidad de fabricación base: 1.
- Ranuras para módulos: 3.
- Requiere electricidad y una receta seleccionada.
- Dispone de conexiones de fluido cuya posición depende de la orientación y la receta.

La refinería no empieza a trabajar hasta que se le asigna una receta. En el modo de información adicional se muestran los puntos de entrada y salida que se utilizarán. La orientación puede cambiarse para que las tuberías encajen en el diseño.

**Problema importante:** las recetas que generan varios productos se detienen si una de sus salidas no puede evacuar el fluido correspondiente. Por ejemplo, aunque todavía haya petróleo crudo disponible, una refinería con la salida de petróleo pesado bloqueada puede dejar de producir petróleo ligero y gas de petróleo.

Más información: [Factorio Wiki: Oil refinery](https://wiki.factorio.com/Oil_refinery).

##### 3.2. Planta química (*Chemical plant*)


La **planta química** procesa fluidos y objetos sólidos mediante recetas químicas. Es el edificio que se utiliza para craquear las fracciones del petróleo y producir plástico, azufre, lubricante, ácido sulfúrico, combustible sólido y otros materiales.

**Características de referencia:**

- Tamaño: 3 × 3 casillas.
- Consumo eléctrico: 210 kW mientras trabaja; consumo en espera aproximado de 7 kW.
- Velocidad de fabricación base: 1.
- Ranuras para módulos: 3.
- Sus conexiones de fluidos tienen posiciones determinadas por la receta; pueden adaptarse mediante la orientación o el volteo admitido por el edificio.
- Los objetos sólidos pueden insertarse o retirarse con insertores desde los lados adecuados; los fluidos utilizan los conectores de tubería.

Una planta química debe tener seleccionada la receta correcta, una red para cada fluido de entrada y espacio para evacuar los productos. Si la receta consume agua, no basta con conectar gas de petróleo: también debe llegar suficiente agua.

Más información: [Factorio Wiki: Chemical plant](https://wiki.factorio.com/Chemical_plant).

##### 3.3. Tuberías (*Pipe*)


Las **tuberías** conectan bombas extractoras, refinerías, plantas químicas, bombas de fluidos y depósitos.

- Se conectan automáticamente a tuberías y puntos de fluido adyacentes compatibles.
- Una red conectada forma un segmento de fluido.
- Es importante mantener separadas las redes de petróleo pesado, petróleo ligero, gas de petróleo, agua y otros fluidos.
- Las tuberías convencionales pueden bloquear el paso de personajes y vehículos; conviene diseñar pasillos y cruces para facilitar el acceso.
- En *Factorio 2.0*, las tuberías se comportan como segmentos de fluido y no como una simulación tradicional en la que cada tramo necesita mantener una corriente física independiente.

##### 3.4. Tubería subterránea (*Pipe to ground*)


Las **tuberías subterráneas** permiten cruzar cintas transportadoras, pasillos y otras líneas de la fábrica sin construir una maraña de tuberías visibles.

- Ocupan una sola casilla cada una.
- Se colocan por parejas para conectar una entrada y una salida subterráneas.
- La distancia máxima de conexión depende de la estructura; la variante estándar permite dejar hasta nueve casillas libres entre extremos.
- El tramo subterráneo no es un depósito adicional: el fluido se mantiene en las entidades conectadas al sistema.

Son útiles para mantener líneas de fluido independientes y liberar espacio en diseños compactos.

##### 3.5. Bomba de fluidos (*Pump*)


La **bomba de fluidos** mueve líquidos en una dirección, controla su circulación y puede servir como válvula. También se utiliza para cargar y descargar vagones cisterna en estaciones ferroviarias.

**Características de referencia:**

- Tamaño: 1 × 2 casillas.
- Consumo eléctrico: 30 kW.
- Caudal nominal máximo: 1200 unidades de fluido por segundo, si las condiciones del sistema lo permiten.
- Solo conecta por sus dos extremos, lo que permite separar líneas paralelas.
- Puede controlarse mediante la red de circuitos.

En una red de *Factorio 2.0*, no es necesario colocar bombas en cada tramo corto de tubería. Su función más importante es dirigir, controlar, aislar o transferir fluidos —por ejemplo, a un tren— y mantener un sistema práctico en instalaciones largas.

Más información: [Factorio Wiki: Pump](https://wiki.factorio.com/Pump).

##### 3.6. Depósito de almacenamiento (*Storage tank*)


El **depósito de almacenamiento** guarda grandes cantidades de un fluido y permite crear una reserva intermedia entre producción y consumo.

**Características de referencia:**

- Tamaño: 3 × 3 casillas.
- Capacidad normal: 25 000 unidades de fluido.
- No consume electricidad para almacenar el fluido.
- Puede conectarse a la red de circuitos para enviar la cantidad almacenada como señal.

Usos habituales:

- Mantener una reserva de petróleo crudo para amortiguar variaciones de extracción.
- Evitar interrupciones breves en la producción cuando cambia la demanda.
- Observar si se acumula petróleo pesado, petróleo ligero o gas de petróleo.
- Controlar bombas mediante circuitos para reservar productos para recetas prioritarias.

Un depósito es un **amortiguador**, no una solución a un desequilibrio permanente. Si un fluido se produce continuamente más rápido de lo que se consume, el depósito acabará llenándose y la refinería puede detenerse.

Más información: [Factorio Wiki: Storage tank](https://wiki.factorio.com/Storage_tank).

##### 3.7. Otras estructuras que pueden intervenir

| Estructura | Función en el procesamiento de fluidos |
|---|---|
| **Bomba extractora** | Obtiene petróleo crudo de un campo petrolífero. |
| **Postes eléctricos** | Alimentan bombas extractoras, refinerías, plantas químicas y bombas de fluidos. |
| **Insertores** | Introducen carbón u otros sólidos en edificios, y retiran productos sólidos. No transportan líquidos a través de tuberías. |
| **Cintas transportadoras** | Mueven carbón, plástico, azufre, combustible sólido y otros objetos sólidos. |
| **Cofres** | Guardan objetos sólidos; no almacenan fluidos directamente. |
| **Envasadora mediante barriles** | Permite convertir ciertos fluidos en objetos transportables por cintas o por la red logística. |
| **Vagón cisterna** | Transporta fluidos en trenes; se carga y descarga con bombas colocadas correctamente. |
| **Red de circuitos** | Permite activar bombas según los niveles de los depósitos o las condiciones de producción. |

---

#### 4. Recetas de refinamiento del petróleo

Las recetas se expresan en unidades por ciclo. Los tiempos corresponden a máquinas con velocidad de fabricación 1, antes de módulos u otras bonificaciones.

##### 4.1. Procesamiento básico del petróleo

| Parámetro | Valor |
|---|---|
| Edificio | Refinería de petróleo |
| Tiempo base | 5 segundos por ciclo |
| Entrada | 100 unidades de petróleo crudo |
| Salida | 45 unidades de gas de petróleo |
| Tecnología | Procesamiento de petróleo |

El **procesamiento básico** es la primera receta de petróleo que se desbloquea. Es sencilla porque solo produce una salida, el gas de petróleo, y no necesita agua adicional. Resulta adecuada para empezar a fabricar plástico y azufre.

Su principal limitación es que no produce petróleo pesado ni petróleo ligero. Cuando la fábrica necesita lubricante o combustible sólido de forma más eficiente, conviene desbloquear el procesamiento avanzado.

##### 4.2. Procesamiento avanzado del petróleo

| Parámetro | Valor |
|---|---|
| Edificio | Refinería de petróleo |
| Tiempo base | 5 segundos por ciclo |
| Entradas | 100 petróleo crudo + 50 agua |
| Salidas | 25 petróleo pesado + 45 petróleo ligero + 55 gas de petróleo |
| Tecnología | Procesamiento avanzado del petróleo |

El **procesamiento avanzado** produce tres fluidos a la vez. Esto proporciona una mayor flexibilidad industrial, pero exige que los tres productos puedan salir de la refinería.

Antes de cambiar una refinería de la receta básica a la avanzada, revisa las tuberías y los conectores del agua. La receta avanzada necesita una entrada adicional y tiene varias salidas que deben conectarse a sus redes respectivas.

##### 4.3. Licuefacción del carbón

| Parámetro | Valor |
|---|---|
| Edificio | Refinería de petróleo |
| Tiempo base | 5 segundos por ciclo |
| Entradas | 10 carbón + 25 petróleo pesado + 50 vapor |
| Salidas | 90 petróleo pesado + 20 petróleo ligero + 10 gas de petróleo |
| Tecnología | Licuefacción del carbón |

La **licuefacción del carbón** permite fabricar productos de petróleo a partir de carbón, sin depender exclusivamente de un campo petrolífero. Sin embargo, la receta estándar consume inicialmente petróleo pesado y vapor. Por eso, el sistema necesita una pequeña reserva inicial de petróleo pesado para arrancar.

Es útil cuando hay carbón abundante o se necesita ampliar la producción de derivados lejos de un campo petrolífero. Antes de emplearla, asegúrate de tener capacidad de vapor y de gestionar las salidas del refinamiento.

##### 4.4. Licuefacción simple del carbón — *Space Age*

La expansión *Space Age* añade una receta de **licuefacción simple del carbón**, especialmente útil en Vulcanus, donde los recursos disponibles difieren de los del planeta de inicio.

| Parámetro | Valor |
|---|---|
| Edificio | Refinería de petróleo |
| Tiempo base | 5 segundos por ciclo |
| Entradas | 10 carbón + 2 calcita + 25 ácido sulfúrico |
| Salida | 50 petróleo pesado |
| Tecnología | Procesamiento de calcita |

Esta receta permite obtener petróleo pesado sin exigir la reserva inicial de petróleo pesado y vapor de la receta estándar. Es una mecánica específica de la expansión y no forma parte de las recetas estándar del juego base.

---

#### 5. Craqueo y equilibrio de productos

##### 5.1. ¿Qué es el craqueo?

El **craqueo** convierte una fracción del petróleo en otra para equilibrar la producción. Es especialmente importante en el procesamiento avanzado, porque la fábrica puede necesitar mucho gas de petróleo mientras acumula petróleo pesado o ligero.

El craqueo se realiza en una planta química con la receta apropiada y necesita agua en ambos procesos principales.

##### 5.2. Craqueo de petróleo pesado a petróleo ligero

| Parámetro | Valor |
|---|---|
| Edificio | Planta química |
| Tiempo base | 2 segundos por ciclo |
| Entradas | 40 petróleo pesado + 30 agua |
| Salida | 30 petróleo ligero |
| Tecnología | Procesamiento avanzado del petróleo |

El craqueo de petróleo pesado permite aprovechar el exceso de esta fracción y convertirla en petróleo ligero. No conviene convertir todo el petróleo pesado de forma incondicional si la fábrica también necesita producir lubricante.

##### 5.3. Craqueo de petróleo ligero a gas de petróleo

| Parámetro | Valor |
|---|---|
| Edificio | Planta química |
| Tiempo base | 2 segundos por ciclo |
| Entradas | 30 petróleo ligero + 30 agua |
| Salida | 20 gas de petróleo |
| Tecnología | Procesamiento avanzado del petróleo |

Esta receta resulta útil cuando la producción de plástico o azufre requiere más gas de petróleo del que proporciona la refinería por sí sola. El petróleo ligero puede convertirse en gas, aunque también se utiliza en la producción de combustible sólido y combustible para cohetes.

##### 5.4. Tabla de recetas principales

| Proceso | Edificio | Entradas | Salidas | Tiempo base |
|---|---|---|---|---:|
| Procesamiento básico | Refinería | 100 petróleo crudo | 45 gas de petróleo | 5 s |
| Procesamiento avanzado | Refinería | 100 petróleo crudo + 50 agua | 25 petróleo pesado + 45 petróleo ligero + 55 gas de petróleo | 5 s |
| Craqueo pesado | Planta química | 40 petróleo pesado + 30 agua | 30 petróleo ligero | 2 s |
| Craqueo ligero | Planta química | 30 petróleo ligero + 30 agua | 20 gas de petróleo | 2 s |
| Licuefacción del carbón | Refinería | 10 carbón + 25 petróleo pesado + 50 vapor | 90 petróleo pesado + 20 petróleo ligero + 10 gas de petróleo | 5 s |
| Licuefacción simple — *Space Age* | Refinería | 10 carbón + 2 calcita + 25 ácido sulfúrico | 50 petróleo pesado | 5 s |

*Los tiempos son de receta y no incluyen módulos, calidad ni bonificaciones de velocidad o productividad.*

---

#### 6. Productos derivados y sus usos

##### 6.1. Gas de petróleo


El **gas de petróleo** es un fluido fundamental para la industria química. Su demanda crece con la fabricación de componentes avanzados y productos que requieren plástico o azufre.

Usos habituales:

- Producción de barras de plástico.
- Producción de azufre.
- Fabricación de combustible sólido, si la fábrica necesita aprovechar un excedente.
- Otras recetas químicas y cadenas de producción especializadas.

En muchas fábricas el gas de petróleo se convierte en un recurso limitante. Conviene priorizar las recetas esenciales, como plástico y azufre, antes de consumirlo en combustible sólido.

##### 6.2. Petróleo pesado


El **petróleo pesado** se produce en el procesamiento avanzado y en recetas de licuefacción. Sus usos principales son:

- Producción de lubricante.
- Craqueo hacia petróleo ligero.
- Producción de combustible sólido en recetas compatibles.
- Otras aplicaciones específicas de la expansión.

El lubricante es especialmente importante para fabricar unidades de motor eléctrico y determinadas estructuras de logística de alto rendimiento. Por eso, no es recomendable craquear automáticamente todo el petróleo pesado que se produzca.

##### 6.3. Petróleo ligero


El **petróleo ligero** se puede utilizar para fabricar combustible sólido, combustible para cohetes y otros productos, o convertirse en gas de petróleo mediante craqueo.

El petróleo ligero es especialmente eficiente para producir combustible sólido. Si se acumula petróleo pesado, normalmente resulta más eficiente craquearlo a petróleo ligero antes de convertirlo en combustible sólido, en lugar de fabricar combustible directamente a partir del petróleo pesado.

##### 6.4. Lubricante


La receta estándar del **lubricante** es:

| Edificio | Entrada | Salida | Tiempo base |
|---|---|---|---:|
| Planta química | 10 petróleo pesado | 10 lubricante | 1 s |

El lubricante se emplea en componentes de maquinaria avanzada, como unidades de motor eléctrico y elementos de transporte de alto nivel. Si estos productos forman parte de la progresión actual, reserva una parte del petróleo pesado para mantener su producción.

##### 6.5. Plástico


La receta estándar de plástico utiliza carbón y gas de petróleo:

| Edificio | Entradas | Salida | Tiempo base |
|---|---|---|---:|
| Planta química | 1 carbón + 20 gas de petróleo | 2 barras de plástico | 1 s |

El plástico es fundamental para los circuitos avanzados y las estructuras de baja densidad, entre otros productos. Como muchas de estas recetas se desbloquean en etapas intermedias y avanzadas, es recomendable mantener una línea de plástico dedicada en lugar de depender solo de una producción ocasional.

##### 6.6. Azufre


La receta estándar de azufre es:

| Edificio | Entradas | Salida | Tiempo base |
|---|---|---|---:|
| Planta química | 30 gas de petróleo + 30 agua | 2 azufre | 1 s |

El azufre se utiliza para producir ácido sulfúrico y otros productos químicos, incluidos materiales necesarios en progresión tecnológica avanzada. La demanda de azufre puede competir indirectamente con la del plástico, ya que ambas recetas consumen gas de petróleo.

##### 6.7. Ácido sulfúrico

El ácido sulfúrico es un fluido utilizado en varias cadenas de producción. En el juego base se fabrica en una planta química a partir de azufre, agua y una placa de hierro.

| Edificio | Entradas | Salida | Tiempo base |
|---|---|---|---:|
| Planta química | 5 azufre + 1 placa de hierro + 100 agua | 50 ácido sulfúrico | 1 s |

El ácido sulfúrico se utiliza en la fabricación de baterías, en la extracción de uranio y en otras recetas. Además, en *Space Age* tiene usos adicionales en recetas y entornos planetarios específicos.

##### 6.8. Combustible sólido


El combustible sólido es un objeto que se obtiene en una planta química a partir de determinadas fracciones del petróleo.

| Fluido de entrada | Cantidad por receta | Combustible sólido producido | Observación |
|---|---:|---:|---|
| Petróleo ligero | 10 | 1 | Generalmente, el uso más eficiente de esta fracción para combustible sólido. |
| Petróleo pesado | 20 | 1 | Puede ser útil si sobra petróleo pesado y no se necesita para lubricante. |
| Gas de petróleo | 20 | 1 | Evítalo cuando el gas sea necesario para plástico o azufre. |

El combustible sólido sirve como combustible para estructuras compatibles y puede formar parte de otras cadenas industriales. Conviene elegir el fluido de entrada según las necesidades de la fábrica en lugar de consumir indiscriminadamente los productos más valiosos.

---

#### 7. Transporte, almacenamiento y control de fluidos

##### 7.1. Diseño de tuberías

La forma más sencilla de transportar petróleo crudo y productos refinados es mediante tuberías. Los principios de diseño más importantes son:

- Mantén redes separadas para cada fluido.
- Comprueba las entradas y salidas de las máquinas con el modo de información adicional.
- Usa tuberías subterráneas para cruzar cintas o mantener pasillos despejados.
- Utiliza bombas para definir una dirección de transferencia, aislar segmentos o gestionar la carga de trenes.
- Evita redes innecesariamente enrevesadas: una distribución legible es más fácil de ampliar y reparar.
- En instalaciones largas, ten en cuenta el límite de distancia de fluido indicado por el juego y coloca bombas si la red lo requiere.

En *Factorio 2.0*, el funcionamiento simplificado de los segmentos de fluido reduce la necesidad de hacer diseños basados en una supuesta pérdida de presión de cada tramo. La claridad de conexiones y la capacidad de entrada/salida siguen siendo fundamentales.

##### 7.2. Almacenamiento en depósitos

Los depósitos son útiles para observar reservas y suavizar pequeñas variaciones de producción. Conviene dedicar uno a cada fluido relevante en lugar de conectarlos de modo que puedan mezclar accidentalmente productos.

Un diseño común puede tener depósitos de petróleo crudo antes de las refinerías y depósitos independientes de petróleo pesado, ligero y gas de petróleo después de ellas. No es necesario poner un depósito en cada tubería; se colocan donde permitan medir y gestionar la producción.

Un tanque lleno no resuelve un exceso permanente de producción. Si no hay consumo suficiente, el fluido seguirá acumulándose hasta detener la máquina que lo produce. La solución adecuada es equilibrar la demanda, ampliar el consumo o utilizar craqueo cuando sea compatible.

##### 7.3. Transporte ferroviario de fluidos

Los vagones cisterna permiten mover grandes volúmenes de fluido entre campos petrolíferos, refinerías y zonas de producción. Para cargar o descargar un vagón cisterna, se utilizan bombas colocadas en las posiciones adecuadas de una estación ferroviaria.

Los trenes son útiles cuando la distancia hace poco práctico un tendido muy largo de tuberías. Para mantener una estación fiable:

- Separa la zona de carga de la de descarga.
- Utiliza bombas orientadas en la dirección correcta.
- Mantén las redes de productos diferentes aisladas.
- Añade tanques de amortiguación si la carga/descarga debe mantener un ritmo estable.
- Comprueba que el tren se detiene en la posición correcta para que las bombas conecten con el vagón.

##### 7.4. Barriles

Los barriles convierten ciertos fluidos en objetos transportables mediante cintas, insertores, cofres o robots logísticos. Para usarlos, una máquina compatible llena barriles vacíos y otra puede vaciarlos en una red de tuberías. El barril vacío puede volver al circuito.

Son útiles cuando se necesita usar una red de objetos o transportar distintos fluidos mediante vagones de carga, pero añaden pasos y materiales frente al uso directo de tuberías o vagones cisterna.

##### 7.5. Control mediante circuitos

La red de circuitos permite automatizar bombas según la cantidad de fluido disponible en un depósito. Por ejemplo, una bomba que dirige petróleo pesado hacia una planta de craqueo puede activarse solo cuando el depósito supera un umbral; así se conserva una reserva para producir lubricante.

Una estrategia común consiste en:

1. Conectar el depósito a un cable rojo o verde.
2. Configurar el depósito para emitir su cantidad de fluido.
3. Conectar la bomba relevante a la misma red.
4. Configurar una condición de activación que responda al nivel de la reserva.

Las condiciones exactas dependen del diseño. Conviene dejar un margen de almacenamiento suficiente para evitar que la bomba se active y desactive constantemente.

---

#### 8. Ejemplo de una refinería automatizada

##### 8.1. Diseño básico con procesamiento de petróleo


Este diagrama muestra la lógica general, no una plantilla exacta de colocación. Las tuberías deben respetar las conexiones físicas y las recetas seleccionadas en cada edificio.

##### 8.2. Diseño con procesamiento avanzado

Cuando se desbloquea el procesamiento avanzado, la refinería necesita una conexión adicional de agua y tres rutas de salida. Un diseño fácil de mantener separa cada fluido nada más salir de la refinería.

- **Petróleo pesado:** se dirige primero a lubricante o a una reserva; el excedente puede craquearse a petróleo ligero.
- **Petróleo ligero:** se reserva para combustible sólido o se craquea a gas de petróleo según la demanda.
- **Gas de petróleo:** se dirige a las cadenas de plástico y azufre antes de emplearse en usos secundarios.

Si uno de los tres fluidos se acumula de manera continua, las refinerías dejarán de procesar. Un depósito y un sistema de craqueo bien configurado proporcionan un margen de operación mayor.

##### 8.3. Ejemplo de control con reserva de lubricante

Imagina que la fábrica necesita lubricante de forma constante para fabricar motores eléctricos, pero también necesita convertir petróleo pesado sobrante en petróleo ligero.

Una estrategia robusta es mantener una planta química dedicada al lubricante y una bomba que habilite el flujo hacia el craqueo solo cuando la reserva de petróleo pesado alcance un umbral definido. Así se evita consumir inmediatamente todo el producto que puede necesitar la cadena de lubricante.

---

#### 9. Estrategias de optimización

##### 9.1. Dimensionar la extracción según la demanda

La extracción de petróleo depende de la suma de rendimientos de las bombas. Antes de construir una gran cantidad de refinerías, revisa cuánto petróleo crudo producen realmente los campos. Si la extracción no puede mantener la demanda, aumentar las refinerías no mejora la producción final.

Si la producción es insuficiente:

- Añade bombas extractoras en casillas petrolíferas libres.
- Utiliza módulos de velocidad compatibles para mejorar el caudal.
- Conecta otros campos a la misma red si la distancia lo justifica.
- Considera el transporte ferroviario cuando los campos estén lejos.

##### 9.2. Evitar cuellos de botella en las salidas

El fallo más frecuente de una refinería de procesamiento avanzado es que uno de sus productos no pueda salir. Para detectarlo, revisa los tres productos a la vez. No basta con que el gas de petróleo se consuma rápidamente si el petróleo pesado se acumula hasta bloquear la receta.

Causas habituales:

- No existe una tubería conectada a una salida.
- La tubería está conectada al producto equivocado.
- El depósito está lleno.
- La planta de craqueo no recibe agua.
- El producto tiene un consumo menor que su producción a largo plazo.

##### 9.3. No craquear todos los fluidos automáticamente

Un diseño donde todo el petróleo pesado se craquea al instante puede dejar a la fábrica sin lubricante. De forma similar, transformar todo el petróleo ligero en gas puede reducir la cantidad disponible para combustible sólido.

Prioriza el producto final que necesita la fábrica y trata el craqueo como una herramienta de equilibrio. La mejor receta depende de si la demanda actual está concentrada en plástico, azufre, lubricante, combustible sólido u otros productos.

##### 9.4. Priorizar el gas de petróleo para la química

El gas de petróleo se necesita para fabricar plástico y azufre, que sirven como ingredientes para muchas recetas de nivel medio y avanzado. Convertirlo en combustible sólido puede ser apropiado si existe excedente, pero suele ser una mala prioridad cuando la producción de plástico o azufre es insuficiente.

Una opción es crear líneas químicas dedicadas a plástico y azufre, y asegurarse de que las tuberías o conexiones que las alimentan no estén bloqueadas por otras instalaciones.

##### 9.5. Utilizar módulos con intención

Las refinerías, las plantas químicas y las bombas extractoras admiten módulos compatibles. La mejor elección depende del objetivo:

- **Velocidad:** aumenta el ritmo de procesamiento o extracción, pero incrementa el consumo eléctrico.
- **Productividad:** permite obtener más producto en recetas compatibles, aunque puede reducir la velocidad.
- **Eficiencia:** reduce el consumo energético, útil cuando la red eléctrica está limitada.
- **Calidad:** en *Factorio 2.x*, puede intervenir en resultados de calidad cuando el edificio y la receta lo admiten.

Mide la producción real antes de instalar módulos en toda la planta. Si el cuello de botella es una tubería mal conectada o un depósito lleno, los módulos no solucionarán el problema.

##### 9.6. Mantener el sistema accesible

Una refinería es más fácil de mantener si sus tuberías son reconocibles, hay espacio para leer los conectores de los edificios y se puede ampliar el número de plantas químicas sin rehacer toda la instalación.

Conviene dejar pasillos de mantenimiento, usar tuberías subterráneas para cruzar líneas de objetos sólidos y colocar los depósitos donde se pueda consultar fácilmente su nivel.

---

#### 10. Errores comunes y soluciones

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| La bomba extractora no produce petróleo. | No está colocada sobre una casilla petrolífera, le falta electricidad o el campo tiene poco rendimiento. | Comprueba el indicador de la casilla, la conexión eléctrica y la colocación. |
| La refinería no funciona. | No tiene receta seleccionada o le falta petróleo crudo, agua o electricidad. | Revisa la receta y los insumos requeridos. |
| La refinería básica funciona, pero la avanzada no arranca. | Falta la conexión de agua adicional o una de las salidas no está conectada correctamente. | Inspecciona los conectores y la orientación del edificio. |
| La refinería se detiene aunque tenga petróleo. | Una de las salidas de la receta múltiple está llena o bloqueada. | Evacúa todos los productos; utiliza depósitos o craqueo para equilibrar la producción. |
| Falta gas de petróleo para plástico. | La demanda es mayor que la producción o se consume en otras recetas. | Amplía la refinación, craquea petróleo ligero o prioriza el consumo químico. |
| Se acumula petróleo pesado. | Hay poca demanda de lubricante y no existe suficiente craqueo. | Reserva lo necesario para lubricante y dirige el excedente hacia el craqueo. |
| Se agota el lubricante. | Todo el petróleo pesado se envía al craqueo o la planta química no recibe suficiente fluido. | Protege una reserva y da prioridad a la receta de lubricante. |
| Se acumula petróleo ligero. | No hay suficiente consumo de combustible sólido ni craqueo. | Revisa la demanda y aumenta el craqueo a gas si ese producto falta. |
| La planta química no produce azufre. | Falta gas de petróleo o agua, o la salida de azufre no puede evacuarse. | Comprueba ambos fluidos y la salida sólida de la planta. |
| El sistema mezcla productos por accidente. | Dos redes que deberían estar separadas comparten una conexión. | Aísla las redes y revisa cada segmento de fluido. |
| La bomba no transfiere fluido. | Está orientada al revés, no tiene electricidad o la condición del circuito la desactiva. | Revisa dirección, alimentación y señales de control. |
| El tren cisterna no se carga o descarga. | La bomba no está alineada con el vagón o el tren no se detiene en la posición correcta. | Ajusta las posiciones de parada, las bombas y la estación. |

---

#### 11. Particularidades de *Space Age*

La expansión *Space Age* incorpora nuevos fluidos y recetas para adaptarse a los recursos y desafíos de planetas distintos.

- **Licuefacción simple del carbón:** utiliza carbón, calcita y ácido sulfúrico para producir petróleo pesado; es útil en Vulcanus, donde las fuentes y recetas habituales no están disponibles de la misma manera.
- **Extracción adicional mediante bombas extractoras:** en planetas específicos, algunas casillas de recursos contienen fluidos distintos del petróleo crudo, como ácido sulfúrico, flúor o salmuera de litio.
- **Nuevas recetas químicas:** la planta química puede fabricar productos relacionados con el funcionamiento de plataformas espaciales y otras cadenas de la expansión.
- **Otros edificios de procesamiento:** ciertas recetas pueden elaborarse también en edificios planetarios especializados, según el producto y la tecnología desbloqueada.

Aunque cambien los recursos y las recetas, los principios de la red de fluidos siguen siendo los mismos: las entradas deben corresponder a la receta, cada salida necesita espacio y los fluidos diferentes deben mantenerse separados.

---

#### 12. Lista de comprobación de una refinería

Antes de considerar terminada una instalación de procesamiento de fluidos, verifica lo siguiente:

- [ ] Las bombas extractoras están colocadas sobre casillas petrolíferas y conectadas a la electricidad.
- [ ] El caudal de petróleo crudo es suficiente para alimentar las refinerías instaladas.
- [ ] Cada refinería tiene la receta correcta seleccionada.
- [ ] La orientación y los conectores de agua, petróleo crudo y productos se han comprobado con la vista de información adicional.
- [ ] Cada fluido tiene su propia red de tuberías.
- [ ] Los productos de cada receta tienen una ruta de salida o una reserva de almacenamiento.
- [ ] Las plantas de craqueo reciben el agua y el fluido necesarios.
- [ ] Hay suficiente capacidad para consumir petróleo pesado, petróleo ligero y gas de petróleo.
- [ ] El lubricante está protegido frente a un craqueo demasiado agresivo del petróleo pesado.
- [ ] El gas de petróleo se asigna primero a productos químicos importantes cuando existe escasez.
- [ ] Las bombas de fluidos están orientadas correctamente y reciben electricidad.
- [ ] Las estaciones ferroviarias de fluidos están alineadas con las bombas y los vagones cisterna.
- [ ] Los depósitos se utilizan para gestionar reservas y detectar desequilibrios, no para ocultarlos indefinidamente.
- [ ] La instalación deja espacio para añadir refinerías y plantas químicas cuando aumente la demanda.

---

#### 13. Fuentes de referencia

Las siguientes páginas de la *Factorio Wiki* contienen estadísticas y recetas que permiten verificar los detalles de las estructuras y de las cadenas químicas:

- [Oil processing — Procesamiento del petróleo](https://wiki.factorio.com/Light_oil_cracking)
- [Oil refinery — Refinería de petróleo](https://wiki.factorio.com/Oil_refinery)
- [Pumpjack — Bomba extractora](https://wiki.factorio.com/Pumpjack)
- [Chemical plant — Planta química](https://wiki.factorio.com/Chemical_plant)
- [Fluid system — Sistema de fluidos](https://wiki.factorio.com/Fluid_system)
- [Pump — Bomba de fluidos](https://wiki.factorio.com/Pump)
- [Storage tank — Depósito de almacenamiento](https://wiki.factorio.com/Storage_tank)
- [Crude oil — Petróleo crudo](https://wiki.factorio.com/Crude_oil)
- [Petroleum gas — Gas de petróleo](https://wiki.factorio.com/Petroleum_gas)
- [Heavy oil — Petróleo pesado](https://wiki.factorio.com/Heavy_oil)
- [Light oil — Petróleo ligero](https://wiki.factorio.com/Light_oil)
- [Lubricant — Lubricante](https://wiki.factorio.com/Infobox:Lubricant)
- [Plastic bar — Barra de plástico](https://wiki.factorio.com/Plastic_bar)
- [Sulfur — Azufre](https://wiki.factorio.com/Sulfur)
- [Storage tanks and the fluid system in Factorio 2.0](https://factorio.com/blog/post/fff-416)


---

### Guía técnica de Factorio: Red logística y de construcción con robots

**Versión de referencia:** *Factorio 2.x*, con las particularidades de *Space Age* señaladas cuando corresponda. Los nombres de algunos elementos pueden variar ligeramente según el idioma configurado en el juego.

Los robots permiten automatizar el transporte de objetos, la construcción de estructuras, la reparación de edificios y la modificación de diseños. A diferencia de las cintas transportadoras, los robots se desplazan por el aire y pueden viajar directamente entre un punto de recogida y uno de entrega, siempre que la tarea esté dentro de una red compatible.

En *Factorio* hay dos tipos principales de robots:

- **Robots logísticos:** trasladan objetos entre cofres logísticos y satisfacen solicitudes de materiales del jugador.
- **Robots de construcción:** construyen entidades fantasma, desmantelan estructuras, aplican mejoras y reparan edificios dañados.

Ambos tipos utilizan roboports para recargar sus baterías y operar dentro de redes de cobertura. Sin embargo, sus funciones son distintas: los robots logísticos se ocupan del movimiento de objetos; los de construcción realizan tareas sobre el mundo.

---

#### 1. Fundamentos de la red de robots

##### 1.1. ¿Qué es una red logística?

Una **red logística** es un conjunto de roboports conectados que coordina el movimiento de robots y los recursos disponibles en cofres logísticos. Los roboports proporcionan cobertura, puntos de recarga y acceso a la red. Los cofres definen cómo pueden utilizarse los objetos almacenados.

Una red correctamente diseñada permite, por ejemplo:

- Llevar placas de hierro desde un cofre proveedor hasta una máquina que las solicite.
- Entregar materiales al inventario del jugador automáticamente.
- Retirar objetos sobrantes del inventario del jugador.
- Suministrar componentes para construir planos (*blueprints*) de forma automática.
- Reparar edificios dañados y reconstruir estructuras destruidas cuando existen fantasmas y materiales disponibles.
- Desmantelar instalaciones y transportar los materiales recuperados a cofres apropiados.

Los robots no eliminan la necesidad de producir materias primas. Solo transportan los objetos que ya existen en el inventario del jugador o en los cofres accesibles de la red, y ejecutan las tareas para las que cuentan con instrucciones y materiales.

##### 1.2. Las dos áreas de un roboport


Un roboport fijo presenta dos zonas de cobertura que cumplen funciones diferentes:

- **Área logística (naranja):** aproximadamente 50 × 50 casillas con un roboport normal. Define la zona donde los robots pueden atender objetivos logísticos y donde los cofres pueden incorporarse a la red.
- **Área de construcción (verde):** aproximadamente 110 × 110 casillas. Permite que los robots de construcción trabajen en entidades y fantasmas dentro de su alcance, aunque esa área sea más grande que la zona logística.

Las áreas se visualizan al seleccionar el roboport. Cuando las áreas logísticas de distintos roboports se tocan o se solapan correctamente, estos pueden enlazarse para formar una red más extensa. Las redes que no están conectadas no comparten automáticamente sus robots ni sus recursos logísticos.

Los robots pueden volar en línea directa entre puntos de la red y no necesitan que haya roboports en cada casilla de su recorrido. Sin embargo, deben disponer de energía para llegar a un punto de recarga, y las estaciones de recarga congestionadas pueden reducir el rendimiento.

Consulta la [Factorio Wiki: Logistic network](https://wiki.factorio.com/Logistic_network) y la [Factorio Wiki: Roboport](https://wiki.factorio.com/Roboport).

##### 1.3. Diferencia entre logística y construcción

| Característica | Robots logísticos | Robots de construcción |
|---|---|---|
| Función principal | Transportar objetos | Construir, reparar, mejorar y desmantelar estructuras |
| Recogida y entrega de objetos | Sí, entre cofres compatibles y solicitudes | Sí, cuando una tarea de construcción o reparación requiere materiales |
| Construcción de entidades fantasma | No | Sí |
| Reparación automática | No | Sí, si hay kits de reparación disponibles y la entidad está dentro del alcance de construcción |
| Solicitudes personales | Pueden entregar objetos al jugador o retirar objetos sobrantes | No son responsables de la logística personal del jugador |
| Uso desde un roboport fijo | Sí | Sí |
| Uso desde un roboport personal equipado por el jugador | No | Sí |

Es posible que ambos tipos de robots estén presentes en la misma red, pero no deben confundirse. Añadir muchos robots logísticos no acelera directamente la construcción de un plano si faltan robots de construcción o materiales accesibles para ellos.

##### 1.4. Cómo se ejecuta una tarea

La secuencia general depende del tipo de robot:

**Para logística:**

1. Un cofre proveedor, un cofre de almacenamiento u otra fuente compatible contiene un objeto.
2. Un destino solicita ese objeto, o el sistema necesita retirar objetos de una fuente.
3. La red asigna un robot logístico disponible.
4. El robot recoge el objeto, lo transporta al destino y lo deposita.
5. Si necesita recargarse, se dirige a un roboport compatible.

**Para construcción:**

1. El jugador coloca un plano o una entidad fantasma, ordena una mejora, marca una estructura para desmantelarla o deja una entidad dañada dentro de la cobertura correspondiente.
2. La red determina qué tareas puede ejecutar.
3. Un robot de construcción disponible obtiene los materiales necesarios de la red, si la tarea los requiere.
4. El robot viaja hasta la tarea y construye, repara, mejora o desmantela.
5. Los materiales recuperados durante un desmantelamiento se transportan a destinos compatibles de la red.

La automatización funciona mejor cuando las zonas de trabajo están bien cubiertas, los cofres cumplen roles claros y los materiales se suministran de forma constante.

---

#### 2. Robots y edificios principales

##### 2.1. Robot logístico (*Logistic robot*)


El **robot logístico** transporta objetos entre cofres logísticos y satisface las solicitudes válidas de la red. No construye edificios ni repara estructuras.

Características de referencia para un robot de calidad normal, sin mejoras de investigación:

- Velocidad de movimiento básica: aproximadamente 3 casillas por segundo.
- Capacidad de carga inicial: 1 objeto por viaje, ampliable mediante la investigación de capacidad de carga de robots.
- Energía almacenada: 1,5 MJ.
- Necesita recargarse periódicamente en un roboport.
- Consume energía durante el vuelo; las mejoras de velocidad pueden incrementar el consumo energético.

Los robots logísticos son útiles para recorridos cortos y transporte flexible de cantidades moderadas. No siempre resultan adecuados para mover enormes flujos industriales: si cientos de robots vuelan continuamente para trasladar placas entre dos puntos, una cinta transportadora o un tren puede ser más sencillo y eficiente.

Los robots logísticos no se defienden de los enemigos. Si sus trayectos atraviesan zonas peligrosas, pueden ser destruidos y perderse tanto robots como objetos transportados.

Más información: [Factorio Wiki: Logistic robot](https://wiki.factorio.com/Logistic_robot).

##### 2.2. Robot de construcción (*Construction robot*)


El **robot de construcción** ejecuta órdenes de construcción y mantenimiento. Entre sus tareas están colocar edificios en posiciones fantasma, reparar entidades dañadas, aplicar cambios indicados por un plan de mejora y desmantelar entidades marcadas.

Sus tareas principales incluyen:

- Construir edificios y estructuras a partir de planos o entidades fantasma.
- Reparar entidades dañadas usando kits de reparación.
- Reemplazar estructuras destruidas cuando existe un fantasma de reconstrucción y están disponibles los materiales.
- Aplicar mejoras especificadas por el planificador de mejoras.
- Desmantelar entidades marcadas y ayudar a transportar los materiales recuperados.

Los robots de construcción que operan desde roboports fijos obtienen materiales de los cofres logísticos accesibles a la red. Los robots controlados desde el **roboport personal** trabajan con los materiales que lleva el jugador en su inventario y no sustituyen a los robots logísticos de la red fija.

Más información: [Factorio Wiki: Construction robot](https://wiki.factorio.com/Construction_robot).

##### 2.3. Roboport fijo (*Roboport*)


El **roboport** es el edificio central de una red logística fija. Define las áreas de cobertura, sirve como estación de recarga y permite añadir robots y kits de reparación a la red.

Características principales de referencia:

- Tamaño: 4 × 4 casillas.
- Consumo eléctrico en espera: alrededor de 50 kW.
- Dispone de cuatro puntos de recarga para robots.
- Área logística: aproximadamente 50 × 50 casillas.
- Área de construcción: aproximadamente 110 × 110 casillas.
- Almacena robots y kits de reparación para incorporarlos a las operaciones de la red.

Una instalación grande puede necesitar más roboports no solo para ampliar la cobertura, sino también para distribuir los puntos de carga. Si demasiados robots compiten por pocos puertos, pasarán más tiempo esperando o desviándose para recargarse.

**Importante:** los roboports necesitan electricidad. Una interrupción eléctrica puede limitar la recarga y reducir la capacidad de trabajo de los robots, aunque estos no se comporten exactamente como una máquina que se detiene en el acto.

##### 2.4. Cofres logísticos

Los cofres logísticos indican qué papel desempeña cada inventario dentro de la red. No son intercambiables: una elección incorrecta puede provocar movimientos innecesarios o que los robots no encuentren los materiales que necesitan.

###### Cofre proveedor activo (*Active provider chest*)


El cofre proveedor activo **intenta vaciar su inventario hacia la red**. Sus contenidos quedan disponibles para que los robots los recojan y los trasladen a destinos válidos, incluso cuando no existe una solicitud específica para esos objetos.

Usos típicos:

- Retirar rápidamente productos de una máquina.
- Vaciar un inventario temporal.
- Sacar de una línea materiales que deben redistribuirse.

No conviene usarlo indiscriminadamente para almacenar productos que deben permanecer cerca de una máquina. Si no se diseña con cuidado, puede generar vuelos innecesarios y dispersar objetos que sería preferible mantener en un lugar concreto.

###### Cofre proveedor pasivo (*Passive provider chest*)


El cofre proveedor pasivo pone sus contenidos a disposición de la red, pero no los empuja a salir por sí mismo. Los robots toman los objetos cuando una solicitud o una tarea válida los necesita.

Es adecuado para:

- Guardar productos fabricados que otros sistemas pueden solicitar.
- Mantener materiales disponibles para construcción.
- Crear depósitos de suministro sin forzar el movimiento inmediato de todos los objetos.

Es una buena opción general para almacenar materiales destinados al consumo de la red.

###### Cofre de almacenamiento (*Storage chest*)


El cofre de almacenamiento guarda objetos que los robots necesitan almacenar cuando no hay una solicitud más específica o cuando el sistema devuelve materiales a la red. También sirve para consolidar materiales sobrantes y objetos recuperados durante el desmantelamiento.

Usos típicos:

- Guardar los objetos que el jugador marca para retirar de su inventario.
- Recibir materiales de estructuras desmanteladas.
- Servir de reserva logística para objetos que no tienen un consumidor inmediato.

Si existen varios cofres de almacenamiento apropiados, la red puede seleccionar un destino de acuerdo con sus reglas internas. Mantener demasiados cofres dispersos puede dificultar encontrar dónde se ha almacenado un recurso.

###### Cofre solicitante (*Requester chest*)


El cofre solicitante pide a la red cantidades configuradas de determinados objetos. Los robots transportan esos objetos desde fuentes disponibles hasta el cofre, siempre que existan recursos suficientes y la red pueda llegar a los puntos pertinentes.

Es útil para:

- Suministrar componentes a una máquina de montaje.
- Entregar ingredientes de forma automática a una línea de producción.
- Preparar estaciones de construcción o mantenimiento.
- Aprovisionar una zona de la fábrica con materiales concretos.

Los cofres solicitantes son especialmente útiles cuando una máquina necesita una selección exacta de materiales y se quiere desacoplarla de las cintas transportadoras tradicionales.

###### Cofre de búfer (*Buffer chest*)


El cofre de búfer combina el comportamiento de suministro y solicitud. Puede pedir ciertos objetos para mantener una reserva local y, al mismo tiempo, poner su contenido a disposición de otras solicitudes de la red, según su configuración.

Usos recomendados:

- Mantener cerca de una zona los materiales necesarios para futuras construcciones.
- Crear reservas locales de piezas frecuentes.
- Alimentar áreas que necesitan materiales disponibles rápidamente, sin tener que depender de un único depósito lejano.

Una opción de configuración permite indicar que un cofre solicitante no tome materiales de cofres de búfer. Revisa esa opción cuando una red no esté usando la fuente que esperabas.

##### 2.5. Roboport personal (*Personal roboport*)


El **roboport personal** es un módulo de equipamiento que se instala en la cuadrícula de equipo de una armadura compatible. Permite que robots de construcción que lleva el jugador ejecuten tareas cerca de él.

Características y limitaciones:

- Solo controla robots de construcción, no robots logísticos.
- Los robots utilizan los materiales que lleva el jugador en el inventario.
- Obtiene electricidad de la cuadrícula de equipo del jugador.
- Cada unidad equipada aumenta el número de robots que pueden trabajar simultáneamente, el alcance efectivo y los puntos de recarga disponibles.
- Es útil para construir o reparar mientras el jugador se desplaza, sin depender de una red logística fija que cubra toda la zona.

Si el jugador utiliza un plano y no lleva suficientes materiales, algunos fantasmas quedarán sin construir hasta que se consigan los objetos que faltan. El roboport personal no puede extraer automáticamente los materiales de cofres lejanos como lo hace una red logística fija.

Más información: [Factorio Wiki: Personal roboport](https://wiki.factorio.com/Personal_roboport).

##### 2.6. Roboport personal MK2 (*Personal roboport MK2*)


El roboport personal MK2 es una versión mejorada para armaduras que admite más actividad de robots y ofrece una cobertura y una capacidad de recarga superiores a la versión normal.

Es apropiado cuando el jugador necesita:

- Construir planos grandes con rapidez.
- Reparar o ampliar instalaciones alejadas.
- Mantener más robots personales en funcionamiento simultáneamente.

Como sigue dependiendo de la cuadrícula de equipo y del inventario del jugador, debe combinarse con baterías suficientes y una reserva razonable de materiales de construcción.

##### 2.7. Kits de reparación (*Repair packs*)


Los **kits de reparación** permiten que los robots de construcción reparen entidades dañadas. Se fabrican con materiales industriales y deben estar disponibles para la red o para las condiciones de construcción pertinentes.

Recomendaciones:

- Mantén kits de reparación en los roboports de zonas importantes o en cofres logísticos accesibles.
- Asegúrate de que la cobertura de construcción incluye los edificios que quieres reparar.
- Recuerda que reparar no sustituye la defensa de una instalación: si los enemigos destruyen un edificio y no queda un fantasma que pueda reconstruirse, quizá sea necesario ordenar la reconstrucción o volver a colocar la entidad.

---

#### 3. Objetos, insumos y tecnologías

##### 3.1. Insumos para montar una red fija

| Objeto o insumo | Función |
|---|---|
| Roboport | Crea cobertura de logística y construcción, recarga robots y mantiene conexiones de la red. |
| Robot logístico | Transporta objetos según solicitudes logísticas. |
| Robot de construcción | Ejecuta construcción, reparación, mejora y desmantelamiento. |
| Cofres logísticos | Definen qué objetos están disponibles, guardados o solicitados. |
| Electricidad | Alimenta los roboports y permite que se recarguen los robots. |
| Kits de reparación | Permiten reparar entidades dañadas. |
| Placas de acero, engranajes y circuitos | Materiales de fabricación habituales para los edificios de robótica. |
| Baterías, motores y circuitos | Componentes de las cadenas de fabricación de robots, según la receta del objeto. |
| Cintas e insertores | Alimentan cofres desde las fábricas y retiran materiales cuando sea necesario. |

Los materiales exactos para fabricar robots y roboports se pueden comprobar en sus recetas del juego. Los componentes intermedios pueden variar en disponibilidad según la progresión tecnológica, la versión y los cambios de balance.

##### 3.2. Objetos para construcción y planos

| Elemento | Uso |
|---|---|
| Plano (*Blueprint*) | Guarda una disposición reutilizable de edificios, cintas, postes y otras entidades. Los robots pueden construirlo cuando se coloca como fantasmas y hay materiales disponibles. |
| Planificador de desmantelamiento | Marca entidades para que los robots de construcción las retiren. |
| Planificador de mejoras | Indica qué edificios deben sustituirse o actualizarse por otros compatibles. |
| Entidades fantasma | Representan edificios aún no construidos; los robots de construcción pueden convertirlas en edificios reales. |
| Kits de reparación | Reponen la salud de entidades dañadas. |
| Materiales del jugador | Insumos que utilizan los robots personales al construir desde la cuadrícula del jugador. |
| Cofres proveedores y de almacenamiento | Fuentes habituales de materiales de construcción para robots de una red fija. |

Los planos no suministran materiales por sí mismos. Solo describen dónde debe colocarse cada entidad. Para completar la construcción, la red necesita que los materiales correctos estén disponibles.

##### 3.3. Tecnologías relevantes

La robótica se desbloquea progresivamente mediante tecnologías de investigación. Los nombres exactos de los requisitos deben verificarse en el árbol del juego según la versión, pero los grupos principales son:

- **Robótica de construcción (*Construction robotics*):** habilita el uso de robots para ejecutar tareas de construcción automatizada y las estructuras asociadas.
- **Robótica logística (*Logistic robotics*):** habilita robots logísticos y funciones básicas de logística mediante robots.
- **Sistema logístico (*Logistic system*):** amplía las opciones de cofres logísticos, incluidos los cofres solicitantes y de búfer.
- **Velocidad de robots de trabajo (*Worker robot speed*):** aumenta la velocidad de vuelo de los robots compatibles.
- **Capacidad de carga de robots de trabajo (*Worker robot cargo size*):** aumenta la cantidad de objetos que pueden transportar por viaje.
- **Logística personal y solicitudes del jugador:** permite configurar objetos que los robots deben entregar al inventario y, según la configuración, retirar del mismo.

Las mejoras de velocidad reducen tiempos de desplazamiento, pero pueden aumentar el consumo energético. Las mejoras de capacidad permiten que cada viaje mueva más objetos, lo cual puede ser especialmente valioso en redes con trayectos frecuentes y solicitudes de grandes cantidades.

---

#### 4. Cómo configurar una red logística

##### 4.1. Construir una red pequeña y conectada


[Roboport — referencia técnica en Factorio Wiki](https://wiki.factorio.com/Roboport)

Para construir una red logística estable:

1. Coloca uno o más roboports en la zona de la fábrica que quieras automatizar.
2. Conecta cada roboport a una red eléctrica adecuada.
3. Verifica que las áreas logísticas naranjas se toquen o solapen para enlazar las zonas que deben compartir recursos.
4. Inserta robots logísticos o de construcción en la red.
5. Coloca cofres logísticos cerca de las máquinas o estaciones que requieren objetos.
6. Comprueba la cobertura y confirma que los robots aparecen como disponibles en la red.

La cobertura de construcción es mayor que la logística, pero no debe usarse como indicador de que un cofre o destino pertenece a la red logística. Para transportar objetos entre cofres, estos deben estar conectados a la red logística correspondiente.

##### 4.2. Crear una estación de suministro

Un ejemplo sencillo para alimentar una máquina de montaje es usar un cofre proveedor pasivo con los ingredientes y un cofre solicitante junto a la máquina.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Suministro de ingredientes | Objetos en la línea de producción | Insertores | Objetos introducidos en la máquina de montaje | — |
| Solicitud logística | Objetos disponibles en un cofre proveedor pasivo | Robots logísticos | Objetos transportados a la red/destino | — |
| Entrega al destino | Objetos transportados por robots logísticos | Cofre solicitante | Objetos disponibles para la máquina de montaje | El diagrama representa una red logística por cofres y robots. |
| Uso de los objetos | Objetos en el cofre solicitante | Insertores y máquina de montaje | Producción alimentada | — |

El cofre proveedor pasivo pone los componentes a disposición de la red. El cofre solicitante pide las cantidades configuradas, y los robots trasladan los objetos entre ambos. Los insertores transfieren después los objetos del cofre a la máquina.

Si la máquina consume los objetos más rápido de lo que los robots pueden suministrarlos, necesitarás ampliar la capacidad de producción, reducir la distancia o aumentar el número de robots y los puntos de recarga.

##### 4.3. Configurar solicitudes del personaje

Las solicitudes logísticas personales permiten pedir cantidades concretas de objetos para el inventario del jugador. Una vez configuradas, la red puede entregar los materiales que falten, siempre que existan en fuentes accesibles y haya robots disponibles.

Las ranuras de basura permiten marcar objetos que no se necesitan para que los robots los retiren del inventario y los transporten a destinos logísticos apropiados. Esta función resulta útil al cambiar de tarea, al transportar componentes sobrantes o al limpiar el inventario antes de construir.

Consejos:

- Solicita cantidades razonables para evitar que el personaje acumule materiales innecesarios.
- Utiliza las ranuras de basura para limitar el inventario a los recursos que necesitas en ese momento.
- Comprueba que los objetos solicitados están disponibles en la red.
- Mantén fuentes de materiales ordenadas para que los robots no tengan que viajar grandes distancias por solicitudes pequeñas.

##### 4.4. Cómo se abastece la construcción automática

Los robots de construcción necesitan acceso a los materiales indicados por cada entidad fantasma. Las fuentes habituales son los cofres proveedores, cofres de almacenamiento y otros depósitos compatibles que formen parte de la red.

Para evitar que un plano quede incompleto:

- Mantén reservas de los objetos más usados, como cintas, insertores, postes, cofres y máquinas.
- Comprueba que la red dispone de suficiente capacidad de roboports y robots.
- Evita que los materiales estén encerrados en cofres que no forman parte de la red.
- Revisa la distancia entre las fuentes y la zona de construcción.
- Si el diseño requiere objetos avanzados, confirma que su línea de producción los fabrica antes de colocar el plano.

---

#### 5. Construcción, reparación y desmantelamiento con robots

##### 5.1. Construir a partir de planos

Los planos permiten guardar disposiciones de fábrica y repetirlas con precisión. Al colocar un plano en el mundo, las estructuras aparecen como entidades fantasma. Los robots de construcción pueden materializarlas si la red tiene cobertura, robots disponibles e insumos suficientes.

Procedimiento recomendado:

1. Crea o importa un plano.
2. Colócalo en una zona cubierta por la red de construcción.
3. Revisa si hay entidades fantasma fuera de cobertura.
4. Verifica los materiales requeridos y los objetos disponibles en cofres de la red.
5. Comprueba el trabajo pendiente si algunas entidades no se construyen.

Los robots no pueden fabricar por su cuenta los componentes que faltan. Si el plano requiere una máquina que todavía no has producido, tendrás que fabricar ese objeto primero y almacenarlo en una fuente accesible.

##### 5.2. Desmantelar estructuras

El planificador de desmantelamiento marca entidades para su retirada. Los robots de construcción pueden desmontarlas automáticamente si están dentro del alcance correspondiente. Los materiales recuperados pasan a formar parte del flujo logístico y pueden terminar en cofres de almacenamiento u otros destinos compatibles.

Para evitar que los materiales recuperados se mezclen de forma desordenada:

- Deja espacio disponible en cofres de almacenamiento.
- Mantén los cofres de suministro claramente organizados.
- Revisa si el desmantelamiento está produciendo objetos que ya no necesitas.
- Evita marcar por accidente entidades críticas o cables que mantienen conectada la red.

##### 5.3. Aplicar mejoras con robots

El planificador de mejoras permite marcar estructuras para su reemplazo por otras variantes compatibles. Por ejemplo, puede utilizarse para sustituir cintas transportadoras por una versión más rápida o para actualizar ciertas máquinas.

Antes de aplicar una mejora a gran escala:

- Confirma que el inventario logístico contiene las entidades nuevas.
- Comprueba que la sustitución no interrumpe una conexión esencial.
- Prueba primero el plano o la herramienta sobre una zona pequeña.
- Ten en cuenta que las mejoras solo se ejecutan si existe una conversión válida entre la entidad antigua y la nueva.

##### 5.4. Reparación automática

Los robots de construcción pueden reparar entidades dañadas dentro del alcance apropiado cuando los kits de reparación están disponibles. Esto incluye maquinaria, postes, muros y otras entidades compatibles.

La reparación no sustituye las defensas. En zonas con ataques frecuentes, las entidades pueden sufrir daño más rápido de lo que los robots pueden repararlas. Además, los robots son vulnerables a enemigos y pueden ser destruidos en el trayecto.

---

#### 6. Rendimiento, capacidad y escalabilidad

##### 6.1. Velocidad de los robots

La velocidad de vuelo afecta al tiempo necesario para completar cada viaje. Las tecnologías de velocidad de robots de trabajo aumentan el movimiento, pero una red con desplazamientos largos puede seguir perdiendo mucho tiempo por recargas y esperas.

Para mejorar el rendimiento:

- Reduce distancias entre fuentes y destinos cuando sea viable.
- Distribuye los roboports para que los robots tengan puntos de recarga próximos a las zonas de trabajo.
- Investiga mejoras de velocidad cuando la robótica sea una parte importante de la producción.
- Evita enviar robots a través de zonas peligrosas.

##### 6.2. Capacidad de carga

Al principio, un robot puede transportar una cantidad pequeña por viaje. La investigación de capacidad de carga permite mover más objetos en una sola entrega.

Esto reduce el número de vuelos necesarios para satisfacer una demanda grande, aunque no soluciona la falta de robots, las fuentes vacías ni una red de recarga saturada.

##### 6.3. Puntos de recarga y congestión

Los robots deben regresar a un roboport para recargar cuando su energía baja. Si una red tiene muchos robots y pocos puntos de recarga, estos pueden acumularse alrededor de los puertos y pasar más tiempo esperando.

Síntomas de congestión:

- Robots que tardan mucho en entregar objetos pese a que las distancias son razonables.
- Muchos robots intentando recargar al mismo tiempo.
- Solicitudes que se completan lentamente durante periodos de alta actividad.
- Robots que viajan repetidamente entre fuentes dispersas.

Soluciones posibles:

- Añadir roboports para incrementar los puntos de recarga.
- Distribuir fuentes de suministro y destinos en distintas zonas conectadas.
- Reducir los viajes innecesarios mediante una mejor configuración de cofres.
- Usar cintas o trenes para flujos masivos y reservar los robots para entregas flexibles.

##### 6.4. ¿Robots o cintas transportadoras?

| Situación | Solución que suele convenir |
|---|---|
| Movimiento constante de grandes cantidades de placas | Cintas transportadoras o trenes |
| Suministrar materiales distintos según una solicitud | Robots logísticos |
| Construir cientos de estructuras según un plano | Robots de construcción |
| Alimentación flexible de máquinas aisladas | Cofres solicitantes y robots logísticos |
| Transporte de artículos que cambian de destino con frecuencia | Red logística |
| Flujo fijo, predecible y de alto volumen | Cintas o trenes |
| Reparación automática y reconstrucción de fantasmas | Robots de construcción |

Los robots no son necesariamente superiores a las cintas. Una fábrica eficiente puede combinar ambos sistemas: cintas para el movimiento masivo y continuo, y robots para el abastecimiento flexible, la construcción y el mantenimiento.

---

#### 7. Estrategias y consejos de optimización

##### 7.1. Asignar un propósito claro a cada cofre

El uso coherente de cofres reduce desplazamientos innecesarios:

- Proveedor pasivo para almacenar productos que la red pueda necesitar.
- Proveedor activo solo cuando convenga retirar el inventario de forma prioritaria.
- Almacenamiento para sobrantes y materiales recuperados.
- Solicitante para obtener cantidades concretas cerca de un consumidor.
- Búfer para mantener reservas locales y, al mismo tiempo, ofrecer objetos a la red según la configuración.

##### 7.2. Evitar redes excesivamente fragmentadas

Si una instalación utiliza varias redes logísticas separadas, sus robots no compartirán libremente recursos. Una máquina puede quedarse sin materiales aunque exista una gran reserva en otra red desconectada.

Antes de ampliar, revisa la cobertura naranja de los roboports y determina si las redes deben estar conectadas o mantenerse independientes. Las redes separadas pueden ser útiles cuando se necesita aislar materiales, pero deben diseñarse intencionalmente.

##### 7.3. Mantener una reserva de construcción

Una red que construye planos grandes necesita acceso a los objetos que aparecen con mayor frecuencia en ellos. Puedes mantener reservas de cintas, insertores, postes, máquinas de montaje, cofres y otros edificios mediante cofres proveedores o de almacenamiento.

No almacenes cantidades enormes de todos los objetos sin una razón: eso ocupa espacio, puede complicar la organización y no acelera una construcción si el cuello de botella está en los robots o en la recarga.

##### 7.4. Controlar el inventario logístico del jugador

Las solicitudes personales son útiles para que el personaje mantenga una carga de trabajo uniforme, pero solicitudes excesivas pueden consumir las reservas de la fábrica.

Configura cantidades que correspondan a tus tareas habituales y utiliza las ranuras de basura para retirar objetos que ya no necesitas. Si el suministro se vuelve lento, revisa primero si los objetos solicitados existen y si las fuentes están conectadas a la red correcta.

##### 7.5. Proteger los robots y las instalaciones

Los robots logísticos y de construcción no están diseñados para defenderse. Evita que sus trayectos pasen por zonas enemigas siempre que sea posible y protege las áreas de construcción en las que operan continuamente.

Además, no dependas exclusivamente de una única zona de roboports si una interrupción de energía podría detener la recarga de todos los robots críticos. La distribución de las estaciones facilita el mantenimiento y reduce los recorridos de recarga.

##### 7.6. Usar la red de circuitos cuando sea necesario

En versiones modernas, los roboports y algunos cofres permiten integrarse con la red de circuitos para leer estadísticas o contenidos logísticos y automatizar decisiones. Esto resulta útil para activar señales, limitar entregas o coordinar sistemas de suministro más avanzados.

Sin embargo, la red de circuitos no reemplaza la cobertura logística ni crea materiales: solo proporciona señales para controlar otros comportamientos. Comprueba las opciones disponibles en la interfaz del edificio según tu versión del juego.

---

#### 8. Errores comunes y cómo solucionarlos

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| Los robots no se mueven entre dos zonas | Las redes logísticas están separadas o los destinos no están dentro de la cobertura adecuada | Revisa las áreas naranjas y la conexión entre roboports. |
| Los robots de construcción no completan un plano | Faltan materiales, robots, cobertura o capacidad de recarga | Comprueba las entidades fantasma, los cofres de suministro y los roboports. |
| Hay materiales en la fábrica, pero no se entregan al cofre solicitante | Los objetos están en una red desconectada o en cofres que no los ofrecen de forma compatible | Verifica la conectividad y el tipo de cofre fuente. |
| El cofre proveedor activo se vacía demasiado rápido | El diseño permite que la red retire todo el inventario | Sustitúyelo por un proveedor pasivo si se necesita mantener una reserva. |
| El cofre de almacenamiento se llena de objetos inútiles | La red acumula sobrantes sin consumidores | Revisa las fuentes, las solicitudes y las reglas de almacenamiento. |
| Los robots tardan mucho en entregar | Trayectos largos, falta de robots, baja capacidad de carga o congestión de recarga | Añade roboports, mejora la investigación y acerca fuentes y destinos. |
| Los robots de construcción personales no construyen un plano | No hay suficientes materiales en el inventario o están desactivados | Revisa el inventario, el estado del roboport personal y la selección de robots personales. |
| Los robots se quedan sin energía o vuelan lentamente | Se encuentran lejos de puntos de recarga o están afectados por la falta de energía | Distribuye mejor los roboports y comprueba el suministro eléctrico. |
| Los robots desaparecen durante los recorridos | Han sido destruidos por enemigos u otros peligros | Protege la zona y evita rutas peligrosas. |
| Los kits de reparación no se usan | No hay kits disponibles o la entidad queda fuera del alcance de construcción | Coloca kits accesibles y revisa el área de cobertura. |

---

#### 9. Particularidades de *Space Age* y Factorio 2.x

*Factorio 2.x* incorpora ajustes a la experiencia logística y a las interfaces de los roboports. Algunas versiones muestran solicitudes del personaje, estadísticas de robots y controles de red de forma diferente a versiones antiguas. Conviene revisar las opciones de los edificios en la versión que se esté utilizando.

En *Space Age*, las plataformas espaciales y la logística entre superficies añaden restricciones y requisitos de transporte propios. No debe asumirse que toda instalación robótica funciona en una plataforma exactamente igual que en el suelo de un planeta: ciertos edificios, materiales y métodos de transporte tienen condiciones específicas según el entorno.

Los principios generales siguen siendo los mismos:

- Los robots logísticos transportan objetos dentro de redes logísticas compatibles.
- Los robots de construcción realizan tareas sobre entidades y fantasmas dentro de su cobertura.
- Los roboports aportan cobertura y recarga.
- Los cofres definen cómo se ofrecen, almacenan y solicitan los objetos.
- La capacidad real de la red depende de las distancias, la cantidad de robots, la energía, la carga disponible y los puntos de recarga.

---

#### 10. Lista de comprobación de una red robótica

Antes de considerar terminada una instalación, verifica lo siguiente:

- [ ] Los roboports tienen suministro eléctrico.
- [ ] Las áreas logísticas naranjas se conectan donde deben compartir recursos.
- [ ] Los objetivos de construcción están dentro de la cobertura de construcción.
- [ ] Hay suficientes robots logísticos para el volumen de transporte previsto.
- [ ] Hay suficientes robots de construcción para ejecutar planos y reparaciones.
- [ ] Los cofres tienen el tipo correcto para su función: proveedor activo, proveedor pasivo, almacenamiento, solicitante o búfer.
- [ ] Los materiales necesarios existen en cofres accesibles a la red.
- [ ] Las solicitudes personales del jugador están configuradas con cantidades razonables.
- [ ] Las fuentes y los destinos no están tan lejos que generen recorridos innecesarios.
- [ ] Hay suficientes puntos de recarga para evitar congestión.
- [ ] La zona está protegida si los robots vuelan cerca de enemigos.
- [ ] Se dispone de kits de reparación para las entidades que necesitan mantenimiento.
- [ ] Los planos se han comprobado para confirmar que todos los edificios y materiales necesarios están disponibles.

---

#### 11. Fuentes de referencia

Estas páginas de la *Factorio Wiki* ofrecen detalles de estadísticas, recetas y comportamiento de la red robótica:

- [Logistic network — Red logística](https://wiki.factorio.com/Logistic_network)
- [Roboport — Roboport fijo](https://wiki.factorio.com/Roboport)
- [Logistic robot — Robot logístico](https://wiki.factorio.com/Logistic_robot)
- [Construction robot — Robot de construcción](https://wiki.factorio.com/Construction_robot)
- [Personal roboport — Roboport personal](https://wiki.factorio.com/Personal_roboport)
- [Active provider chest — Cofre proveedor activo](https://wiki.factorio.com/Active_provider_chest)
- [Passive provider chest — Cofre proveedor pasivo](https://wiki.factorio.com/Passive_provider_chest)
- [Storage chest — Cofre de almacenamiento](https://wiki.factorio.com/Storage_chest)
- [Requester chest — Cofre solicitante](https://wiki.factorio.com/Requester_chest)
- [Buffer chest — Cofre de búfer](https://wiki.factorio.com/Buffer_chest)
- [Repair pack — Kit de reparación](https://wiki.factorio.com/Repair_pack)


---

### Guía técnica de Factorio: Red de circuitos (cables rojos/verdes y combinadores)

**Versión de referencia:** *Factorio 2.x*, con las funciones específicas de *Space Age* y de las versiones posteriores identificadas cuando corresponda.

La **red de circuitos** es un sistema de control y automatización que permite a las estructuras intercambiar señales numéricas. Con esas señales, una fábrica puede medir existencias, activar o detener máquinas, regular bombas, controlar insertores, gestionar estaciones ferroviarias, seleccionar recetas y construir sistemas lógicos más complejos.

A diferencia de las cintas transportadoras, que desplazan objetos físicos, los cables de circuito transmiten **información**. Un cable puede comunicar que hay 500 placas de hierro en una red, que un tanque contiene cierta cantidad de petróleo o que una señal de control debe estar activa. Los combinadores procesan esa información para tomar decisiones o realizar cálculos.

Esta guía explica la mecánica, los edificios implicados, las señales disponibles, los ejemplos de uso y los errores más comunes.
Esta guía explica la mecánica, los edificios implicados, las señales disponibles, los ejemplos de uso y los errores más comunes.

---

#### 1. Fundamentos de la red de circuitos

##### 1.1. ¿Qué es una red de circuitos?

Una **red de circuitos** es un conjunto de entidades conectadas mediante cables rojos o verdes. Las entidades pueden emitir señales, recibirlas o realizar ambas funciones.

Cada señal es un par formado por un identificador y un valor numérico. Por ejemplo:

- `Placa de hierro = 500` representa una cantidad de 500 placas de hierro detectada por la red.
- `Petróleo crudo = 12000` representa una señal de 12 000 unidades de petróleo crudo.
- `Señal verde = 1` puede actuar como una orden de activación creada por el jugador.
- `Señal A = 0` indica que esa señal no tiene un valor distinto de cero en la red.

Las señales se transmiten en canales diferentes: una red puede transportar al mismo tiempo cantidades de muchos objetos, fluidos y señales virtuales. Si dos dispositivos conectados emiten la misma señal, sus valores se suman.

Por ejemplo, si dos cofres conectados a la misma red verde tienen 100 y 250 placas de hierro, la señal de placas de hierro observada en esa red será **350**.

Fuente principal: [Factorio Wiki: Circuit network](https://wiki.factorio.com/Circuit_network).

##### 1.2. Diferencia entre los cables rojos y verdes

Los cables rojos y verdes tienen el mismo propósito general, pero cada color forma una red independiente.

- **Cable rojo:** comunica las señales de las entidades conectadas por cables rojos.
- **Cable verde:** comunica las señales de las entidades conectadas por cables verdes.
- **Ambos colores en una misma entidad:** la entidad puede leer las señales de las dos redes. En un receptor, los valores que recibe por rojo y verde se suman cuando la configuración lee ambas redes.

Dos cables rojos que se conectan entre sí pertenecen a la misma red roja. Dos cables verdes conectados entre sí forman una única red verde. Sin embargo, un cable rojo y uno verde que llegan al mismo poste no se convierten automáticamente en una red común: siguen siendo dos redes distintas.

Esta separación permite transportar información diferente por cada color. Por ejemplo, se puede usar el cable rojo para leer existencias locales y el verde para distribuir una orden general.


##### 1.3. Los cables de circuito no son cables eléctricos de cobre

Los **cables rojos y verdes transportan señales**, no electricidad. No sustituyen a los cables eléctricos que conectan generadores y consumidores.

Los postes eléctricos pueden cumplir dos funciones diferentes:

- Distribuir energía a las máquinas dentro de su cobertura eléctrica.
- Servir como punto de conexión para extender una red roja o verde.

Por eso, un combinador necesita estar conectado eléctricamente para funcionar, pero la red de circuitos y la red eléctrica siguen siendo sistemas distintos.

En *Factorio 2.0*, los cables rojos y verdes son elementos abstractos para realizar conexiones: no se consumen como materiales al tender o retirar una conexión. Las conexiones también pueden conservarse en planos (*blueprints*) y ser construidas por robots.

##### 1.4. Cómo se crea una red

El procedimiento general es el siguiente:

1. Seleccionar el cable rojo o verde desde la barra de accesos rápidos o mediante el atajo correspondiente.
2. Hacer clic en una entidad compatible para crear una conexión.
3. Hacer clic en otra entidad compatible, o utilizar un poste eléctrico como punto intermedio.
4. Abrir la entidad conectada y configurar qué señales debe enviar, leer o utilizar para activar su comportamiento.
5. Comprobar los valores que circulan por la red, pasando el cursor sobre entidades compatibles o sobre los postes conectados.

En la configuración predeterminada habitual, `Alt + R` selecciona el cable rojo y `Alt + G` el verde. Los atajos pueden cambiarse desde las opciones de controles.

El alcance físico de una conexión directa es limitado. Para tender señales a mayores distancias se pueden utilizar postes eléctricos como puntos de conexión, respetando el alcance del cable entre conexiones.

##### 1.5. Las señales se suman

Una regla fundamental es que las señales del mismo tipo se suman dentro de una red.

Ejemplo:

| Fuente conectada | Placas de hierro emitidas |
|---|---:|
| Cofre A | 100 |
| Cofre B | 250 |
| Cofre C | 75 |
| **Total de la red** | **425** |

Si se conectan los tres cofres con cable verde, todos los dispositivos que lean esa red verde observarán un valor total de 425 placas de hierro.

Si un insertor solo está conectado al cofre A mediante cable rojo, y no tiene otro camino rojo hacia los cofres B y C, leerá 100 placas de hierro en esa red roja. Sin embargo, si el receptor tiene conectadas las dos redes y lee ambas, sumará las señales pertinentes de los dos colores.

Esta regla es muy útil para medir inventarios, pero también puede causar resultados inesperados si dos emisores publican la misma señal y el diseñador no quería sumarlas.

---

#### 2. Señales y lógica básica

##### 2.1. Señales de objetos y fluidos

Los dispositivos compatibles pueden emitir las cantidades de objetos o fluidos que contienen. Cada clase de objeto o fluido dispone de su propio canal.

Ejemplos habituales:

- Cofres: contenido almacenado.
- Tanques de almacenamiento: cantidad de fluido.
- Insertores: objetos que tienen en la mano, de forma puntual o continua.
- Cintas transportadoras: objetos que pasan por ellas, según el modo de lectura.
- Máquinas de montaje, refinerías y plantas químicas: contenido de sus ranuras, receta o estado de trabajo, según la configuración disponible.
- Taladros mineros: recursos estimados restantes, según el modo configurado.
- Acumuladores: porcentaje de carga.
- Paradas de tren: contenido del tren detenido, señales de control y datos de la estación configurados.

Los dispositivos no emiten todas estas señales automáticamente en todos los casos. Es necesario activar la opción de lectura correspondiente en la interfaz de la entidad.

##### 2.2. Señales virtuales

Las **señales virtuales** son canales que no representan directamente un objeto físico. Se pueden utilizar como variables y órdenes creadas por el jugador.

Entre las señales más habituales se encuentran:

- Números del 0 al 9.
- Letras del alfabeto.
- Señales de colores.
- Señales lógicas como `Todo`, `Cualquiera` y `Cada uno`.
- Otras señales virtuales que ayudan a construir estados, contadores y sistemas de control.

Una señal virtual puede utilizarse, por ejemplo, para representar `Petición de hierro`, `Alarma`, `Modo nocturno` o `Estación habilitada`. El significado lo establece el diseño del circuito, no el juego automáticamente.

##### 2.3. Señales comodín: Todo, Cualquiera y Cada uno

Los comodines permiten configurar un circuito que trabaja con distintas señales sin seleccionar manualmente cada tipo de objeto.

- **Cada uno (*Each*):** aplica una operación o condición a cada señal no nula que corresponda. Es útil para filtrar o transformar muchas clases de objetos con una sola configuración.
- **Cualquiera (*Anything*):** comprueba si al menos una señal satisface una condición, según el modo utilizado.
- **Todo (*Everything*):** comprueba si todas las señales consideradas cumplen una condición. El conjunto de señales evaluadas depende de la configuración de la entidad o combinador.

Por ejemplo, un combinador decisor que evalúa `Cada uno > 0` puede emitir las señales de todos los objetos presentes que tengan un valor positivo, en lugar de tener que configurar una comparación distinta para hierro, cobre, carbón y piedra.

Conviene comprobar el resultado en la interfaz del combinador, ya que el comportamiento exacto cambia según si el comodín se utiliza como entrada, como condición o como salida.

##### 2.4. Condiciones y comparaciones

Los receptores pueden utilizar comparaciones como:

- `>`: mayor que.
- `<`: menor que.
- `=`: igual a.
- `≥`: mayor o igual que.
- `≤`: menor o igual que.
- `≠`: distinto de.

Una condición puede comparar una señal con un valor fijo o con otra señal. Por ejemplo:

- `Placas de hierro < 100`: activa el dispositivo cuando la cantidad indicada por la red es inferior a 100.
- `Petróleo crudo > 20000`: cumple la condición cuando la lectura supera las 20 000 unidades.
- `Señal A = Señal B`: comprueba si los valores de ambas señales coinciden.

Las condiciones que se configuran en máquinas y combinadores deben verificarse teniendo en cuenta qué redes están seleccionadas como entrada. Una comparación correcta sobre una red equivocada no producirá el comportamiento previsto.

---

#### 3. Combinadores

Los **combinadores** son edificios que leen señales y generan nuevas señales. Son el elemento principal para transformar una red sencilla de sensores y receptores en un sistema de control más elaborado.

##### 3.1. Combinador constante (*Constant combinator*)


El **combinador constante** emite los valores que el jugador configura, sin necesidad de recibir una señal de entrada.

Usos habituales:

- Definir cantidades objetivo de un inventario.
- Emitir una orden virtual para activar una máquina.
- Configurar recetas o peticiones en sistemas logísticos.
- Probar un circuito antes de conectarlo a una fábrica.
- Proporcionar valores de referencia a otros combinadores.

Ejemplo: configurar `Placas de hierro = 1000` y utilizar esa señal como objetivo de inventario. Después, otro dispositivo puede comparar la cantidad real de hierro con el objetivo.

El combinador constante emite sus señales a las redes a las que está conectado. No permite seleccionar de forma independiente qué señales se envían por rojo y cuáles por verde desde un único edificio. Para emitir valores distintos por cada color, se suelen utilizar dos combinadores constantes separados.

##### 3.2. Combinador aritmético (*Arithmetic combinator*)


El **combinador aritmético** realiza una operación matemática con los valores de entrada y emite el resultado en una señal seleccionada.

Puede realizar operaciones como:

- Suma (`+`).
- Resta (`−`).
- Multiplicación (`×`).
- División (`÷`).
- Módulo o resto (`%`).
- Potencia (`^`).
- Desplazamientos de bits (`<<`, `>>`).
- Operaciones bit a bit AND (`&`), OR (`|`) y XOR (`^`).

Ejemplos prácticos:

- Multiplicar una señal por 2 para escalar un valor.
- Restar las existencias reales de la cantidad objetivo para calcular cuánto falta.
- Convertir un valor de un canal en otro para reutilizarlo como señal de control.
- Calcular restos y dividir cantidades para crear indicadores numéricos.

Cuando se utiliza `Cada uno` como una de las entradas y también como salida, la operación se aplica por separado a cada señal no nula. Por ejemplo, multiplicar `Cada uno × 2` con salida `Cada uno` duplica cada señal manteniendo su tipo de canal.

**Precaución con la división:** el resultado se trunca a un número entero. Por ejemplo, `21 / 10` produce `2`, no `2,1`.

El combinador aritmético introduce una latencia de un tick de juego antes de que su salida esté disponible. Esto importa en contadores, circuitos de memoria y circuitos de sincronización.

##### 3.3. Combinador decisor (*Decider combinator*)


El **combinador decisor** evalúa una o más condiciones y, cuando se cumplen, emite las señales definidas en sus salidas.

Por ejemplo:

- Si `Placas de hierro < 100`, emitir `Señal A = 1`.
- Si `Petróleo crudo > 20000`, emitir `Señal verde = 1`.
- Si el contenido de un depósito supera un umbral, emitir una señal para habilitar una bomba.

En Factorio 2.0, un único combinador decisor puede contener varias condiciones combinadas mediante **AND** y **OR**, además de múltiples salidas. Esto reduce la necesidad de construir largas cadenas de combinadores para reglas sencillas.

El decisor puede emitir una cantidad constante o copiar el valor de entrada de las señales seleccionadas. Con el comodín `Cada uno`, puede evaluar señales distintas y conservar el canal de cada señal que cumple la condición.

Al igual que el combinador aritmético, el combinador decisor tiene una latencia de un tick.

##### 3.4. Combinador selector (*Selector combinator*)


El **combinador selector** permite elegir, contar o transformar señales según un modo de funcionamiento.

Entre los modos disponibles en Factorio 2.x están:

- **Seleccionar entrada:** ordena señales por valor y emite una de ellas, como la señal de valor más alto o más bajo. Puede seleccionarse una posición en la lista ordenada.
- **Contar entradas:** cuenta cuántas señales distintas existen en la entrada, sin sumar sus cantidades como si fueran un único inventario.
- **Entrada aleatoria:** transmite una señal elegida de forma aleatoria con el intervalo configurado.
- **Tamaño de pila:** emite información sobre el tamaño de pila de los objetos compatibles.
- **Capacidad de cohete:** emite información sobre la capacidad de carga en cohete para objetos compatibles.
- **Transferencia o selección de calidad:** en versiones que incluyen las funciones correspondientes, permite filtrar o transformar señales según la calidad.
- **Tiempo:** en versiones que incluyen este modo, puede emitir señales de ticks de juego, ticks del día y duración del día.

Los modos disponibles dependen de la versión y de las funciones de la partida. Algunas opciones están relacionadas con *Space Age* o con versiones posteriores de Factorio 2.x.

El selector es útil para construir sistemas que eligen automáticamente el recurso más escaso o más abundante, seleccionan una señal para mostrarla o consultan propiedades de objetos sin añadir un gran número de combinadores.

---

#### 4. Edificios que pueden enviar o recibir señales

No todos los edificios admiten las mismas funciones. Algunos pueden informar de su estado, otros aceptan órdenes de activación y otros permiten ambas cosas.

##### 4.1. Cofres y tanques de almacenamiento

Los cofres y tanques son fuentes habituales de información.

- Un cofre puede emitir la cantidad de cada objeto que contiene.
- Un tanque puede emitir la cantidad de fluido almacenado.
- Conectar varias unidades a una misma red permite calcular la suma de sus existencias.
- La señal puede utilizarse para detener insertores, activar fabricación o evitar que los almacenes se llenen de un producto.

Los tanques y cofres no suelen tomar decisiones por sí mismos: son sensores de inventario que proporcionan datos a los receptores.

##### 4.2. Insertores


Los insertores pueden configurarse para:

- Activarse o desactivarse según una condición.
- Leer el contenido que llevan en la mano.
- Emitir la lectura en modo **pulso** o **mantener**.
- Cambiar sus filtros según señales, cuando la opción está disponible.
- Ajustar el tamaño de pila mediante una señal, si se habilita el control correspondiente.

**Pulso** significa que el inserter emite la señal durante un solo tick cuando recoge un objeto. **Mantener** significa que la señal se conserva mientras el objeto permanece en su mano.

Ejemplo: conectar un cofre a un insertor y configurarlo para activarse cuando `Placas de hierro < 100`. La condición mantiene el insertor trabajando mientras el inventario observado esté por debajo del umbral, y se detiene cuando la señal alcanza o supera el objetivo.

##### 4.3. Cintas transportadoras y divisores

En Factorio 2.x, las cintas compatibles con circuitos pueden leer el contenido y activarse o desactivarse por condición. Según el modo seleccionado, la cinta puede emitir:

- Un pulso cuando un objeto entra.
- Una lectura continua mientras los objetos están sobre la cinta.
- En modos compatibles, un recuento de los objetos que circulan por una línea más amplia.

Desde versiones de Factorio 2.x, los divisores (*splitters*) también admiten conexión a la red de circuitos para ajustar prioridades o filtros mediante señales. Esto permite diseñar redes de transporte que reaccionan a la demanda o al tipo de material requerido.

##### 4.4. Máquinas de montaje, refinerías y plantas químicas

Las máquinas de fabricación compatibles pueden informar sobre su contenido, receta y estado de trabajo. También pueden controlarse mediante señales para habilitarse o seleccionar recetas dentro de las opciones que ofrece la versión.

Esto permite construir líneas flexibles para fabricar distintos productos con los mismos insumos. No obstante, cambiar dinámicamente la receta requiere que todos los materiales necesarios estén disponibles y que la máquina tenga una salida apropiada para el producto seleccionado.

##### 4.5. Taladros, bombas y estructuras de fluidos

Los circuitos pueden utilizarse para:

- Habilitar o deshabilitar taladros mineros según existencias o señales externas.
- Leer las reservas estimadas de un yacimiento mediante las opciones del taladro.
- Activar bombas de fluidos cuando un tanque alcanza un nivel determinado.
- Regular la distribución de petróleo entre procesos de refinamiento.
- Habilitar bombas extractoras de petróleo u otros fluidos cuando las condiciones lo requieran.
- Leer cantidades de fluido en tuberías o la temperatura de ciertos sistemas compatibles.

En el refinamiento, una estrategia habitual es controlar las bombas que alimentan las plantas químicas según las cantidades almacenadas de petróleo pesado, petróleo ligero, gas de petróleo o lubricante.

##### 4.6. Faros, altavoces programables y paneles de visualización

Las **lámparas** pueden encenderse según una condición de circuito y mostrar colores a partir de señales de color. Son útiles para crear indicadores de estado: por ejemplo, una luz verde cuando hay suficientes placas y una roja cuando falta material.

Los **altavoces programables** pueden emitir sonidos o alertas cuando una señal cumple una condición.

Los **paneles de visualización** pueden mostrar etiquetas o mensajes configurados según las señales recibidas, lo que facilita construir cuadros de control visibles dentro de la fábrica.

##### 4.7. Trenes, señales ferroviarias y paradas

La red de circuitos también permite gestionar el transporte ferroviario.

- Las paradas de tren pueden emitir señales del tren detenido y del tráfico configurado.
- Una parada puede habilitarse o deshabilitarse por condición.
- Se pueden establecer límites o prioridades de trenes mediante señales en las versiones compatibles.
- Las señales ferroviarias pueden exponer su estado o controlarse según condiciones, según el tipo de señal.

Esto permite automatizar estaciones de carga, evitar que los trenes se dirijan a estaciones sin demanda y mejorar la distribución de materiales entre zonas distantes.

##### 4.8. Interruptor de alimentación y acumuladores

El **interruptor de alimentación** (*Power switch*) permite conectar o desconectar dos redes eléctricas bajo una condición de circuito. Es especialmente útil para sistemas que necesitan energía prioritaria o de respaldo.

Los acumuladores pueden emitir su porcentaje de carga. Esa señal se puede usar para encender una central de respaldo cuando el almacenamiento está bajo o para desconectar cargas no esenciales.

##### 4.9. Radar y transmisión de señales sin cable directo

En Factorio 2.x, los radares compatibles con circuitos pueden transmitir señales sin necesidad de conectar físicamente todas las entidades con un cable largo. Según el modo, la señal puede circular por los radares de una superficie o por un canal seleccionable entre superficies o plataformas compatibles.

Esta función resulta útil para comunicar información entre puntos remotos, pero debe diseñarse con claridad para evitar combinar señales de diferentes sistemas por error.

Más detalles sobre edificios compatibles: [Factorio Wiki: Circuit network](https://wiki.factorio.com/Circuit_network).

---

#### 5. Ejemplos prácticos de automatización

##### 5.1. Mantener una reserva mínima de placas de hierro

**Objetivo:** detener el suministro de hierro a un cofre cuando la reserva sea suficiente y reanudarlo cuando falte material.

Configuración:

1. Conectar el cofre de destino a un cable rojo.
2. Conectar el mismo cable al insertor que introduce placas en el cofre.
3. Abrir la configuración del insertor y activar el control por circuito.
4. Configurar la condición `Placas de hierro < 1000`.

Resultado esperado: el insertor trabaja cuando hay menos de 1000 placas de hierro en el inventario medido y se desactiva al alcanzar el umbral.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Lectura del inventario | Cantidad de hierro en el cofre | Cofre de hierro conectado mediante cable rojo | Señal de hierro | — |
| Control de inserción | Señal de hierro | Insertor controlado por circuito | Inserción habilitada o deshabilitada | Condición del esquema: `Hierro < 1000`. |

Si el cofre está conectado a una red que suma otros depósitos, el umbral se aplicará al total de hierro de esa red. Si se necesita medir un único cofre, debe mantenerse una red local separada.

##### 5.2. Activar una bomba según el nivel de un tanque

**Objetivo:** mover un fluido cuando el tanque de origen supera cierto nivel.

Configuración:

1. Conectar el tanque a una red de circuito.
2. Conectar la bomba al mismo circuito.
3. Activar el control de la bomba mediante circuito.
4. Establecer, por ejemplo, `Petróleo crudo > 20000`.

La bomba se activará mientras la señal de petróleo crudo supere las 20 000 unidades. El umbral debe adaptarse a la capacidad del tanque y al consumo de la fábrica.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Lectura del fluido | Cantidad de petróleo crudo en el tanque | Tanque de petróleo conectado mediante cable verde | Señal de petróleo crudo | — |
| Control de bombeo | Señal de petróleo crudo | Bomba controlada por circuito | Bombeo habilitado o deshabilitado | Condición del esquema: `Petróleo crudo > 20000`. |

##### 5.3. Usar el combinador decisor como filtro

**Objetivo:** retransmitir solo señales positivas.

Configuración ilustrativa:

- Entrada: señales de una red de inventario.
- Condición: `Cada uno > 0`.
- Salida: `Cada uno`, copiando los valores de entrada.

El combinador emite únicamente las señales que cumplen la condición. Puede usarse antes de otro circuito para eliminar señales negativas, ceros u objetos que no deben llegar a un receptor.

##### 5.4. Calcular cuánto material falta

**Objetivo:** comparar una cantidad objetivo con el inventario real.

Supongamos que se necesita mantener 1000 placas de hierro y que la red de inventario emite 650.

Una solución puede emitir el objetivo con un combinador constante y restar las existencias mediante un combinador aritmético:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Definición del objetivo | Objetivo de inventario: Hierro = 1000 | Combinador constante | Señal objetivo | — |
| Lectura de existencias | Inventario: Hierro = 650 | Red de inventario | Señal de existencias | — |
| Cálculo del faltante | Señal objetivo y señal de existencias | Combinador aritmético | Hierro requerido = 350 | El esquema calcula objetivo menos existencias. |

El resultado `350` representa la cantidad que falta para alcanzar el objetivo. Para evitar señales de petición negativas cuando hay excedente, se puede añadir un combinador decisor que solo emita el resultado si es mayor que cero.

Al construir el circuito, separa las redes de entrada y salida para no sumar por accidente el objetivo, el inventario y el resultado en el mismo canal. La configuración de las entradas roja y verde del combinador es importante para conseguir la resta prevista.

##### 5.5. Activar la energía de respaldo por nivel de carga

**Objetivo:** desconectar una central de vapor cuando los acumuladores están cargados y volver a conectarla cuando baja la reserva.

Configuración conceptual:

1. Conectar un acumulador a un cable de circuito.
2. Activar la salida de su señal de porcentaje de carga.
3. Conectar la señal al interruptor que separa la central de vapor de la red principal.
4. Configurar el interruptor para conectar la central cuando la carga sea inferior a un umbral, por ejemplo, 20 %.

Esta configuración ayuda a ahorrar combustible cuando hay energía solar suficiente. El umbral se debe ajustar según la demanda y el tamaño del campo solar. Para evitar oscilaciones frecuentes alrededor del umbral, pueden utilizarse condiciones de encendido y apagado diferentes mediante un circuito con memoria.

##### 5.6. Controlar el craqueo de petróleo

Una fábrica puede utilizar señales de los tanques para abrir o cerrar bombas que alimentan las plantas de craqueo.

Ejemplo conceptual:

- Si el petróleo pesado supera cierto nivel y falta petróleo ligero, permitir el craqueo de petróleo pesado.
- Si el gas de petróleo ya está por encima del objetivo, reducir la conversión de petróleo ligero.
- Si falta lubricante, mantener suficiente petróleo pesado para la producción de lubricante antes de convertirlo.

No conviene copiar umbrales universales: deben establecerse según la cantidad de refinerías, la demanda de productos y los volúmenes de almacenamiento.

##### 5.7. Controlar estaciones de tren por demanda

Una estación puede emitir el contenido de un tren detenido o recibir una señal para habilitarse o ajustar su límite de trenes. Esto permite impedir que lleguen trenes a una estación de descarga cuando ya no hay espacio, o abrir una estación de carga solo cuando el inventario local supera una cantidad mínima.

Para diseños avanzados, conviene usar señales distintas para separar la **cantidad de material**, el **estado de la estación** y la **orden de control**. Mezclar los tres valores en un único canal puede crear condiciones incorrectas.

---

#### 6. Planificación y buenas prácticas

##### 6.1. Diseñar señales con un propósito definido

Antes de colocar cables, decide qué información debe transmitir cada red. Por ejemplo:

- Red roja: inventario local de una estación.
- Red verde: peticiones de materiales o señales de control global.
- Señales virtuales: órdenes de habilitación, alarmas o estados de un sistema.

No es obligatorio usar rojo y verde de una manera concreta, pero una convención consistente facilita leer y depurar los circuitos.

##### 6.2. Separar entrada, procesamiento y salida

En circuitos complejos, utiliza una estructura clara:

1. **Entrada:** cofres, tanques, cintas o entidades que publican datos.
2. **Procesamiento:** combinadores que filtran, comparan o calculan.
3. **Salida:** insertores, bombas, máquinas o interruptores que reaccionan a las señales.

Si la misma red conecta todas las etapas, las salidas pueden regresar a las entradas y cambiar los valores que se están calculando. Esto suele ser la causa de muchos errores.

##### 6.3. Utilizar correctamente rojo y verde

Los colores permiten tener dos redes independientes en los mismos edificios y postes. Una técnica habitual es usar una red para transportar valores medidos y otra para enviar señales de control.

No conectes ambos colores entre sí salvo que quieras que el receptor lea ambos conjuntos de señales. Aunque las redes sigan siendo independientes, una entidad conectada a las dos puede sumar sus valores.

##### 6.4. Evitar bucles de realimentación accidentales

Los combinadores aritméticos y decisores tienen conexiones de entrada y salida diferenciadas. Si la salida se conecta de nuevo a la entrada, la señal resultante puede volver al circuito y modificarse una y otra vez.

Esto puede ser intencional en contadores, temporizadores o celdas de memoria, pero es peligroso en una operación que suma constantemente un valor a sí mismo.

Ejemplo de riesgo:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada de señal | Hierro | Entrada del combinador aritmético | Señal de hierro | — |
| Operación aritmética | Señal de hierro | Combinador aritmético | Hierro + 1 -> Hierro | La operación vuelve a la misma entrada en el esquema. |
| Realimentación | Señal de salida del combinador | Cable conectado de vuelta a la entrada | Señal que se vuelve a procesar | Ejemplo de bucle de realimentación; puede ser intencional o accidental. |

En este esquema, la señal de hierro puede aumentar repetidamente en lugar de representar el inventario físico. No conectes salida y entrada de esta forma salvo que comprendas cómo se controla el ciclo.

##### 6.5. Considerar la latencia

El combinador aritmético y el decisor introducen un tick de latencia, aproximadamente 1/60 de segundo a la velocidad de simulación normal. Un circuito con varios combinadores en serie puede tardar varios ticks en propagar un cambio.

En la mayoría de las automatizaciones de inventario esta demora es insignificante. En temporizadores, pulsos o circuitos de memoria sí puede ser determinante.

##### 6.6. Observar los valores antes de complicar el circuito

Cuando un sistema no funciona:

- Pasa el cursor sobre los postes y entidades conectadas para ver las señales, cuando la entidad permita mostrarlas.
- Comprueba qué señales lee el combinador y desde qué color de cable.
- Revisa que las entradas y salidas estén conectadas en el lado correcto del combinador.
- Prueba primero con un combinador constante para emitir señales conocidas.
- Comprueba si el valor es cero, negativo, duplicado o sumado con otra fuente.
- Introduce los combinadores de uno en uno para localizar qué etapa cambia el valor esperado.

##### 6.7. Mantener planos y descripciones legibles

Los circuitos elaborados pueden ser difíciles de entender meses después de construirlos. Pon nombres a señales virtuales cuando sea posible, agrega descripciones a combinadores y conserva planos con etiquetas sobre el objetivo de la lógica.

Una red clara no solo funciona: también permite ampliarla sin tener que reconstruir todo el razonamiento desde cero.

---

#### 7. Errores comunes y soluciones

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| El receptor no reacciona a la señal. | No está conectado, lee otro canal o la condición nunca se cumple. | Verifica el cable, la condición y los valores en la interfaz. |
| El inventario aparece duplicado. | Dos emisores de la misma señal están conectados a la red. | Comprueba qué entidades publican la señal y separa las redes si hace falta. |
| El valor es la suma de dos redes que se querían separar. | El receptor lee señales rojas y verdes a la vez. | Configura qué color lee el receptor o reestructura las conexiones. |
| Una red roja parece conectarse con una verde. | Un mismo dispositivo está conectado a ambas, aunque las redes siguen siendo distintas. | Comprueba la configuración de lectura de cada señal y recuerda que el receptor puede sumarlas. |
| El combinador aritmético produce un valor inesperado. | Operandos, canales o colores de entrada incorrectos. | Revisa ambas entradas, la operación y el canal de salida. |
| El decisor no emite ninguna señal. | No se cumple la condición, se lee el canal equivocado o la señal es cero. | Comprueba la comparación y prueba con una señal constante. |
| Una señal crece sin límite. | Se ha creado un bucle de realimentación sin un mecanismo de control. | Separa entrada y salida o diseña intencionalmente una lógica de memoria. |
| Un insertor trabaja de manera intermitente. | La condición cambia cerca del umbral o hay señales que se suman. | Ajusta el umbral y revisa las señales que llegan al receptor. |
| Una bomba no se activa. | La señal de fluido no alcanza el umbral o el circuito no está conectado a la bomba. | Comprueba el tanque, la señal y la condición de activación. |
| La estación de tren se deshabilita de forma inesperada. | La señal de control comparte canal con inventario u otras señales. | Separa los canales de control y comprueba la condición de la estación. |
| El sistema de control tarda en reaccionar. | Hay varios combinadores en serie o la condición depende de una salida con latencia. | Simplifica el circuito o contempla los ticks de demora. |

---

#### 8. Diferencias y funciones de Factorio 2.x y Space Age

Las versiones modernas de Factorio añadieron funciones que amplían las posibilidades de los circuitos tradicionales.

- **Selector combinator:** incorporado en Factorio 2.0, permite seleccionar señales, contar entradas y consultar propiedades de objetos.
- **Decider combinator actualizado:** la interfaz de 2.0 permite múltiples condiciones y salidas, además de seleccionar qué redes de entrada se utilizan en cada parte de la lógica.
- **Más entidades compatibles:** en Factorio 2.x las cintas, los divisores, los hornos, las máquinas de fabricación y otras estructuras cuentan con opciones de circuito que facilitan el control directo.
- **Paneles de visualización:** permiten mostrar mensajes o etiquetas vinculadas a señales.
- **Radar conectado al circuito:** permite transmitir señales a otros radares en modos compatibles, incluso entre superficies o plataformas cuando se selecciona un canal adecuado.
- **Más información sobre recetas y estados de fabricación:** las máquinas compatibles pueden emitir información de receta, contenido, finalización de trabajo o estado.
- **Modos avanzados del selector:** versiones posteriores añaden opciones como el tiempo o la selección de calidad; algunas funciones dependen de las características de la partida y de *Space Age*.

Aunque se añaden opciones, los fundamentos siguen siendo los mismos: las redes rojas y verdes son independientes, las señales se suman dentro de una red, y los combinadores procesan señales según una configuración específica.

---

#### 9. Lista de comprobación para una red de circuitos

Antes de considerar terminado un circuito, verifica lo siguiente:

- [ ] Las entidades están conectadas con el color de cable correcto.
- [ ] Las redes roja y verde permanecen separadas cuando se necesita conservar señales independientes.
- [ ] Los dispositivos emisores tienen activada la opción de lectura necesaria.
- [ ] Los receptores utilizan la señal y el umbral correctos.
- [ ] Las cantidades repetidas no se suman accidentalmente desde varios emisores.
- [ ] Los combinadores tienen entrada, operación y salida configuradas correctamente.
- [ ] Las condiciones de los decisores utilizan AND/OR según la lógica prevista.
- [ ] No existen bucles de realimentación no deseados.
- [ ] Se contempla la latencia en circuitos de temporización o memoria.
- [ ] Los dispositivos que requieren electricidad reciben suministro.
- [ ] Los planos incluyen las conexiones de circuito y las descripciones útiles.
- [ ] El diseño se probó con señales sencillas antes de conectarse al sistema de producción completo.

---

#### 10. Fuentes de referencia

Para consultar las estadísticas, las opciones actuales y ejemplos adicionales, utiliza estas páginas de la *Factorio Wiki* y la documentación oficial del juego:

- [Circuit network — Red de circuitos](https://wiki.factorio.com/Circuit_network)
- [Red wire — Cable rojo](https://wiki.factorio.com/Red_wire)
- [Green wire — Cable verde](https://wiki.factorio.com/Green_wire)
- [Combinators — Combinadores](https://wiki.factorio.com/Combinators)
- [Constant combinator — Combinador constante](https://wiki.factorio.com/Constant_combinator)
- [Arithmetic combinator — Combinador aritmético](https://wiki.factorio.com/Arithmetic_combinator)
- [Decider combinator — Combinador decisor](https://wiki.factorio.com/Decider_combinator)
- [Selector combinator — Combinador selector](https://wiki.factorio.com/Selector_combinator)
- [Circuit network cookbook — Ejemplos prácticos](https://wiki.factorio.com/Circuit-network_Cookbook)
- [Friday Facts #384 — Combinators 2.0, blog oficial de Factorio](https://factorio.com/blog/post/fff-384)


---

### Guía técnica de Factorio: Red de trenes, estaciones y semáforos

**Versión de referencia:** *Factorio 2.x*. Las características exclusivas de *Space Age*, especialmente las vías elevadas, están identificadas por separado. Las estadísticas indicadas corresponden a objetos de calidad normal y pueden variar si se utilizan mods o configuraciones distintas.

Los trenes permiten transportar grandes cantidades de materiales entre zonas alejadas de la fábrica. Una red ferroviaria bien diseñada conecta minas, fundiciones, refinerías, centros de fabricación y depósitos sin obligar a construir cintas transportadoras durante distancias enormes.

El sistema se compone de vías, locomotoras, vagones, estaciones y señales ferroviarias. Las vías definen por dónde circulan los trenes; las locomotoras los mueven; los vagones transportan objetos o fluidos; las estaciones establecen destinos; y los semáforos coordinan el tráfico para evitar que varios trenes ocupen el mismo tramo de vía al mismo tiempo.

---

#### 1. Fundamentos de la red ferroviaria

##### 1.1. ¿Qué es una red de trenes?

Una **red ferroviaria** es un conjunto conectado de vías, desvíos, estaciones y señales por el que circulan uno o más trenes. Puede utilizarse manualmente o automatizarse mediante los horarios de las locomotoras.

La red ferroviaria se vuelve especialmente útil cuando las distancias son grandes o se necesita mover un volumen importante de recursos. Un tren con varios vagones puede transportar mucha más carga por viaje que una sola cinta transportadora, aunque requiere una infraestructura inicial más costosa y una planificación cuidadosa de las rutas.

Las aplicaciones más comunes son:

- Transportar mineral desde yacimientos remotos hasta las fundiciones.
- Llevar placas de hierro, cobre y acero a las zonas de fabricación.
- Mover petróleo y otros fluidos mediante vagones cisterna.
- Distribuir carbón, piedra y otros materiales a múltiples fábricas.
- Crear una red de estaciones de carga y descarga reutilizable por muchos trenes.
- Conectar regiones separadas del mapa sin tener que construir una cinta continua entre ellas.

Consulta la [Factorio Wiki: Railway](https://wiki.factorio.com/Railway) o su [versión en español](https://wiki.factorio.com/Railway/es).

##### 1.2. Los componentes básicos

Un tren automatizado necesita varios elementos que cumplen funciones diferentes:

1. **Vías:** permiten que el tren se desplace por una ruta conectada.
2. **Locomotora:** aporta el movimiento y necesita combustible.
3. **Vagones:** transportan la carga; pueden ser de objetos o de fluidos.
4. **Paradas de tren:** ofrecen destinos con nombre para los horarios automáticos.
5. **Señales ferroviarias:** dividen la red en bloques y regulan el acceso a ellos.
6. **Sistemas de carga y descarga:** insertores para objetos y bombas para fluidos.
7. **Horario del tren:** define las paradas y las condiciones que determinan cuándo partir.

Una vía y una locomotora bastan para conducir manualmente un tren. Para automatizar el transporte se necesita una ruta conectada, destinos válidos, combustible y un horario configurado correctamente.

##### 1.3. Progresión e importancia en la fábrica

Al principio, las cintas transportadoras suelen ser suficientes para unir máquinas cercanas. A medida que se explotan nuevos yacimientos y se amplía la producción, las distancias entre las minas y la fábrica aumentan. En ese punto, el ferrocarril permite transportar recursos en grandes cantidades y centralizar el procesamiento.

La infraestructura ferroviaria también ayuda a separar la producción en módulos. Por ejemplo, una mina puede cargar mineral en una estación; varios trenes pueden llevarlo a una zona de fundición; y desde allí otros trenes pueden distribuir placas a las fábricas consumidoras.

El beneficio depende de una condición clave: la red tiene que mover la carga sin atascarse. Una línea larga con estaciones mal ubicadas, rutas sin señales o trenes que no puedan salir de los cruces puede convertirse en un cuello de botella.

##### 1.4. Cómo circula un tren automatizado

El proceso general es el siguiente:

1. La locomotora recibe combustible y el tren se forma conectando una locomotora y los vagones necesarios.
2. Se construyen vías continuas que unan los destinos previstos.
3. Se colocan las paradas de tren en las ubicaciones donde el tren debe cargar o descargar.
4. Se configuran las señales para que el tráfico pueda compartir la infraestructura sin conflictos.
5. Se crea un horario desde la interfaz de la locomotora, con las estaciones en el orden deseado.
6. Se agregan condiciones de espera para que el tren no salga hasta estar cargado, descargado o haber cumplido el tiempo necesario.
7. Se cambia el tren al modo automático y se comprueba que complete el recorrido.

Ejemplo de transporte de mineral:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Carga de mineral | Yacimiento de hierro | Estación de carga | Tren listo para transportar mineral | — |
| Transporte ferroviario | Mineral cargado en el tren | Vía principal y señales | Tren en ruta | — |
| Descarga | Tren con mineral | Estación de descarga | Mineral descargado | — |
| Procesamiento | Mineral descargado | Fundición | Material procesado | — |

---

#### 2. Vías y planificación de rutas

##### 2.1. Vía ferroviaria (*Rail*)


Las **vías** constituyen la infraestructura física por la que circulan los trenes. Se construyen usando el planificador ferroviario (*rail planner*) o colocando piezas de vía manualmente.

Aspectos importantes:

- Las vías pueden ser rectas, curvas y diagonales, según la geometría de la red.
- El planificador facilita la construcción de trazados largos y la conexión con tramos existentes.
- Las vías se ajustan a una cuadrícula ferroviaria; no siempre pueden desplazarse una casilla sin rehacer la geometría.
- Las curvas necesitan espacio suficiente para respetar el radio de giro de los trenes.
- Los desvíos deben conectar correctamente los segmentos que el tren necesita recorrer.
- Las señales, estaciones y algunos elementos de control se colocan junto a segmentos de vía compatibles.

Un tren no puede atravesar una interrupción de vía, aunque en el mapa los segmentos parezcan estar muy cerca. Ante una ruta que no funciona, comprueba primero la continuidad física y después el señalizado.

Fuente: [Factorio Wiki: Rail](https://wiki.factorio.com/Straight_rail).

##### 2.2. Vía simple y vía doble

Hay dos diseños habituales para las líneas de transporte.

**Vía simple:** ambos sentidos utilizan el mismo tramo. Es económica en materiales, pero exige organizar el tráfico con cuidado. Si dos trenes avanzan en sentidos opuestos y no existe un apartadero adecuado, pueden quedar esperando uno frente al otro.

**Vía doble:** una vía se reserva para cada sentido, normalmente con un trazado de ida y otro de vuelta. Consume más materiales y espacio, pero simplifica la señalización y suele ofrecer mayor capacidad cuando circulan muchos trenes.

En redes pequeñas puede ser conveniente empezar con una vía simple y apartaderos. Para redes con tráfico frecuente, una vía doble suele resultar más fácil de ampliar y depurar.

##### 2.3. Desvíos e intersecciones

Los desvíos permiten que el tren elija entre distintas rutas. Las intersecciones conectan vías y pueden convertirse en los puntos más congestionados de una red.

Recomendaciones:

- Mantén las intersecciones compactas, pero deja espacio suficiente para que el tren complete el giro.
- Evita situar una estación justo dentro de una intersección.
- No permitas que un tren tenga que detenerse sobre las vías que cruzan otras rutas.
- Separa las vías de entrada y salida de las estaciones cuando el volumen de tráfico lo justifique.
- Comprueba las rutas desde la perspectiva de cada sentido de circulación.

Cruzar dos líneas rectas visualmente no siempre crea una conexión funcional. Para que exista un desvío, la geometría de las vías debe formar un empalme que los trenes puedan recorrer.

##### 2.4. Vías elevadas (*Space Age*)


La expansión *Space Age* incorpora vías elevadas, que permiten pasar por encima de instalaciones, obstáculos y cuerpos de agua compatibles. Se construyen con rampas ferroviarias y soportes.

- **Rampas ferroviarias (*Rail ramps*):** permiten que la vía ascienda o descienda entre el nivel del suelo y el elevado.
- **Soportes ferroviarios (*Rail supports*):** sostienen los tramos elevados en los puntos requeridos.
- **Vías elevadas:** forman un segundo nivel de circulación, útil para separar rutas o cruzar zonas donde una vía terrestre sería difícil.


En la versión actual de *Space Age*, los soportes se requieren a intervalos regulares; el planificador ferroviario puede colocarlos automáticamente al construir el trazado. Se pueden colocar señales sobre tramos elevados compatibles, pero **las paradas de tren no se pueden colocar sobre rampas** y deben ubicarse en tramos apropiados.

Las vías elevadas no son necesarias para una red estándar del juego base. Son una ampliación útil cuando las intersecciones a nivel ocupan demasiado espacio o el terreno dificulta construir una ruta continua.

Más información: [Rail ramp](https://wiki.factorio.com/Rail_ramp), [Rail support](https://wiki.factorio.com/Rail_support) y [Space Age](https://wiki.factorio.com/Space_Age).

---

#### 3. Locomotoras y vagones

##### 3.1. Locomotora (*Locomotive*)


La **locomotora** es el vehículo que mueve el tren por las vías. Necesita combustible y puede conducirse manualmente o configurarse para circular automáticamente entre estaciones.

Características básicas de calidad normal:

- Tamaño: 2 × 6 casillas.
- Inventario interno para combustible: 3 ranuras.
- Consume combustible compatible, como carbón, combustible sólido, combustible para cohetes o combustible nuclear.
- Los combustibles de mayor calidad energética pueden proporcionar mejoras a la aceleración y a la velocidad máxima.
- Es posible conectar más de una locomotora a un mismo tren.

La orientación importa. Un tren automático normalmente se desplaza hacia delante con respecto a la locomotora que encabeza la marcha. Para que un tren pueda hacer recorridos automáticos en ambos sentidos por una vía terminal, suele colocarse una locomotora en cada extremo, orientadas hacia fuera.

La locomotora puede recibir combustible mediante insertores cuando está estacionada en una parada adecuada. Conviene incluir una estación de repostaje en redes largas, especialmente si los trenes recorren muchas rutas y no siempre vuelven a la base.

Fuente: [Factorio Wiki: Locomotive](https://wiki.factorio.com/Locomotive).

##### 3.2. Vagón de carga (*Cargo wagon*)


El **vagón de carga** transporta objetos sólidos. Se conecta a una locomotora o a otros vagones para formar un tren.

Características principales:

- Tamaño: 2 × 6 casillas.
- Capacidad básica: 40 ranuras de inventario en calidad normal.
- Puede cargarse y descargarse con insertores colocados a los lados de la vía.
- Permite limitar su inventario para que solo acepte los objetos previstos.
- Los artículos transportados dependen del tamaño de pila de cada objeto.

En una estación bien preparada, los insertores pueden transferir objetos directamente entre cofres, cintas transportadoras y el vagón. El diseño de carga y descarga debe respetar la posición de los vagones cuando el tren se detiene, porque los insertores solo alcanzan las ranuras que quedan dentro de su radio.

Un vagón de carga es especialmente útil para minerales, placas, carbón, circuitos, materiales de construcción y cualquier otro objeto sólido que necesite transporte de gran volumen.

Fuente: [Factorio Wiki: Cargo wagon](https://wiki.factorio.com/Cargo_wagon).

##### 3.3. Vagón de fluidos (*Fluid wagon*)


El **vagón de fluidos** transporta líquidos y gases compatibles con el sistema de fluidos del juego. Se utiliza principalmente para petróleo crudo, derivados del petróleo, agua y otros fluidos industriales.

Características básicas de calidad normal:

- Tamaño: 2 × 6 casillas.
- Capacidad: 50.000 unidades de fluido.
- Puede utilizar hasta tres bombas conectadas a un solo vagón, una por cada sección de su depósito.
- Para cargarlo y descargarlo, el vagón debe quedar alineado correctamente sobre una vía recta en la estación.
- Las bombas deben conectarse a las tuberías que transportan el fluido correspondiente.

Los fluidos no se cargan con insertores. Una estación de fluidos necesita bombas y tuberías, y debe evitar conexiones que mezclen productos incompatibles.

Fuente: [Factorio Wiki: Fluid wagon](https://wiki.factorio.com/Fluid_wagon).

##### 3.4. Cómo se conectan los vehículos

Un tren debe tener como mínimo una locomotora. Para mover carga, se añaden uno o más vagones. En las posiciones válidas, los vehículos se acoplan automáticamente cuando se colocan correctamente sobre la vía; también es posible conectar vehículos existentes mediante los controles de conducción del juego.

La longitud del tren afecta a:

- El espacio necesario en las estaciones.
- La distancia que debe quedar libre después de una señal.
- El tamaño de los apartaderos.
- La capacidad transportada por viaje.
- La aceleración y el frenado.
- La cantidad de insertores y bombas que conviene instalar en las estaciones.

Los trenes más largos transportan más carga por viaje, pero necesitan más espacio para detenerse y pueden bloquear intersecciones si las señales se colocan demasiado cerca.

---

#### 4. Estaciones y paradas de tren

##### 4.1. Parada de tren (*Train stop*)


La **parada de tren** crea un destino con nombre que puede incluirse en los horarios de las locomotoras. Las paradas se utilizan para organizar los lugares de carga, descarga, repostaje y espera.

Características y comportamiento:

- Tamaño: 2 × 2 casillas.
- Puede tener un nombre personalizado.
- Debe colocarse en el lado correcto de la vía para la dirección de llegada prevista.
- Desde la interfaz de la parada se puede consultar qué trenes se dirigen a ella o están asociados a ella.
- Permite establecer un límite de trenes para controlar cuántos pueden reservarla como destino.
- Puede habilitarse o deshabilitarse mediante la red de circuitos, si se desea una logística dinámica.

En una vía de sentido único, la parada se coloca a la derecha del tren en el sentido de llegada. La orientación de la parada puede comprobarse al pasar el cursor sobre ella y observando la dirección indicada por la interfaz.

Fuente: [Factorio Wiki: Train stop](https://wiki.factorio.com/Train_stop).

##### 4.2. Estaciones de carga y descarga

Una parada define dónde se detiene el tren, pero la carga y descarga necesitan estructuras adicionales.

**Para objetos sólidos:**

- Insertores que transfieren materiales entre el vagón y las cintas o cofres.
- Cintas transportadoras para desplazar la carga hacia la fábrica.
- Cofres de reserva para desacoplar el ritmo de los insertores del ritmo de consumo de las máquinas.
- Espacio suficiente para que los insertores puedan alcanzar el vagón cuando el tren está detenido.

Un vagón de carga puede atenderse con hasta doce insertores adyacentes en la configuración normal: seis por lado, siempre que el diseño permita colocarlos. El número apropiado depende del volumen de carga, la velocidad de los insertores y el tiempo que el tren permanece en la estación.

**Para fluidos:**

- Bombas alineadas con los depósitos del vagón.
- Tuberías conectadas a la red de fluidos.
- Depósitos opcionales para almacenar el producto antes o después del transporte.
- Espacio recto suficiente para que el vagón quede bien alineado con las bombas.

Una estación debe tener vías de entrada y salida que permitan al tren detenerse sin bloquear la ruta principal. Para estaciones con mucho tráfico, conviene crear un tramo de acceso y un tramo de salida separados de la línea principal.

##### 4.3. Nombres de estación y destinos compartidos

Los trenes usan los nombres de las paradas para buscar destinos. Si varias paradas tienen el mismo nombre, un tren puede seleccionar una parada disponible según el cálculo de rutas y penalizaciones del sistema, teniendo en cuenta los trenes presentes y otros factores.

Esto permite diseñar sistemas con estaciones intercambiables, por ejemplo, varias paradas llamadas `Mineral de hierro - Carga`. Sin embargo, no significa que los trenes siempre elijan la parada que está físicamente más cerca en el mapa.

Buenas prácticas para los nombres:

- Usa un patrón coherente, como `Hierro - Carga`, `Hierro - Descarga`, `Petróleo - Carga` y `Petróleo - Descarga`.
- Distingue claramente las estaciones que cargan de las que descargan.
- Si varias estaciones comparten nombre, asegúrate de que todas ofrecen el recurso esperado.
- Evita duplicar nombres para estaciones que cumplen funciones diferentes, salvo que el sistema esté diseñado deliberadamente para ello.
- Utiliza iconos en los nombres si facilitan la lectura de la red.

##### 4.4. Límites de trenes y estaciones activas

El **límite de trenes** de una parada controla cuántos trenes pueden reservarla como destino. No es lo mismo que el número de trenes físicamente presentes en el área: evita que demasiados trenes se dirijan a una estación que no tiene espacio para recibirlos.

Por ejemplo, una estación con un solo andén de carga suele funcionar bien con un límite de 1. Si tiene dos andenes compatibles y puede atender dos trenes a la vez, puede establecerse un límite de 2.

La red de circuitos también puede utilizarse para modificar límites o habilitar y deshabilitar estaciones en función de inventarios y condiciones. Esto ayuda a enviar trenes solamente a lugares que realmente necesitan una entrega o que tienen capacidad para recibirla.

Ten presente que reducir un límite por debajo de las reservas ya existentes no siempre redirige inmediatamente todos los trenes que ya están en camino.

---

#### 5. Señales ferroviarias y bloques

##### 5.1. ¿Qué es un bloque ferroviario?

Las señales dividen la red de vías en **bloques**. Un bloque es una sección conectada de vía delimitada por señales, cruces u otros límites relevantes del trazado. El sistema evita que más de un tren ocupe el mismo bloque al mismo tiempo en circulación automática.

Un tren largo puede ocupar varios bloques simultáneamente. Por eso no basta con que la locomotora haya pasado una señal: todo el tren debe haber despejado el bloque para que otros trenes puedan utilizarlo.

Los colores de los bloques que se muestran al usar una señal ayudan a detectar qué partes de la vía pertenecen a la misma sección de tráfico.

##### 5.2. Señal ferroviaria normal (*Rail signal*)


La **señal ferroviaria normal** controla la entrada al bloque siguiente. Se utiliza para dividir una vía en secciones y permitir que varios trenes circulen por una misma línea sin entrar en el mismo bloque a la vez.

Estados generales:

- **Verde:** el bloque protegido está libre para reservarse.
- **Amarillo:** el tren ha reservado el bloque y se encuentra comprometido con esa entrada; puede no poder detenerse antes de la señal.
- **Rojo:** el bloque no está disponible, normalmente porque está ocupado o reservado por otro tren.
- **Parpadeante:** la señal no está colocada en una situación válida o no puede separar correctamente los bloques.

Las señales se colocan a la derecha de la vía respecto al sentido de marcha previsto. Si una vía debe permitir circulación automática en ambos sentidos, normalmente se colocan señales opuestas en ambos lados de los puntos que controlan ese tramo.

Fuente: [Factorio Wiki: Rail signal](https://wiki.factorio.com/Rail_signal).

##### 5.3. Señal ferroviaria en cadena (*Rail chain signal*)


La **señal en cadena** mira más allá del bloque siguiente y comprueba las señales de salida de la ruta que el tren pretende seguir. De este modo puede impedir que un tren entre en un cruce si no tiene espacio para salir de él.

Se utiliza especialmente:

- Antes de las intersecciones.
- En entradas a cruces donde un tren podría bloquear otras rutas.
- Antes de tramos donde el tren no debe detenerse, como un cruce estrecho.
- En apartaderos o segmentos compartidos donde conviene garantizar que el tren pueda continuar hasta una salida libre.

La regla inicial más útil es: **señal en cadena antes del cruce y señal normal después del cruce**. La señal en cadena evita que un tren entre sin tener salida; la señal normal después del cruce marca el punto a partir del cual otro tren puede reservar el siguiente bloque.

La señal en cadena puede mostrar distintos estados, incluidos azul cuando algunas salidas están disponibles y otras bloqueadas. El comportamiento depende de la ruta que cada tren intenta recorrer.

Fuente: [Factorio Wiki: Rail chain signal](https://wiki.factorio.com/Rail_chain_signal).

##### 5.4. Cómo señalizar una intersección

Un diseño de referencia es:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada al cruce | Tren que se aproxima al cruce | Señal en cadena | Permiso de entrada condicionado a la salida disponible | La señal en cadena se sitúa antes del cruce en el esquema. |
| Control de salida | Tren dentro de la zona del cruce | Señales ferroviarias normales | Bloques de salida controlados | El esquema muestra señales normales después del cruce. |
| Continuación del recorrido | Tren que supera el cruce | Vía de salida | Tren encaminado a la salida | — |

El esquema es conceptual: una intersección real puede tener varias entradas y salidas, por lo que cada ruta debe señalizarse de acuerdo con sus posibles movimientos. En general, se coloca una señal en cadena en cada entrada del área de conflicto y señales normales en las salidas, después de que el tren haya despejado el cruce.

No coloques señales normales dentro de una intersección si eso permite que un tren se detenga en medio y bloquee el tráfico transversal. Si el tren necesita esperar, es preferible que se detenga antes de entrar.

##### 5.5. Señalización de vías de doble sentido

En una vía bidireccional compartida por trenes automáticos, las señales deben permitir el acceso desde ambos sentidos, y los apartaderos deben tener longitud suficiente para que un tren pueda abandonar la vía principal.

Los errores frecuentes en estas redes incluyen:

- Señalizar un lado de la vía y olvidar el sentido contrario.
- Crear bloques tan largos que un tren ocupa una gran parte de la ruta mientras espera.
- Colocar señales de manera que dos trenes entren en el mismo tramo compartido desde extremos opuestos.
- Diseñar un apartadero más corto que el tren que debe detenerse en él.

Para redes con mucho tráfico, una vía por sentido suele ser más sencilla de mantener. Si se utiliza una vía simple, deben comprobarse de manera especial los apartaderos, las señales en cadena y los posibles bloqueos mutuos.

##### 5.6. Prevenir bloqueos mutuos (*deadlocks*)

Un **bloqueo mutuo** ocurre cuando un conjunto de trenes se impide mutuamente avanzar: cada uno espera que se libere un tramo ocupado por otro tren, pero ninguno puede moverse para liberarlo.

Para reducir el riesgo:

- Evita que los trenes esperen dentro de intersecciones.
- Deja espacio suficiente entre señales y estaciones para alojar el tren completo.
- Usa señales en cadena en las entradas a cruces.
- Procura que los trenes puedan abandonar la estación sin bloquear la línea principal.
- No permitas que un tren se detenga sobre un desvío importante.
- Inspecciona los apartaderos de las vías simples y verifica que el tren pueda entrar y salir sin bloquear al contrario.

Las señales no corrigen una geometría de vía defectuosa. Una red sin rutas conectadas o con estaciones orientadas incorrectamente seguirá fallando aunque tenga muchas señales.

---

#### 6. Horarios y automatización de trenes

##### 6.1. Crear un horario

El horario se configura desde la interfaz de una locomotora del tren. Se añaden las paradas que debe visitar y se especifica la condición de espera que debe cumplirse antes de salir de cada una.

Por ejemplo, un tren de mineral puede tener este horario:

1. `Hierro - Carga`: esperar hasta que la carga esté llena.
2. `Hierro - Descarga`: esperar hasta que la carga esté vacía.
3. Repetir el ciclo.

Al activar el modo automático, el tren recorre las paradas en el orden definido y vuelve al inicio del horario al completar la lista.

##### 6.2. Condiciones de espera habituales

Las condiciones permiten adaptar el funcionamiento del tren a las necesidades de la fábrica.

| Condición | Uso típico |
|---|---|
| **Carga llena** | El tren espera hasta que los inventarios de carga estén llenos. No incluye el combustible de las locomotoras. |
| **Carga vacía** | El tren no parte hasta que todos los inventarios de carga estén vacíos. |
| **Tiempo transcurrido** | Mantiene el tren en la estación durante un tiempo mínimo o fijo. |
| **Inactividad** | Espera hasta que deja de haber transferencia de objetos durante el intervalo indicado. |
| **Cantidad de un objeto** | Espera a que la carga alcance una cantidad determinada de un producto. |
| **Cantidad de fluido** | Comprueba la cantidad total de un fluido transportado en los vagones cisterna. |
| **Condición de circuito** | Utiliza una señal de la red de circuitos conectada a la parada. |
| **Combustible** | Comprueba las reservas de combustible de una o varias locomotoras. |
| **Carga presente** | Comprueba que el tren lleva al menos algún objeto o fluido. |

Las condiciones pueden combinarse mediante operadores lógicos **Y** y **O**. Por ejemplo, puede hacerse que el tren salga cuando la carga esté llena **o** haya transcurrido un tiempo máximo. Esta combinación evita que el tren quede esperando indefinidamente cuando la producción es lenta.

##### 6.3. Trenes de doble sentido

Un tren con una sola locomotora puede dar marcha atrás de forma manual, pero la circulación automática se diseña normalmente para avanzar en la dirección de la locomotora que encabeza la formación. Para una ruta de ida y vuelta por la misma vía, se puede colocar una locomotora en cada extremo, orientadas en sentidos opuestos.

En una red de doble vía, un patrón de circulación claro suele simplificar mucho la señalización. Antes de añadir más trenes, comprueba que todos pueden orientarse y llegar a las paradas previstas.

##### 6.4. Interrupciones de horario y grupos (*Factorio 2.x*)

En *Factorio 2.x*, los horarios permiten usar **interrupciones** para insertar paradas temporales cuando se cumple una condición determinada, por ejemplo, cuando el destino está lleno o no existe una ruta válida. También se pueden asignar trenes a grupos para mantener horarios compartidos.

Estas herramientas son útiles en redes con muchos trenes, pero no son necesarias para una instalación inicial. Primero conviene conseguir que un solo tren cumpla su horario correctamente; después se pueden añadir reglas más avanzadas.

##### 6.5. Ejemplo: tren de placas de hierro

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Estación de carga: Hierro - Carga | Placas de hierro disponibles | Insertores y vagón | Vagón cargado | Condición de espera del esquema: carga llena O 30 segundos transcurridos. |
| Estación de descarga: Hierro - Descarga | Placas de hierro en el vagón | Insertores y estación de descarga | Placas descargadas hacia la fábrica | Condición de espera del esquema: carga vacía. |
| Repetición del horario | Tren con horario configurado | Sistema de horarios ferroviarios | Servicio automatizado repetido | El tren repite el horario automáticamente. |

El tiempo de espera máximo es útil en una estación que no siempre recibe suficiente mineral. Sin él, un tren podría quedarse en la estación más tiempo del deseado a la espera de llenar por completo sus vagones.

---

#### 7. Objetos e insumos necesarios

##### 7.1. Materiales para construir la infraestructura

Las recetas exactas pueden consultarse en la ficha de cada objeto, pero una red ferroviaria requiere principalmente los siguientes materiales y productos intermedios.

| Objeto o recurso | Para qué se utiliza |
|---|---|
| **Placas de hierro** | Forman parte de la fabricación de vías, insertores y distintos edificios ferroviarios. |
| **Placas de acero** | Se usan en locomotoras, vagones y estructuras avanzadas. |
| **Piedra** | Ingrediente para fabricar vías. |
| **Varillas de hierro** | Se emplean en recetas de vías y señales. |
| **Engranajes de hierro** | Forman parte de componentes mecánicos, incluidas algunas piezas ferroviarias. |
| **Circuitos electrónicos** | Se usan en señales y diversos elementos de automatización. |
| **Motores** | Ingrediente de locomotoras y otros dispositivos mecánicos. |
| **Insertores** | Cargan y descargan objetos de los vagones. |
| **Bombas** | Transfieren fluidos entre tuberías y vagones cisterna. |
| **Combustible** | Alimenta las locomotoras. |
| **Postes eléctricos** | Pueden alimentar insertores eléctricos y otros equipos de las estaciones. |
| **Cintas transportadoras** | Llevan objetos desde la estación hasta la fábrica o la zona de carga. |
| **Cofres** | Sirven como buffers para objetos sólidos. |
| **Tuberías y depósitos** | Conducen y almacenan fluidos en las estaciones correspondientes. |

Para construir una red inicial, prioriza las vías, una locomotora, al menos un vagón, las paradas necesarias y una cantidad suficiente de combustible. Las señales pasan a ser esenciales cuando circulan varios trenes por vías compartidas o intersecciones.

##### 7.2. Recursos transportados

| Tipo de carga | Vagón recomendado | Aplicaciones frecuentes |
|---|---|---|
| Mineral de hierro | Vagón de carga | Alimentar fundiciones alejadas de la mina. |
| Mineral de cobre | Vagón de carga | Suministrar cobre a fundiciones y líneas electrónicas. |
| Carbón | Vagón de carga | Combustible, industria química y generación de vapor. |
| Piedra y ladrillos | Vagón de carga | Construcción y producción de materiales. |
| Placas de hierro, cobre y acero | Vagón de carga | Suministrar productos intermedios a varias zonas de fabricación. |
| Circuitos y componentes | Vagón de carga | Mover productos de alto valor o de producción centralizada. |
| Petróleo crudo | Vagón de fluidos | Conectar campos petrolíferos con refinerías. |
| Petróleo ligero, pesado y gas de petróleo | Vagón de fluidos | Llevar productos refinados a procesos químicos. |
| Agua y otros fluidos industriales | Vagón de fluidos | Transportar fluidos cuando la red de tuberías no es práctica. |

El tren no transforma los recursos por sí mismo. Transporta la carga entre instalaciones, donde se extrae, procesa, almacena o consume.

##### 7.3. Insumos para poner el tren en funcionamiento

Antes de probar una línea, revisa que estén disponibles:

- Combustible en la locomotora.
- Un recorrido continuo entre todas las estaciones del horario.
- Estaciones colocadas y orientadas correctamente.
- Vagones que coincidan con el tipo de carga.
- Insertores y cintas en las estaciones de carga y descarga de objetos.
- Bombas y tuberías para las estaciones de fluidos.
- Señales suficientes para los bloques y cruces de la red.
- Un horario que incluya los destinos correctos y condiciones de espera razonables.

---

#### 8. Diseños ferroviarios recomendados

##### 8.1. Línea simple de dos estaciones

Una red inicial puede conectar una mina y una fábrica con una vía directa y un tren. Si no hay más tráfico, se puede empezar probando la conducción manual y después configurar las paradas automáticas.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Carga en la mina | Material disponible en la mina | Estación de carga | Tren cargado | — |
| Trayecto por vía única | Tren cargado | Una vía ferroviaria | Tren transportado a la fábrica | La ruta conecta la mina y la fábrica. |
| Descarga en la fábrica | Material en el tren | Estación de descarga | Material disponible en la fábrica | — |

Con más de un tren o con tráfico en ambos sentidos, deben añadirse señales y, si se comparte la vía, un método para que los trenes no entren al mismo tramo desde direcciones opuestas. En una línea de un solo tren es más fácil diagnosticar las primeras dificultades.

##### 8.2. Red de doble vía

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Trayecto de ida | Tren en estación de origen | Vía de ida | Tren en dirección al destino | — |
| Trayecto de vuelta | Tren en estación de destino | Vía de vuelta | Tren hacia el origen | — |
| Cruce de sentidos | Trenes en ambos sentidos | Apartaderos | Tráfico ferroviario distribuido | El esquema representa apartaderos a lo largo de las vías. |

Una vía por sentido reduce las esperas entre trenes opuestos. Se necesitan señales en los bloques para permitir que varios trenes utilicen cada vía manteniendo una distancia segura entre ellos.

En tramos largos, se pueden añadir señales normales a intervalos adecuados. La separación debe considerar la longitud de los trenes y el espacio que necesitan para despejar por completo cada bloque.

##### 8.3. Estación con apartadero

La estación debe situarse fuera de la vía principal cuando sea posible. Así, un tren parado para cargar o descargar no bloquea el paso de los demás.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Circulación por la vía principal | Tren en la vía principal | Vía principal y señalización | Tren continúa por la línea principal | — |
| Acceso a la estación | Tren que necesita detenerse en la estación | Desvío y vía de estación | Tren encaminado a la estación | El esquema muestra una rama que sale de la vía principal. |

En una red con mucho tráfico, se puede diseñar una entrada y una salida dedicadas. Para un solo tren, la solución puede ser más sencilla; para varias líneas, conviene estandarizar el diseño de las estaciones para que los trenes compartan los mismos patrones de circulación.

##### 8.4. Estación de carga de objetos

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Reserva de carga | Yacimiento o producción | Cinta transportadora o cofres de reserva | Material disponible para cargar | — |
| Carga del vagón | Material de reserva | Insertores | Material transferido al vagón de carga detenido | — |
| Tracción | Vagón de carga detenido | Locomotora | Tren listo para transporte | — |

Los insertores deben colocarse en posiciones en las que puedan alcanzar el vagón cuando esté detenido en la parada. Comprueba los límites del vagón y el número de insertores para evitar que algunos queden fuera de alcance.

##### 8.5. Estación de fluidos

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Extracción y conducción | Campo de petróleo | Tuberías y bombas | Petróleo transportado hacia la estación | — |
| Carga de fluido | Petróleo conducido por tuberías | Bombas y vagón de fluidos | Vagón de fluidos cargado | — |
| Descarga y transporte final | Fluido en el vagón | Estación, bombas y tuberías | Fluido suministrado a refinería o depósitos | — |

El vagón debe quedar alineado sobre un tramo recto de vía para que las bombas puedan transferir el fluido. Si no hay transferencia, revisa la alineación, el sentido de las bombas, la conexión de las tuberías y que el tren haya parado en el punto correcto.

---

#### 9. Estrategias de optimización

##### 9.1. Establecer un patrón de nombres

Un patrón de nombres coherente reduce los errores al crear horarios. Utiliza nombres que identifiquen el recurso y la función de la estación, por ejemplo, `Cobre - Carga` y `Cobre - Descarga`.

Si se duplican nombres para estaciones equivalentes, comprueba que todos los destinos sean realmente compatibles. Una parada que se habilita sin tener material disponible puede atraer trenes innecesarios y generar esperas.

##### 9.2. Mantener estaciones fuera de la línea principal

Siempre que el terreno lo permita, construye las paradas en apartaderos o ramales. Esto deja libre la línea principal mientras un tren carga o descarga y reduce las colas que se forman detrás de una estación ocupada.

En rutas de alto tráfico, considera separar la zona de espera, el acceso a la estación, la zona de carga y la salida. Así, un tren puede esperar fuera del cruce principal sin impedir el paso a los demás.

##### 9.3. Ajustar la longitud de los trenes

Los trenes más largos reducen la cantidad de viajes necesaria para transportar el mismo volumen, pero sus estaciones requieren más espacio y sus movimientos afectan a más bloques.

Para escoger una longitud adecuada, considera:

- Cuánta producción genera la estación por minuto.
- Cuánto consume la fábrica receptora.
- El tiempo de viaje entre ambos destinos.
- La capacidad de los insertores o bombas.
- La longitud disponible en las estaciones y apartaderos.
- La cantidad de trenes que utilizarán el mismo corredor.

No es necesario que todos los trenes de la red tengan la misma longitud, pero estandarizar las formaciones facilita reutilizar estaciones y planos.

##### 9.4. Evitar estaciones saturadas

Utiliza límites de trenes en las paradas para evitar que varios trenes se dirijan a una estación que solo puede recibir uno. Si una estación tiene dos vías de carga independientes, su límite puede ajustarse de acuerdo con su capacidad real.

Cuando uses estaciones del mismo nombre, habilita solo las que puedan atender carga o descarga de forma útil. La combinación de límites y disponibilidad dinámica puede mejorar la distribución, pero se debe verificar con trenes reales porque las reservas de destino y las rutas pueden producir comportamientos distintos de los esperados.

##### 9.5. Señalizar según el tamaño del tren

Las señales demasiado cercanas pueden permitir que un tren entre en un bloque sin tener espacio para detenerse antes del siguiente obstáculo. Un bloque corto también puede quedar ocupado por un tren que todavía no ha despejado una intersección.

Diseña los bloques y apartaderos pensando en el tren más largo que circulará por la ruta. En los cruces, utiliza señales en cadena en las entradas y señales normales en las salidas, evitando paradas en zonas compartidas.

##### 9.6. Automatizar la carga y descarga

Las condiciones de carga llena y carga vacía suelen ser suficientes para los primeros trenes. Si el flujo de material es irregular, una condición de tiempo máximo o de inactividad permite evitar esperas demasiado largas.

Las condiciones de circuito son útiles cuando la estación solo debe recibir un tren si el inventario cae por debajo de un límite o cuando se quiere activar una parada en función de una necesidad real de producción.

##### 9.7. Repostaje y mantenimiento

En redes largas, los trenes necesitan una estrategia de repostaje. Puede crearse una estación dedicada o suministrar combustible en las estaciones en las que el tren ya se detiene durante suficiente tiempo.

Si se usa una parada común para repostar, asegúrate de que su posición no bloquee una vía importante. Los insertores deben tener acceso al inventario de combustible de la locomotora mientras el tren está estacionado.

---

#### 10. Problemas frecuentes y soluciones

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| El tren no encuentra la estación. | La estación está deshabilitada, el nombre no coincide o el horario apunta a un destino inexistente. | Revisa el nombre, la disponibilidad y el horario. |
| La locomotora muestra que no hay ruta. | Hay una interrupción de vía, un desvío incorrecto o una señal que impide entrar desde ese sentido. | Sigue la ruta desde el origen hasta el destino y revisa cada empalme y señal. |
| El tren no se detiene en la parada prevista. | La estación está al lado incorrecto de la vía, está mal orientada o no forma parte del horario. | Comprueba la orientación y el horario. |
| Dos trenes se bloquean entre sí. | Comparten una vía simple sin un apartadero eficaz o reservan bloques de forma incompatible. | Añade apartaderos, señales en cadena o separa los sentidos mediante doble vía. |
| Un tren se detiene dentro de un cruce. | La señalización permite entrar sin que la salida esté libre. | Coloca señales en cadena en las entradas y señales normales después de la intersección. |
| La estación principal se convierte en una cola. | Los trenes esperan en la vía principal o la estación tiene menos capacidad que la demanda. | Crea un ramal de espera, limita trenes o amplía la estación. |
| El vagón de carga no recibe objetos. | Los insertores no alcanzan el vagón, no hay materiales o el tren está en una posición inesperada. | Revisa la colocación de los insertores, la cinta de entrada y la alineación del tren. |
| El vagón de fluidos no se llena. | Las bombas no están alineadas o las tuberías no suministran el fluido. | Comprueba la vía recta, la posición del vagón, las bombas y el circuito de tuberías. |
| El tren se queda sin combustible. | No hay suministro en la ruta o la estación de repostaje no está incluida o no funciona. | Añade una estación de repostaje y automatiza el suministro. |
| La estación recibe demasiados trenes. | No hay límite de trenes o varias estaciones comparten un nombre sin gestionar su capacidad. | Ajusta el límite de trenes y las condiciones de activación. |
| Un tren ocupa varios bloques durante mucho tiempo. | La señalización tiene tramos demasiado largos o el tren está esperando en una zona crítica. | Ajusta los bloques y reubica la zona de espera fuera de los cruces. |

##### 10.1. Diagnóstico paso a paso

Cuando un tren no funciona como se espera, revisa los problemas en un orden sistemático:

1. **Combustible:** confirma que la locomotora tiene combustible.
2. **Vías:** recorre visualmente toda la ruta para encontrar segmentos desconectados.
3. **Desvíos:** comprueba que la ruta permite el giro que necesita el tren.
4. **Señales:** verifica que la entrada a cada bloque esté permitida desde el sentido correcto.
5. **Paradas:** confirma que estén activas, colocadas correctamente y con el nombre previsto.
6. **Horario:** revisa los destinos en el orden correcto y las condiciones de espera.
7. **Carga:** verifica que los insertores o bombas puedan transferir los materiales.
8. **Longitud y espacio:** comprueba que el tren quepa en la estación y que pueda despejar las intersecciones.

Este procedimiento ayuda a detectar si el fallo corresponde a la vía, la automatización, la señalización o la logística de la carga sin rehacer toda la red.

---

#### 11. Seguridad y circulación en el mapa

Los trenes pueden desplazarse a gran velocidad y causan mucho daño dentro del juego. Evita permanecer sobre las vías o cruzarlas sin comprobar si se aproxima un tren.

- Usa pasos peatonales claramente identificables cuando la fábrica tenga tráfico frecuente.
- Revisa las señales antes de cruzar una vía.
- No dejes vehículos ni edificios móviles bloqueando las vías.
- Diseña cruces que permitan observar la llegada de los trenes.
- Evita que las estaciones se sitúen en zonas por las que los personajes deban cruzar constantemente.
- En redes grandes, considera construir pasos a distinto nivel si el espacio y los recursos lo permiten.

En *Space Age*, las vías elevadas pueden separar el tráfico ferroviario del movimiento a nivel del suelo. Esto mejora la planificación de cruces y permite superar ciertos obstáculos, aunque necesita rampas y soportes adecuados.

---

#### 12. Lista de comprobación para una red ferroviaria

Antes de dar por terminada una línea, verifica los siguientes puntos:

- [ ] Las vías forman un recorrido continuo entre todas las estaciones del horario.
- [ ] La locomotora dispone de combustible y está orientada correctamente.
- [ ] Los vagones son adecuados para el tipo de carga.
- [ ] Las paradas están bien orientadas respecto al sentido de llegada.
- [ ] Los nombres de estación son coherentes y coinciden con el horario.
- [ ] Las condiciones de espera evitan que el tren parta demasiado pronto o quede detenido indefinidamente.
- [ ] Las señales están en el lado correcto de la vía.
- [ ] Las señales en cadena protegen las entradas a los cruces.
- [ ] Las señales normales están situadas después de los cruces, en zonas donde el tren pueda detenerse con seguridad.
- [ ] Los bloques y apartaderos tienen espacio para el tren más largo que utilizará la ruta.
- [ ] Las estaciones de carga tienen suficientes insertores y capacidad de transporte.
- [ ] Las estaciones de fluidos tienen bombas alineadas y tuberías conectadas correctamente.
- [ ] Los límites de trenes evitan que demasiados trenes se dirijan a una estación pequeña.
- [ ] La red tiene rutas de espera y salida que no bloquean la línea principal.
- [ ] Si se utilizan vías elevadas, las rampas y los soportes están colocados correctamente.

---

#### 13. Fuentes de referencia

Estas páginas de la *Factorio Wiki* contienen estadísticas, recetas y explicaciones detalladas de las mecánicas ferroviarias:

- [Railway — Red ferroviaria](https://wiki.factorio.com/Railway)
- [Railway en español](https://wiki.factorio.com/Railway/es)
- [Rail — Vías](https://wiki.factorio.com/Straight_rail)
- [Locomotive — Locomotora](https://wiki.factorio.com/Locomotive)
- [Cargo wagon — Vagón de carga](https://wiki.factorio.com/Cargo_wagon)
- [Fluid wagon — Vagón de fluidos](https://wiki.factorio.com/Fluid_wagon)
- [Train stop — Parada de tren](https://wiki.factorio.com/Train_stop)
- [Rail signal — Señal ferroviaria](https://wiki.factorio.com/Rail_signal)
- [Rail chain signal — Señal ferroviaria en cadena](https://wiki.factorio.com/Rail_chain_signal)
- [Railway: Train path finding — Búsqueda de rutas](https://wiki.factorio.com/Railway/Train_path_finding)
- [Rail ramp — Rampa ferroviaria de *Space Age*](https://wiki.factorio.com/Rail_ramp)
- [Rail support — Soporte ferroviario de *Space Age*](https://wiki.factorio.com/Rail_support)
- [Space Age — Expansión](https://wiki.factorio.com/Space_Age)


---

### Guía técnica de Factorio: Generación y expansión de la contaminación

**Versión de referencia:** *Factorio 2.x*, con las mecánicas de *Space Age* señaladas por separado cuando corresponde. Los valores indicados para edificios corresponden a configuraciones normales, a plena potencia y sin modificadores, salvo que se especifique lo contrario.

La **contaminación** es una simulación ambiental abstracta que representa el impacto de la actividad industrial del jugador. No es un objeto que se recoja ni se transporte: los edificios la generan mientras funcionan, se distribuye por el mapa y es absorbida gradualmente por el terreno, los árboles y ciertas estructuras de la fauna nativa.

Su importancia va más allá de lo visual. En *Nauvis*, la nube de contaminación puede provocar ataques de mordedores (*biters*) y escupidores (*spitters*), y la contaminación industrial generada también contribuye al **factor de evolución** de los enemigos. La gestión de la contaminación forma parte de la planificación de una fábrica eficiente y defendible.

#### 1. Fundamentos de la contaminación

#### 1.1. ¿Qué representa la contaminación?

La contaminación es un valor abstracto que el juego registra por zonas del mapa. Se muestra como una nube roja con bordes escalonados en el mapa y en el minimapa cuando la visualización está habilitada.

Sus efectos principales son:

- **Expansión ambiental:** la contaminación pasa desde zonas con más contaminación a zonas vecinas.
- **Absorción:** el terreno natural, los árboles y los nidos enemigos reducen parte de la contaminación presente.
- **Ataques enemigos:** los nidos absorben contaminación y la utilizan para reunir grupos de ataque que se dirigen hacia la fuente de contaminación.
- **Evolución de los enemigos:** en Nauvis, la contaminación total generada por la maquinaria del jugador contribuye a aumentar el factor de evolución, junto con el tiempo transcurrido y la destrucción de nidos.
- **Daño a los árboles:** cuando la contaminación acumulada de una zona es suficientemente alta, los árboles pueden perder hojas y adquirir un aspecto grisáceo.

La nube que se ve en el mapa no equivale a la contaminación acumulada durante toda la partida. Es la cantidad que permanece en distintas zonas después de la generación, dispersión y absorción.

#### 1.2. Cómo se representa en el mapa


Para consultar la nube dentro del juego:

1. Abre el mapa.
2. Activa la visualización de información alternativa si no se ve la nube; por defecto, se utiliza la tecla **Alt** para alternar esta vista.
3. Comprueba qué áreas industriales quedan cubiertas por la nube y en qué dirección se extiende.
4. Observa la ubicación de los nidos enemigos cercanos y los posibles corredores de aproximación.

La configuración de generación del mapa permite habilitar o deshabilitar la contaminación y ajustar parámetros relacionados con su difusión, absorción y daño a los árboles. Por eso, los valores pueden variar entre partidas personalizadas.

#### 1.3. Los *chunks* o sectores del mapa

El juego calcula la contaminación por **chunks**, sectores de **32 × 32 casillas**. Cada sector guarda su propia cantidad de contaminación y la actualiza aproximadamente cada 64 ticks del juego.

En la configuración normal, un sector comienza a difundir contaminación cuando alcanza **15 unidades**. La contaminación puede transferirse a los cuatro sectores adyacentes —norte, sur, este y oeste— mediante una tasa predeterminada del **2 % por actualización**. No se transfiere directamente en diagonal en una sola operación, aunque puede alcanzar zonas diagonales al continuar propagándose por sectores intermedios.

Este sistema explica por qué la nube no tiene un contorno circular perfectamente uniforme: los sectores tienen forma cuadrada, cada terreno absorbe una cantidad distinta y los obstáculos geográficos no detienen necesariamente la simulación abstracta de contaminación.

#### 1.4. Generación, dispersión y absorción no son lo mismo

Es importante diferenciar tres procesos:

| Proceso | Qué significa | Qué lo modifica |
|---|---|---|
| **Generación** | Contaminación que crean las máquinas al trabajar. | Tipo de edificio, uso de energía, módulos y actividad de la máquina. |
| **Dispersión** | Parte de la contaminación pasa a sectores vecinos. | Cantidad acumulada en cada sector y configuración de difusión. |
| **Absorción** | Contaminación que se elimina del mapa. | Terreno natural, árboles, nidos y modificadores de la partida. |

Reducir la nube que llega a los nidos puede disminuir la frecuencia o el tamaño de los ataques, pero **no revierte por sí solo la evolución acumulada por la contaminación que ya generó la fábrica**. La evolución utiliza la contaminación producida por la maquinaria, no la contaminación que queda después de que el terreno la absorba.

#### 2. Edificios que generan contaminación

La contaminación se genera principalmente en estructuras que extraen recursos, queman combustible, funden materiales o procesan productos. La cifra por minuto depende de la máquina y de la potencia a la que esté trabajando.

#### 2.1. Edificios contaminantes del juego base

| Edificio | Contaminación por minuto a plena potencia | Actividad asociada |
|---|---:|---|
| Horno de piedra (*Stone furnace*) | 2 | Fundición con combustible. |
| Horno de acero (*Steel furnace*) | 4 | Fundición con combustible. |
| Horno eléctrico (*Electric furnace*) | 1 | Fundición eléctrica. |
| Taladro minero a combustión (*Burner mining drill*) | 12 | Extracción de minerales mediante combustible. |
| Taladro minero eléctrico (*Electric mining drill*) | 10 | Extracción de minerales. |
| Bomba extractora (*Pumpjack*) | 10 | Extracción de petróleo crudo. |
| Máquina de ensamblaje 1 (*Assembling machine 1*) | 4 | Fabricación automática. |
| Máquina de ensamblaje 2 (*Assembling machine 2*) | 3 | Fabricación automática. |
| Máquina de ensamblaje 3 (*Assembling machine 3*) | 2 | Fabricación automática. |
| Caldera (*Boiler*) | 30 | Combustión para producir vapor. |
| Refinería de petróleo (*Oil refinery*) | 6 | Procesamiento de petróleo. |
| Planta química (*Chemical plant*) | 4 | Recetas químicas. |
| Centrífuga (*Centrifuge*) | 4 | Procesamiento de uranio. |

Los valores son tasas de referencia a plena potencia, no una garantía de que todas las máquinas emitan esa cantidad de manera permanente. Una máquina inactiva o que trabaja a una carga inferior puede contaminar menos.

#### 2.2. ¿Qué estructuras contaminan más?

Las calderas y los taladros a combustión están entre los edificios habituales que más contaminación generan por minuto. En cambio, el horno eléctrico puede emitir menos contaminación directa que los hornos de combustible, aunque consume electricidad y la generación de esa electricidad puede tener su propio impacto.

Eso no significa que toda fábrica deba sustituir automáticamente cualquier máquina por otra de menor emisión. También hay que considerar el combustible, la potencia disponible, el espacio, el rendimiento de producción y la etapa de la partida.

#### 2.3. Edificios de *Space Age*

En *Space Age* existen estructuras adicionales con sus propias tasas de contaminación. Algunos valores de referencia a plena potencia son:

| Edificio | Contaminación por minuto |
|---|---:|
| Fundición (*Foundry*) | 6 |
| Planta electromagnética (*Electromagnetic plant*) | 4 |
| Planta criogénica (*Cryogenic plant*) | 6 |
| Recicladora (*Recycler*) | 2 |
| Gran taladro minero (*Big mining drill*) | 40 |
| Torre de calentamiento (*Heating tower*) | 100 |
| Biolaboratorio (*Biolab*) | 8 |
| Trituradora (*Crusher*) | 1 |

La torre de calentamiento destaca por su elevada contaminación directa durante la operación. Para evaluar el impacto de una instalación avanzada, conviene sumar las emisiones de todas las máquinas que funcionan de forma simultánea, en lugar de fijarse en un único edificio.

#### 2.4. Módulos y consumo energético

Los módulos pueden alterar la contaminación final. Los módulos que muestran un modificador de contaminación positivo aumentan el **multiplicador de contaminación**, no una cantidad fija añadida a la tasa base. El cálculo depende de la contaminación base, del multiplicador de contaminación y del multiplicador de consumo energético.

Por eso, una máquina acelerada o configurada con módulos que aumentan el consumo puede convertirse en una fuente importante de contaminación. Por el contrario, un módulo de eficiencia puede reducir el consumo energético y, cuando la fórmula y el edificio lo permiten, reducir también la contaminación asociada.

Al comparar dos configuraciones, no basta con comparar la contaminación por edificio: conviene calcular la contaminación por unidad de producto, teniendo en cuenta velocidad, módulos y energía consumida.

#### 3. Expansión de la nube de contaminación

#### 3.1. Cómo se propaga

Cuando un sector supera el umbral de difusión, envía una parte de su contaminación a sus cuatro vecinos cardinales. Los sectores vecinos también pueden devolver contaminación al sector original si tienen valores superiores o similares, por lo que la variación neta depende de la cantidad existente en todos ellos y de la absorción local.

Ejemplo conceptual:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Dispersión hacia el norte | Contaminación del sector emisor | Propagación ambiental | Sector norte | — |
| Dispersión hacia el oeste | Contaminación del sector emisor | Propagación ambiental | Sector oeste | — |
| Dispersión hacia el este | Contaminación del sector emisor | Propagación ambiental | Sector este | — |
| Dispersión hacia el sur | Contaminación del sector emisor | Propagación ambiental | Sector sur | — |

Este esquema representa las direcciones de difusión. La cantidad que recibe cada sector no tiene por qué ser idéntica a la que pierde el sector de origen, ya que cada zona tiene su propio nivel de contaminación y absorción.

#### 3.2. ¿Qué determina el tamaño de la nube?

El tamaño visible de la nube depende de varios factores:

- La contaminación que generan las máquinas por unidad de tiempo.
- Cuánto tiempo lleva funcionando la fábrica.
- La absorción del terreno sobre el que se extiende la nube.
- La presencia y el estado de los árboles.
- La contaminación absorbida por los nidos enemigos.
- Los parámetros de contaminación elegidos al crear el mapa.

Una fábrica con una tasa de emisión moderada puede generar una nube extensa si funciona durante mucho tiempo, si está en un terreno que absorbe poco o si tiene pocas zonas naturales que amortigüen su contaminación.

#### 3.3. El terreno y su capacidad de absorción

Los sectores del mapa eliminan lentamente parte de la contaminación. La absorción depende del tipo de casillas que los componen. En la configuración por defecto de *Factorio 2.x*, el agua y ciertas superficies naturales absorben más contaminación que algunas superficies artificiales.

Algunos valores orientativos por sector completo y por minuto son:

| Tipo de terreno | Absorción aproximada por sector por minuto |
|---|---:|
| Agua y tipos de agua relacionados | 1,536 unidades |
| Hierba | 1,106 unidades |
| Tierra y tierra seca | 1,106 unidades |
| Arena y desierto rojo | 0,922 unidades |
| Terreno nuclear | 0,768 unidades |
| Ladrillos de piedra, hormigón y relleno de terreno | 0 unidades |

La absorción total de un sector depende de la proporción de cada tipo de casilla que contiene. Estos valores son referencias para la configuración normal y pueden cambiar con los modificadores de generación del mapa.

**Consecuencia práctica:** cubrir un terreno natural con ladrillos, hormigón o relleno puede reducir su absorción ambiental en esas casillas. La pavimentación facilita el tránsito y el diseño industrial, pero no debe considerarse una herramienta para eliminar contaminación.

#### 3.4. Los árboles como absorbedores


Los árboles absorben pequeñas cantidades de contaminación de manera continua. Los árboles con follaje denso absorben más que los que han perdido hojas, y los árboles muertos o secos absorben cantidades muy pequeñas.

Cuando un sector supera el umbral de contaminación que provoca daño a los árboles —60 unidades en la configuración normal— algunos árboles pueden perder etapas de hojas o volverse más grises. Cada transición de daño absorbe una cantidad adicional de contaminación, pero un árbol muy deteriorado termina absorbiendo menos que uno sano.

Por ello, un bosque puede ayudar a amortiguar la contaminación, pero **no constituye un sistema de filtrado suficiente para una fábrica grande**. Además, talar el bosque elimina parte de esa absorción y puede aumentar la contaminación que permanece en el sector.

#### 3.5. Los nidos enemigos y la absorción

Los nidos de enemigos absorben contaminación de las zonas donde se encuentran. Esa contaminación se utiliza para reunir unidades y formar grupos de ataque; la absorción por los nidos no debe confundirse con la aparición normal de nuevos enemigos, que sigue sus propias reglas.

En Nauvis, cuando el sector de un nido tiene contaminación elevada, la absorción puede acelerar la preparación de un grupo de ataque. El grupo reunido se dirige hacia la fuente de contaminación y puede atacar edificios durante el recorrido.

Por tanto, que los nidos absorban contaminación no significa que la zona quede protegida: los nidos convierten parte de la contaminación absorbida en actividad ofensiva. Por este motivo, es importante considerar tanto la extensión de la nube como la ubicación de los nidos cercanos.

#### 4. Contaminación y ataques enemigos

#### 4.1. Cómo se activa un ataque

En el juego base, los mordedores y escupidores detectan indirectamente la actividad industrial a través de la contaminación. Los nidos absorben contaminación y reúnen unidades para atacar la fuente. Los enemigos que atraviesan zonas contaminadas pueden intentar llegar al origen y destruir las estructuras que lo producen.

La contaminación es uno de los factores que influye en los ataques, pero no determina por sí sola todas las interacciones enemigas. Los enemigos también pueden encontrar y atacar instalaciones por otros motivos relacionados con su comportamiento y la configuración de la partida.

#### 4.2. Cantidad de contaminación necesaria para reunir unidades

Como referencia, los valores de contaminación asociados a la incorporación de una unidad adicional a un grupo de ataque son los siguientes:

| Enemigo | Contaminación asociada |
|---|---:|
| Mordedor pequeño | 4 |
| Mordedor mediano | 20 |
| Mordedor grande | 80 |
| Mordedor behemoth | 400 |
| Escupidor pequeño | 4 |
| Escupidor mediano | 12 |
| Escupidor grande | 30 |
| Escupidor behemoth | 200 |

Estos valores no indican que un nido vaya a lanzar automáticamente un ataque al llegar a una cifra exacta de contaminación. Los nidos absorben contaminación según sus reglas, seleccionan unidades de acuerdo con el factor de evolución y reúnen el grupo antes de enviarlo. El tamaño de los grupos y los tiempos también dependen de las reglas de la partida.

#### 4.3. Factor de evolución: una diferencia crucial

En Nauvis, el factor de evolución aumenta por tres fuentes principales:

1. **Tiempo transcurrido:** el paso del tiempo contribuye al progreso de los enemigos.
2. **Contaminación generada:** la contaminación total que producen las máquinas del jugador aumenta la evolución.
3. **Destrucción de nidos:** destruir nidos también incrementa el factor de evolución.

La distinción más importante es que la evolución aumenta por la **contaminación producida**, no por la cantidad de contaminación que termina acumulándose en el mapa. Que los árboles absorban contaminación o que la nube se disperse no borra la contribución de las emisiones a la evolución.

Por ejemplo, apagar una línea de calderas reduce las emisiones futuras de esa línea. Sin embargo, absorber la contaminación ya producida con árboles no revierte la evolución que la producción anterior haya generado.

El factor de evolución influye en qué tipos y tamaños de enemigos pueden aparecer. A medida que aumenta, la defensa de la fábrica debe adaptarse a enemigos más resistentes o con capacidades de ataque superiores.

#### 4.4. Cómo observar la producción y absorción


La pestaña **Contaminación** del panel de estadísticas de producción permite estudiar las tasas de generación y absorción durante diferentes intervalos temporales. Es útil para detectar qué edificios aportan más contaminación y observar si el nivel producido crece con rapidez.

Úsala para comparar periodos antes y después de modificar la fábrica. Si la contaminación aumenta a pesar de haber reducido el uso de calderas, puede que otra parte de la producción haya crecido o que se hayan añadido edificios con emisiones importantes.

#### 5. Objetos, insumos y recursos relacionados

La contaminación no se fabrica ni se transporta como un objeto: es un valor del sistema ambiental. Sin embargo, varios recursos y estructuras determinan cuánta contaminación se genera o cómo se gestiona.

| Elemento | Papel en la mecánica |
|---|---|
| **Carbón y otros combustibles** | Proporcionan energía a las máquinas de combustión, como calderas y hornos. La combustión está asociada a emisiones. |
| **Electricidad** | Alimenta taladros, ensambladoras, refinerías y otras máquinas eléctricas; la contaminación de cada edificio depende de sus valores de operación. |
| **Módulos de eficiencia** | Pueden reducir el consumo energético y contribuir a reducir emisiones de edificios compatibles. |
| **Módulos con modificador de contaminación** | Pueden incrementar la contaminación final de una máquina según sus multiplicadores. |
| **Árboles** | Absorben parte de la contaminación, aunque su capacidad disminuye si pierden follaje. |
| **Terreno natural** | Absorbe contaminación de forma gradual según el tipo de casilla. |
| **Hormigón, ladrillos y relleno de terreno** | Son útiles para infraestructura, pero las casillas artificiales de estos tipos no absorben contaminación en la configuración normal. |
| **Nidos enemigos** | Absorben contaminación para reunir grupos de ataque. |
| **Centrales eléctricas** | Las calderas y otras fuentes pueden producir emisiones propias; reemplazarlas o complementarlas cambia el perfil de contaminación. |

#### 5.1. No existe un filtro industrial estándar en el juego base

En el juego base, no hay una estructura estándar que funcione como un filtro universal que capture y elimine toda la contaminación industrial. La reducción normal se consigue principalmente controlando las emisiones futuras, ajustando la actividad de la fábrica y aprovechando la absorción natural.

Los mods pueden añadir filtros, árboles cultivables o edificios especiales, pero esas opciones no deben confundirse con las mecánicas del juego base.

#### 6. Estrategias para controlar la contaminación

#### 6.1. Reducir las emisiones en origen

El primer método consiste en identificar cuáles son los edificios que más contaminación producen y evaluar si pueden operar de forma más eficiente.

- Evita mantener máquinas funcionando sin una necesidad de producción real.
- Revisa el uso de calderas, hornos y taladros a combustión, especialmente si muchas unidades trabajan de manera constante.
- Cuando sea conveniente, compara hornos eléctricos con hornos de combustible teniendo en cuenta tanto las emisiones directas como la forma de generar la electricidad.
- Utiliza módulos de eficiencia en los edificios compatibles cuando ayuden a reducir consumo y emisiones.
- Revisa los módulos de productividad o velocidad, ya que sus efectos pueden alterar el consumo energético y el multiplicador de contaminación.

No hay una solución universal: una máquina más rápida podría fabricar el mismo número de productos en menos tiempo, mientras que un edificio acelerado que trabaja de forma continua puede consumir más potencia. Evalúa la contaminación por producto terminado, además de la tasa por minuto.

#### 6.2. Planificar la ubicación de la fábrica

La posición de la base afecta a qué nidos quedan dentro de la nube y cuánto terreno natural rodea la industria. Al explorar el mapa, observa la distribución de los nidos y las rutas de expansión previstas antes de comprometerte con una gran instalación.

- Evita expandir una línea muy contaminante sin revisar la cercanía de nidos.
- Mantén espacio para ampliar defensas y acceder a las áreas de producción.
- Ten en cuenta que una explotación minera distante puede crear una zona de actividad industrial que también atraiga ataques.
- No elimines árboles por defecto alrededor de toda la base: conservados, pueden absorber contaminación.

Los bosques no sustituyen la defensa ni garantizan que la nube permanezca lejos de los nidos. Su utilidad depende de la densidad, el estado de los árboles, la ubicación de la fábrica y la tasa de emisiones.

#### 6.3. Aprovechar una red eléctrica más limpia

Las fuentes de energía no tienen el mismo perfil de contaminación directa. La energía solar no produce contaminación durante su generación, mientras que la generación a vapor basada en calderas sí genera emisiones. La energía nuclear tampoco produce contaminación por combustión durante el funcionamiento del reactor, aunque la extracción, el procesamiento y la logística de los materiales tienen su propia actividad industrial.

Por eso, ampliar una red solar o nuclear puede reducir la contaminación directa de la generación eléctrica en comparación con una central de calderas, siempre que el cambio sustituya efectivamente parte de la producción basada en combustibles.

#### 6.4. No pavimentar pensando que eliminará la contaminación

Las carreteras y superficies pavimentadas facilitan el desplazamiento y la construcción, pero no son un método de descontaminación. En la configuración normal, las casillas de ladrillos de piedra, hormigón y relleno tienen absorción nula. Cubrir grandes áreas naturales puede reducir la absorción de esos sectores.

Utiliza pavimento donde sea útil para la logística, pero no lo consideres una mejora para la absorción ambiental.

#### 6.5. Observar las tendencias en vez de una sola captura del mapa

La nube cambia con el tiempo. Una captura instantánea no muestra si la contaminación está creciendo, disminuyendo o simplemente desplazándose a otro sector.

Una rutina práctica consiste en:

1. Consultar las estadísticas de contaminación después de ampliar una línea industrial.
2. Observar la extensión de la nube en el mapa.
3. Comprobar si los nidos cercanos están recibiendo contaminación.
4. Comparar la producción antes y después de hacer cambios.
5. Repetir la revisión cuando se incorpore una nueva central, mina o planta química.

#### 7. Configuración de la contaminación al crear el mapa


Las opciones avanzadas de generación permiten ajustar aspectos de la contaminación. Los nombres exactos pueden variar ligeramente según el idioma o la versión.

| Parámetro | Efecto general |
|---|---|
| **Contaminación habilitada** | Activa o desactiva la simulación normal de contaminación. |
| **Modificador de absorción** | Cambia cuánto contaminante absorben los árboles y el terreno. |
| **Modificador de coste de ataque** | Ajusta cómo la contaminación absorbida contribuye a la preparación de ataques enemigos. |
| **Daño mínimo a los árboles** | Define el umbral a partir del cual empieza el deterioro de los árboles por contaminación. |
| **Contaminación absorbida por árbol dañado** | Controla cuánta contaminación absorbe el árbol al avanzar una etapa de daño. |
| **Tasa de difusión** | Controla la proporción de contaminación transferida a sectores vecinos. |
| **Factor de evolución por contaminación** | Controla el ritmo con el que las emisiones contribuyen a la evolución enemiga. |

Modificar estos parámetros cambia considerablemente la dificultad y el comportamiento ambiental de una partida. En una partida estándar, los valores predeterminados ofrecen la referencia de esta guía; los escenarios personalizados y los mods pueden ser diferentes.

#### 8. Mecánica especial de *Space Age*: esporas en Gleba

Las **esporas** de Gleba son una forma de contaminación diferente de la contaminación industrial habitual de Nauvis. Se representan con cuadrados verdes en el mapa y están vinculadas a la recolección agrícola.

- Cosechar un árbol de yumako genera 15 esporas por cosecha.
- Cosechar un jellystem genera 15 esporas por cosecha.
- La torre agrícola genera 4 esporas de manera continua según la mecánica del edificio.
- Las esporas atraen a los pentápodos, la fauna hostil de Gleba.
- Las esporas también contribuyen al cálculo local del factor de evolución en Gleba.
- Los raft de huevos de pentápodos y la mayoría de los tipos de terreno absorben esporas; las biochambers no las absorben.

A diferencia de la contaminación industrial de Nauvis, las máquinas corrientes de Gleba no producen esporas, con la excepción de la torre agrícola. Además, desde la actualización 2.0.43, los árboles no sufren daño por esporas ni las absorben a raíz del daño por contaminación.

Por lo tanto, una instalación agrícola en Gleba debe planificarse teniendo en cuenta tanto el ritmo de cosecha como la generación de esporas y la proximidad de nidos de pentápodos.

Más información: [Factorio Wiki: Pollution](https://wiki.factorio.com/Pollution).

#### 9. Errores comunes y soluciones

| Problema | Causa probable | Qué revisar |
|---|---|---|
| La nube roja crece pese a que la fábrica parece estable. | Las emisiones acumuladas superan la absorción y la nube sigue difundiéndose. | Comprueba la producción por minuto y la tendencia en el mapa. |
| Los enemigos atacan aunque haya árboles alrededor. | Los árboles absorben solo una cantidad limitada y pueden haberse deteriorado. | Revisa la densidad y el estado del bosque, la intensidad de las emisiones y la ubicación de los nidos. |
| La evolución sigue aumentando después de que la nube disminuye. | La evolución depende de las emisiones generadas históricamente, no solo de la contaminación visible. | Reduce las emisiones futuras; la absorción del terreno no revierte la evolución previa. |
| Una zona pavimentada absorbe menos contaminación. | Las casillas artificiales pueden tener absorción nula. | Considera el efecto de pavimentar sobre la absorción natural del sector. |
| Un edificio emite más contaminación tras instalar módulos. | El módulo puede modificar el consumo de energía o el multiplicador de contaminación. | Revisa los modificadores y compara emisiones por unidad producida. |
| Los nidos parecen consumir contaminación. | Los nidos absorben contaminación para reunir unidades de ataque. | Considera que esa absorción puede contribuir a formar ataques. |
| La contaminación de la fábrica no coincide con la de un vídeo o una guía. | La versión, el escenario, los módulos o los parámetros del mapa pueden ser diferentes. | Comprueba la versión del juego y los ajustes de generación. |
| En Gleba aparece una nube verde. | Son esporas, no la contaminación industrial normal. | Revisa cosechas agrícolas, torres agrícolas y la proximidad de pentápodos. |

#### 10. Lista de comprobación para una fábrica con contaminación controlada

- [ ] La visualización de contaminación está habilitada y se revisa el mapa con regularidad.
- [ ] Se conoce la tasa aproximada de contaminación de las principales líneas industriales.
- [ ] Las emisiones de calderas, taladros, hornos y plantas químicas se consideran al ampliar la fábrica.
- [ ] Los módulos se eligen teniendo en cuenta su efecto sobre consumo y contaminación.
- [ ] Se conservan los árboles útiles cuando no interfieren con la expansión o la logística.
- [ ] La ubicación de los nidos cercanos se tiene en cuenta al planificar nuevas instalaciones.
- [ ] La defensa de la fábrica puede responder a los ataques que atrae la contaminación.
- [ ] La producción de energía se evalúa como parte del impacto total de la industria.
- [ ] Se recuerda que absorber o dispersar contaminación reduce la nube, pero no borra la evolución ya generada.
- [ ] En Gleba, se controla la producción de esporas como mecánica distinta.

#### 11. Fuentes de referencia

Estas páginas permiten verificar los valores, las reglas de difusión y las relaciones entre contaminación y evolución:

- [Factorio Wiki — Pollution (inglés)](https://wiki.factorio.com/Pollution): difusión, absorción, árboles, edificios emisores y esporas.
- [Factorio Wiki — Contaminación (español)](https://wiki.factorio.com/Pollution/es): descripción general en español; algunas tablas pueden estar desactualizadas respecto a Factorio 2.x.
- [Factorio Wiki — Map generator](https://wiki.factorio.com/Map_generator): configuración de absorción, difusión y parámetros de contaminación.
- [Factorio Wiki — Enemies](https://wiki.factorio.com/Enemies): comportamiento y tipos de enemigos.
- [Factorio Wiki — Nests / Evolution](https://wiki.factorio.com/Nests): relación entre generación de contaminación, destrucción de nidos y evolución.
- [Factorio Runtime Docs — PollutionMapSettings](https://lua-api.factorio.com/latest/concepts/PollutionMapSettings.html): valores predeterminados de los parámetros de contaminación en la API de Factorio.


---

### Guía técnica de Factorio: Sistema de enemigos (Biters/Spitters) y su evolución

**Versión de referencia:** *Factorio 2.x*. Las diferencias de *Space Age* se indican por separado. Las estadísticas concretas pueden variar según la versión, la calidad de las entidades cuando corresponda y la configuración del mapa.

El sistema de enemigos de *Factorio* representa la presión que ejerce el ecosistema nativo sobre la expansión industrial del jugador. En Nauvis, las principales amenazas son los **Biters**, los **Spitters** y los **Worms**. Los nidos generan criaturas, la contaminación puede atraer ataques y el **factor de evolución** modifica qué tipos de enemigos aparecen y con qué frecuencia.

Esta mecánica conecta la expansión de la fábrica con la gestión de contaminación, la exploración del mapa y la planificación defensiva. Comprenderla permite anticipar cambios en la dificultad y reducir interrupciones en la producción.

---

#### 1. Fundamentos del sistema de enemigos

##### 1.1. ¿Qué son los enemigos?

Los enemigos son criaturas y estructuras nativas del planeta que pueden dañar al personaje y a las instalaciones del jugador. En Nauvis se organizan principalmente alrededor de nidos orgánicos y zonas infestadas.

Los tres grupos principales del juego base son:

- **Biters:** enemigos de combate cercano que se aproximan al objetivo y lo atacan cuerpo a cuerpo.
- **Spitters:** enemigos a distancia que arrojan ácido; sus ataques pueden dejar charcos temporales que dañan y ralentizan.
- **Worms:** criaturas estáticas que protegen los alrededores de los nidos y atacan a distancia cuando un objetivo entra en su alcance.

Los enemigos aparecen como puntos rojos en el mapa y sus nidos se distinguen por su forma orgánica. El tamaño y la resistencia de las criaturas varían a medida que progresa la evolución.


##### 1.2. Conceptos que conviene distinguir

- **Nido o spawner:** estructura que genera Biters o Spitters. En Nauvis existen nidos de ambos tipos.
- **Enemigo individual:** criatura generada por un nido o que participa en una incursión o expansión.
- **Factor de evolución:** valor global de cada superficie que va de 0 a menos de 1 e influye en los tipos de enemigos y sus capacidades de aparición.
- **Contaminación:** emisión industrial que se propaga por el mapa. En Nauvis, cuando llega a zonas infestadas, puede alimentar la preparación de ataques.
- **Expansión enemiga:** proceso por el que grupos de criaturas se desplazan a nuevas áreas y pueden establecer nuevos nidos o Worms.
- **Incursión:** grupo de enemigos que se reúne y se dirige hacia la fábrica u objetivos relacionados con ella.

La evolución no es una barra de salud ni un nivel individual de cada criatura: es una variable del entorno que determina las probabilidades de aparición y otros aspectos del comportamiento enemigo.

##### 1.3. Cómo se relacionan contaminación, evolución y ataques

El sistema se entiende mejor como varios procesos conectados:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Emisiones industriales | Actividad de producción industrial | Edificios industriales | Contaminación | — |
| Propagación ambiental | Contaminación | Dispersión por el entorno | Contaminación en zonas cercanas | — |
| Llegada a nidos | Contaminación propagada | Nidos enemigos | Grupos de ataque preparados | — |
| Ataque a la fábrica | Grupos de ataque preparados | Enemigos | Enemigos que se dirigen a la fábrica | — |
| Contribución a la evolución | Contaminación/actividad señalada en el esquema | Factor de evolución | Aparición de variantes más fuertes | Rama separada mostrada en el diagrama original. |

El paso del tiempo también aumenta la evolución, aunque la contaminación producida y la destrucción de nidos son otras contribuciones importantes. Por eso, una fábrica puede experimentar enemigos más avanzados incluso si mantiene la contaminación bajo control.

---

#### 2. Tipos de enemigos en Nauvis

##### 2.1. Biters

Los **Biters** se desplazan hacia los objetivos y atacan a corta distancia. Tienen cuatro tamaños; los ejemplares mayores poseen más salud, causan más daño y pueden resultar mucho más resistentes.


[Biters — referencia técnica en Factorio Wiki](https://wiki.factorio.com/Enemies#Biters)

| Variante | Salud de referencia | Regeneración aproximada | Contaminación requerida para incorporarse a un ataque* | Características generales |
|---|---:|---:|---:|---|
| **Pequeño** | 15 | 0,6/s | 4 | Variante inicial; menos resistente. |
| **Mediano** | 75 | 0,6/s | 20 | Más resistente y rápido que el pequeño. |
| **Grande** | 375 | 1,2/s | 80 | Mucha más salud; puede golpear objetivos situados tras muros de un bloque de grosor. |
| **Behemoth** | 3000 | 6/s | 400 | Variante de gran resistencia, asociada a una evolución elevada. |

**La cifra de contaminación es un valor de referencia de la mecánica de reunión de grupos; no representa la salud, el daño ni la contaminación total de una base. Los valores concretos deben interpretarse según la versión y configuración del mapa.*

Los Biters siguen rutas hacia el objetivo elegido y pueden atacar estructuras que les bloqueen el paso si no encuentran una ruta conveniente. Los grupos de mayor tamaño pueden ejercer presión considerable sobre las infraestructuras periféricas.

##### 2.2. Spitters

Los **Spitters** son enemigos de ataque a distancia. Lanzan proyectiles ácidos hacia su objetivo y pueden dejar charcos que continúan causando daño durante un periodo. El ácido también ralentiza temporalmente al personaje y a los vehículos.


[Spitters — referencia técnica en Factorio Wiki](https://wiki.factorio.com/Enemies#Spitters)

| Variante | Salud de referencia | Alcance aproximado | Contaminación requerida para incorporarse a un ataque* | Características generales |
|---|---:|---:|---:|---|
| **Pequeño** | 10 | 13 casillas | 4 | Frágil, pero puede atacar desde lejos. |
| **Mediano** | 50 | 14 casillas | 12 | Proyectiles más dañinos que los de la variante pequeña. |
| **Grande** | 200 | 15 casillas | 30 | Mayor alcance y daño ácido. |
| **Behemoth** | 1500 | 16 casillas | 200 | Variante muy resistente con ataques ácidos potentes. |

**Valores de referencia del sistema de ataques documentado en la Wiki; no son umbrales universales para que un enemigo aparezca.*

La diferencia esencial con los Biters es la distancia: un Spitter puede dañar estructuras sin necesidad de alcanzar el contacto directo. Además, los charcos ácidos hacen que permanecer en una zona impactada resulte peligroso.

##### 2.3. Worms

Los **Worms** son enemigos estáticos que actúan como defensores naturales de los nidos. No persiguen al jugador como los Biters: disparan proyectiles ácidos cuando un objetivo entra en su alcance.


[Worms — referencia técnica en Factorio Wiki](https://wiki.factorio.com/Enemies#Worms)

| Variante | Salud de referencia | Alcance aproximado | Observaciones |
|---|---:|---:|---|
| **Pequeño** | 200 | 25 casillas | Puede ser peligroso aunque aparezca en zonas iniciales. |
| **Mediano** | 500 | 30 casillas | Puede aparecer por expansión cuando la evolución supera aproximadamente 0,3. |
| **Grande** | 1500 | 38 casillas | Puede aparecer por expansión cuando la evolución supera aproximadamente 0,5. |
| **Behemoth** | 3000 | 48 casillas | Puede aparecer por expansión cuando la evolución supera aproximadamente 0,9. |

En la generación inicial del mapa, los Worms también pueden depender de la distancia respecto de la zona de inicio. Por eso, algunas áreas alejadas pueden contener variantes más fuertes incluso antes de que el jugador observe una evolución muy elevada.

Los proyectiles ácidos de los Worms pueden afectar al personaje y a los vehículos; los charcos temporales ralentizan y dañan a las unidades móviles. Las estructuras del mapa no reciben el mismo daño persistente por los charcos, aunque sí pueden resultar dañadas por los proyectiles.

##### 2.4. Nidos de Biters y Spitters

Los **nidos** son las estructuras que generan enemigos. Deben distinguirse de los Worms: los nidos producen unidades, mientras que los Worms defienden su zona de forma estática.


Aspectos importantes:

- Los nidos pueden generar unidades de forma continuada.
- Su actividad se relaciona con el sistema de ataques y el uso de contaminación disponible en la zona.
- La salud máxima de los nidos aumenta con el factor de evolución, hasta alcanzar aproximadamente diez veces su salud inicial en la evolución máxima.

- Destruir un nido contribuye a la evolución global; por tanto, su eliminación puede resolver un foco local de presión, pero también incrementa el factor de evolución.
- Los nidos y los Worms pueden aparecer de nuevo como resultado de la expansión enemiga.

##### 2.5. Diferencias entre Biters, Spitters y Worms

| Aspecto | Biters | Spitters | Worms |
|---|---|---|---|
| Movimiento | Se aproximan al objetivo | Se aproximan y mantienen distancia de ataque | Permanecen en una ubicación fija |
| Ataque característico | Contacto directo | Proyectiles ácidos | Proyectiles ácidos |
| Efecto del ácido | No es su ataque habitual | Puede dañar y ralentizar | Puede dañar y ralentizar |
| Función en el ecosistema | Presión directa sobre objetivos | Presión a distancia | Defensa territorial de áreas con nidos |
| Escalado de tamaño | Cuatro variantes | Cuatro variantes | Cuatro variantes |

---

#### 3. Factor de evolución

##### 3.1. ¿Qué representa?

El **factor de evolución** es una variable comprendida entre 0 y menos de 1. A medida que aumenta, los nidos pueden generar enemigos de mayor tamaño y las condiciones de expansión se vuelven más exigentes.

El factor de evolución no se puede revertir. Por ello, planifica las defensas y la progresión tecnológica antes de que aumente la amenaza.

El factor puede aumentar por tres fuentes principales:

1. **Paso del tiempo:** aporta una evolución gradual incluso si el jugador no produce mucha contaminación.
2. **Contaminación generada:** la producción industrial contribuye al avance de la evolución.
3. **Destrucción de nidos enemigos:** destruir un nido incrementa la evolución, aunque no se destruyan Worms.

La evolución es acumulativa y no disminuye de forma natural. Reducir la contaminación puede limitar una fuente de crecimiento futuro, pero no devuelve el factor de evolución a un nivel anterior.

##### 3.2. Umbrales de aparición orientativos

Los siguientes valores muestran cuándo empiezan a resultar posibles algunas variantes. No significan que todas las criaturas cambien instantáneamente al alcanzar ese porcentaje: la composición se determina mediante probabilidades y puede depender de la distancia y de las reglas de generación del mapa.

| Factor de evolución aproximado | Cambio relevante |
|---:|---|
| **0 %** | La partida comienza con las variantes más pequeñas de Biters; las condiciones del mapa determinan también qué nidos aparecen en el entorno. |
| **20 %** | Empiezan a ser posibles los Biters medianos. |
| **25 %** | Empiezan a ser posibles los Spitters pequeños. |
| **30 %** | Los Worms medianos pueden generarse mediante expansión enemiga. |
| **40 %** | Empiezan a ser posibles los Spitters medianos. |
| **50 %** | Se vuelven posibles los Biters grandes, los Spitters grandes y los Worms grandes mediante expansión. |
| **90 %** | Se vuelven posibles los Behemoths y los Worms Behemoth mediante expansión. |


Los gráficos son más útiles que un umbral aislado porque muestran la transición gradual entre variantes. A medida que avanza la evolución, las probabilidades de los tipos pequeños disminuyen y las variantes más grandes ocupan una fracción mayor de las apariciones.

##### 3.3. Por qué la evolución no aumenta de forma lineal

El juego combina las contribuciones de tiempo, contaminación y destrucción de nidos y transforma el resultado para mantener el factor dentro del intervalo de 0 a menos de 1. Por tanto, no basta con sumar porcentajes individuales estimados.

En valores altos, el factor se aproxima progresivamente a 1. Esto no significa que deje de importar la evolución, sino que las subidas de la variable mostrada se hacen cada vez menores mientras que los enemigos de mayor tamaño ya están habilitados.

En la configuración predeterminada documentada por la Wiki, las contribuciones internas de referencia incluyen:

| Fuente | Parámetro de referencia antes de aplicar la transformación final | Interpretación |
|---|---:|---|
| Tiempo | 0,000004 por segundo | La evolución avanza lentamente aun sin contaminación. |
| Contaminación | 0,0000009 por unidad | La producción industrial contribuye al avance. |
| Nido destruido | 0,002 por nido | Cada nido destruido aporta una contribución adicional. |

Estas cifras describen los parámetros internos de la configuración de referencia y **no** deben interpretarse como incrementos fijos directos del factor visible. Los ajustes de generación del mapa pueden modificarlas y la transformación acumulada hace que el cambio efectivo dependa del nivel de evolución actual.

##### 3.4. Cómo consultar la evolución

En versiones actuales, el comando de consola `/evolution` muestra el factor de evolución y un desglose de sus fuentes. En partidas donde el uso de consola esté permitido, este método permite observar cuánto se atribuye al tiempo, a la contaminación y a los nidos destruidos.

También pueden revisarse las estadísticas de producción para entender cuánto genera la fábrica y las estadísticas de enemigos eliminados para observar la frecuencia de los encuentros. La interfaz exacta y la disponibilidad de estadísticas pueden variar entre versiones y configuraciones.

---

#### 4. Contaminación y preparación de ataques

##### 4.1. Cómo se relaciona con los nidos

En Nauvis, la contaminación producida por la fábrica se propaga por el entorno. Cuando la nube de contaminación llega a las zonas de los nidos, puede ser utilizada para reunir Biters y Spitters y preparar ataques.

Los nidos no necesitan estar pegados a la fábrica para convertirse en una amenaza: basta con que la contaminación se extienda hasta ellos y que puedan reunirse grupos. El alcance efectivo depende de la producción industrial, la propagación de la contaminación, el terreno y los elementos que la absorben.

##### 4.2. Preparación de incursiones

Según el comportamiento documentado, los grupos de ataque se reúnen periódicamente. El intervalo entre ataques se selecciona aleatoriamente dentro de un rango y los enemigos reunidos se desplazan hacia un punto de reunión antes de dirigirse a sus objetivos.

La navegación no es idéntica en todos los casos. Las criaturas intentan seguir rutas convenientes; si los obstáculos dificultan demasiado el paso, pueden atacar elementos que bloquean su camino.

Una base situada cerca de varios nidos o dentro de una nube de contaminación amplia puede recibir ataques más frecuentes que una instalación pequeña cuya contaminación no alcanza zonas infestadas.

##### 4.3. Contaminación generada frente a contaminación visible

Es útil distinguir:

- **Contaminación producida:** la cantidad generada por las máquinas durante un intervalo. Contribuye a la evolución aunque parte de la contaminación no llegue a extenderse lejos.
- **Nube de contaminación:** la contaminación que se desplaza por el mapa. Su posición es relevante para determinar si los nidos pueden aprovecharla para organizar ataques.
- **Contaminación absorbida:** parte de la contaminación se elimina por árboles, terreno y otros mecanismos del ecosistema. La absorción puede cambiar la extensión visible de la nube, pero no equivale a borrar la contaminación que ya se produjo como contribución a la evolución.

##### 4.4. Estrategias de gestión de contaminación

- Revisa periódicamente el mapa de contaminación y observa qué grupos de nidos quedan dentro de la nube.
- Ten en cuenta el impacto de nuevas calderas, hornos y otras instalaciones antes de ampliar la fábrica.
- Sitúa los servicios críticos de modo que una interrupción periférica no paralice toda la producción.
- Planifica perímetros y rutas de acceso alrededor de las áreas industriales expuestas.
- Mantén espacio para reparar estructuras y ampliar la infraestructura de protección.
- No asumas que controlar la contaminación detiene la evolución por completo: el tiempo y la destrucción de nidos también influyen.

---

#### 5. Expansión enemiga

##### 5.1. ¿Qué es la expansión?

La expansión enemiga permite que los grupos de criaturas establezcan nuevos nidos y posiciones de Worms en áreas que antes no estaban ocupadas. Esto hace que el territorio peligroso pueda cambiar durante la partida.

En Nauvis, el proceso documentado selecciona periódicamente una región del mapa como destino de expansión. A continuación, un grupo de enemigos sale de los nidos cercanos y se dirige hacia el área escogida. Si alcanza el destino, sus miembros pueden transformarse en nuevas estructuras de enemigos tras un intervalo.


##### 5.2. Factores que influyen en la expansión

La expansión está influida por:

- La disponibilidad de grupos procedentes de nidos cercanos.
- La evolución, que modifica la frecuencia y el tamaño de los grupos según la configuración.
- La presencia de estructuras de jugador y nidos enemigos alrededor de la región considerada.
- La posibilidad de desplazarse hasta el destino y encontrar posiciones compatibles con el terreno.

En la documentación de referencia, los intentos de expansión tienen intervalos aleatorios de aproximadamente 1 a 4 minutos y los grupos suelen reunir entre 5 y 20 unidades. Estos valores no deben considerarse un reloj exacto para todas las partidas o superficies: la configuración, la versión y el planeta afectan el comportamiento.

##### 5.3. Cómo reconocer un problema de expansión

Algunos indicios son:

- Aparecen nidos nuevos fuera de las zonas infestadas que se conocían al principio.
- Se observan enemigos desplazándose en grupo por zonas periféricas.
- Aumentan los incidentes en puestos mineros o líneas de transporte alejadas.
- La contaminación empieza a alcanzar nuevos sectores del mapa.

Es recomendable revisar el mapa periódicamente, especialmente cuando se amplían las minas, los campos de producción y las líneas ferroviarias.

---

#### 6. Estructuras y elementos relacionados con la mecánica

##### 6.1. Nidos de Biters y Spitters

Los nidos son los principales puntos de origen de los enemigos en Nauvis. Sus funciones son generar criaturas y participar en la dinámica de ataques y expansión. Su salud aumenta con la evolución y pueden volver a aparecer en zonas que los enemigos hayan colonizado.

##### 6.2. Worms

Los Worms defienden zonas alrededor de los nidos. A diferencia de las criaturas móviles, permanecen en una ubicación fija y atacan a distancia cuando el jugador se aproxima. Su alcance y resistencia crecen con su tamaño.

##### 6.3. Edificios industriales que afectan indirectamente a los enemigos

Los edificios siguientes no crean enemigos, pero cambian la presión que ejerce el ecosistema:

| Edificio o elemento | Relación con el sistema enemigo |
|---|---|
| **Calderas** | Producen contaminación al quemar combustible. |
| **Hornos de combustible** | Pueden contribuir a la contaminación durante la producción. |
| **Taladros mineros y máquinas de montaje** | Su actividad forma parte de la carga industrial de la fábrica; las emisiones exactas dependen del edificio. |
| **Paneles solares** | Generan electricidad sin contaminación operativa directa, lo que puede reducir emisiones en comparación con fuentes de combustión. |
| **Árboles y terreno** | Pueden contribuir a la absorción o propagación local de contaminación, según el tipo de terreno y sus reglas. |
| **Radar** | Permite explorar y supervisar zonas del mapa, incluidas áreas potencialmente amenazadas. |
| **Muros y perímetros** | Forman parte del diseño físico de la base y pueden proteger instalaciones, aunque los enemigos pueden atacar obstáculos cuando no encuentran una ruta conveniente. |

##### 6.4. Infraestructura de supervisión y protección

Aunque las defensas concretas dependen de la etapa de progresión, una base bien organizada suele beneficiarse de:

- Perímetros claros alrededor de las instalaciones críticas.
- Espacio para mantenimiento y reconstrucción.
- Supervisión de las áreas exteriores mediante el mapa y la exploración.
- Separación funcional entre producción principal y puestos periféricos.
- Suministro redundante de energía y materiales donde una interrupción sería grave.

La intención de esta sección es planificar la continuidad de la fábrica y la gestión del territorio, no detallar armamento o tácticas de combate.

---

#### 7. Consejos de gestión y errores comunes

##### 7.1. No confundir evolución y frecuencia de ataques

La evolución determina qué tipos de enemigos pueden aparecer y afecta ciertos aspectos de su generación. La contaminación que alcanza los nidos es un factor importante para la preparación de incursiones. Por ello, una evolución elevada no significa necesariamente que un ataque vaya a producirse inmediatamente, y una nube de contaminación extensa no significa que todos los enemigos sean de tipo Behemoth.

##### 7.2. No asumir que la evolución se puede revertir

Reducir las emisiones futuras ayuda a limitar una de las contribuciones a la evolución, pero el factor no disminuye de forma natural. Es conveniente dimensionar la infraestructura de protección pensando en la dificultad que puede alcanzar la partida, no solo en las criaturas que aparecen al principio.

##### 7.3. No ignorar la expansión de nidos

Limitarse a revisar el área inmediatamente visible alrededor de la base puede dejar sin supervisión recursos lejanos y puestos mineros. La expansión enemiga puede crear nuevos puntos de presión en zonas que antes eran seguras. Usa el mapa y la exploración para actualizar la evaluación del territorio.

##### 7.4. Considerar los Spitters en el diseño del perímetro

Los enemigos a distancia no necesitan llegar hasta el borde de una estructura para causar problemas. Un perímetro debe valorarse no solo como obstáculo físico, sino también en relación con las distancias desde las que pueden atacar los enemigos y con la continuidad de los servicios esenciales.

##### 7.5. Tener en cuenta la regeneración

Las criaturas y los nidos pueden regenerar salud. Si una amenaza se queda fuera de combate durante un tiempo, la salud recuperada puede alterar lo que se había previsto sobre su resistencia. Las estadísticas de salud mostradas en las tablas corresponden a valores de referencia, no a un cálculo completo de la duración de un enfrentamiento.

##### 7.6. Preparar instalaciones periféricas

Los puestos de minería y transporte alejados pueden quedar aislados por ataques o por nuevos nidos. Diseña estos puestos pensando en cómo detectar una interrupción, acceder a la zona para repararla y evitar que el fallo se propague hacia toda la cadena de producción.

---

#### 8. Diferencias de *Space Age*

La expansión *Space Age* incorpora ecosistemas distintos en otros planetas. El sistema de evolución de Nauvis no debe extrapolarse automáticamente a todas esas criaturas.

##### 8.1. Evolución por superficie

En *Space Age*, el factor de evolución se calcula de forma independiente para cada superficie planetaria. El desglose puede consultarse con `/evolution` en la consola cuando corresponda.

##### 8.2. Gleba: Pentapods y esporas

En Gleba, los enemigos nativos incluyen **Wrigglers**, **Strafers** y **Stompers**, no los Biters y Spitters clásicos de Nauvis. La producción de esporas, especialmente asociada a las torres agrícolas, reemplaza la contaminación industrial convencional como variable ecológica para la evolución en esa superficie.

Las máquinas industriales trasladadas desde Nauvis no emiten la contaminación estándar ni esporas al funcionar en Gleba; la producción de esporas de las torres agrícolas es una fuente importante en la evolución de la superficie.

##### 8.3. Vulcanus: Demolishers

En Vulcanus hay **Demolishers**, criaturas grandes con territorios definidos. Construir dentro de su territorio puede provocarlas y hacer que ataquen las estructuras cercanas. Su comportamiento es distinto del sistema de nidos y contaminación de Nauvis.

Por lo tanto, en una guía centrada en Biters/Spitters, las mecánicas de Pentapods y Demolishers deben tratarse como sistemas planetarios relacionados, pero diferentes.

---

#### 9. Lista de comprobación

Antes de ampliar una fábrica en Nauvis, revisa los siguientes puntos:

- [ ] Conoces la ubicación de los nidos más cercanos y de las zonas infestadas dentro del mapa explorado.
- [ ] Has revisado hasta dónde se extiende la contaminación industrial.
- [ ] Comprendes que el tiempo, la contaminación producida y la destrucción de nidos contribuyen a la evolución.
- [ ] Has considerado la aparición progresiva de Biters y Spitters medianos, grandes y Behemoths.
- [ ] Tienes en cuenta el alcance de los Spitters y Worms al planificar zonas de infraestructura crítica.
- [ ] Supervisas los puestos mineros y de transporte alejados de la base principal.
- [ ] Dejas espacio para mantenimiento y ampliación del perímetro.
- [ ] Comprendes que los nidos pueden expandirse y volver a establecerse en nuevas zonas.
- [ ] En *Space Age*, identificas las mecánicas específicas de cada planeta en lugar de asumir que todas funcionan como Nauvis.

---

#### 10. Fuentes de referencia

Las siguientes páginas contienen las estadísticas, comportamiento y parámetros de aparición documentados para esta mecánica:

- [Factorio Wiki: Enemies](https://wiki.factorio.com/Enemies) — resumen general de enemigos, estadísticas y diferencias de *Space Age*.
- [Factorio Wiki: Pollution](https://wiki.factorio.com/Pollution) — generación y propagación de contaminación.
- [Factorio Wiki: Biter spawner](https://wiki.factorio.com/Enemies#Nests) — nidos y generación de enemigos.
- [Factorio Wiki: Worms](https://wiki.factorio.com/Enemies#Worms) — variantes, alcance y comportamiento de los Worms.


---

### Guía técnica de Factorio: Defensas pasivas, vigilancia y reparación automática

**Versión de referencia:** *Factorio 2.x*. Las mecánicas de *Space Age* se indican cuando corresponde. Esta guía se centra en estructuras no armadas y sistemas de mantenimiento: muros, puertas, radares, robots de construcción y reparación de la fábrica.

#### 1. Introducción

La protección de una fábrica no depende únicamente de estructuras de combate. También es importante construir límites físicos, vigilar las zonas cercanas, detectar daños y automatizar las reparaciones. Estos sistemas ayudan a reducir interrupciones, conservar edificios valiosos y mantener operativas las áreas alejadas de la base principal.

Los componentes tratados en esta guía cumplen funciones distintas:

- **Muros:** bloquean el paso y actúan como barrera física alrededor de la fábrica.
- **Puertas:** permiten atravesar un muro sin dejar una abertura permanentemente libre.
- **Radares:** revelan el terreno, actualizan el mapa y ayudan a observar cambios en zonas exploradas.
- **Robopuertos:** crean áreas de trabajo para robots de construcción, que pueden levantar estructuras, reemplazarlas y repararlas.
- **Packs de reparación:** consumibles que restauran la salud de edificios dañados.
- **Robots de construcción:** automatizan construcción, reparación, sustitución y desmantelamiento dentro del alcance de una red compatible.

Estas herramientas no hacen invulnerable una fábrica. Funcionan mejor cuando se combinan con un diseño ordenado, una red eléctrica fiable, inventarios de materiales y vigilancia periódica.

#### 2. Muros


#### 2.1. Función

El **muro de piedra** (*Wall*) es una estructura de un solo cuadro que impide que el personaje y los enemigos terrestres atraviesen su posición. Puede utilizarse para delimitar el perímetro de una fábrica, separar zonas de trabajo o formar pasillos y barreras alrededor de instalaciones importantes.

Los muros tienen **350 puntos de salud** en calidad normal y una resistencia considerable a ciertos tipos de daño. Se construyen con ladrillos de piedra y se desbloquean mediante la tecnología correspondiente.

#### 2.2. Características principales

| Característica | Valor o función |
|---|---|
| Tamaño | 1 × 1 casilla |
| Salud base | 350 |
| Material principal | Ladrillo de piedra |
| Electricidad | No requiere |
| Combustible | No requiere |
| Función esencial | Bloquear el paso y proteger zonas detrás del muro |

Los valores pueden cambiar con la calidad del objeto y la versión del juego. Consulta la ficha oficial si necesitas comparar estadísticas exactas.

#### 2.3. Diseño y limitaciones

- Sigue el contorno del área que necesitas proteger, dejando espacio para ampliar la fábrica.
- Evita crear recintos que dificulten la circulación del jugador, los trenes o los robots de construcción.
- Añade puertas en accesos frecuentes para no tener que desmontar partes del muro cada vez que necesites atravesarlo.
- Revisa periódicamente el estado de los segmentos que hayan recibido daños.
- No supongas que un muro impide todo tipo de daño: su función principal es bloquear el paso y ofrecer una barrera física. Algunos ataques a distancia pueden superar una barrera, y ciertos enemigos pueden alcanzar estructuras cercanas al muro.

Los muros no se consideran unidades militares para la lógica de objetivos de los enemigos. Si un muro obstruye la ruta de un enemigo hacia un objetivo, este puede atacarlo para despejar el camino. Por eso, el muro debe tratarse como una capa de protección y retraso, no como una garantía absoluta.

#### 3. Puertas


#### 3.1. ¿Qué es una puerta?

La **puerta** (*Gate*) es una variante del muro que abre un paso temporal cuando el jugador o un vehículo compatible se aproxima. Una vez que ya no es necesario el paso, vuelve a cerrarse automáticamente.

Las puertas comparten con el muro normal una salud base de 350 puntos y resistencias similares. Varias puertas colocadas una junto a otra pueden formar una abertura más larga. También se pueden colocar sobre raíles para crear pasos compatibles con el tránsito ferroviario.

#### 3.2. Comportamiento

- Se abre automáticamente para el jugador cuando se acerca a pie o en un vehículo compatible.
- Las puertas ferroviarias pueden abrirse para trenes cuyo recorrido previsto atraviesa la puerta.
- No se abre normalmente para los enemigos; si un enemigo necesita atravesarla y no encuentra otra ruta, puede atacarla.
- Puede conectarse a la red de circuitos mediante un cable rojo o verde conectado a un muro adyacente. Esto permite configurar señales de apertura y controlar su comportamiento con condiciones.

#### 3.3. Recomendaciones

- Coloca puertas donde el jugador necesite entrar con frecuencia: accesos a la fábrica, caminos de mantenimiento y cruces de raíles.
- Evita dejar una puerta abierta por una condición de circuito que no se haya revisado.
- En un cruce ferroviario, comprueba el comportamiento con trenes en modo manual y automático antes de confiar en la instalación.
- Mantén los accesos visibles y despejados para que sea sencillo identificar cualquier daño o anomalía.

#### 4. Radares y vigilancia del mapa


#### 4.1. Función

El **radar** revela una zona cercana de forma continua y explora gradualmente una región mucho mayor. No repara ni bloquea el paso, pero es esencial para observar el terreno, descubrir recursos, revisar zonas periféricas y mantener información cartográfica actualizada.

#### 4.2. Características principales

| Característica | Valor base |
|---|---|
| Tamaño | 3 × 3 casillas |
| Consumo eléctrico nominal | 300 kW |
| Cobertura continua | 7 × 7 chunks, aproximadamente 224 × 224 casillas |
| Exploración a larga distancia | Descubre progresivamente chunks más alejados |
| Combustible | No requiere; necesita electricidad |

El radar muestra las entidades detectadas en el área explorada, incluidos enemigos, estructuras y vehículos. La información de áreas que ya no están siendo actualizadas puede quedar desfasada: el mapa conserva la última imagen conocida hasta que la zona vuelva a explorarse.

#### 4.3. Uso práctico para proteger la fábrica

- Instala radares en zonas periféricas para mantener actualizados el mapa y los alrededores de la fábrica.
- Revisa las zonas exploradas cuando cambien los límites de la contaminación o cuando se amplíe la producción.
- Comprueba que la red eléctrica tiene capacidad suficiente para alimentar todos los radares.
- Utiliza varios radares cuando el objetivo sea explorar más rápidamente regiones extensas.
- Recuerda que el radar no sustituye la observación en tiempo real de toda la superficie: los sectores lejanos pueden mostrar datos antiguos.

En la versión 2.0.7 y posteriores, los radares dejaron de ser objetivos militares prioritarios para los enemigos. Aun así, pueden sufrir daños si obstruyen el recorrido de una unidad enemiga hacia otro objetivo.

#### 5. Robopuerto y red de construcción


#### 5.1. Función

El **robopuerto** (*Roboport*) sirve como punto de descanso y recarga para robots logísticos y de construcción. También define áreas en las que los robots pueden realizar tareas. Para una estrategia de protección pasiva, su utilidad más importante es permitir que robots de construcción reparen y reconstruyan estructuras automáticamente.

#### 5.2. Áreas de cobertura

El robopuerto crea dos áreas distintas:

- **Área logística:** 50 × 50 casillas. Aquí pueden operar las funciones propias de la red logística y los robots pueden acceder a las entidades logísticas compatibles.
- **Área de construcción:** 110 × 110 casillas. Los robots de construcción pueden construir, desmantelar o reparar estructuras dentro de esta área, según las tareas disponibles.

Dos robopuertos forman una misma red cuando sus áreas logísticas se tocan y se conectan correctamente. Si sus redes no están conectadas, no comparten robots como una sola red.

#### 5.3. Electricidad y recarga

Un robopuerto necesita electricidad. Su consumo en reposo es de aproximadamente 50 kW, y también suministra energía a las estaciones de carga de robots. Cuando muchos robots trabajan al mismo tiempo, la demanda eléctrica y la espera para recargar pueden crecer.

- Comprueba que los robopuertos están conectados a una red eléctrica estable.
- Si hay muchos robots esperando para recargarse, añade capacidad de recarga con más robopuertos distribuidos en el área.
- Mantén conectadas las áreas logísticas de los robopuertos que deban compartir recursos.
- No asumas que colocar robopuertos cerca es suficiente: revisa las líneas de conexión de la red logística.

#### 5.4. Reparación automatizada

Los packs de reparación depositados en un robopuerto quedan disponibles para los robots de construcción que pueden utilizar esa red. Cuando una estructura cubierta por la red sufre daño y hay robots y packs disponibles, los robots pueden repararla automáticamente.

La reparación automática necesita todos estos elementos:

1. Un área de construcción que cubra el edificio dañado.
2. Robots de construcción disponibles en la red.
3. Packs de reparación disponibles.
4. Suministro eléctrico para la red de robopuertos y la recarga de robots.

Si falta alguno de estos elementos, la reparación puede retrasarse o no realizarse.

#### 6. Robots de construcción


#### 6.1. Funciones

Los **robots de construcción** (*Construction robots*) son unidades voladoras que automatizan tareas estructurales. Entre otras funciones, pueden:

- Construir edificios y baldosas indicados por planos o entidades fantasma, siempre que los objetos necesarios estén disponibles.
- Reparar edificios dañados mediante packs de reparación.
- Reemplazar estructuras destruidas cuando la red dispone del objeto necesario en un cofre proveedor o de almacenamiento compatible.
- Retirar objetos marcados con un planificador de desmantelamiento.

Los robots no son invulnerables y necesitan energía para volar. Pueden desplazarse lentamente cuando su batería está muy baja, así que una red con una elevada cantidad de tareas debe disponer de suficiente capacidad de carga.

#### 6.2. Reconstrucción automática

Después de investigar la tecnología **Robótica de construcción**, las estructuras destruidas pueden dejar una entidad fantasma. Si existe el objeto necesario en los cofres adecuados y hay robots disponibles, la red puede reconstruir la estructura.

Para automatizar la reposición de un muro dañado o destruido:

1. Construye los robopuertos de forma que el perímetro esté dentro del área de construcción.
2. Mantén robots de construcción disponibles.
3. Guarda un excedente de los objetos que se puedan reconstruir, como muros y puertas, en cofres compatibles con la red.
4. Conserva packs de reparación en los robopuertos.
5. Revisa que haya electricidad y que los robopuertos formen la red prevista.

La red no puede reconstruir cualquier estructura si no dispone del objeto correcto. Una red con robots pero sin materiales almacenados solo podrá realizar las tareas para las que tenga los recursos necesarios.

#### 7. Packs de reparación


#### 7.1. Función

El **pack de reparación** (*Repair pack*) es un objeto consumible que restaura la salud de una estructura ya colocada. El jugador puede utilizarlo manualmente y los robots de construcción pueden emplearlo para reparar automáticamente edificios cubiertos por una red compatible.

En las estadísticas actuales de la wiki, un pack estándar repara hasta **600 puntos de salud**. La cantidad exacta puede cambiar con la calidad del objeto y las reglas de la versión.

#### 7.2. Fabricación y almacenamiento

Los packs de reparación se fabrican a partir de componentes intermedios, por lo que es útil mantener una pequeña línea de producción estable cuando la fábrica se expande. Consulta la receta del juego o la wiki para comprobar los ingredientes exactos de tu versión.

Buenas prácticas:

- Mantén un inventario de packs de reparación accesible.
- Introduce packs en los robopuertos mediante insertores si quieres automatizar su reposición.
- Comprueba regularmente el stock cuando haya varias zonas de construcción activas.
- Considera aumentar la producción de packs si el sistema de reparación los consume más rápido de lo que se fabrican.

#### 8. Tecnologías relevantes

Las tecnologías disponibles y sus costes pueden depender de la versión y de los ajustes de partida. Las siguientes investigaciones están relacionadas con la protección pasiva, la vigilancia o el mantenimiento de estructuras.

| Tecnología o desbloqueo | Qué permite o mejora | Aplicación práctica |
|---|---|---|
| **Muro de piedra** | Desbloquea la construcción de muros | Crear perímetros y barreras físicas |
| **Puerta** | Desbloquea puertas | Mantener accesos transitables dentro de los muros |
| **Radar** | Desbloquea la vigilancia electrónica del mapa | Revisar zonas periféricas y recursos |
| **Robótica de construcción** | Desbloquea robots de construcción y sus tareas automatizadas | Reparar, construir y reconstruir estructuras |
| **Robótica logística** | Habilita la logística mediante robots y cofres logísticos compatibles | Facilitar el suministro de materiales a una red de construcción |
| **Mejoras de robots** | Pueden aumentar velocidad o capacidad de carga según la investigación | Reducir tiempos de desplazamiento y mejorar el flujo de tareas |

Para conocer los requisitos exactos de cada investigación, consulta el árbol tecnológico dentro de tu partida: puede variar con la versión, los escenarios, los mods y la expansión instalada.

#### 9. Ejemplo: perímetro con reparación automática

El siguiente esquema muestra la organización conceptual de una zona protegida con muros, puertas, vigilancia y mantenimiento automatizado.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Vigilancia periférica | Estado del entorno | Radar periférico | Información para detectar amenazas | — |
| Protección del perímetro | Amenazas exteriores | Muro perimetral y puerta de acceso | Perímetro protegido con acceso controlado | — |
| Cobertura de construcción | Entidades y zona que deben mantenerse | Robopuerto y área de construcción | Área cubierta por robots de construcción | — |
| Suministro de mantenimiento | Packs de reparación y robots disponibles | Robopuerto/red de construcción | Reparación automática de entidades dañadas | — |
| Zona protegida | Instalaciones de producción | Fábrica e instalaciones dentro del perímetro | Infraestructura resguardada | Esquema espacial conceptual, no una plantilla exacta de casillas. |

Es un esquema lógico y no una distribución exacta de casillas. En la construcción real, revisa la cobertura verde de los robopuertos, la conexión entre redes y la ubicación de los radares.

#### 9.1. Secuencia de implementación

1. Delimita el área que necesitas mantener protegida.
2. Construye un muro perimetral y coloca puertas en los accesos necesarios.
3. Instala robopuertos cuya área de construcción cubra las estructuras prioritarias.
4. Añade robots de construcción y packs de reparación.
5. Guarda material de repuesto en cofres compatibles para permitir reconstrucciones.
6. Instala radares y asegúrate de que tienen electricidad.
7. Comprueba los daños, la cobertura y el estado de la red después de ampliar la fábrica.

#### 10. Buenas prácticas y errores comunes

#### 10.1. Mantener cobertura completa

Una línea de muro puede estar dentro del alcance visual de un robopuerto y, sin embargo, tener problemas si los robots no pueden acceder al material de reparación o si el área no está conectada como se esperaba. Activa la visualización de cobertura y revisa cada segmento del perímetro.

#### 10.2. No depender de una única reserva

Si todos los packs de reparación se encuentran en un solo robopuerto, una interrupción de la red o una demanda alta puede dejar otras zonas sin suministro. Distribuye los suministros o automatiza su reposición según la escala del sistema.

#### 10.3. Separar vigilancia y mantenimiento

Los radares mantienen actualizado el mapa, pero no reparan edificios. Los robopuertos facilitan las tareas de construcción, pero no sustituyen por sí solos la exploración cartográfica. Utiliza cada edificio para la función que cumple.

#### 10.4. Evitar redes de robots congestionadas

Si hay demasiadas tareas, pocos robots o escasos puntos de recarga, las reparaciones tardarán. Añade robots gradualmente, comprueba que los robopuertos están conectados y evita extender una única red sobre distancias innecesariamente largas sin evaluar los tiempos de viaje.

#### 10.5. Dejar espacio para la expansión

Un perímetro puede volverse difícil de mantener si bloquea el paso de trenes, impide ampliar cintas o deja las máquinas demasiado cerca de la barrera. Reserva accesos, corredores y espacio para extender el área de construcción.

#### 10.6. No confundir reparación con reconstrucción

Los packs de reparación restauran salud; no sustituyen un objeto que ya fue destruido. Para reconstruir una estructura destruida, la red necesita una entidad fantasma o una orden de construcción, robots disponibles y el objeto requerido en un cofre compatible.

#### 11. Tabla de diagnóstico

| Problema | Causa probable | Qué comprobar |
|---|---|---|
| Los robots no reparan una estructura dañada | No hay packs, robots o cobertura | Inventario de packs, robots disponibles y área de construcción |
| Una estructura destruida no se reconstruye | No hay material disponible o no existe una tarea de reconstrucción | Entidad fantasma, cofres compatibles y objetos en stock |
| Los robots se quedan esperando para recargar | Hay demasiados robots para la capacidad de carga local | Cantidad y distribución de robopuertos; energía disponible |
| Dos robopuertos no comparten robots | Sus áreas logísticas no se conectan | Cobertura naranja y línea de conexión entre robopuertos |
| El radar deja de actualizar el mapa | Falta energía o se reduce su suministro | Conexión eléctrica y capacidad de generación |
| Una puerta no responde como se espera | Está abierta por el jugador, vehículo o circuito, o la condición no es la esperada | Estado de la puerta, proximidad y configuración de circuitos |
| El perímetro se daña repetidamente | La barrera no cubre bien una zona o los daños no se reparan con rapidez | Continuidad del muro, acceso de robots, packs y materiales de repuesto |
| Se agotan los packs de reparación | El uso supera la producción o reposición | Fabricación, insertores, inventarios de robopuertos y demanda de reparación |

#### 12. Lista de comprobación

- [ ] Los muros forman un perímetro continuo en las zonas necesarias.
- [ ] Las puertas están en accesos útiles y funcionan como se espera.
- [ ] Los radares tienen electricidad y cubren las zonas que interesa vigilar.
- [ ] La red de robopuertos tiene cobertura suficiente sobre las estructuras prioritarias.
- [ ] Las áreas logísticas de los robopuertos se conectan donde deben compartir recursos.
- [ ] Hay robots de construcción disponibles y suficientes puntos de recarga.
- [ ] Los robopuertos reciben packs de reparación.
- [ ] Existen materiales de repuesto para reconstruir las estructuras destruidas.
- [ ] La producción de packs y los inventarios se revisan periódicamente.
- [ ] El diseño deja espacio para ampliar la fábrica sin invalidar el perímetro.

#### 13. Fuentes de referencia

Las páginas de la *Factorio Wiki* contienen fichas de objetos, recetas y mecánicas más detalladas:

- [Wall — Muro](https://wiki.factorio.com/Wall)
- [Gate — Puerta](https://wiki.factorio.com/Gate)
- [Radar — Radar](https://wiki.factorio.com/Radar)
- [Roboport — Robopuerto](https://wiki.factorio.com/Roboport)
- [Construction robot — Robot de construcción](https://wiki.factorio.com/Construction_robot)
- [Repair pack — Pack de reparación](https://wiki.factorio.com/Repair_pack)
- [Construction robotics — Tecnología de robótica de construcción](https://wiki.factorio.com/Construction_robotics_(research))
- [Logistic network — Red logística](https://wiki.factorio.com/Logistic_network)
- [Military units and structures — Objetivos y estructuras](https://wiki.factorio.com/Military_units_and_structures)

---

### Guía técnica de Factorio: Módulos (velocidad, productividad, eficiencia) y balizas (*beacons*)

**Versión de referencia:** *Factorio 2.x*. Las funciones exclusivas de la expansión *Space Age* se identifican por separado. Las cifras de los módulos corresponden a calidad normal, salvo que se indique lo contrario.

Los **módulos** son objetos que se insertan en edificios compatibles para modificar su rendimiento. Permiten que una máquina produzca más rápido, obtenga productos adicionales a partir de los mismos insumos o consuma menos energía. Las **balizas** transmiten ciertos efectos de módulos a otras máquinas cercanas, lo que permite construir líneas de producción compactas y especializadas.

Esta mecánica es importante en la etapa intermedia y avanzada de una fábrica: cuando ya existe una cadena de producción estable, los módulos permiten aumentar su capacidad, reducir el consumo de recursos escasos o controlar la energía y la contaminación.

---

#### 1. Fundamentos de los módulos

##### 1.1. ¿Qué es un módulo?

Un módulo es un objeto mejorador que se coloca en una ranura disponible de una máquina. No todos los edificios admiten módulos; la interfaz del edificio indica cuántas ranuras tiene y qué efectos son posibles para la receta elegida.

Los tres grupos principales del juego base son:

- **Módulos de velocidad:** aumentan la velocidad de trabajo, a cambio de elevar el consumo energético.
- **Módulos de productividad:** aumentan la cantidad total de productos obtenidos por una cantidad determinada de insumos, pero normalmente reducen la velocidad y elevan el consumo.
- **Módulos de eficiencia:** reducen el consumo energético de la máquina y, por esa vía, también pueden disminuir la contaminación asociada a su funcionamiento.

En *Factorio 2.x* con *Space Age* existe además una categoría de **módulos de calidad**, que aumenta la probabilidad de obtener objetos de mayor calidad en recetas compatibles. No reemplazan los tres tipos principales, sino que añaden otra opción de optimización.

Página general: [Factorio Wiki: Module](https://wiki.factorio.com/Module).

##### 1.2. Cómo se aplican los efectos

Los efectos de varios módulos se combinan principalmente de forma aditiva. Por ejemplo, dos módulos de velocidad de nivel 1 suman sus bonificaciones de velocidad; no se multiplican una a continuación de la otra.

No obstante, el resultado final depende de los límites de la máquina y de las reglas del efecto. En particular:

- La velocidad final combina la velocidad base del edificio con los modificadores de la máquina y sus módulos.
- La productividad se aplica solamente a las recetas y edificios que admiten ese efecto. Suele acumular progreso en una barra de productividad y genera productos extra cuando la barra completa un ciclo.
- El consumo energético tiene un límite inferior: los módulos de eficiencia no pueden reducir por sí solos el consumo de una máquina eléctrica por debajo del 20 % de su consumo base.
- Los efectos negativos y positivos también se suman. Por ejemplo, un módulo de productividad puede reducir la velocidad, mientras que uno de velocidad puede compensarla parcialmente.
- Un módulo no crea por sí solo los materiales que faltan: la producción seguirá limitada por el suministro de insumos, la capacidad de las cintas, los insert ores, la energía y el espacio de almacenamiento de salida.

##### 1.3. Fabricación e investigación

Los módulos se desbloquean mediante investigación y se fabrican en edificios de producción compatibles. Los niveles superiores requieren más materiales y, por lo general, módulos de nivel inferior o componentes electrónicos más avanzados.

Como regla general, la cadena de fabricación utiliza componentes electrónicos y materiales industriales, como:

- **Circuitos electrónicos:** componentes básicos de automatización.
- **Circuitos avanzados:** necesarios para módulos y versiones de mayor nivel.
- **Plástico y otros productos intermedios:** necesarios en recetas de módulos y en cadenas electrónicas.
- **Módulos de nivel inferior:** necesarios para fabricar ciertas versiones superiores.

Los ingredientes y tiempos exactos pueden variar según el nivel del módulo y la versión del juego. Consulta las fichas individuales de la wiki antes de dimensionar una fábrica de módulos:

- [Speed module](https://wiki.factorio.com/Speed_module)
- [Productivity module](https://wiki.factorio.com/Productivity_module)
- [Efficiency module](https://wiki.factorio.com/Efficiency_module)
- [Beacon](https://wiki.factorio.com/Beacon)

---

#### 2. Módulos de velocidad

##### 2.1. Función

Los **módulos de velocidad** aumentan la velocidad a la que trabaja una máquina compatible. Son útiles cuando una fábrica necesita más productos por segundo y no resulta conveniente —o posible— construir más máquinas.


##### 2.2. Efectos por nivel

| Módulo | Bonificación de velocidad | Aumento del consumo eléctrico | Penalización de calidad en *Factorio 2.x* |
|---|---:|---:|---:|
| Módulo de velocidad 1 | +20 % | +50 % | −1 % |
| Módulo de velocidad 2 | +30 % | +60 % | −1,5 % |
| Módulo de velocidad 3 | +50 % | +70 % | −2,5 % |

Los módulos de velocidad aumentan la producción por unidad de tiempo, pero elevan el consumo de energía por segundo y, normalmente, la contaminación de la máquina. El efecto puede seguir siendo útil por producto fabricado, porque la máquina también termina más rápido cada ciclo.

En *Factorio 2.x*, la penalización de calidad reduce el efecto positivo de los módulos de calidad en recetas compatibles; no puede llevar la probabilidad de calidad por debajo de su mínimo permitido.

##### 2.3. Cuándo conviene utilizarlos

- Cuando la máquina es el cuello de botella de una línea.
- Cuando el recurso de entrada es abundante y el objetivo principal es aumentar el caudal de salida.
- Cuando una instalación compacta debe abastecer a muchas máquinas consumidoras.
- En edificios que procesan recetas largas, siempre que las cintas y la energía puedan sostener el aumento de producción.
- En combinación con módulos de productividad, si la velocidad perdida por productividad está reduciendo demasiado el rendimiento global.

##### 2.4. Precauciones

No coloques módulos de velocidad en todas las máquinas sin comprobar la red. Un aumento de la velocidad puede elevar el consumo energético y la contaminación, y crear nuevos cuellos de botella aguas abajo. Una ensambladora más rápida no mejora el rendimiento de la fábrica si los insert ores no pueden suministrarle ingredientes o la cinta de salida ya está saturada.

---

#### 3. Módulos de productividad

##### 3.1. Función

Los **módulos de productividad** permiten obtener una cantidad adicional de productos sin consumir una cantidad proporcional de insumos. Cada vez que la máquina completa una receta, el módulo añade progreso a una barra de productividad. Cuando esa barra alcanza el 100 %, se genera un conjunto adicional de productos.


Esta bonificación es especialmente valiosa en recetas que consumen muchos recursos o forman parte de cadenas industriales largas. Si se producen productos adicionales en una etapa intermedia, la fábrica necesita extraer, transportar y procesar menos materias primas para lograr la misma producción final.

##### 3.2. Efectos por nivel

| Módulo | Bonificación de productividad | Aumento del consumo eléctrico | Penalización de velocidad | Aumento de contaminación propio del efecto |
|---|---:|---:|---:|---:|
| Módulo de productividad 1 | +4 % | +40 % | −5 % | +5 % |
| Módulo de productividad 2 | +6 % | +60 % | −10 % | +7 % |
| Módulo de productividad 3 | +10 % | +80 % | −15 % | +10 % |

La bonificación de productividad y la penalización de velocidad no son lo mismo. La productividad permite generar más unidades por insumo, mientras que la penalización reduce la cantidad de ciclos que la máquina completa por unidad de tiempo. El resultado neto depende de la receta, el nivel del módulo y los demás modificadores aplicados.

##### 3.3. Restricciones de uso

Los módulos de productividad no son válidos en todas las recetas. Normalmente se permiten en determinados procesos que producen artículos intermedios, pero existen excepciones definidas por el juego.

- La interfaz de la máquina muestra las ranuras disponibles y rechaza los módulos no permitidos para la receta activa.
- Si se cambia la receta a otra incompatible, el edificio puede expulsar los módulos de productividad insertados.
- Las balizas no admiten módulos de productividad.
- El hecho de que una máquina acepte módulos no significa que todas sus recetas acepten productividad.

Antes de diseñar una línea, comprueba la compatibilidad de la receta concreta en la [Factorio Wiki: Productivity module](https://wiki.factorio.com/Productivity_module).

##### 3.4. Cuándo conviene utilizarlos

- En recetas con consumo elevado de minerales, placas, fluidos u otros materiales costosos.
- En la producción de artículos intermedios que se utilizan en muchas recetas posteriores, siempre que la receta admita productividad.
- En fases avanzadas donde la prioridad es ahorrar recursos o reducir la cantidad de extracción y procesamiento necesaria.
- En líneas con espacio suficiente para compensar la menor velocidad de las máquinas.

##### 3.5. Ejemplo conceptual

Supongamos una máquina cuya receta consume 100 unidades de insumo para fabricar 100 productos. Si el proceso recibe una bonificación de productividad total del 20 %, puede generar 120 productos con esos mismos 100 insumos, siempre que la receta permita el efecto y no se apliquen otras reglas especiales.

El beneficio no significa que la máquina trabaje necesariamente un 20 % más rápido. Puede tardar más por las penalizaciones de velocidad y, aun así, consumir menos recursos por producto final.

---

#### 4. Módulos de eficiencia

##### 4.1. Función

Los **módulos de eficiencia** reducen el consumo energético de una máquina. Como la contaminación de muchos edificios eléctricos está relacionada con su consumo de energía, también pueden ayudar a disminuir la contaminación generada por la producción.


##### 4.2. Efectos por nivel

| Módulo | Reducción del consumo eléctrico nominal |
|---|---:|
| Módulo de eficiencia 1 | −30 % |
| Módulo de eficiencia 2 | −40 % |
| Módulo de eficiencia 3 | −50 % |

Los módulos del mismo tipo acumulan sus efectos. En máquinas compatibles, el consumo de energía no puede bajar del 20 % de su valor base mediante efectos de eficiencia. Este límite evita reducirlo a cero.

La regla del 20 % no significa que los módulos de eficiencia sean inútiles por encima del 80 % de reducción: otros módulos pueden aumentar el consumo, y el resultado combina los modificadores aplicables antes de respetar el límite mínimo.

##### 4.3. Cuándo conviene utilizarlos

- En máquinas que consumen mucha electricidad y trabajan durante periodos prolongados.
- En zonas donde se quiere reducir el consumo eléctrico agregado.
- En instalaciones cercanas a zonas sensibles a la contaminación, cuando la reducción de consumo también reduce la contaminación de la máquina.
- En fábricas con una red energética limitada o con un costo elevado de ampliar la generación.
- En máquinas que tienen que funcionar con regularidad, ya que el beneficio acumulado se obtiene durante toda su operación.

##### 4.4. Límites y consideraciones

La eficiencia no aumenta directamente la velocidad ni la productividad. Su ventaja es reducir la energía usada para operar la máquina. En algunos casos, una combinación de productividad y velocidad puede producir más artículos por unidad de energía que una combinación centrada exclusivamente en eficiencia; para saberlo hay que comparar el rendimiento real de la receta.

Los efectos de eficiencia también se aplican de acuerdo con las reglas del edificio. Por ejemplo, reducen el consumo de nutrientes de la biochamber en *Space Age*, además del consumo energético en máquinas eléctricas compatibles.

---

#### 5. Tabla comparativa de los módulos

| Criterio | Velocidad | Productividad | Eficiencia |
|---|---|---|---|
| Objetivo principal | Más productos por segundo | Más productos por insumo | Menor consumo de energía |
| Efecto sobre velocidad | Aumenta | Reduce | No aumenta directamente |
| Efecto sobre consumo | Lo aumenta | Lo aumenta | Lo reduce |
| Efecto sobre productividad | No concede productividad directa | La aumenta en recetas compatibles | No la aumenta directamente |
| Efecto sobre contaminación | Suele aumentarla por el mayor consumo | La aumenta según el modificador del módulo y el consumo | Puede reducirla al reducir el consumo |
| Uso característico | Eliminar un cuello de botella | Ahorrar recursos en procesos compatibles | Reducir consumo y contaminación |
| Limitación principal | Más demanda eléctrica y posible congestión aguas abajo | Menor velocidad y restricciones por receta | No aporta directamente más velocidad ni más productos por insumo |

No existe un módulo universalmente mejor. La elección depende del problema que se pretende resolver: capacidad, consumo de materias primas, energía o contaminación.

---

#### 6. Edificios que admiten módulos

Los módulos se pueden colocar en distintos edificios industriales que dispongan de ranuras. La compatibilidad depende del edificio y, en algunos casos, de la receta seleccionada.

| Edificio | Ejemplos de aplicación |
|---|---|
| **Máquina de montaje** | Aumentar o modificar el rendimiento de recetas de componentes y objetos. |
| **Taladro minero eléctrico** | Aumentar el caudal extraído, mejorar el aprovechamiento del yacimiento o reducir el consumo. |
| **Horno eléctrico** | Acelerar la fundición, mejorar la productividad de recetas compatibles o reducir el consumo energético. |
| **Refinería de petróleo** | Modificar el rendimiento de recetas de procesamiento de petróleo. |
| **Planta química** | Ajustar recetas químicas, como plástico, azufre o productos derivados del petróleo. |
| **Centrífuga** | Ajustar procesos compatibles de uranio y otras recetas disponibles. |
| **Laboratorio** | Modificar el rendimiento de investigación cuando la configuración lo permite. |
| **Edificios industriales de *Space Age*** | Las fundiciones, plantas electromagnéticas, plantas criogénicas y biochambers tienen usos específicos de módulos, según sus ranuras y recetas. |

Las máquinas a combustión no pueden beneficiarse de la misma forma que edificios eléctricos compatibles; las balizas tampoco afectan a edificios que no admiten módulos. Revisa la interfaz de cada edificio para conocer el número de ranuras y los tipos aceptados.

---

#### 7. Balizas (*beacons*)

##### 7.1. ¿Qué es una baliza?

La **baliza** es un edificio eléctrico que transmite los efectos de los módulos que contiene a máquinas compatibles cercanas. No necesita recibir ingredientes para fabricar productos y no produce artículos: modifica el rendimiento de edificios dentro de su área de influencia.


**Características de la baliza de calidad normal:**

- Tamaño: 3 × 3 casillas.
- Área de suministro: 9 × 9 casillas.
- Ranuras: 2 módulos.
- Consumo eléctrico: 480 kW.
- Transmite efectos a edificios compatibles que se encuentren dentro de su área.

La baliza permite usar un conjunto de módulos para afectar a varias máquinas, ahorrando módulos frente a la opción de insertar módulos en cada máquina. A cambio, consume electricidad de forma continua mientras está conectada y no puede transmitir cualquier tipo de módulo.

##### 7.2. Qué módulos acepta

Las balizas pueden contener **módulos de velocidad** y **módulos de eficiencia**. No aceptan módulos de productividad ni de calidad.

- **Velocidad en balizas:** aumenta la velocidad de las máquinas alcanzadas, a cambio de un mayor consumo energético y posibles efectos sobre contaminación y calidad.
- **Eficiencia en balizas:** reduce el consumo de las máquinas afectadas, dentro de los límites aplicables.
- **Productividad:** debe insertarse directamente en máquinas y recetas compatibles; no puede transmitirse desde una baliza.
- **Calidad:** en *Space Age*, los módulos de calidad tampoco pueden instalarse en balizas.

Las balizas no transmiten efectos a otras balizas ni reducen su propio coste energético con sus propios módulos.

##### 7.3. Transmisión de efectos y rendimientos decrecientes

Una máquina cubierta por una baliza no recibe, en general, el efecto completo de cada módulo instalado en ella. El efecto se multiplica por un factor de transmisión. En *Factorio 2.0*, ese factor depende de la eficiencia de distribución de la baliza y del número de balizas que afectan a la máquina.

Para una baliza de calidad normal, la eficiencia de distribución es 1,5. Cuando varias balizas afectan a la misma máquina, el efecto total aumenta, pero cada baliza adicional aporta menos rendimiento marginal. Por eso, suele ser eficiente colocar balizas entre varias máquinas, en vez de rodear una sola máquina con una gran cantidad de ellas sin medir el resultado.

Las balizas de mayor calidad mejoran la eficiencia de transmisión y consumen menos energía que las de calidad normal. Estos detalles importan en construcciones avanzadas que usan objetos de distinta calidad.

##### 7.4. Cómo ubicar las balizas

- Agrupa máquinas compatibles para que una misma baliza afecte a varias.
- Comprueba visualmente el área de suministro antes de fijar un plano de construcción.
- Asegura que las máquinas puedan recibir los insumos y evacuar los productos con independencia del espacio ocupado por las balizas.
- Calcula el consumo eléctrico agregado de todas las balizas de la instalación.
- Evita balizas que trabajan constantemente para máquinas casi siempre inactivas, salvo que el diseño tenga una razón concreta para ello.
- No asumas que añadir más balizas siempre es eficiente; compara la mejora de velocidad con el costo adicional de energía y módulos.

##### 7.5. Esquema conceptual de una línea con balizas

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Emisión de efectos de módulos | Módulos de velocidad | Primera baliza | Efectos de módulos transmitidos | — |
| Emisión de efectos de módulos | Módulos de velocidad | Segunda baliza | Efectos de módulos transmitidos | — |
| Recepción de efectos | Efectos transmitidos por las balizas | Máquinas A, B y C | Bonificaciones recibidas por las máquinas compatibles | Las máquinas deben estar dentro del área de suministro y ser compatibles con los módulos transmitidos. |

El diagrama muestra la idea general de la transmisión; no representa una disposición exacta de casillas. La ubicación óptima depende del tamaño de las máquinas, el espacio para cintas, las conexiones eléctricas y el número de balizas superpuestas.

---

#### 8. Ejemplos de configuraciones

##### 8.1. Priorizar la velocidad de una línea de montaje

Una cadena que fabrica componentes con recetas largas puede beneficiarse de módulos de velocidad en máquinas con pocas ranuras. Si hay varias máquinas juntas, las balizas pueden compartir módulos de velocidad entre ellas.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Entrada de producción | Ingredientes | Cintas de entrada | Ingredientes transportados | — |
| Fabricación con efectos de balizas | Ingredientes y efectos de balizas de velocidad | Máquinas de montaje | Productos fabricados | — |
| Salida de producción | Productos fabricados | Cinta de salida de productos | Productos transportados | — |

Antes de aumentar la velocidad, confirma que las cintas pueden entregar todos los insumos, que las máquinas posteriores pueden consumir el producto y que la red eléctrica soporta el nuevo consumo.

##### 8.2. Ahorrar recursos con productividad

Para una receta compatible, instala módulos de productividad directamente en la máquina. Una bonificación de productividad implica que parte de la producción se obtiene sin consumir una cantidad adicional de ingredientes, lo que puede reducir la demanda total de minería, fundición y transporte.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Consumo de insumos | Insumos de una receta | Máquina con módulos de productividad | Menos insumos por unidad final, según el esquema | — |
| Producción adicional | Mismos insumos disponibles | Máquina con módulos de productividad | Más productos totales por los mismos insumos | — |

La contrapartida es una velocidad de trabajo menor y un consumo por ciclo mayor. Si se necesita mantener una producción por segundo elevada, puede ser conveniente combinar productividad con velocidad o compensar mediante más máquinas.

##### 8.3. Reducir el consumo eléctrico

Los módulos de eficiencia son útiles cuando una instalación trabaja de forma constante y consume mucha electricidad. Su efecto es especialmente interesante en fábricas grandes donde pequeñas reducciones en muchas máquinas se suman.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Aplicación de eficiencia | Máquina compatible | Módulos de eficiencia | Consumo eléctrico reducido | — |
| Efecto energético | Menor consumo eléctrico | Red eléctrica de la fábrica | Menor presión sobre la generación de energía | — |

Si el objetivo principal es reducir la contaminación, revisa también el tipo de edificio y la relación entre su consumo y su generación de contaminación. La eficiencia disminuye el consumo, pero no elimina necesariamente la contaminación de toda la cadena.

##### 8.4. Combinar módulos y balizas

En una instalación avanzada, una opción frecuente consiste en insertar módulos de productividad en las máquinas que producen componentes valiosos y usar balizas para transmitir velocidad. Así se busca mantener el ahorro de recursos por productividad sin aceptar toda la reducción de velocidad.

La combinación debe evaluarse según la receta, porque los efectos sobre el consumo, la velocidad y la productividad interactúan y la baliza consume energía adicional. No existe una disposición óptima universal para todas las fábricas.

---

#### 9. Estrategias y consejos de optimización

##### 9.1. Identifica el problema antes de colocar módulos

- Si falta producción por segundo, revisa primero los cuellos de botella y considera velocidad.
- Si se consume demasiada materia prima, comprueba si la receta admite productividad.
- Si el problema es la demanda eléctrica o la contaminación, evalúa eficiencia.
- Si muchas máquinas están agrupadas, analiza si las balizas permiten compartir módulos.
- Si el problema ocurre en una cinta, un insertor o el suministro de ingredientes, los módulos no lo solucionarán por sí solos.

##### 9.2. Mide el rendimiento total, no solo el porcentaje mostrado

Un aumento de velocidad no implica necesariamente el mismo porcentaje de aumento en la producción final: puede haber falta de insumos, límites de inserción, bloqueo de salida o escasez de potencia. Del mismo modo, una bonificación de productividad puede ahorrar recursos, aunque la máquina produzca menos artículos por segundo debido a su penalización de velocidad.

##### 9.3. No sobrecargues la red eléctrica

Los módulos de velocidad y productividad pueden aumentar el consumo de energía. Las balizas también consumen electricidad aun cuando las máquinas afectadas estén detenidas. Después de mejorar una línea, comprueba la gráfica de producción y consumo de la red para asegurarte de que las nuevas demandas no ralentizan toda la fábrica.

##### 9.4. Optimiza la productividad en etapas de alto costo

La productividad resulta especialmente valiosa en etapas donde un producto se utiliza en muchas cadenas posteriores, o donde producirlo requiere materiales costosos. Obtener productos extra puede evitar parte de la minería, fundición y fabricación de componentes necesaria para satisfacer una demanda final.

##### 9.5. Usa la eficiencia donde aporte valor real

La eficiencia puede ser excelente para controlar el consumo, pero el costo de fabricar módulos de nivel superior también importa. No necesitas reemplazar inmediatamente todos los módulos: empieza por las máquinas que consumen más energía o las zonas donde la reducción de consumo resulte más útil.

##### 9.6. Diseña planos que permitan ampliaciones

En diseños con balizas, deja corredores para cintas y postes, mantén accesibles las máquinas y evita diseños en los que sustituir un módulo exija reconstruir toda la zona. La modularidad facilita probar configuraciones y cambiar la mezcla de módulos conforme cambie la demanda.

---

#### 10. Errores comunes y soluciones

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| La producción no aumenta tras instalar módulos de velocidad. | Falta de insumos, salida bloqueada o límite de transporte. | Comprueba cintas, insertores y capacidad de las máquinas posteriores. |
| La fábrica se queda sin electricidad. | Los módulos de velocidad/productividad o las balizas elevaron el consumo. | Amplía la generación o usa módulos de eficiencia donde corresponda. |
| Un módulo de productividad no puede colocarse. | La receta o edificio no admite productividad. | Comprueba las restricciones de la receta en la interfaz y la wiki. |
| La máquina parece producir menos por segundo con productividad. | La bonificación de productividad viene acompañada de penalización de velocidad. | Compara productos por insumo y productos por segundo; usa velocidad o más máquinas si hace falta. |
| Una baliza no afecta a una máquina. | El edificio no admite módulos, está fuera del área o la receta no permite el efecto relevante. | Revisa la cobertura y compatibilidad de la máquina. |
| El consumo aumenta sin un beneficio claro. | Se instalaron módulos de velocidad sin necesidad o se colocaron balizas sobre máquinas inactivas. | Mide la demanda real y elimina los efectos que no mejoran el objetivo. |
| La calidad de los productos baja al usar módulos de velocidad. | En *Factorio 2.x*, los módulos de velocidad penalizan la probabilidad de calidad. | Si la calidad es prioritaria, compara la configuración con módulos de calidad y evita una penalización innecesaria. |
| Se instalan demasiadas balizas alrededor de una máquina. | No se consideró el rendimiento decreciente de cada baliza adicional. | Agrupa máquinas para compartir efectos y compara el beneficio marginal. |

---

#### 11. Particularidades de *Space Age* y de la calidad

La expansión *Space Age* añade nuevas máquinas, recetas y objetos de distinta calidad. Algunas máquinas avanzadas tienen más ranuras o bonificaciones integradas, y las reglas de compatibilidad dependen del edificio y de la receta concreta.

Los **módulos de calidad** son exclusivos de esta expansión. Aumentan la probabilidad de producir objetos de calidad superior en recetas compatibles, pero reducen ligeramente la velocidad de trabajo. Su efecto no se puede transmitir mediante una baliza, ya que las balizas no admiten módulos de calidad.

La calidad de una máquina, de sus módulos y de la baliza puede mejorar ciertos aspectos de la producción. En diseños avanzados, la calidad puede combinarse con velocidad y productividad para aumentar el rendimiento de pocas máquinas, pero debe tenerse en cuenta el costo de conseguir módulos y edificios de mayor calidad.

La baliza también tiene bonificaciones según su propia calidad: las de calidad superior presentan una eficiencia de transmisión más alta y un consumo energético inferior. Esto puede importar en diseños densos con muchas balizas, aunque conseguirlas requiere una cadena de fabricación y selección de calidad adecuada.

Para ver los valores vigentes y las compatibilidades específicas, consulta [Factorio Wiki: Quality](https://wiki.factorio.com/Quality) y [Factorio Wiki: Beacon](https://wiki.factorio.com/Beacon).

---

#### 12. Lista de comprobación

Antes de considerar terminada una instalación con módulos y balizas, comprueba lo siguiente:

- [ ] El edificio admite módulos y dispone de ranuras disponibles.
- [ ] La receta seleccionada acepta el tipo de módulo que quieres utilizar.
- [ ] La combinación de módulos responde a un objetivo claro: velocidad, productividad o eficiencia.
- [ ] La producción de entrada puede sostener el nuevo rendimiento.
- [ ] Las cintas, insertores y máquinas posteriores pueden procesar la salida adicional.
- [ ] La red eléctrica soporta el consumo añadido de los módulos de velocidad/productividad y de las balizas.
- [ ] La reducción de energía con módulos de eficiencia respeta el límite mínimo aplicable.
- [ ] Las balizas están dentro del alcance de las máquinas y se colocaron para afectar a varios edificios cuando sea posible.
- [ ] Se consideró el rendimiento decreciente cuando varias balizas afectan a una misma máquina.
- [ ] Si se emplean módulos de calidad en *Space Age*, se tuvo en cuenta la penalización de calidad de los módulos de velocidad.
- [ ] Se revisó el rendimiento real tras aplicar los módulos, en lugar de basarse únicamente en los porcentajes nominales.

---

#### 13. Fuentes de referencia

- [Module — descripción general de los módulos](https://wiki.factorio.com/Module)
- [Speed module — módulo de velocidad](https://wiki.factorio.com/Speed_module)
- [Productivity module — módulo de productividad](https://wiki.factorio.com/Productivity_module)
- [Efficiency module — módulo de eficiencia](https://wiki.factorio.com/Efficiency_module)
- [Beacon — baliza](https://wiki.factorio.com/Beacon)
- [Productivity — mecánica de productividad](https://wiki.factorio.com/Productivity)
- [Quality — sistema de calidad de *Factorio 2.x* y *Space Age*](https://wiki.factorio.com/Quality)


---

### Guía técnica de Factorio: Uso de planos (*blueprints*) y librerías de construcción

**Versión de referencia:** *Factorio 2.x*. Las funciones de parametrización y los elementos que dependen de *Space Age* se indican cuando corresponde.

Los **planos** (*blueprints*) permiten guardar una distribución de edificios, entidades y baldosas para reproducirla en distintas zonas de la fábrica. En lugar de construir una instalación desde cero cada vez, el jugador puede colocar su plano, generar estructuras fantasma y dejar que los robots de construcción completen el trabajo cuando dispongan de los materiales necesarios.

La **librería de planos** organiza estos diseños y permite conservarlos entre partidas. Junto con los libros de planos, los planos de mejora y los planificadores de desmantelamiento, transforma los diseños repetibles en una herramienta esencial para escalar una fábrica.

---

#### 1. Fundamentos de los planos

##### 1.1. ¿Qué es un plano?

Un plano es una plantilla reutilizable que registra una disposición de edificios y otros elementos compatibles. Puede representar, por ejemplo:

- Una línea de hornos para fundir hierro o cobre.
- Un bloque de generación solar con acumuladores.
- Un módulo de producción de circuitos.
- Una estación ferroviaria con sus vías y señales.
- Una instalación química con sus tuberías y conexiones.
- Una sección de muros, puertas, radares y edificios de apoyo.

Al colocar un plano sobre el terreno, el juego muestra **estructuras fantasma** (*ghosts*): indican qué debería construirse y dónde. Esos fantasmas sirven como guía para construir manualmente o como pedidos que pueden completar los robots de construcción.

Un plano no es una fábrica completa en funcionamiento por sí solo. Los edificios que representa necesitan los recursos, la electricidad, el combustible, las conexiones y la logística adecuados para operar.

##### 1.2. ¿Qué información puede guardar?

Según los elementos incluidos, un plano puede conservar posiciones, orientaciones y configuraciones importantes de la instalación. Por ejemplo, puede guardar:

- La posición y orientación de edificios.
- Las baldosas incluidas en la selección, como hormigón o ladrillos de piedra.
- La receta seleccionada en máquinas compatibles.
- Los módulos instalados en edificios que admiten módulos.
- Configuraciones compatibles de filtros, señales o comportamientos de entidades.
- Conexiones de circuitos y determinados datos de configuración de la instalación.
- Elementos ferroviarios y horarios de tren cuando la selección los incluye y la versión los admite.

El plano **no duplica automáticamente el inventario real** de cofres, máquinas o vagones. Su propósito es reproducir el diseño y las configuraciones compatibles, no copiar las existencias almacenadas dentro de cada edificio.

##### 1.3. ¿Por qué son importantes?

Los planos aportan varias ventajas a la progresión de la fábrica:

- **Repetibilidad:** permiten volver a construir diseños que ya fueron probados.
- **Escalabilidad:** facilitan ampliar una línea productiva por módulos.
- **Consistencia:** reducen errores de colocación y diferencias entre instalaciones equivalentes.
- **Velocidad de construcción:** evitan ubicar manualmente cada edificio de una instalación grande.
- **Colaboración:** permiten compartir diseños entre partidas o jugadores, según dónde se guarden.
- **Mantenimiento:** hacen más sencilla la sustitución o expansión de módulos estandarizados.

##### 1.4. Plano, fantasma y construcción terminada

Estos tres estados no son lo mismo:

1. **Plano:** la plantilla guardada que puede reutilizarse.
2. **Fantasma:** la representación de un edificio o baldosa pendiente de construcción en el mundo.
3. **Construcción terminada:** la entidad física que se ha colocado manualmente o que han construido los robots.

Colocar el plano no significa que todos los edificios ya estén construidos. Para que los robots actúen, necesitan cobertura de una red de construcción, materiales disponibles y robots operativos. Sin robots, el jugador puede usar los fantasmas como guía para colocar los edificios a mano.


---

#### 2. Crear, editar y colocar un plano

##### 2.1. Crear un plano a partir de una instalación existente

El procedimiento habitual es:

1. Abre la herramienta de planos desde la barra de accesos rápidos o la interfaz correspondiente.
2. Selecciona un rectángulo que cubra los edificios y elementos que deseas copiar.
3. Revisa el área marcada: comprueba que no falten cintas, postes, insertores, tuberías, señales o baldosas importantes.
4. En la ventana de configuración, asigna un nombre descriptivo e iconos reconocibles.
5. Revisa el resumen de componentes si necesitas conocer los materiales que exige la construcción.
6. Confirma la creación del plano y guárdalo en tu inventario, un libro o la librería.

El tamaño máximo de un plano puede llegar a **10.000 × 10.000 baldosas**, aunque una plantilla de ese tamaño suele ser poco práctica para manejar, revisar y construir. En general, es preferible dividir instalaciones grandes en módulos manejables.

##### 2.2. Nombrar e identificar los planos

Un nombre útil describe tanto la función como la escala o las condiciones del diseño. Por ejemplo:

- `Fundición - hierro - 24 hornos`
- `Energía - solar - bloque estándar`
- `Trenes - estación de carga - 2 vías`
- `Circuitos - producción de verdes - módulo A`
- `Minería - hierro - salida por cinta rápida`

Los iconos ayudan a encontrar diseños de un vistazo. Es recomendable utilizar categorías consistentes y evitar nombres genéricos como `Plano nuevo` o `Diseño 4`.

En la descripción se pueden añadir requisitos importantes: nivel de cinta, tipo de combustible, materiales especiales, consumo eléctrico, dirección de entrada o condiciones para enlazar el módulo con otras instalaciones.

##### 2.3. Colocar un plano

Para usar un plano guardado:

1. Selecciónalo desde la barra rápida, el inventario, un libro o la librería.
2. Mueve el cursor al lugar donde quieres construirlo.
3. Comprueba la vista previa: orientación, colisiones, accesos y conexiones con la fábrica existente.
4. Gira el diseño según sea necesario.
5. Colócalo para crear las estructuras fantasma.
6. Proporciona los materiales necesarios y deja que los robots construyan, o sigue las siluetas manualmente.

Por defecto, `R` rota el plano. `H` y `V` permiten reflejarlo horizontal o verticalmente. Sin embargo, los planos que contienen señales ferroviarias, señales de cadena o estaciones de tren no se pueden reflejar; solo pueden rotarse. Los atajos pueden modificarse en la configuración del juego.

Los planos también pueden colocarse desde la vista del mapa en zonas exploradas, lo que permite diseñar expansiones a distancia sin desplazarse primero hasta cada punto.

##### 2.4. Colocación forzada y obstáculos

El juego dispone de modos de colocación que ayudan cuando hay vegetación, rocas u otras estructuras en el área:

- **Colocación normal:** los edificios incompatibles o los obstáculos pueden impedir la colocación de determinadas partes del plano.
- **Colocación forzada (`Shift` al colocar, con controles predeterminados):** marca árboles y rocas bajo el plano para desmantelamiento y puede añadir fantasmas de relleno al construir sobre agua. En *Space Age*, la superficie apropiada puede ser plataforma de hielo o cimentación según el planeta.
- **Colocación superforzada (`Ctrl` + `Shift`, con controles predeterminados):** también marca otras construcciones que interfieren para desmantelamiento. Sobre cintas transportadoras, el juego puede intentar sustituir los tramos que interfieren por fantasmas de cintas subterráneas del mismo nivel, cuando sea posible.

Estos modos deben usarse con cuidado: pueden marcar para desmantelamiento elementos que se pretendía conservar. Antes de confirmar, revisa la zona y comprueba qué objetos quedarían afectados.


---

#### 3. Librería de planos

##### 3.1. ¿Qué es la librería?

La **librería de planos** es un inventario virtual para almacenar y organizar planos, libros de planos, planificadores de desmantelamiento y planificadores de mejora. A diferencia de los objetos guardados en un cofre, los elementos de la librería no ocupan espacio normal del inventario y no se pierden porque un cofre sea destruido durante un ataque o un accidente.

La interfaz se abre desde el botón que aparece sobre el minimapa o con la tecla `B` en la configuración predeterminada. Si se han cambiado los controles, la tecla puede ser distinta.

##### 3.2. Las dos bibliotecas principales

En la interfaz estándar aparecen dos apartados con propósitos distintos:

| Apartado | Alcance | Uso recomendado |
|---|---|---|
| **Mis planos (*My blueprints*)** | Es personal y está disponible en las distintas partidas guardadas del jugador; no se comparte automáticamente con otros jugadores de una partida multijugador. | Guardar diseños personales que se quieren reutilizar en diferentes mundos. |
| **Planos de la partida (*Game blueprints*)** | Está asociado a la partida guardada actual; en multijugador, puede compartirse con los jugadores de la misma fuerza. | Mantener una colección común para un mundo o una partida cooperativa. |

Puedes colocar accesos directos a planos de la librería en la barra rápida. Estos accesos no necesariamente crean copias: normalmente apuntan al elemento guardado. Por tanto, editar el plano mediante el acceso puede modificar el original de la librería. Si necesitas experimentar sin alterar el diseño maestro, crea una copia independiente.

##### 3.3. Organización de la librería

Una librería grande resulta mucho más útil si se estructura con categorías coherentes. Una forma de organizarla es:

- **Logística:** cintas, divisores, cintas subterráneas e insertores.
- **Producción:** líneas de montaje, fundición, productos químicos y circuitos.
- **Energía:** vapor, solar, nuclear y redes eléctricas.
- **Minería:** módulos de extracción y transporte de recursos.
- **Ferrocarril:** tramos rectos, curvas, cruces, estaciones y patios de espera.
- **Ciencia:** módulos para fabricar y transportar paquetes de ciencia.
- **Defensa y mantenimiento:** perímetros, radares, reparación y edificios de apoyo.
- **Plantillas auxiliares:** bloques de baldosas, postes y transiciones entre sistemas.

El orden y los nombres no están impuestos por el juego: el objetivo es que la colección sea fácil de buscar y que se pueda ampliar sin perder consistencia.

##### 3.4. Seguridad y conservación

Los planos guardados en la librería están separados del inventario ordinario. En cambio, un plano o libro que solo se encuentra en un cofre es un objeto físico y puede perderse si el contenedor se destruye.

Mantén los diseños más importantes en la librería y usa copias cuando pruebes modificaciones importantes. En multijugador, acuerda quién puede editar la colección común y cómo se van a nombrar las versiones actualizadas.


---

#### 4. Libros de planos (*Blueprint books*)

##### 4.1. Función de un libro

Un **libro de planos** agrupa muchos planos y otros planificadores dentro de un solo objeto. Puede contener otros libros, por lo que se pueden construir jerarquías de categorías. El libro ocupa un solo espacio de inventario, mientras que su contenido se organiza internamente.

Un libro puede guardar, por ejemplo, un conjunto completo de módulos de minería, fundición y transporte, o una colección de segmentos de vías y estaciones ferroviarias.

##### 4.2. Crear una estructura útil

Una estructura de ejemplo podría ser:

- **Biblioteca de la fábrica**
  - **01 - Logística**
    - Cintas y divisores
    - Cintas subterráneas
    - Insertores y filtros
  - **02 - Producción**
    - Fundición
    - Circuitos
    - Productos químicos
  - **03 - Energía**
    - Vapor
    - Solar y acumuladores
    - Nuclear
  - **04 - Ferrocarril**
    - Vías rectas y curvas
    - Estaciones
    - Cruces y señalización
  - **05 - Ciencia**
    - Ciencia inicial
    - Ciencia avanzada

El ejemplo es una convención organizativa, no una estructura obligatoria. Adáptala a la escala de la fábrica y a las categorías que uses con más frecuencia.

##### 4.3. Seleccionar planos desde un libro

Los planos pueden usarse directamente desde un libro. Con los controles predeterminados, `Shift` + rueda del ratón permite cambiar el plano activo mientras el libro está seleccionado. En los libros anidados, el cambio recorre también los contenidos de los libros internos.

Los libros pueden duplicarse. La copia es independiente, por lo que modificar o borrar la copia no cambia automáticamente el libro original. Esta función es útil para crear una nueva versión de un diseño conservando la plantilla anterior.

##### 4.4. Aplicar un planificador de mejora a un libro

En versiones compatibles, el libro dispone de una función para aplicar un **planificador de mejora** a las entidades de todos los planos contenidos en el libro. Esto facilita actualizar una colección completa, por ejemplo al pasar de un nivel de cintas a otro, siempre que los objetos del plano sean compatibles con la mejora configurada.

Antes de aplicar cambios masivos, duplica el libro y comprueba un subconjunto de diseños. No todas las entidades tienen una sustitución válida y algunos diseños necesitan una revisión manual posterior.

---

#### 5. Planos parametrizados en Factorio 2.x

##### 5.1. ¿Qué es la parametrización?

En *Factorio 2.x*, los planos compatibles pueden configurarse con **parámetros** que se eligen o completan al colocar el diseño. En lugar de mantener decenas de planos casi idénticos, un diseño parametrizado puede permitir ajustar ciertos valores o elementos para adaptar la misma plantilla a distintas necesidades.

Por ejemplo, una plantilla de montaje puede diseñarse para que el jugador elija una receta y los materiales o configuraciones asociados, siempre que esos elementos estén preparados como parámetros compatibles.

##### 5.2. Cuándo conviene parametrizar

La parametrización suele ser útil cuando:

- Un mismo diseño de máquina se reutiliza con recetas diferentes.
- La instalación se construye en distintas escalas o con diferentes entradas.
- Se necesita adaptar nombres o configuraciones de entidades dentro de una plantilla.
- Se quiere mantener una única plantilla base en vez de duplicar muchas variantes.

No todos los elementos o comportamientos se pueden parametrizar. La disponibilidad depende del tipo de entidad, los campos configurables y la versión del juego. Es importante revisar los parámetros que aparecen en la interfaz y probar el plano antes de usarlo en una instalación importante.

##### 5.3. Ventajas y límites

**Ventajas:** menos duplicación, configuraciones reutilizables y mantenimiento más sencillo de diseños similares.

**Límites:** un plano parametrizado sigue necesitando entradas, salidas, espacio y recursos compatibles. La parametrización no crea automáticamente un sistema de abastecimiento, no resuelve colisiones y no garantiza que una receta sea viable en cualquier ubicación.

---

#### 6. Importar, exportar y compartir planos

##### 6.1. Cadenas de planos (*blueprint strings*)

El juego permite exportar planos y libros como una **cadena de texto** que puede copiarse, guardarse y compartirse. Para importar una cadena, se utiliza el comando de importación disponible en la barra de accesos rápidos y se pega el contenido en el cuadro correspondiente.

En el formato habitual, la cadena representa datos estructurados del plano que se comprimen y codifican para facilitar su transporte. En *Factorio 2.0* también se admite la importación de JSON sin comprimir en los casos contemplados por el juego.

La cadena puede ser muy larga si contiene un plano complejo o un libro con muchos elementos. Al compartirla, asegúrate de copiar el contenido entero y comprobar que corresponde a una versión compatible del juego.

##### 6.2. Flujo de exportación

1. Abre el plano o libro que quieres compartir.
2. Usa la función de exportación de cadena de la interfaz.
3. Copia la cadena completa.
4. Guárdala en un archivo de texto o compártela por el medio acordado.
5. Añade una descripción externa cuando sea útil: versión del juego, propósito, materiales requeridos o limitaciones.

##### 6.3. Flujo de importación

1. Copia la cadena completa del plano o libro.
2. Abre el comando de importación de planos desde la interfaz del juego.
3. Pega la cadena en el cuadro de importación.
4. Confirma y revisa el elemento que aparece en el cursor.
5. Guarda el plano importado en la librería o en un libro adecuado.
6. Antes de construirlo en una fábrica importante, inspecciona la orientación, los ingredientes, las recetas y las conexiones.

La importación no garantiza que el plano funcione igual en cualquier versión, con cualquier mod o en cualquier planeta. Puede contener entidades o recetas que no estén disponibles en la partida actual.

##### 6.4. Compartir dentro de una partida multijugador

Para colaborar, acordad una librería común de la partida, una convención de nombres y un proceso para actualizar los diseños. Evitad modificar sin aviso los planos compartidos que otros jugadores usan como referencias. Cuando sea necesario, guardad una copia con un sufijo de versión, como `v2` o `v3`, hasta validar la actualización.


---

#### 7. Herramientas relacionadas: desmantelamiento y mejora

##### 7.1. Planificador de desmantelamiento (*Deconstruction planner*)

El planificador de desmantelamiento marca entidades para que los robots de construcción las retiren. No es un plano de construcción, pero se puede guardar en la librería y dentro de libros.

Se utiliza para tareas como:

- Retirar máquinas obsoletas.
- Eliminar vegetación o rocas de una zona de expansión.
- Limpiar una sección de fábrica antes de colocar un nuevo diseño.
- Marcar grupos de objetos para retirarlos de forma sistemática.

El alcance del planificador depende de las entidades seleccionadas y de las opciones configuradas. Antes de confirmar una selección amplia, revisa qué elementos quedarán marcados y asegúrate de no incluir edificios que deban conservarse.

##### 7.2. Planificador de mejora (*Upgrade planner*)

El planificador de mejora indica qué entidades deben sustituirse por otras. Por ejemplo, puede ayudar a reemplazar un tipo de cinta o un nivel de máquina por otro que esté disponible y configurado en el planificador.

Los robots pueden realizar sustituciones cuando el edificio de destino y los materiales necesarios estén disponibles y las condiciones permitan reemplazar la entidad. Algunos cambios requieren intervención manual o una revisión de las conexiones y del espacio.

##### 7.3. Cómo se complementan

| Herramienta | Función principal | Ejemplo de uso |
|---|---|---|
| Plano | Reproduce una distribución. | Construir otra línea de fundición idéntica. |
| Libro de planos | Organiza un conjunto de diseños. | Guardar todas las variantes de una estación ferroviaria. |
| Librería | Conserva y comparte plantillas. | Reutilizar diseños entre partidas. |
| Planificador de desmantelamiento | Marca objetos para retirarlos. | Limpiar un área ocupada por instalaciones antiguas. |
| Planificador de mejora | Define sustituciones de entidades. | Actualizar cintas o máquinas de un diseño existente. |

---

#### 8. Robots y materiales necesarios para construir los planos

##### 8.1. Requisitos de los robots de construcción

Los robots no pueden construir cualquier fantasma en cualquier lugar sin apoyo logístico. Para que un plano se complete automáticamente, deben cumplirse las condiciones relevantes:

- Los fantasmas tienen que quedar dentro del alcance de una red de construcción válida o de un roboport personal compatible.
- Debe haber robots de construcción disponibles y operativos.
- La red debe disponer de los objetos necesarios en cofres que puedan abastecer la construcción, como cofres proveedores activos o pasivos, cofres de almacenamiento y cofres de búfer compatibles.
- El diseño debe poder colocarse físicamente: los obstáculos, el terreno, los requisitos de los edificios y las conexiones siguen siendo relevantes.

En una red personal, el jugador también debe disponer de robots, energía para el equipo que los utiliza y materiales accesibles de acuerdo con las reglas de esa red.

##### 8.2. Consultar el coste de construcción

Al crear o editar un plano, revisa el resumen de componentes. Te permite estimar cuántos edificios y materiales hacen falta para construirlo. Para instalaciones grandes, conviene comparar el coste con las existencias disponibles y separar los diseños en módulos si una sola expansión necesita demasiados recursos a la vez.

No olvides recursos que suelen pasarse por alto, como módulos, postes eléctricos, señales ferroviarias, tuberías, baldosas, insertores o cofres. Un plano puede ser correcto en cuanto a distribución y, aun así, quedarse incompleto si falta uno de esos componentes.

##### 8.3. Edificios fantasma y conexiones incompletas

Que todos los fantasmas se hayan construido no garantiza que la instalación funcione. Después de la construcción, comprueba:

- Que los edificios están conectados a electricidad cuando corresponde.
- Que las cintas, tuberías y vías forman rutas continuas.
- Que los insertadores están orientados correctamente.
- Que las máquinas tienen las recetas correctas y reciben insumos.
- Que los postes y roboports cubren las zonas necesarias.
- Que las señales ferroviarias y circuitos se comportan como se espera.

---

#### 9. Estrategias y consejos de organización

##### 9.1. Construir módulos repetibles

Diseña instalaciones con límites claros de entrada y salida. Un módulo de fundición, por ejemplo, debería indicar por dónde recibe mineral y combustible, por dónde sale la placa y dónde requiere electricidad. Cuanto más estandarizadas sean las interfaces, más fácil será combinar varias copias.

##### 9.2. Mantener un plano maestro

Guarda una versión estable como plantilla principal y crea copias para experimentar. Una vez probada una actualización, sustituye el diseño de referencia o guarda la nueva versión con un nombre explícito. Esto reduce el riesgo de que la librería contenga muchos planos casi iguales sin indicar cuál es el recomendado.

##### 9.3. Usar cuadrícula y alineación en grandes instalaciones

En fábricas amplias, la alineación consistente permite conectar módulos sin ajustes manuales constantes. Si el diseño admite ajuste a cuadrícula, configura dimensiones y desplazamientos que correspondan con el tamaño del módulo. Prueba la alineación antes de producir muchas copias: una cuadrícula incorrecta puede repetir un error por toda la fábrica.

##### 9.4. Crear planos que se puedan ampliar

Al diseñar un plano, deja espacio para futuras conexiones, postes, tuberías y vías. Una instalación excesivamente compacta puede ser eficiente al principio, pero difícil de actualizar cuando necesita más capacidad.

##### 9.5. Validar los planos antes de reutilizarlos

Prueba cada diseño en una zona segura o una partida de prueba. Comprueba la producción real, el consumo energético, la capacidad de transporte, las recetas y los posibles bloqueos. El hecho de que el plano sea visualmente correcto no garantiza que la instalación esté equilibrada.

##### 9.6. Documentar los requisitos

Incluye en el nombre o la descripción los datos que más afectan a su uso:

- Nivel de cintas y tipo de insertores.
- Ingredientes, fluidos o combustible requeridos.
- Receta seleccionada en cada máquina.
- Potencia eléctrica aproximada.
- Espacio y orientación de entrada y salida.
- Compatibilidad con el juego base o con *Space Age*.
- Limitaciones conocidas y versión del diseño.

##### 9.7. No confundir el plano con la automatización completa

Un plano permite copiar una estructura; no crea automáticamente los yacimientos, los recursos, el suministro energético ni la red logística que necesita. Es recomendable construir primero una copia pequeña, verificarla y luego expandirla mediante robots.

---

#### 10. Problemas frecuentes y soluciones

| Problema | Causa probable | Solución recomendada |
|---|---|---|
| Los robots no construyen algunos fantasmas. | Están fuera del alcance, faltan robots o materiales, o la construcción está bloqueada. | Revisa la cobertura de construcción, los recursos y los obstáculos. |
| Faltan objetos al construir un plano. | No hay suficientes materiales en la red o en el inventario personal. | Comprueba el resumen de componentes y abastece la red. |
| El plano no encaja con la fábrica existente. | Orientación incorrecta, desalineación o colisiones. | Gira el plano, revisa la cuadrícula y comprueba las interfaces del módulo. |
| No se puede reflejar un plano ferroviario. | Incluye estaciones o señales que solo admiten rotación. | Usa la rotación permitida o crea una variante diseñada para el otro lado. |
| Una modificación de un acceso rápido cambia el plano guardado. | El acceso es una referencia al plano original de la librería. | Duplica el plano antes de editarlo si quieres mantener intacta la plantilla. |
| No se encuentra un plano en otra partida. | Se guardó en la librería de la partida y no en la personal. | Comprueba ambas pestañas de la librería. |
| La cadena de importación falla. | Se copió incompleta o proviene de una versión o mod incompatible. | Vuelve a copiar la cadena entera y verifica versión y contenido. |
| La instalación se construye, pero no funciona. | El plano no incluye todos los requisitos o hay problemas de logística y energía. | Revisa recetas, electricidad, tuberías, cintas, conexiones y suministro de materiales. |
| La librería está llena de variantes difíciles de distinguir. | No hay convención de nombres ni control de versiones. | Agrupa por categorías, añade descripciones y conserva una plantilla principal. |
| El planificador de mejora no cambia todos los edificios esperados. | La entidad de origen no tiene una sustitución compatible o faltan recursos. | Revisa el planificador y comprueba qué entidades admiten mejora. |

---

#### 11. Lista de comprobación para un sistema de planos

Antes de dar por terminado un plano o una librería, verifica los siguientes puntos:

- [ ] El área seleccionada contiene todos los edificios y conexiones necesarios.
- [ ] Se revisaron los iconos, el nombre y la descripción del plano.
- [ ] La plantilla conserva las configuraciones compatibles que se esperan reutilizar.
- [ ] La orientación y la alineación se probaron en el terreno donde se usará.
- [ ] Las colisiones y el terreno se revisaron antes de la construcción.
- [ ] La red de construcción dispone de robots, materiales y cobertura suficientes.
- [ ] Se han tenido en cuenta las cintas, tuberías, postes, señales y conexiones auxiliares.
- [ ] Los diseños principales están guardados en la librería, no solo en cofres.
- [ ] Los planos personales y los de la partida se almacenan en el apartado correcto.
- [ ] Las plantillas importantes tienen copias de respaldo antes de cambios grandes.
- [ ] Las cadenas importadas se validaron para la versión del juego y los mods utilizados.
- [ ] Los diseños compartidos tienen nombres y versiones coherentes.
- [ ] La instalación construida se probó para comprobar que realmente funciona.

---

#### 12. Fuentes de referencia

Estas páginas de la *Factorio Wiki* y del sitio oficial proporcionan detalles de uso, interfaz y compatibilidad:

- [Blueprint — Planos](https://wiki.factorio.com/Blueprint)
- [Blueprint library — Librería de planos](https://wiki.factorio.com/Blueprint_library)
- [Blueprint book — Libros de planos](https://wiki.factorio.com/Blueprint_book)
- [Blueprint string format — Formato de cadena de planos](https://wiki.factorio.com/Blueprint_string_format)
- [Deconstruction planner — Planificador de desmantelamiento](https://wiki.factorio.com/Deconstruction_planner)
- [Upgrade planner — Planificador de mejora](https://wiki.factorio.com/Upgrade_planner)
- [Friday Facts #356 — Blueprint library for real](https://factorio.com/blog/post/fff-356)


---

### Guía técnica de Factorio: Árbol de tecnologías y paquetes de ciencia

**Versión de referencia:** *Factorio 2.x*. Las funciones exclusivas de la expansión *Space Age* aparecen identificadas por separado. Las recetas, tecnologías y cifras pueden variar con versiones futuras, mods o ajustes de partida; para una receta concreta, la interfaz del juego y la wiki enlazada son la referencia final.

El **árbol de tecnologías** organiza la progresión de *Factorio*: permite desbloquear edificios, recetas, mejoras y nuevas posibilidades de producción mediante investigación. Para investigar, la fábrica debe producir **paquetes de ciencia**, transportarlos a los laboratorios y suministrarlos de manera continua.

Esta mecánica conecta prácticamente todos los sistemas del juego. La minería proporciona materias primas; la fundición las convierte en placas; la logística transporta componentes; el ensamblado fabrica paquetes; la energía mantiene las máquinas funcionando; y los laboratorios consumen esos paquetes para desbloquear tecnologías.

#### 1. Fundamentos del árbol de tecnologías

#### 1.1. ¿Qué es una tecnología?

Una **tecnología** es una investigación que, al completarse, desbloquea una capacidad nueva o mejora una ya existente. Sus efectos pueden incluir:

- **Nuevas recetas:** permiten fabricar objetos o materiales que antes no estaban disponibles.
- **Nuevos edificios:** desbloquean máquinas, instalaciones de producción y estructuras logísticas.
- **Mejoras de rendimiento:** aumentan la velocidad, el daño, la productividad, el alcance u otras estadísticas.
- **Nuevas formas de expansión:** permiten automatizar trenes, construir plataformas espaciales, acceder a otros planetas o fabricar productos avanzados.
- **Investigaciones infinitas:** mejoras repetibles que pueden seguir investigándose después de alcanzar el nivel inicial, normalmente con un coste que aumenta por nivel.

La investigación no suele fabricar directamente el objeto desbloqueado. Por ejemplo, investigar una cinta transportadora permite crear la receta de la cinta; después hay que producir sus insumos y fabricarla en el edificio apropiado.

#### 1.2. Cómo se organiza el árbol

El árbol de tecnologías no es una lista lineal. Es una red de tecnologías conectadas por **prerrequisitos**. Una tecnología puede exigir que se haya completado una o varias investigaciones anteriores y que se disponga de determinados tipos de paquetes de ciencia.


[Ver el árbol y las tecnologías en la Factorio Wiki](https://wiki.factorio.com/Technologies)

Cada ficha de tecnología suele mostrar:

- Su nombre e icono.
- Los prerrequisitos que deben completarse antes.
- Los paquetes de ciencia que necesita.
- La cantidad de unidades de investigación.
- El tiempo base de cada unidad de investigación.
- Los edificios, recetas o efectos que se desbloquean.
- Si la investigación tiene niveles repetibles.

Algunas tecnologías se pueden investigar usando únicamente los paquetes básicos. Otras requieren combinaciones de ciencia química, militar, de producción, de utilidad, espacial o de los paquetes adicionales de *Space Age*.

#### 1.3. Coste de investigación y consumo de paquetes

El coste de una tecnología se expresa normalmente en **unidades de investigación**. Cada unidad consume un paquete de cada tipo requerido por la tecnología en cada laboratorio que complete esa unidad.

Por ejemplo, si una investigación cuesta 100 unidades y requiere ciencia de automatización y ciencia logística, se necesitan 100 paquetes de automatización y 100 de logística para completar el coste, sin contar efectos de productividad u otras modificaciones.

El tiempo base de investigación tampoco es necesariamente el tiempo total que tardará la fábrica. La duración efectiva depende del número de laboratorios, la velocidad de investigación, los módulos instalados y la continuidad del suministro de ciencia.

Una forma simplificada de estimar la duración es:

\[
\text{Duración (s)} \approx \frac{T \times P}{L \times S}
\]

Donde:

- \(T\) es el tiempo base por unidad mostrado por la tecnología.
- \(P\) es el número de unidades de investigación.
- \(L\) es la cantidad de laboratorios que pueden trabajar en paralelo.
- \(S\) es la velocidad efectiva de cada laboratorio, incluyendo las bonificaciones aplicables.

Esta estimación supone que todos los laboratorios tienen energía, reciben los paquetes necesarios y trabajan continuamente. Si falta un tipo de ciencia, la investigación no avanza a su ritmo teórico.

#### 1.4. Por qué la ciencia define la progresión

Los paquetes de ciencia representan el nivel de desarrollo industrial que la fábrica ya puede sostener. Los primeros son relativamente sencillos; los posteriores necesitan cadenas de producción cada vez más largas y materiales más avanzados.

La progresión típica del juego base es:

1. Automatizar la producción inicial y la ciencia de automatización.
2. Desbloquear y automatizar la logística básica para producir ciencia logística.
3. Incorporar ciencia militar cuando se necesiten tecnologías de defensa o la investigación la exija.
4. Desarrollar petróleo y productos químicos para crear ciencia química.
5. Escalar la producción metalúrgica y de componentes para producir ciencia de producción y de utilidad.
6. Construir y lanzar cohetes para conseguir ciencia espacial en el juego base.
7. En *Space Age*, explorar otros planetas y automatizar los paquetes científicos específicos de cada entorno.

No todas las partidas requieren seguir exactamente el mismo orden. La demanda de defensa, energía, logística y producción puede hacer que unas tecnologías sean más urgentes que otras.

#### 2. Laboratorios y edificios de investigación

#### 2.1. Laboratorio (*Lab*)


El **laboratorio** consume paquetes de ciencia y los transforma en progreso de investigación. Es la estructura central de esta mecánica.

**Características principales de un laboratorio de calidad normal:**

- Tamaño: 3 × 3 casillas.
- Consumo eléctrico nominal: 60 kW.
- Velocidad base de investigación: 1.
- Dos ranuras para módulos.
- Requiere una conexión eléctrica y los paquetes de ciencia indicados por la tecnología activa.

Se pueden utilizar varios laboratorios simultáneamente para avanzar más rápido en una misma tecnología. La investigación activa es común a toda la fábrica: los laboratorios no investigan tecnologías distintas en paralelo, pero sí contribuyen juntos a la investigación seleccionada.

Los laboratorios pueden recibir paquetes mediante insertores y cintas transportadoras. También es posible encadenar laboratorios, ya que los paquetes colocados en uno pueden pasar a los laboratorios conectados detrás. Aun así, una distribución con cintas y divisores suele ser más fácil de ampliar y diagnosticar.

Más información: [Factorio Wiki: Lab](https://wiki.factorio.com/Lab).

#### 2.2. BioLab (*Space Age*)


[Ver información del BioLab en la Factorio Wiki](https://wiki.factorio.com/Biolab)

El **BioLab** es un laboratorio avanzado exclusivo de *Space Age*, asociado a la progresión de Gleba. Tiene una velocidad de investigación mayor que un laboratorio normal y utiliza una mecánica especializada de ciencia.

Es una instalación avanzada: no es necesaria para comenzar a investigar y no reemplaza el requisito de fabricar paquetes de ciencia. Su valor aparece cuando la fábrica ya ha desarrollado las cadenas planetarias y quiere ampliar la capacidad de investigación.

Antes de diseñar una línea alrededor del BioLab, comprueba en el juego los requisitos de construcción, los tipos de ciencia compatibles y las estadísticas de su calidad concreta, porque las cifras y bonificaciones pueden cambiar según el edificio y la versión.

#### 2.3. Edificios de apoyo a la producción de ciencia

Los paquetes se fabrican usando los edificios normales de producción; no existe una única máquina universal para todas las recetas.

| Edificio o estructura | Uso dentro de la cadena científica |
|---|---|
| **Máquina de ensamblaje** | Fabrica gran parte de los paquetes de ciencia y sus componentes. |
| **Planta química** | Produce azufre, plástico, ácido sulfúrico y otros insumos de recetas científicas. |
| **Refinería de petróleo** | Convierte petróleo crudo en fluidos necesarios para productos químicos. |
| **Hornos** | Producen placas metálicas y otros materiales utilizados en componentes. |
| **Centrífuga** | Procesa uranio para cadenas de energía y productos que pueden intervenir en tecnologías avanzadas. |
| **Cohete y silo de cohete** | En el juego base, el lanzamiento de un satélite produce ciencia espacial. |
| **Plataforma espacial** | En *Space Age*, transporta materiales y permite elaborar ciencia espacial y otras recetas en el espacio. |
| **Máquinas planetarias** | En *Space Age*, producen insumos y paquetes especializados de cada planeta. |
| **Cintas, insertores y cofres** | Mueven, alimentan y almacenan los paquetes y sus ingredientes. |
| **Postes eléctricos** | Suministran energía a las máquinas de producción y a los laboratorios. |

#### 3. Paquetes de ciencia del juego base

Los paquetes de ciencia son objetos que se consumen en los laboratorios. La mayoría se produce en máquinas de ensamblaje a partir de materiales intermedios. La cantidad de salida y el tiempo de elaboración dependen de la receta.

Las recetas de esta sección corresponden al juego base de *Factorio 2.x*. Los nombres y los ingredientes se muestran para orientar el diseño; los costes tecnológicos y las recetas exactas pueden consultarse en los enlaces de cada paquete.

#### 3.1. Ciencia de automatización (roja)


La **ciencia de automatización** es el primer paquete científico de la partida y permite comenzar la investigación automatizada.

- **Receta típica:** 1 engranaje de hierro + 1 placa de cobre → 1 paquete.
- **Tiempo de receta:** 5 segundos, antes de aplicar la velocidad de la máquina.
- **Edificio habitual:** máquina de ensamblaje compatible.
- **Etapa:** inicio de la partida.

Su producción permite investigar tecnologías fundamentales y comenzar a construir una fábrica capaz de crear sus propios materiales de investigación. Al principio puede fabricarse manualmente, pero automatizarla pronto evita dedicar tiempo del jugador a repetir recetas.

Usos principales: investigar tecnologías iniciales, desbloquear automatización básica y construir una primera línea de ciencia.

#### 3.2. Ciencia logística (verde)


La **ciencia logística** representa una progresión hacia una fábrica con transporte automatizado.

- **Receta típica:** 1 inserter + 1 cinta transportadora → 1 paquete.
- **Tiempo de receta:** 6 segundos.
- **Edificio habitual:** máquina de ensamblaje.
- **Etapa:** inicio y expansión temprana.

Su receta crea una relación directa con la logística: hay que fabricar insertores y cintas y, por tanto, mantener una producción estable de hierro, cobre y engranajes.

Se utiliza junto con los paquetes que solicite cada tecnología. Muchas investigaciones tempranas combinan ciencia roja y verde, lo que hace conveniente alimentar los laboratorios con ambas líneas.

#### 3.3. Ciencia militar (gris)


La **ciencia militar** permite investigar tecnologías relacionadas con armamento, munición, defensas y otras capacidades militares.

- **Receta típica:** cargador de munición perforante + granada + 2 muros de piedra → 2 paquetes.
- **Tiempo de receta:** 10 segundos.
- **Edificio habitual:** máquina de ensamblaje.
- **Etapa:** expansión temprana o intermedia, según las necesidades de defensa.

Se fabrica en lotes de dos paquetes por receta. Esto no significa que todas las investigaciones militares sean baratas: el coste total depende del precio de la tecnología y de los tipos de ciencia requeridos.

Para producirla sin interrupciones, conviene separar o equilibrar la fabricación de munición, granadas y muros. Si un ingrediente se queda sin suministro, la producción de ciencia militar se detendrá aunque los demás estén disponibles.

#### 3.4. Ciencia química (azul)


La **ciencia química** introduce una cadena industrial más compleja, porque depende de productos derivados del petróleo y componentes avanzados.

- **Receta típica:** 3 circuitos avanzados + 2 unidades de motor + 1 azufre → 2 paquetes.
- **Tiempo de receta:** 24 segundos.
- **Edificio habitual:** máquina de ensamblaje.
- **Etapa:** intermedia.

Entre los materiales necesarios aparecen productos que requieren petróleo procesado, circuitos electrónicos y fabricación de motores. Esto obliga a coordinar varias instalaciones y a sostener una buena disponibilidad de electricidad, fluidos y metales.

La ciencia química abre el acceso a una gran cantidad de tecnologías industriales. Al planificar su línea, no hay que olvidar la capacidad de la refinería, las plantas químicas y el suministro de componentes electrónicos.

#### 3.5. Ciencia de producción (púrpura)


La **ciencia de producción** es un paquete avanzado con ingredientes procedentes de cadenas industriales costosas.

- **Receta típica:** 30 raíles + 1 horno eléctrico + 1 módulo de productividad → 3 paquetes.
- **Tiempo de receta:** 21 segundos.
- **Edificio habitual:** máquina de ensamblaje.
- **Etapa:** avanzada.

La receta exige ya disponer de una fábrica con transporte ferroviario, producción de componentes electrónicos y capacidad para fabricar hornos eléctricos y módulos. Aunque el lote produce tres unidades, no debe ignorarse el coste de todos los materiales intermedios.

Una instalación de ciencia de producción suele necesitar producción específica de raíles, módulos y hornos eléctricos, además de un almacenamiento o transporte fiable para los paquetes terminados.

#### 3.6. Ciencia de utilidad (amarilla)


La **ciencia de utilidad** se fabrica mediante otra cadena avanzada de componentes.

- **Receta típica:** 3 estructuras de baja densidad + 2 unidades de procesamiento + 1 marco de robot volador → 3 paquetes.
- **Tiempo de receta:** 21 segundos.
- **Edificio habitual:** máquina de ensamblaje.
- **Etapa:** avanzada.

Las estructuras de baja densidad requieren materiales como cobre y acero; las unidades de procesamiento dependen de circuitos y productos químicos, y los marcos de robot exigen una cadena de motores y otros componentes. Por eso es importante disponer de líneas estables de cobre, plástico, acero y circuitos.

La ciencia de utilidad permite desarrollar tecnologías avanzadas y suele compartir cadenas de suministro con otras partes importantes de la fábrica.

#### 3.7. Ciencia espacial


La ciencia espacial se obtiene después de desarrollar la infraestructura de cohetes.

**Juego base:** el lanzamiento de un cohete con un satélite produce 1000 paquetes de ciencia espacial. Por tanto, la fuente principal es una cadena de producción que termina en el silo de cohete y el satélite.

**Space Age:** cambia esta mecánica. La ciencia espacial puede fabricarse en una plataforma espacial a partir de ingredientes que se obtienen, procesan o transportan en el espacio. La receta y el rendimiento son diferentes de los del juego base.

Este paquete se utiliza especialmente en investigaciones avanzadas e infinitas. La cantidad de lanzamientos y la eficiencia de la cadena dependen del diseño del cohete, los materiales disponibles y el ritmo de fabricación de satélites.

Consulta la [Factorio Wiki: Space science pack](https://wiki.factorio.com/Space_science_pack) para distinguir las recetas de cada versión.

#### 4. Paquetes adicionales de *Space Age*

La expansión añade cinco paquetes científicos nuevos asociados a planetas o entornos avanzados. Su producción necesita maquinaria, recursos y procesos propios de cada lugar. Estos paquetes hacen que la progresión tecnológica dependa de explorar y establecer operaciones fuera de Nauvis.

#### 4.1. Ciencia metalúrgica — Vulcanus


La **ciencia metalúrgica** se produce en Vulcanus y está relacionada con las nuevas tecnologías de metalurgia. Su cadena aprovecha recursos y procesos especializados, incluidos los fluidos metálicos fundidos y productos avanzados disponibles en ese planeta.

**Edificios y recursos asociados:** fundiciones, sistemas para manejar fluidos metálicos, máquinas de ensamblaje y recursos planetarios como calcita, además de los componentes que indique la receta concreta.

Su función es desbloquear tecnologías que amplían la producción metalúrgica y hacen posible una fabricación más eficiente de materiales. Como algunas recetas se apoyan en procesos de fundición y moldeo, conviene distinguir entre el coste del paquete de ciencia y las cadenas de producción que lo abastecen.

Más información: [Metallurgic science pack](https://wiki.factorio.com/Metallurgic_science_pack).

#### 4.2. Ciencia electromagnética — Fulgora


La **ciencia electromagnética** se obtiene en Fulgora y está ligada a la producción eléctrica y a tecnologías electromagnéticas avanzadas.

La cadena utiliza intermediarios propios de ese entorno y puede apoyarse en la planta electromagnética, que es una máquina especializada de la expansión. La logística de recursos reciclados es importante en Fulgora, por lo que el diseño debe contemplar la clasificación y el uso eficiente de los materiales resultantes del reciclaje.

Consulta [Electromagnetic science pack](https://wiki.factorio.com/Electromagnetic_science_pack) para ver los ingredientes y las recetas exactas vigentes.

#### 4.3. Ciencia agrícola — Gleba


La **ciencia agrícola** se produce en Gleba a partir de productos biológicos, incluidos bioflujo (*bioflux*) y huevos de pentápodo, con la receta disponible en el juego.

Su principal particularidad es que **se deteriora con el tiempo**. Los ingredientes frescos producen paquetes más frescos; a medida que pierden frescura, los paquetes ofrecen menos valor científico y los laboratorios deben consumir más para generar la misma cantidad de progreso.

Consejos esenciales:

- Evita fabricar grandes reservas que no vayan a utilizarse pronto.
- Coordina la producción biológica con el consumo de los laboratorios.
- Da prioridad a transportar y consumir los paquetes mientras están frescos.
- Vigila también la frescura de los ingredientes previos, no solo la del paquete terminado.

Más información: [Agricultural science pack](https://wiki.factorio.com/Agricultural_science_pack).

#### 4.4. Ciencia criogénica — Aquilo


La **ciencia criogénica** se produce en Aquilo y se relaciona con tecnologías avanzadas de bajas temperaturas, procesamiento de fluidos y componentes especializados.

La cadena exige una infraestructura capaz de sostener las condiciones y los insumos de Aquilo. Deben considerarse la producción local, el transporte interplanetario, los fluidos necesarios y las máquinas compatibles con las recetas criogénicas.

Más información: [Cryogenic science pack](https://wiki.factorio.com/Cryogenic_science_pack).

#### 4.5. Ciencia de promethium — más allá del borde del sistema solar


La **ciencia de promethium** es un paquete de final de juego que se fabrica en plataformas espaciales. Su producción combina materiales procedentes de distintas etapas de la expansión, incluidos fragmentos de asteroides de promethium y componentes avanzados.

Se utiliza para la investigación de **productividad de investigación**, una tecnología infinita que reduce indirectamente el esfuerzo científico necesario para progresar. Esta cadena es especialmente compleja porque exige alcanzar regiones espaciales peligrosas, recoger recursos escasos y coordinar productos procedentes de varios planetas.

Más información: [Promethium science pack](https://wiki.factorio.com/Promethium_science_pack).

#### 5. Comparación de los paquetes de ciencia

| Paquete | Juego base o expansión | Etapa aproximada | Característica principal |
|---|---|---|---|
| **Automatización (rojo)** | Base | Inicial | Ingredientes relativamente sencillos; inicia la investigación automática. |
| **Logística (verde)** | Base | Inicial | Requiere cintas e insertores y acompaña la automatización de la fábrica. |
| **Militar (gris)** | Base | Inicial/intermedia | Requiere materiales de combate y habilita tecnologías militares. |
| **Química (azul)** | Base | Intermedia | Necesita productos derivados del petróleo y componentes avanzados. |
| **Producción (púrpura)** | Base | Avanzada | Requiere raíles, hornos eléctricos y módulos de productividad. |
| **Utilidad (amarillo)** | Base | Avanzada | Requiere estructuras de baja densidad, unidades de procesamiento y marcos de robot. |
| **Espacial** | Base y *Space Age*, con recetas diferentes | Avanzada | En el juego base se obtiene del cohete y satélite; en *Space Age* se produce en una plataforma. |
| **Metalúrgica** | *Space Age* | Planetaria avanzada | Asociada a Vulcanus y las tecnologías metalúrgicas. |
| **Electromagnética** | *Space Age* | Planetaria avanzada | Asociada a Fulgora y a la producción electromagnética. |
| **Agrícola** | *Space Age* | Planetaria avanzada | Asociada a Gleba; se deteriora y pierde valor científico con el tiempo. |
| **Criogénica** | *Space Age* | Planetaria avanzada | Asociada a Aquilo y a procesos criogénicos. |
| **Promethium** | *Space Age* | Final de juego | Se produce en plataformas espaciales y permite la investigación de productividad de investigación. |

#### 6. Diseñar una línea de producción científica

#### 6.1. Flujo de producción general

Una instalación científica automatizada necesita producir los ingredientes, convertirlos en paquetes y distribuirlos hacia los laboratorios.

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Obtención de recursos | Yacimientos y recursos | Minería y extracción | Materias primas | — |
| Procesamiento inicial | Materias primas | Fundición y refinamiento | Materiales procesados | — |
| Fabricación intermedia | Materiales procesados | Producción de componentes intermedios | Componentes intermedios | — |
| Fabricación avanzada | Componentes intermedios | Máquinas de ensamblaje o plantas especializadas | Productos fabricados | — |
| Preparación de ciencia | Productos fabricados | Línea de producción de paquetes de ciencia | Paquetes de ciencia | — |
| Distribución de ciencia | Paquetes de ciencia | Cintas e insertores | Paquetes suministrados a laboratorios | — |
| Investigación | Paquetes de ciencia | Laboratorios | Tecnología investigada | — |

Es recomendable separar las cadenas de producción de cada paquete, al menos en las etapas tempranas. Las recetas comparten materiales y componentes, por lo que una deficiencia de placas de hierro, circuitos, plástico o acero puede afectar a varias líneas científicas al mismo tiempo.

#### 6.2. Automatizar los paquetes de ciencia

Para automatizar una línea:

1. Desbloquea la receta correspondiente mediante la investigación que la habilite.
2. Coloca una máquina de ensamblaje compatible con la receta.
3. Suministra todos los ingredientes con cintas, insertores o logística de robots.
4. Retira los paquetes terminados mediante una cinta o un insertor.
5. Transporta los paquetes a los laboratorios que los consumen.
6. Comprueba que la producción de ingredientes y la capacidad de transporte alcanzan el ritmo de investigación deseado.

Los laboratorios consumen los tipos de paquetes requeridos por la tecnología activa. Si una investigación necesita tres clases de ciencia y solo dos llegan a los laboratorios, el suministro incompleto limitará el progreso.

#### 6.3. Distribución hacia varios laboratorios


Hay varias formas habituales de llevar ciencia a los laboratorios:

- **Cintas con carriles separados:** apropiadas cuando se quiere asegurar que cada tipo de paquete tenga una ruta predecible.
- **Divisores y filtros:** permiten distribuir paquetes de forma más controlada o separar tipos de objetos.
- **Insertores entre cofres y laboratorios:** útiles en instalaciones compactas o alimentadas por logística de robots.
- **Laboratorios encadenados:** pueden pasar paquetes desde un laboratorio a otro, aunque se debe cuidar la orientación y la transferencia.
- **Robots logísticos:** facilitan la distribución flexible cuando la fábrica ya dispone de una red de logística suficiente.

Un sistema robusto debe evitar que una cinta saturada o un tipo de paquete acumulado bloquee el suministro de los demás.

#### 6.4. Calcular el consumo científico

Para estimar cuántos paquetes por segundo necesita una línea de investigación, hay que conocer cuántos laboratorios están trabajando, la velocidad de investigación y el tiempo por unidad.

Si una tecnología requiere un paquete de cada tipo por unidad y se investigan \(P\) unidades, el consumo total de cada tipo requerido será \(P\), antes de considerar productividad u otros modificadores. Si la investigación necesita varios paquetes de un tipo por unidad, esa cantidad se multiplica por el coste de la tecnología.

La producción de ciencia debe dimensionarse según la **demanda de investigación real**. Si se construyen muchas máquinas de paquetes pero pocos laboratorios pueden consumirlos, se acumulará inventario. Si hay más laboratorios de los que la fábrica puede abastecer, estos pasarán tiempo inactivos.

#### 6.5. Equilibrar la línea

Una fábrica científica equilibrada necesita coordinar:

- La velocidad de producción de los ingredientes intermedios.
- El número y la velocidad de las máquinas que fabrican los paquetes.
- La capacidad de las cintas y los insertores.
- La cantidad de laboratorios que consumen cada paquete.
- El suministro eléctrico de toda la instalación.
- El almacenamiento disponible para gestionar cambios de demanda.

Cuando una tecnología termina, los paquetes de ciencia que ya se encuentran en los laboratorios pueden utilizarse para la investigación siguiente si esta requiere los mismos tipos. Por ello, no es necesario vaciar siempre los laboratorios entre investigaciones.

#### 7. Módulos, velocidad de investigación y productividad

#### 7.1. Módulos compatibles con laboratorios

Los laboratorios normales disponen de dos ranuras para módulos. Las mejoras dependen del tipo de módulo y de la configuración de la partida.

- **Módulos de velocidad:** aumentan el ritmo de investigación, normalmente con mayor consumo eléctrico.
- **Módulos de productividad:** pueden aumentar el progreso científico generado por los paquetes consumidos, según las reglas de la versión del juego.
- **Módulos de eficiencia:** ayudan a reducir el consumo de electricidad.
- **Calidad:** los edificios de calidad superior pueden ofrecer estadísticas mejoradas, de acuerdo con el sistema de calidad de *Factorio 2.x*.

Los módulos solo son una parte del diseño: la escasez de paquetes, la potencia eléctrica y la logística suelen ser limitaciones más importantes que la velocidad nominal de los laboratorios.

#### 7.2. Investigación de velocidad de laboratorio

El árbol incluye investigaciones de **velocidad de investigación de laboratorios**. Sus niveles aumentan la bonificación aplicable a la velocidad de investigación, aunque cada nivel exige sus propios paquetes y cantidades de investigación.

La bonificación de velocidad no crea paquetes adicionales. En general, permite avanzar más rápido con la misma infraestructura de laboratorios, siempre que la fábrica pueda producir ciencia al ritmo de consumo.

#### 7.3. Productividad de investigación en *Space Age*

La productividad de investigación es una mejora avanzada que aumenta el progreso científico efectivo. En *Space Age*, la ciencia de promethium se utiliza para esta investigación infinita. Aunque el paquete se emplea directamente en esa tecnología, sus efectos ayudan indirectamente a las demás investigaciones.

#### 8. Estrategias y consejos de optimización

#### 8.1. Automatizar pronto la ciencia básica

La producción manual de paquetes sirve para arrancar, pero no escala. Automatiza los paquetes rojo y verde cuanto antes y conecta sus máquinas a una fuente estable de materias primas. De ese modo, los laboratorios pueden seguir investigando sin que el jugador deba fabricar cada lote a mano.

#### 8.2. Dar prioridad a las tecnologías que desbloquean nuevas cadenas

Una investigación útil no es solo la que mejora una estadística, sino también la que habilita un recurso o edificio necesario para el siguiente paso de la fábrica. Antes de comenzar una investigación costosa, revisa sus efectos y sus prerrequisitos.

#### 8.3. Evitar producir paquetes sin demanda

La investigación no siempre consume todos los paquetes a la misma velocidad. Si un tipo se acumula, puedes reducir temporalmente su producción o ajustar las cintas. Mantener inventarios enormes de productos caros ocupa espacio y puede ocultar un desbalance de la fábrica.

#### 8.4. Mantener separadas las cadenas complejas

La ciencia química, de producción y de utilidad comparten ingredientes con otras líneas industriales. Una buena práctica es reservar capacidad específica para esos ingredientes o definir prioridades mediante divisores, cofres, circuitos y redes de robots.

#### 8.5. Usar el almacenamiento con cuidado

Los cofres pueden compensar variaciones temporales de suministro, pero no resuelven una producción insuficiente de forma permanente. Los paquetes ordinarios pueden almacenarse para más tarde; la ciencia agrícola es una excepción relevante en *Space Age*, porque se deteriora y pierde valor científico con el tiempo.

#### 8.6. Escalar a partir de la demanda de laboratorios

Antes de ampliar una línea científica, consulta cuántos laboratorios trabajan y a qué ritmo consumen los paquetes. Si los laboratorios están frecuentemente vacíos, aumenta la producción del tipo deficitario. Si están llenos o esperan a que se complete otra investigación, no suele ser útil ampliar todas las líneas indiscriminadamente.

#### 8.7. Preparar la ciencia espacial como una cadena industrial

En el juego base, la ciencia espacial no es solo una receta de ensamblaje: forma parte de la producción de satélites, cohetes, combustible, estructuras y componentes avanzados. Es recomendable organizar el silo de cohete y las líneas asociadas como un bloque industrial que pueda ampliarse.

En *Space Age*, la plataforma espacial añade la necesidad de recolectar asteroides, procesar materiales y transportar carga entre superficies planetarias. La ciencia espacial y los paquetes planetarios deben diseñarse teniendo en cuenta esos costes logísticos.

#### 8.8. Planificar la ciencia de los planetas

En *Space Age*, cada planeta introduce nuevos recursos y reglas logísticas. Conviene crear una producción que use los recursos locales y transportar a Nauvis o a las otras instalaciones los materiales que realmente necesite la fábrica. Para la ciencia agrícola, la frescura es crítica; para la ciencia de promethium, la disponibilidad y la logística de los componentes espaciales son restricciones mayores.

#### 9. Errores comunes y soluciones

| Problema | Causa posible | Solución recomendada |
|---|---|---|
| La investigación no avanza. | Falta un tipo de paquete, no hay electricidad o no hay laboratorios disponibles. | Comprueba la tecnología seleccionada, el contenido de los laboratorios y la red eléctrica. |
| Un paquete no se fabrica. | Falta un ingrediente o la máquina no tiene la receta correcta. | Inspecciona la receta y asegura el suministro de todos los insumos. |
| Se acumula un tipo de ciencia mientras otros escasean. | Las líneas de producción tienen ritmos diferentes. | Equilibra las cantidades de máquinas o distribuye los materiales mediante divisores y prioridades. |
| Hay muchos laboratorios, pero la investigación no acelera. | El suministro de paquetes limita el trabajo o la energía es insuficiente. | Amplía la ciencia deficitaria y revisa el consumo eléctrico. |
| Los paquetes llegan a algunos laboratorios, pero no a todos. | Las cintas, los insertores o los divisores distribuyen mal los objetos. | Revisa la orientación de los insertores y la capacidad de transporte. |
| La ciencia química se detiene. | Falta plástico, azufre, circuitos avanzados, motores o capacidad de refinamiento. | Comprueba toda la cadena química, no solo la máquina del paquete. |
| La ciencia de producción es demasiado costosa. | Sus insumos avanzados no tienen producción dedicada. | Separa líneas de raíles, módulos de productividad y hornos eléctricos. |
| La ciencia espacial no aparece tras lanzar el cohete. | En el juego base falta un satélite o se está usando la lógica de producción de otra versión. | Comprueba la receta del satélite y distingue la mecánica base de la de *Space Age*. |
| La ciencia agrícola pierde rendimiento. | Los paquetes se deterioran antes de consumirse. | Reduce el almacenamiento, mejora el transporte y coordina la producción con el consumo. |
| La ciencia de promethium no puede producirse. | Falta acceso a plataformas y recursos espaciales avanzados. | Comprueba la cadena de exploración, recolección y transporte requerida. |

#### 10. Lista de comprobación para una fábrica científica

- [ ] La tecnología que desbloquea cada receta de ciencia está investigada.
- [ ] Todas las máquinas de ensamblaje tienen la receta correcta seleccionada.
- [ ] La fábrica produce todos los ingredientes necesarios de forma estable.
- [ ] Las cintas, insertores y redes de robots llevan los paquetes a los laboratorios.
- [ ] Todos los laboratorios disponen de electricidad y de los tipos de ciencia requeridos.
- [ ] La capacidad de producción de cada paquete coincide con la demanda de investigación.
- [ ] Las líneas complejas cuentan con suministro suficiente de circuitos, motores, plástico, acero y otros componentes.
- [ ] La producción de ciencia espacial incluye la infraestructura necesaria para la versión de juego utilizada.
- [ ] En *Space Age*, los recursos planetarios, las plataformas y los paquetes especializados se integran en un plan logístico coherente.
- [ ] La ciencia agrícola se consume antes de perder demasiada frescura.
- [ ] Se revisa el árbol de tecnologías para priorizar desbloqueos que habiliten nuevas cadenas de producción.

#### 11. Fuentes de referencia

Las siguientes páginas de la *Factorio Wiki* permiten consultar las tecnologías, recetas y características actuales de la mecánica:

- [Technologies — Árbol de tecnologías](https://wiki.factorio.com/Technologies)
- [Research — Investigación](https://wiki.factorio.com/Research)
- [Science pack — Paquetes de ciencia](https://wiki.factorio.com/Science_pack)
- [Lab — Laboratorio](https://wiki.factorio.com/Lab)
- [Biolab — Laboratorio biológico de *Space Age*](https://wiki.factorio.com/Biolab)
- [Automation science pack — Ciencia de automatización](https://wiki.factorio.com/Automation_science_pack)
- [Logistic science pack — Ciencia logística](https://wiki.factorio.com/Logistic_science_pack)
- [Military science pack — Ciencia militar](https://wiki.factorio.com/Military_science_pack)
- [Chemical science pack — Ciencia química](https://wiki.factorio.com/Chemical_science_pack)
- [Production science pack — Ciencia de producción](https://wiki.factorio.com/Production_science_pack)
- [Utility science pack — Ciencia de utilidad](https://wiki.factorio.com/Utility_science_pack)
- [Space science pack — Ciencia espacial](https://wiki.factorio.com/Space_science_pack)
- [Metallurgic science pack — Ciencia metalúrgica](https://wiki.factorio.com/Metallurgic_science_pack)
- [Electromagnetic science pack — Ciencia electromagnética](https://wiki.factorio.com/Electromagnetic_science_pack)
- [Agricultural science pack — Ciencia agrícola](https://wiki.factorio.com/Agricultural_science_pack)
- [Cryogenic science pack — Ciencia criogénica](https://wiki.factorio.com/Cryogenic_science_pack)
- [Promethium science pack — Ciencia de promethium](https://wiki.factorio.com/Promethium_science_pack)


---

### Guía técnica de Factorio: Lanzamiento del cohete espacial y ciencia espacial

**Versión de referencia:** *Factorio 2.x*. Las diferencias de la expansión *Space Age* se indican por separado. Las recetas y cifras pueden variar con la versión, la calidad de los edificios y los módulos instalados; cuando corresponda, se especifican los valores base.

El lanzamiento del cohete representa un hito importante en la progresión de *Factorio*. En el juego base, permite completar el objetivo principal de la partida y obtener paquetes de ciencia espacial si se envía un satélite. En *Space Age*, el cohete también es la infraestructura esencial para trasladar objetos y al personaje desde Nauvis hasta plataformas orbitales y, desde ellas, continuar la exploración del sistema.

Esta guía explica cómo se desbloquea y construye el silo, cómo se fabrican las partes del cohete, qué diferencia hay entre el juego base y *Space Age*, cómo se obtiene la ciencia espacial y cómo diseñar una línea de lanzamiento automatizada y fiable.

---

#### 1. Fundamentos del lanzamiento espacial

##### 1.1. ¿Qué es el lanzamiento de un cohete?

El lanzamiento espacial es un proceso industrial que reúne numerosas cadenas de producción avanzadas en un mismo objetivo. El jugador debe investigar la tecnología correspondiente, fabricar un silo de cohete, suministrar los materiales necesarios y producir suficientes partes de cohete para que la estructura lo ensamble.

El cohete no se fabrica en una máquina de montaje normal: **las partes y el cohete se producen dentro del silo de cohete**. Los recursos para esas partes sí provienen de otras cadenas automatizadas.

El proceso general es:

1. Investigar la tecnología del silo y las tecnologías de fabricación que requiere.
2. Construir el silo de cohete y conectarlo a la red eléctrica.
3. Producir y suministrar unidades de procesamiento, estructuras de baja densidad y combustible para cohetes.
4. Dejar que el silo fabrique las partes y ensamble el cohete.
5. Preparar una carga compatible con el modo de juego.
6. Lanzar el cohete y aprovechar el resultado: completar el objetivo principal o transportar materiales a la órbita.

##### 1.2. Diferencias entre el juego base y *Space Age*

Aunque ambos modos usan el silo de cohete, la función de los lanzamientos es distinta.

| Aspecto | Juego base | *Space Age* |
|---|---|---|
| Propósito principal | Completar el objetivo principal y obtener ciencia espacial mediante satélites. | Transportar al personaje y objetos hacia plataformas orbitales y conectar planetas. |
| Cantidad de partes por cohete | 100 partes. | 50 partes. |
| Materiales por parte de cohete | 10 unidades de cada uno de los tres ingredientes básicos de la receta. | 1 unidad de cada uno de los tres ingredientes básicos de la receta. |
| Materiales para las partes de un cohete completo, sin bonificaciones | 1000 unidades de cada ingrediente de la receta. | 50 unidades de cada ingrediente de la receta. |
| Satélite | Se lanza para obtener 1000 paquetes de ciencia espacial. | El satélite no se usa en el sistema de lanzamiento de *Space Age*. |
| Plataforma espacial | No está presente. | Se crea con un paquete inicial de plataforma espacial y actúa como fábrica orbital y transporte. |
| Ciencia espacial | Se obtiene al lanzar un satélite. | Se fabrica en una plataforma espacial a partir de recursos recuperados de asteroides. |

En el juego base, la receta de una parte de cohete consume **10 unidades de estructura de baja densidad, 10 de combustible para cohetes y 10 unidades de procesamiento**. Al necesitar 100 partes, una producción sin bonificaciones requiere 1000 unidades de cada uno de esos tres ingredientes para las partes del cohete. A esto hay que sumar los materiales necesarios para fabricar el silo y, si se busca ciencia espacial, el satélite.

En *Space Age*, una parte de cohete consume una unidad de cada uno de esos tres ingredientes y bastan 50 partes. Por eso, el coste de materiales de un cohete es considerablemente menor que en el juego base. La expansión también añade una tecnología infinita de productividad de partes de cohete que reduce aún más los costes a largo plazo.

Fuentes: [Rocket silo — Factorio Wiki](https://wiki.factorio.com/Rocket_silo), [Rocket part — Factorio Wiki](https://wiki.factorio.com/Rocket_part).

##### 1.3. ¿Qué aporta la ciencia espacial?

La ciencia espacial amplía la progresión tecnológica y permite investigar mejoras que requieren una gran cantidad de recursos. Su papel exacto depende del modo de juego:

- **En el juego base**, el silo lanza un satélite y entrega 1000 paquetes de ciencia espacial al puerto de aterrizaje de carga. Los paquetes se utilizan en laboratorios para investigar tecnologías infinitas, por ejemplo mejoras que pueden continuar investigándose más allá de los niveles iniciales.
- **En *Space Age***, la ciencia espacial se produce en una plataforma orbital. Se utiliza para las investigaciones posteriores que hacen posible avanzar en la expansión, incluidas las tecnologías vinculadas a otros planetas.

La ciencia espacial convierte la infraestructura de lanzamiento en algo más que una meta final: en el juego base es una fuente repetible de investigación avanzada, mientras que en *Space Age* forma parte de la cadena de progresión necesaria para expandirse por el sistema.

---

#### 2. Edificios y estructuras del lanzamiento

##### 2.1. Silo de cohete (*Rocket silo*)


El **silo de cohete** es la estructura central de esta mecánica. Ensambla las partes del cohete, prepara la carga y realiza el lanzamiento.

**Características de referencia para calidad normal:**

- Tamaño: 9 × 9 casillas.
- Consumo eléctrico nominal: 4 MW.
- Velocidad de fabricación base: 1.
- Dispone de 4 ranuras para módulos.
- Requiere una tecnología avanzada y una gran cantidad de materiales para fabricarse.
- Las partes de cohete se almacenan internamente en el silo; no se pueden retirar como objetos normales.

El silo trabaja de manera continua cuando tiene electricidad y dispone de los ingredientes requeridos. Una vez que acumula suficientes partes, ensambla el cohete y lo prepara para recibir su carga.

**Advertencia:** si se deconstruye el silo, las partes de cohete almacenadas en él se pierden. Por eso, conviene verificar su contenido antes de mover o sustituir la instalación.

El silo admite módulos de productividad y de velocidad compatibles. Debido al coste de sus recetas, es uno de los edificios que más puede beneficiarse de una mejora de productividad. Los módulos que aumentan la velocidad también elevan normalmente el consumo eléctrico, por lo que hay que revisar la red de energía.

Fuente: [Rocket silo — Factorio Wiki](https://wiki.factorio.com/Rocket_silo).

##### 2.2. Puerto de aterrizaje de carga (*Cargo landing pad*)


El puerto de aterrizaje de carga es una estructura importante en ambos modos, pero su función cambia.

**En el juego base:**

- Es necesario tener uno colocado para poder lanzar un cohete.
- Recibe los 1000 paquetes de ciencia espacial generados por un lanzamiento con satélite.
- Solo puede haber un puerto de aterrizaje de carga colocado en la superficie a la vez.
- Puede descargar sus objetos mediante insertores como un contenedor proveedor pasivo.

**En *Space Age*:**

- Sirve como punto de recepción de objetos enviados desde plataformas espaciales que están en órbita sobre el planeta.
- Puede usar solicitudes logísticas para indicar qué materiales se necesitan en la superficie.
- Si el inventario se llena, los objetos entregados pueden quedar en el suelo alrededor de la estructura; conviene ampliar la capacidad o retirar los materiales con rapidez.
- Puede ampliarse con estructuras de carga compatibles para aumentar capacidad y mejorar la descarga.

Fuente: [Cargo landing pad — Factorio Wiki](https://wiki.factorio.com/Cargo_landing_pad).

##### 2.3. Máquinas de montaje para los componentes


El silo fabrica las partes de cohete, pero **no fabrica por sí solo todos los ingredientes**. Las cadenas de producción habituales utilizan máquinas de montaje y otras instalaciones para producir:

- **Unidades de procesamiento:** circuitos avanzados necesarios para las partes de cohete y muchas otras tecnologías.
- **Estructuras de baja densidad:** componentes ligeros y resistentes usados en cohetes y otros productos avanzados.
- **Combustible para cohetes:** combustible procesado que se utiliza como ingrediente de las partes de cohete.
- **Satélite:** carga necesaria para obtener ciencia espacial en el juego base.
- **Paquete inicial de plataforma espacial:** objeto de *Space Age* necesario para crear una nueva plataforma orbital.

Las máquinas de montaje deben contar con sus ingredientes de entrada, electricidad y espacio libre para retirar la producción. La línea debe dimensionarse para que el silo no permanezca detenido esperando uno de los componentes.

##### 2.4. Insertores, cintas y cofres

Los **insertores** y las **cintas transportadoras** llevan las unidades de procesamiento, estructuras de baja densidad y combustible para cohetes hasta las entradas del silo. También pueden retirar productos de edificios auxiliares y moverlos hacia cofres de almacenamiento.

Los cofres permiten crear una reserva de materiales, pero no solucionan una falta de producción sostenida. Si el silo consume materiales más rápido de lo que se fabrican, el almacenamiento se vaciará y el ritmo de lanzamientos caerá.

En *Space Age*, los insertores y la red logística también pueden utilizarse para responder a solicitudes de carga de una plataforma orbital y automatizar los envíos.

##### 2.5. Estructuras exclusivas de *Space Age*

###### Plataforma espacial (*Space platform*) y centro de plataforma (*Space platform hub*)


[Ver la página de referencia de las plataformas espaciales en Factorio Wiki](https://wiki.factorio.com/Space_platform)

Las plataformas espaciales son fábricas que operan en órbita. Se construyen a partir de un **paquete inicial de plataforma espacial**, que se lanza desde el silo a una órbita seleccionada.

Sus funciones principales son:

- Transportar objetos y al personaje entre superficies planetarias y órbitas.
- Recoger y procesar asteroides.
- Fabricar ciencia espacial y otros productos que solo pueden elaborarse en el espacio.
- Servir como nodo logístico para solicitar objetos a las superficies planetarias y entregar carga.

El centro de plataforma almacena objetos, registra solicitudes y coordina buena parte de la logística orbital. La plataforma debe disponer de una base de cimientos espaciales para colocar edificios y expandirse.

Fuente: [Space platform — Factorio Wiki](https://wiki.factorio.com/Space_platform).

###### Cimientos de plataforma espacial (*Space platform foundation*)


Los cimientos amplían el área donde se pueden colocar edificios en una plataforma. Su uso no es necesario para el lanzamiento inicial de todos los objetos, pero sí para construir una fábrica orbital funcional y darle espacio para los colectores, trituradoras, hornos, cintas y defensas que correspondan.

La plataforma debe conservar una superficie conectada con el centro; no se pueden dejar huecos que rompan la continuidad de los cimientos.

Fuente: [Space platform foundation — Factorio Wiki](https://wiki.factorio.com/Space_platform_foundation).

###### Colector de asteroides (*Asteroid collector*) y trituradora (*Crusher*)


En *Space Age*, la ciencia espacial no se obtiene mediante satélites. La plataforma recoge fragmentos de asteroides con un colector y los procesa mediante trituradoras. Parte de los recursos resultantes se convierte en ingredientes para recetas espaciales; ciertos productos, como el mineral de hierro, pueden necesitar fundición antes de incorporarse a otras recetas.

El colector debe orientarse y colocarse para recoger los fragmentos disponibles en la trayectoria orbital. Las trituradoras necesitan recibir los fragmentos correctos y tener una salida para los productos. Una plataforma sin espacio suficiente, sin energía o sin una logística de objetos bien organizada puede dejar de producir ciencia espacial.

Fuentes: [Space science pack — Factorio Wiki](https://wiki.factorio.com/Space_science_pack), [Space Age — Factorio Wiki](https://wiki.factorio.com/Space_Age).

---

#### 3. Objetos e insumos requeridos

##### 3.1. Ingredientes de una parte de cohete

La receta de una parte de cohete cambia entre el juego base y *Space Age*.

| Ingrediente | Juego base por parte | *Space Age* por parte | Función |
|---|---:|---:|---|
| Unidad de procesamiento | 10 | 1 | Circuito avanzado requerido para la estructura y los sistemas del cohete. |
| Estructura de baja densidad | 10 | 1 | Componente ligero empleado en la estructura del vehículo. |
| Combustible para cohetes | 10 | 1 | Ingrediente energético para fabricar cada parte de cohete. |
| Partes de cohete necesarias para completar el cohete | 100 | 50 | El silo las acumula internamente hasta tener suficientes para ensamblar el vehículo. |

**Coste total de ingredientes para las partes de un cohete, antes de considerar productividad:**

| Modo | Unidades de procesamiento | Estructuras de baja densidad | Combustible para cohetes |
|---|---:|---:|---:|
| Juego base | 1000 | 1000 | 1000 |
| *Space Age* | 50 | 50 | 50 |

Estos totales no incluyen el coste de construir el silo, fabricar la carga útil ni cubrir las pérdidas o variaciones debidas a la planificación de la fábrica. La productividad puede reducir el consumo real de ingredientes por lanzamiento.

Fuente: [Rocket part — Factorio Wiki](https://wiki.factorio.com/Rocket_part).

##### 3.2. Productos y componentes importantes

| Objeto | Cómo se consigue | Para qué sirve |
|---|---|---|
| **Unidad de procesamiento** | Cadena de fabricación electrónica avanzada | Ingrediente de las partes de cohete y de muchos edificios avanzados. |
| **Estructura de baja densidad** | Máquina de montaje o recetas metalúrgicas compatibles en *Space Age* | Ingrediente de las partes de cohete y otros productos de alta tecnología. |
| **Combustible para cohetes** | Cadena de fabricación de combustible | Ingrediente de las partes de cohete; también puede utilizarse como combustible en dispositivos compatibles. |
| **Parte de cohete** | Fabricada directamente dentro del silo | Se acumula para ensamblar el cohete completo. No puede retirarse normalmente del silo. |
| **Satélite** | Fabricado en máquinas de montaje en el juego base | Carga que produce 1000 paquetes de ciencia espacial al lanzarse. No se usa en *Space Age*. |
| **Paquete de ciencia espacial** | Lanzamiento con satélite en el juego base; fabricación en órbita en *Space Age* | Se utiliza en laboratorios para investigaciones que lo requieren. |
| **Paquete inicial de plataforma espacial** | Cadena de fabricación avanzada en *Space Age* | Permite crear una plataforma espacial en la órbita elegida. |
| **Fragmentos de asteroide** | Recogidos por colectores en una plataforma | Se procesan en el espacio para obtener recursos que alimentan las recetas orbitales. |
| **Cimientos de plataforma espacial** | Fabricación y envío logístico en *Space Age* | Amplían la superficie edificable de la plataforma. |

##### 3.3. Satélite y carga útil en el juego base


En el juego base, el satélite es una carga útil que se fabrica con componentes avanzados. No se consume en las recetas de ciencia habituales: se introduce en el cohete antes del lanzamiento y, al lanzarlo, genera una entrega de **1000 paquetes de ciencia espacial** al puerto de aterrizaje de carga.

El satélite es costoso, por lo que conviene preparar una línea de producción específica para sus componentes. Antes de lanzar, comprueba que el puerto de aterrizaje de carga existe y tiene capacidad disponible. Si el inventario del puerto está lleno, los paquetes pueden terminar en el suelo en lugar de almacenarse donde esperabas.

La primera vez que se lanza un cohete se muestra la pantalla de finalización del juego base; el jugador puede terminar la partida o continuar para seguir ampliando la fábrica y lanzar más cohetes.

Fuente: [Satellite — Factorio Wiki](https://wiki.factorio.com/Satellite).

##### 3.4. Paquete de ciencia espacial


El **paquete de ciencia espacial** es un objeto de investigación que se consume en laboratorios.

- En el juego base, se obtiene como resultado de lanzar un satélite y recibir la entrega en el puerto de aterrizaje de carga.
- En *Space Age*, se fabrica en una plataforma espacial a partir de recursos obtenidos y procesados de asteroides.

En el juego base, la ciencia espacial se utiliza principalmente para tecnologías infinitas. En *Space Age*, es parte de la progresión científica que permite investigar tecnologías posteriores, por lo que la producción orbital debe mantenerse de forma estable si se quiere continuar investigando.

Fuente: [Space science pack — Factorio Wiki](https://wiki.factorio.com/Space_science_pack).

---

#### 4. Cómo preparar y realizar un lanzamiento

##### 4.1. Paso 1: investigar la tecnología

Investiga la tecnología del silo de cohete y las tecnologías que habilitan sus ingredientes. La investigación requiere paquetes de ciencia avanzados y una fábrica capaz de producir componentes electrónicos, estructuras de baja densidad y combustible para cohetes.

En el juego base, el silo se desbloquea relativamente tarde y es una de las últimas grandes metas industriales de la partida. En *Space Age*, se desbloquea antes porque el transporte orbital es necesario para alcanzar los demás planetas.

##### 4.2. Paso 2: construir el silo en una zona adecuada

El silo ocupa un área de 9 × 9 casillas y necesita una conexión eléctrica robusta. Antes de construirlo, deja espacio para:

- Las cintas o los insertores de entrada de los tres ingredientes principales.
- La retirada o el transporte de los productos fabricados en las líneas auxiliares.
- Los postes eléctricos y las conexiones de automatización.
- El puerto de aterrizaje de carga, en el juego base, o la logística de carga de plataformas, en *Space Age*.
- La ampliación futura, si el objetivo es lanzar cohetes con frecuencia.

El silo consume aproximadamente 4 MW a plena actividad. Comprueba la capacidad de la red eléctrica antes de ponerlo en marcha; una red insuficiente puede ralentizar el montaje de partes y afectar a otras máquinas.

##### 4.3. Paso 3: mantener el suministro de componentes

El silo necesita recibir unidades de procesamiento, estructuras de baja densidad y combustible para cohetes. Como estas recetas son complejas y pueden competir con otras cadenas de producción, conviene separar la producción de los ingredientes principales en líneas visibles y fáciles de ampliar.

Una forma práctica de diseñar la cadena es:

| Proceso / Etapa | Insumos Requeridos | Edificio / Máquina | Producto Resultante | Observación Técnica |
| --- | --- | --- | --- | --- |
| Preparación de componentes | Unidades de procesamiento | Línea de fabricación | Unidades de procesamiento disponibles | — |
| Preparación de componentes | Estructuras de baja densidad | Línea de fabricación | Estructuras de baja densidad disponibles | — |
| Preparación de combustible | Combustible para cohetes | Línea de fabricación | Combustible para cohetes disponible | — |
| Construcción de partes | Unidades de procesamiento, estructuras de baja densidad y combustible para cohetes | Silo de cohete | Partes de cohete | — |
| Montaje del cohete | Partes de cohete | Silo de cohete | Cohete completo | — |

Este esquema muestra el flujo de materiales, no la colocación exacta de cintas e insertores. En una fábrica real, asegúrate de que cada ingrediente tenga suficiente caudal y que la salida de las máquinas de montaje no quede bloqueada.

##### 4.4. Paso 4: preparar la carga útil

La carga útil depende del modo de juego.

**Juego base:** coloca un satélite en el silo para producir ciencia espacial. En las versiones actuales, un lanzamiento requiere una carga válida; la carga habitual es el satélite. Asegúrate de que el puerto de aterrizaje de carga esté construido.

**Space Age:** usa el silo para enviar objetos a una plataforma orbital o para lanzar un paquete inicial de plataforma espacial. La carga y el destino se gestionan desde la interfaz del silo y las solicitudes logísticas. Para enviar diferentes materiales, revisa las solicitudes de la plataforma y el método de carga seleccionado.

##### 4.5. Paso 5: lanzar y comprobar el resultado

Cuando el cohete esté construido y la carga esté preparada, el lanzamiento puede iniciarse desde la interfaz del silo. Según el modo y la configuración, es posible habilitar el envío automático cuando haya carga lista.

Después del lanzamiento:

- En el juego base, verifica que el puerto de aterrizaje de carga recibió los paquetes de ciencia espacial.
- En *Space Age*, comprueba que la plataforma recibió la carga esperada, que el paquete inicial creó la plataforma correcta o que el personaje llegó a la órbita prevista.
- Revisa las reservas del silo y las líneas de producción para que el siguiente cohete no se quede sin ingredientes.

---

#### 5. Automatización de lanzamientos y logística orbital

##### 5.1. Automatizar el suministro del silo

Una línea automatizada debe mantener un flujo constante de componentes. Para ello:

- Utiliza cintas o insertores para suministrar los materiales de la receta.
- Reserva capacidad de producción para que la demanda de otras máquinas no vacíe los ingredientes del silo.
- Emplea cofres como amortiguador de producción cuando haya variaciones en los tiempos de fabricación.
- Comprueba el estado de la red eléctrica y evita que el silo quede sin potencia.
- Considera usar módulos de productividad compatibles para reducir la cantidad de ingredientes necesaria por lanzamiento.

El objetivo no es solo llenar el silo una vez, sino construir una infraestructura que pueda repetir el proceso sin intervención manual continua.

##### 5.2. Automatización en el juego base

En el juego base, la automatización suele centrarse en el suministro de materiales, la fabricación del satélite y la recepción de paquetes de ciencia espacial. El silo puede configurarse para lanzar automáticamente cuando el cohete esté listo y haya una carga disponible.

Si se automatizan lanzamientos repetidos, el puerto de aterrizaje debe tener espacio y un sistema de insertores que traslade los paquetes a las cintas o cofres que alimentan los laboratorios. De lo contrario, la entrega puede quedar bloqueada o dejar objetos en el suelo.

##### 5.3. Automatización en *Space Age*

En *Space Age*, el silo forma parte de la red logística interplanetaria. Las plataformas tienen solicitudes de carga y el silo puede ser abastecido con materiales pedidos por el centro de la plataforma. La automatización puede funcionar con insertores y logística de la superficie o con carga manual desde la interfaz.

Ten en cuenta que algunos métodos de envío tienen limitaciones distintas. La selección manual permite decidir la mezcla de objetos para una carga, mientras que el llenado automático basado en solicitudes está pensado para responder a las necesidades de la plataforma. Verifica la interfaz de la versión actual del juego, ya que la disponibilidad de opciones depende del método de carga y de la configuración logística.

##### 5.4. Crear y mantener una fábrica orbital

Para producir ciencia espacial en *Space Age*, una plataforma necesita una combinación funcional de edificios:

1. **Centro de plataforma:** gestiona inventario y solicitudes.
2. **Cimientos de plataforma espacial:** proporcionan superficie edificable.
3. **Colector de asteroides:** recoge fragmentos durante el viaje orbital.
4. **Trituradoras:** procesan los fragmentos en recursos utilizables.
5. **Hornos y máquinas de montaje compatibles:** convierten los recursos en ingredientes y fabrican productos espaciales, incluida la ciencia espacial.
6. **Cintas e insertores:** distribuyen los materiales dentro de la plataforma.
7. **Energía y defensas adecuadas:** mantienen la fábrica operativa durante los desplazamientos orbitales.

La capacidad de producción depende del trayecto, de los tipos de asteroides disponibles, de la velocidad de procesamiento y del espacio que tenga la plataforma. Una plataforma que solo recoge fragmentos, pero no puede procesarlos o almacenar sus productos, no producirá ciencia de manera continua.

---

#### 6. Costes, productividad y capacidad de producción

##### 6.1. El coste industrial de un cohete

La receta de las partes de cohete exige ingredientes que también se utilizan en otros componentes avanzados. En particular, las unidades de procesamiento y las estructuras de baja densidad pueden convertirse en cuellos de botella si la fábrica las comparte entre muchas líneas.

En el juego base, la exigencia de 100 partes y 10 unidades de cada ingrediente por parte supone una gran inversión de materias primas. En *Space Age*, el coste de 50 partes y una unidad de cada ingrediente por parte reduce significativamente la cantidad de materiales que necesita el lanzamiento.

El coste total real de un lanzamiento será superior al coste de las partes cuando se suman el silo, las máquinas auxiliares, las cintas, los postes eléctricos, los cofres y la carga útil.

##### 6.2. Módulos de productividad

El silo admite cuatro módulos. Los **módulos de productividad** pueden aumentar el número de productos obtenidos por cada receta compatible, lo que reduce el consumo efectivo de materiales por unidad producida. Los módulos de velocidad pueden aumentar la velocidad de fabricación y suelen incrementar el consumo de energía.

La elección debe considerar la demanda y no solo la velocidad del edificio. Debido al alto coste de los ingredientes de cohete, la productividad suele ser atractiva, especialmente cuando hay módulos de nivel alto o bonificaciones de tecnología disponibles.

##### 6.3. Tecnología de productividad de partes de cohete

*Space Age* incorpora una tecnología infinita de productividad de partes de cohete. Cada nivel mejora el rendimiento de los ingredientes utilizados al producir partes de cohete, lo cual reduce el coste por lanzamiento a largo plazo.

Esta mejora es relevante para fábricas que envían grandes volúmenes de materiales a órbita o que necesitan lanzar cohetes de forma continua para mantener una red de plataformas.

##### 6.4. Ritmo de lanzamiento y fases de animación

El ritmo de lanzamientos no depende únicamente de la velocidad de fabricación de las partes. El silo también debe preparar el cohete, recibir la carga y completar la secuencia de lanzamiento. Por eso, añadir módulos de velocidad no reduce todas las fases en la misma proporción.

En *Space Age*, el silo puede mantener partes para el siguiente cohete y, en determinadas condiciones, acortar el intervalo entre lanzamientos. Sin embargo, para aprovechar esa capacidad debe tener suficiente producción de ingredientes y una logística que mantenga la carga y el transporte al día.

##### 6.5. Dimensionar las líneas de suministro

Para evitar que un ingrediente limite toda la instalación:

- Calcula el consumo de unidades de procesamiento, estructuras de baja densidad y combustible para cohetes según el número de partes que deseas fabricar por minuto.
- Revisa la capacidad de las cintas transportadoras y los insertores.
- Comprueba que los ingredientes no se desvíen a otras líneas hasta el punto de agotar el suministro del silo.
- Amplía primero el componente que realmente limita la producción, en vez de añadir máquinas al azar.
- Utiliza estadísticas de producción para comparar el consumo del silo con lo que fabrican las líneas auxiliares.

---

#### 7. Estrategias y consejos prácticos

##### 7.1. Construir por módulos

Diseña el área alrededor del silo de manera que cada ingrediente tenga una ruta clara. Por ejemplo, puedes reservar una línea para estructuras de baja densidad, otra para unidades de procesamiento y una tercera para combustible. La organización por módulos facilita duplicar una sección sin alterar todo el sistema.

##### 7.2. Crear reservas de componentes avanzados

Antes de poner en marcha lanzamientos frecuentes, acumula cierta reserva de los tres ingredientes principales. Esto permite que el silo siga trabajando durante las interrupciones momentáneas de alguna cadena de producción.

Sin embargo, no dependas únicamente de las reservas. Si el consumo medio del silo supera la producción media de un ingrediente, el inventario se agotará tarde o temprano.

##### 7.3. Mantener despejada la salida de ciencia espacial

En el juego base, utiliza insertores para retirar los paquetes del puerto de aterrizaje de carga y distribuirlos a los laboratorios o al almacenamiento. Comprueba que hay espacio en el destino y que las cintas no estén bloqueadas.

En *Space Age*, planifica la salida de ciencia espacial desde la plataforma. Si la ciencia se acumula en el centro de plataforma y el resto del inventario no tiene capacidad, puede limitar la logística y reducir el ritmo de producción.

##### 7.4. No invertir toda la fábrica en el primer lanzamiento

El silo y sus componentes cuestan muchos recursos, pero también es necesario mantener las cadenas que producen energía, circuitos, estructuras y combustible. Conviene evitar que la línea del cohete consuma tantos recursos que detenga la producción de otros elementos imprescindibles para la fábrica.

##### 7.5. Aprovechar las bonificaciones de productividad

Debido a la elevada cantidad de materiales que consume un cohete, una mejora de productividad puede ahorrar muchos recursos a lo largo de varios lanzamientos. Prioriza las mejoras que reduzcan el coste real de los ingredientes y considera los módulos de velocidad cuando el problema sea el tiempo de fabricación y la red eléctrica tenga margen suficiente.

##### 7.6. En *Space Age*, diseñar la plataforma según el destino

Las plataformas pueden cumplir funciones diferentes: transportar carga, transportar al personaje o producir ciencia espacial. No todas necesitan la misma distribución.

- Una plataforma de transporte necesita capacidad de carga y una logística de solicitudes clara.
- Una plataforma científica necesita colectores, trituradoras, almacenamiento y máquinas de fabricación.
- Una plataforma para trayectos largos debe considerar la disponibilidad de fragmentos, la energía y la protección frente a asteroides.

Mantén las líneas de producción y el inventario de la plataforma sencillos al principio; después, añade capacidad cuando los datos de producción indiquen que es necesaria.

---

#### 8. Errores frecuentes y soluciones

| Problema | Causa posible | Solución |
|---|---|---|
| El silo no comienza a fabricar partes. | Falta uno de los ingredientes, falta electricidad o no está desbloqueada la receta/tecnología necesaria. | Comprueba las entradas, la energía y la investigación. |
| Las partes de cohete no avanzan con rapidez. | La producción de unidades de procesamiento, estructuras de baja densidad o combustible está por debajo del consumo requerido. | Identifica el ingrediente limitante y amplía esa línea. |
| La red eléctrica se vuelve insuficiente al activar el silo. | El silo consume una potencia considerable y otras máquinas compiten por la misma red. | Añade generación o reserva capacidad antes de lanzar. |
| Se desmonta un silo y desaparece el progreso del cohete. | Las partes almacenadas dentro del silo se pierden al deconstruirlo. | No deconstruyas el silo hasta terminar el lanzamiento o acepta la pérdida de las partes. |
| El lanzamiento del juego base no produce ciencia. | No se colocó un satélite válido como carga o el puerto de aterrizaje de carga no está disponible. | Comprueba la carga y que haya un puerto colocado en la superficie. |
| Los paquetes de ciencia espacial aparecen en el suelo. | El inventario del puerto de aterrizaje de carga está lleno. | Retira la ciencia con insertores o amplía el almacenamiento de destino. |
| El silo de *Space Age* no entrega materiales a la plataforma. | La plataforma no está en órbita sobre el planeta, no tiene una solicitud válida o el método de carga no corresponde a lo solicitado. | Revisa el destino, las solicitudes y la configuración del silo. |
| La plataforma orbital no produce ciencia espacial. | No recoge suficientes asteroides, faltan trituradoras u otros ingredientes o la salida de productos está bloqueada. | Comprueba los colectores, el procesamiento, la electricidad y el espacio de inventario. |
| La fábrica orbital se queda sin recursos. | La producción de los fragmentos o su procesamiento no iguala la demanda. | Ajusta la velocidad de trituración, almacenamiento y fabricación, o modifica el diseño de la plataforma. |
| Los módulos de velocidad no mejoran el ritmo de lanzamientos como se esperaba. | Las animaciones de preparación y lanzamiento añaden tiempo que no se reduce en la misma proporción que la receta. | Evalúa el ciclo completo y asegúrate de que el silo tenga suficientes ingredientes y carga. |

---

#### 9. Diferencias avanzadas de *Space Age*

##### 9.1. El cohete como transporte, no solo como meta

En el juego base, el lanzamiento está asociado con el hito final y la producción de ciencia espacial. En *Space Age*, los cohetes constituyen el sistema de transporte que conecta las superficies planetarias con las plataformas en órbita. Los objetos se envían desde silos a centros de plataforma, mientras que las plataformas pueden realizar entregas a los puertos de aterrizaje de las superficies.

Esta diferencia transforma la planificación logística: los recursos de la superficie no solo deben abastecer a la fábrica local, sino también atender las solicitudes de las plataformas.

##### 9.2. Ciencia espacial fabricada en órbita

La ciencia espacial de *Space Age* se fabrica a partir de recursos recogidos de asteroides. El flujo básico consiste en recolectar fragmentos, triturarlos y procesar las salidas en las máquinas adecuadas antes de fabricar el paquete. Ciertas recetas necesitan que uno de los productos minerales se funda primero.

La plataforma debe tener espacio para las estructuras y cintas. Es importante diseñar la salida de residuos y subproductos porque una trituradora puede seguir recibiendo fragmentos y llenar el inventario si no existe una ruta para sus resultados.

##### 9.3. Expansión y capacidad de las plataformas

Una plataforma inicial solo proporciona una base pequeña. Para realizar producción avanzada se requieren cimientos adicionales, máquinas, cintas, colectores y espacio para los sistemas necesarios en el trayecto. La masa y el tamaño de la plataforma también influyen en su capacidad de transporte y en la planificación del movimiento orbital.

La infraestructura debe ampliarse en pasos: primero conseguir un flujo estable de recursos, después procesarlos y finalmente aumentar la capacidad de ciencia o de carga.

##### 9.4. El objetivo final de la expansión

*Space Age* no termina simplemente con el primer lanzamiento. El objetivo global se amplía hacia la exploración planetaria y la construcción de una plataforma capaz de llegar al borde del sistema solar. Por eso, el primer cohete de la expansión es el inicio de una nueva etapa y no el cierre de la progresión.

Fuente: [Space Age — Factorio Wiki](https://wiki.factorio.com/Space_Age).

---

#### 10. Lista de comprobación antes de lanzar

- [ ] Se investigó la tecnología del silo de cohete.
- [ ] El silo está conectado a una red eléctrica con capacidad suficiente.
- [ ] Las líneas de producción suministran unidades de procesamiento, estructuras de baja densidad y combustible para cohetes.
- [ ] Se conoce la cantidad de partes requerida por el modo de juego: 100 en el juego base o 50 en *Space Age*.
- [ ] El silo tiene suficientes ingredientes para completar el cohete.
- [ ] En el juego base, hay un puerto de aterrizaje de carga colocado y capacidad para recibir los paquetes de ciencia.
- [ ] En el juego base, el satélite está listo si se desea obtener ciencia espacial.
- [ ] En *Space Age*, está seleccionada la plataforma o la carga de destino adecuada.
- [ ] Las rutas de cinta e insertores permiten cargar y descargar los objetos.
- [ ] Hay suficiente espacio de almacenamiento y las salidas no están bloqueadas.
- [ ] Las reservas de componentes avanzados son suficientes para mantener los demás procesos de la fábrica.
- [ ] Se comprobó que los módulos y las bonificaciones instaladas no exceden la capacidad eléctrica disponible.
- [ ] El silo no se deconstruirá mientras contenga partes de cohete que se quieran conservar.

---

---

#### 12. Fuentes de referencia

Las siguientes páginas de la *Factorio Wiki* describen las recetas, los edificios y las mecánicas de esta guía:

- [Rocket silo — Silo de cohete](https://wiki.factorio.com/Rocket_silo)
- [Rocket part — Parte de cohete](https://wiki.factorio.com/Rocket_part)
- [Satellite — Satélite](https://wiki.factorio.com/Satellite)
- [Space science pack — Paquete de ciencia espacial](https://wiki.factorio.com/Space_science_pack)
- [Cargo landing pad — Puerto de aterrizaje de carga](https://wiki.factorio.com/Cargo_landing_pad)
- [Space platform — Plataforma espacial](https://wiki.factorio.com/Space_platform)
- [Space platform foundation — Cimientos de plataforma espacial](https://wiki.factorio.com/Space_platform_foundation)
- [Space platform starter pack — Paquete inicial de plataforma espacial](https://wiki.factorio.com/Space_platform_starter_pack)
- [Low density structure — Estructura de baja densidad](https://wiki.factorio.com/Low_density_structure)
- [Rocket fuel — Combustible para cohetes](https://wiki.factorio.com/Rocket_fuel)
- [Science pack — Paquetes de ciencia](https://wiki.factorio.com/Science_pack)
- [Space Age — Expansión](https://wiki.factorio.com/Space_Age)

---

### Guía técnica de Factorio: Gestión del inventario personal y equipamiento (*Spidertron*, armaduras)

**Versión de referencia:** *Factorio 2.x*. Las características exclusivas de la expansión *Space Age* se identifican por separado. Las dimensiones de las cuadrículas y las bonificaciones numéricas indicadas corresponden a objetos de calidad normal, salvo cuando se presenta una tabla de calidad.

La gestión del inventario y del equipamiento determina cuánto material puede transportar el jugador, qué herramientas tiene disponibles y qué funciones adicionales puede utilizar durante la exploración y la construcción. A medida que la fábrica crece, desplazarse con un inventario bien organizado y disponer de una armadura preparada puede ahorrar viajes, facilitar la construcción con robots y reducir interrupciones.

Esta guía explica el inventario del personaje, las armaduras, los módulos de equipamiento, la logística personal y las particularidades del *Spidertron* como vehículo móvil configurable.

---

#### 1. Fundamentos del inventario y el equipamiento

##### 1.1. Inventario personal

El inventario personal es el conjunto de espacios donde el personaje guarda los objetos que recoge, fabrica o transporta. Puede contener materias primas, placas, componentes, herramientas de construcción, planos, objetos de equipamiento y otros artículos.

La capacidad total disponible depende de varios factores:

- **Espacios base del personaje:** los espacios normales del inventario.
- **Tecnologías de capacidad:** determinadas investigaciones aumentan el número de espacios disponibles.
- **Armadura equipada:** las armaduras avanzadas agregan espacio de inventario como parte de sus características.
- **Equipamiento de cinturón de herramientas (*Toolbelt equipment*):** objeto de *Space Age* que puede añadir espacios adicionales dentro de una cuadrícula de equipamiento compatible.
- **Calidad del objeto:** en *Factorio 2.x*, una armadura o un módulo de mayor calidad puede ofrecer una cuadrícula o una bonificación superior.

El inventario no es lo mismo que la barra rápida (*quickbar*). La barra rápida guarda accesos directos a objetos, planos y herramientas; no constituye una reserva adicional de materiales. Si el objeto que representa un acceso directo se acaba, el acceso continúa visible, pero no crea unidades nuevas.

##### 1.2. Interfaz del personaje

La interfaz del personaje incluye zonas diferenciadas para el inventario, la armadura y el equipamiento. Según la interfaz y las funciones desbloqueadas, también pueden aparecer secciones para logística personal y filtros de objetos sobrantes.

Las acciones más comunes son:

- **Mover objetos:** arrastrar pilas entre espacios, cofres, edificios e inventarios.
- **Transferir pilas rápidamente:** utilizar los controles de transferencia rápida de la interfaz para mover objetos sin arrastrarlos manualmente.
- **Filtrar espacios:** reservar ranuras para determinados artículos y mantener ordenados materiales de uso frecuente.
- **Abrir la cuadrícula de equipamiento:** hacer clic derecho sobre una armadura equipada o sobre el vehículo compatible para configurar sus módulos.
- **Usar filtros logísticos:** definir qué objetos deberían entregarse al personaje y qué exceso debería retirarse mediante robots logísticos.

Los nombres exactos de algunos botones pueden variar según la traducción y la configuración de controles.

##### 1.3. Inventario, armadura y cuadrícula: tres cosas distintas

Es importante no confundir estos elementos:

| Elemento | Función | Ejemplo |
|---|---|---|
| **Inventario** | Guarda objetos transportados. | Placas de hierro, cintas y postes eléctricos. |
| **Ranura de armadura** | Determina la armadura que lleva el personaje y sus bonificaciones. | Armadura modular o armadura de potencia. |
| **Cuadrícula de equipamiento** | Acepta módulos que otorgan funciones adicionales. | Baterías, paneles solares portátiles, exoesqueletos y robots de construcción personales. |
| **Barra rápida** | Ofrece accesos directos para seleccionar objetos o herramientas. | Plano de una fundición o una cinta transportadora. |
| **Logística personal** | Pide objetos a la red logística y permite retirar sobrantes. | Mantener una reserva de postes eléctricos y retirar exceso de piedra. |

La cuadrícula de equipamiento no almacena materias primas comunes: sus espacios aceptan módulos de equipamiento compatibles. De forma similar, aumentar el inventario no incrementa por sí mismo la potencia eléctrica ni el área cubierta por robots; esas funciones dependen de los módulos instalados.

---

#### 2. Gestión práctica del inventario

##### 2.1. Priorizar materiales de uso frecuente

Durante la construcción manual suele ser útil mantener un conjunto de materiales esenciales accesibles. La selección concreta depende de la etapa de la fábrica, pero puede incluir:

- Cintas transportadoras, postes eléctricos e insertores.
- Cofres, ensambladoras y materiales de construcción habituales.
- Rieles, señales y postes para ampliar una red ferroviaria.
- Tuberías y bombas para trabajos con fluidos.
- Reparadores y materiales necesarios para mantener infraestructura.
- Planos (*blueprints*) de instalaciones que se construyen con frecuencia.

No es necesario cargar grandes cantidades de todos los materiales de forma permanente. Los objetos que se usan poco pueden quedar en cofres o en la red logística y solicitarse cuando hagan falta.

##### 2.2. Reservar espacios y evitar un inventario lleno

Un inventario lleno dificulta recoger productos, abrir espacio para objetos de construcción y transportar materiales obtenidos durante una expedición. Para evitarlo:

1. Reserva espacios para los materiales prioritarios si utilizas filtros.
2. Evita llevar grandes cantidades de recursos que pueden solicitarse a una red logística cercana.
3. Deposita los materiales sobrantes en cofres o devuélvelos a la red logística.
4. Utiliza la opción de basura logística para los objetos de los que quieras desprenderte, siempre que haya una red accesible.
5. Antes de retirar una armadura con bonificación de inventario, comprueba que los objetos caben en los espacios restantes.

**Importante:** desde las versiones modernas, si el inventario no tiene espacio suficiente para conservar los objetos, el juego impide retirar la armadura que aporta esos espacios adicionales. Esto evita que el equipo o los objetos alojados en las ranuras adicionales se pierdan o se desplacen inesperadamente.

##### 2.3. Filtros y reservas de materiales

Los filtros permiten reservar espacios para determinados tipos de objetos. Son útiles cuando se quiere impedir que las transferencias rápidas llenen las ranuras destinadas a materiales prioritarios.

Una organización práctica puede separar:

- **Construcción:** cintas, postes, insertores y edificios habituales.
- **Logística:** cofres, robots y componentes de red cuando se necesitan para una instalación.
- **Exploración:** materiales para reparación y construcción de emergencia.
- **Objetos temporales:** recursos que se recogen durante una tarea y deben depositarse al regresar.

El inventario debe responder al trabajo en curso. Una configuración preparada para ampliar una fundición no tiene por qué ser la ideal para recorrer una zona alejada de la base.

##### 2.4. Tecnología de capacidad y calidad

El árbol tecnológico incluye investigaciones que aumentan la capacidad del personaje. En *Space Age*, el equipamiento de cinturón de herramientas añade otra forma de ampliar el inventario mediante la cuadrícula de equipamiento.

La calidad añade una dimensión adicional: las versiones de mayor calidad pueden disponer de bonificaciones de inventario y cuadrículas mayores. Eso no significa que toda mejora de calidad sea necesaria desde el inicio; conviene priorizarla cuando la capacidad adicional justifique los materiales y el proceso de fabricación.

---

#### 3. Armaduras: función y progresión

##### 3.1. ¿Qué aporta una armadura?

La armadura ocupa la ranura de armadura del personaje y reduce determinados tipos de daño según sus resistencias. Las armaduras avanzadas también ofrecen una cuadrícula para instalar módulos, además de una bonificación al inventario.

La progresión habitual pasa de las armaduras sencillas, que priorizan la protección, a las armaduras modulares y de potencia, que permiten configurar el equipo para construcción, exploración y movilidad.

##### 3.2. Armadura ligera y armadura pesada

Las armaduras ligera y pesada son opciones iniciales del juego base.

- **Armadura ligera:** proporciona protección básica y requiere pocos recursos en comparación con las armaduras avanzadas.
- **Armadura pesada:** ofrece más protección y resulta útil antes de desbloquear armaduras modulares.

Estas armaduras no cuentan con una cuadrícula de equipamiento modular. Por tanto, no pueden instalarse directamente en ellas baterías personales, paneles solares portátiles, exoesqueletos ni módulos de robot de construcción.

Consulta la [Factorio Wiki: Armor](https://wiki.factorio.com/Armor).

##### 3.3. Armadura modular (*Modular armor*)


La **armadura modular** es el primer tipo de armadura que permite instalar equipamiento. Su cuadrícula normal mide **5 × 5 casillas** y la armadura aporta **10 espacios de inventario adicionales**.

Sus principales ventajas son:

- Permite adaptar el personaje a tareas específicas.
- Admite módulos de generación y almacenamiento de energía.
- Puede incorporar visión nocturna, exoesqueletos, escudos y robots de construcción personales.
- Sirve como primera plataforma para aprender a equilibrar espacio, potencia y módulos.

El límite principal es su cuadrícula relativamente pequeña. Los módulos que ocupan superficies grandes, como ciertos generadores portátiles, pueden dejar poco espacio para otros sistemas.

Más información: [Factorio Wiki: Modular armor](https://wiki.factorio.com/Modular_armor).

##### 3.4. Armadura de potencia (*Power armor*)


La **armadura de potencia** amplía las posibilidades de la armadura modular. En calidad normal ofrece una cuadrícula de **6 × 8 casillas** y una bonificación de **20 espacios de inventario**.

Es apropiada para diseños equilibrados que combinan movilidad, generación eléctrica, escudos y construcción mediante robots personales. Su espacio adicional facilita integrar más de una función sin sacrificar tantos módulos como en la armadura modular.

Más información: [Factorio Wiki: Power armor](https://wiki.factorio.com/Power_armor).

##### 3.5. Armadura de potencia MK2 (*Power armor MK2*)

La **armadura de potencia MK2** es una mejora avanzada del juego base. En calidad normal cuenta con una cuadrícula de **10 × 10 casillas** y añade **30 espacios de inventario**.

Su principal ventaja es el equilibrio entre protección, capacidad y espacio para módulos. Permite instalar combinaciones más completas de generadores, baterías, exoesqueletos, escudos y roboports personales.

Es una opción adecuada para jugadores que construyen grandes zonas de la fábrica con planos, recorren grandes distancias y necesitan llevar una configuración de equipamiento flexible.

Más información: [Factorio Wiki: Power armor MK2](https://wiki.factorio.com/Power_armor_MK2).

##### 3.6. Armadura mecánica (*Mech armor*) — *Space Age*


La **armadura mecánica** es exclusiva de *Space Age*. En calidad normal ofrece una cuadrícula de **10 × 12 casillas** y añade **50 espacios de inventario**.

Su característica diferencial es que permite al personaje volar automáticamente al caminar hacia ciertos obstáculos o terrenos que normalmente impedirían el paso, como árboles, edificios, agua o lava. El vuelo no tiene un límite de duración ni exige energía, aunque determinadas condiciones especiales pueden desactivarlo temporalmente.

La armadura mecánica proporciona una gran cuadrícula y una bonificación de inventario elevada, por lo que resulta especialmente útil para tareas de construcción y desplazamiento entre áreas separadas. No puede retirarse mientras el personaje está volando.

Más información: [Factorio Wiki: Mech armor](https://wiki.factorio.com/Mech_armor).

##### 3.7. Comparación de armaduras

| Armadura (calidad normal) | Cuadrícula de equipamiento | Bonificación de inventario | ¿Admite módulos? | Característica principal |
|---|---:|---:|---|---|
| Ligera | No tiene | 0 | No | Protección inicial. |
| Pesada | No tiene | 0 | No | Mayor protección que la ligera. |
| Modular | 5 × 5 | +10 espacios | Sí | Primer acceso al equipamiento configurable. |
| Potencia | 6 × 8 | +20 espacios | Sí | Más espacio para módulos. |
| Potencia MK2 | 10 × 10 | +30 espacios | Sí | Gran cuadrícula y protección avanzada. |
| Mecánica (*Space Age*) | 10 × 12 | +50 espacios | Sí | Cuadrícula muy amplia y capacidad de vuelo. |

Los valores de inventario y dimensiones de la cuadrícula cambian con la calidad de la armadura. La tabla muestra los valores de calidad normal para facilitar la comparación de la progresión básica.

---

#### 4. Módulos de equipamiento personal

Los módulos de equipamiento se colocan en la cuadrícula de una armadura compatible. Algunos producen energía; otros almacenan energía, modifican la movilidad, permiten desplegar robots o mejoran la visión. Varios efectos se acumulan, siempre que haya espacio y energía suficientes.

##### 4.1. Generación de energía

| Módulo | Tamaño habitual | Función | Consideraciones |
|---|---:|---|---|
| **Panel solar portátil** | 1 × 1 | Genera electricidad mientras hay luz. | Pequeño, pero produce poca potencia por casilla. La producción depende del planeta y del ciclo de iluminación. |
| **Reactor de fisión portátil** | 4 × 4 | Genera energía para el equipamiento. | Ocupa mucho espacio; requiere una armadura con cuadrícula suficiente. |
| **Reactor de fusión portátil** | 4 × 4 | Proporciona una fuente de energía potente para equipos avanzados. | Es una opción avanzada; suele aprovecharse mejor en armaduras grandes. |

Sin un módulo que genere electricidad, los componentes eléctricos de la cuadrícula no pueden realizar trabajo de forma sostenida. Las baterías almacenan energía, pero no crean energía por sí mismas.

##### 4.2. Almacenamiento de energía

| Módulo | Tamaño | Función |
|---|---:|---|
| **Batería personal** | 1 × 2 | Guarda energía generada por paneles u otros módulos. |
| **Batería personal MK2** | 1 × 2 | Ofrece más capacidad que la batería personal básica. |
| **Baterías de calidad superior** | 1 × 2 | En las versiones compatibles, aumentan las prestaciones del módulo. |

Las baterías son especialmente útiles cuando se combina un generador con módulos que consumen mucha energía. Una cuadrícula con gran capacidad, pero sin generación suficiente, puede dejar de proporcionar energía a los módulos durante tareas continuadas.

##### 4.3. Movimiento: exoesqueleto


El **exoesqueleto** aumenta la velocidad de movimiento cuando recibe energía. Cada módulo ocupa **2 × 4 casillas** en una cuadrícula compatible y consume electricidad al estar activo.

Los efectos de varios exoesqueletos se acumulan. Esto es útil para desplazarse entre áreas de construcción, recoger materiales o recorrer grandes instalaciones. Sin embargo, instalar muchos exoesqueletos ocupa espacio que podría destinarse a baterías, escudos o roboports personales, y aumenta la demanda de energía.

Más información: [Factorio Wiki: Exoskeleton](https://wiki.factorio.com/Exoskeleton).

##### 4.4. Protección y visión

| Módulo | Tamaño habitual | Función |
|---|---:|---|
| **Escudo de energía** | 2 × 2 | Absorbe daño mientras dispone de energía. |
| **Escudo de energía MK2** | 2 × 2 | Aporta más protección que la versión básica. |
| **Visión nocturna** | 2 × 2 | Mejora la visibilidad durante la noche. |
| **Inmunidad a cintas** | 1 × 1 | Evita que las cintas transportadoras desplacen al personaje o vehículo compatible. |

Los escudos consumen energía para recargarse. Si el suministro de energía resulta insuficiente, la recuperación puede volverse lenta. La visión nocturna también depende de disponer de energía, aunque su consumo es relativamente pequeño en comparación con otros equipos.

##### 4.5. Robotport personal (*Personal roboport*)


El **robotport personal** permite al personaje desplegar robots de construcción desde su inventario o desde los recursos disponibles para la tarea. Se instala en una armadura con cuadrícula y funciona como una base robótica móvil.

Sus funciones principales son:

- Construir instalaciones indicadas por planos, si el personaje dispone de los edificios necesarios.
- Reparar estructuras cuando hay robots de construcción y materiales apropiados.
- Retirar estructuras marcadas para deconstrucción.
- Operar dentro del área cubierta por el sistema de roboports personales y de acuerdo con el número de robots permitido.

**Distinción importante:** el robotport personal controla **robots de construcción**, no sustituye a la red de robots logísticos que entrega objetos al personaje. La logística personal requiere una red logística disponible, mientras que el robotport personal se ocupa de tareas de construcción.

Cada robotport personal añade capacidad de despliegue, alcance de construcción y puntos de recarga. Colocar varios puede permitir que más robots trabajen simultáneamente, pero exige suficiente energía y espacio para los módulos.

Más información: [Factorio Wiki: Personal roboport](https://wiki.factorio.com/Personal_roboport).

##### 4.6. Equipamiento de cinturón de herramientas — *Space Age*

El **equipamiento de cinturón de herramientas** (*Toolbelt equipment*) es exclusivo de *Space Age*. Ocupa **3 × 1 casillas** y aumenta el inventario del personaje o vehículo compatible. Varias unidades pueden acumular su bonificación.

Este módulo es especialmente interesante para armaduras grandes y para el *Spidertron*, ya que aprovecha una franja de cuadrícula que a veces quedaría difícil de utilizar con módulos de otras dimensiones. Las versiones de mayor calidad aumentan la bonificación de inventario.

Más información: [Factorio Wiki: Toolbelt equipment](https://wiki.factorio.com/Toolbelt_equipment).

---

#### 5. Equilibrar la cuadrícula de equipamiento

##### 5.1. No basta con llenar todas las casillas

Una cuadrícula llena no es necesariamente una cuadrícula eficiente. Hay que elegir módulos compatibles con la tarea y comprobar que la producción y el almacenamiento de energía alcanzan para mantenerlos activos.

Antes de colocar módulos, revisa:

- **Espacio:** cada objeto ocupa una superficie fija, por lo que la forma de la cuadrícula importa tanto como el número de casillas libres.
- **Generación:** paneles solares, reactores portátiles u otras fuentes deben producir energía suficiente.
- **Almacenamiento:** las baterías ayudan a cubrir picos de consumo o momentos con poca generación.
- **Consumo:** módulos como los exoesqueletos y los escudos eléctricos pueden demandar potencia mientras trabajan.
- **Función prioritaria:** movilidad, construcción robótica, protección o capacidad de transporte.

##### 5.2. Configuraciones orientativas

Estas configuraciones son ejemplos conceptuales, no recetas obligatorias. Deben adaptarse al tamaño de la cuadrícula, la calidad de los objetos y la disponibilidad tecnológica.

**Configuración para construir con planos**

- Uno o más robotports personales.
- Una fuente de energía adecuada y baterías cuando hagan falta.
- Espacio de inventario reservado para los edificios del plano.
- Exoesqueletos si se necesita desplazarse rápidamente entre distintas áreas.

**Configuración para explorar**

- Fuente de energía y almacenamiento.
- Módulos de movilidad si son necesarios.
- Visión nocturna y protección según el entorno.
- Materiales de construcción y reparación en el inventario, sin llenar todos los espacios con objetos de uso poco frecuente.

**Configuración para maximizar capacidad**

- En *Space Age*, instalar equipamiento de cinturón de herramientas en la cuadrícula compatible.
- Aprovechar la calidad superior cuando el coste adicional sea razonable.
- Mantener los módulos que generan y almacenan energía solo si la tarea también requiere otros módulos eléctricos.

##### 5.3. Errores comunes al configurar equipamiento

- Instalar muchos módulos que consumen energía sin añadir generación ni baterías suficientes.
- Dedicar toda la cuadrícula a un único efecto y descubrir que falta espacio para el robotport o la visión nocturna.
- Usar paneles solares portátiles como única fuente para módulos de gran consumo, sin considerar la producción variable durante el ciclo de iluminación.
- Olvidar que los módulos instalados dejan de funcionar cuando la armadura no está equipada o cuando el vehículo no está activo.
- Retirar una armadura con bonificación de inventario mientras los espacios extra contienen objetos.

---

#### 6. Logística personal e inventario automatizado

##### 6.1. ¿Qué es la logística personal?

La logística personal permite indicar qué objetos debería mantener el personaje. Cuando el jugador está dentro de una red logística válida y la función está activada, los robots logísticos pueden entregar artículos solicitados y retirar los objetos marcados para eliminación.

La tecnología **Robótica logística** desbloquea las funciones básicas de solicitudes logísticas personales y eliminación automática, además de las estructuras y robots necesarios para establecer redes logísticas.

Esta mecánica reduce la necesidad de cargar siempre los materiales de construcción. Por ejemplo, el personaje puede solicitar una cantidad determinada de postes, cintas y ensambladoras. Mientras permanezca conectado a una red logística que disponga de esos materiales, los robots pueden completar las cantidades solicitadas.

Más información: [Factorio Wiki: Logistic network](https://wiki.factorio.com/Logistic_network) y [Logistic robotics](https://wiki.factorio.com/Logistic_robotics_%28research%29).

##### 6.2. Solicitudes y eliminación automática

Una solicitud logística define la cantidad deseada de un objeto. Según la configuración, puede establecer un mínimo que se quiere mantener y un máximo que no se desea superar.

La **eliminación automática** (*auto-trash*) permite marcar objetos para retirarlos de la entidad logística. Es útil para vaciar materiales que ya no se necesitan, pero hay que configurarla con atención para no retirar recursos que se utilizarán enseguida.

Buenas prácticas:

- Solicita cantidades que reflejen una tarea real de construcción.
- Mantén reservas razonables de materiales esenciales.
- Usa grupos logísticos para alternar conjuntos de solicitudes según el trabajo.
- Evita solicitar grandes cantidades de productos raros si la red logística tiene existencias limitadas.
- Comprueba que la función logística esté activada y que el personaje esté dentro del alcance de una red operativa.

##### 6.3. Red logística: estructuras necesarias

| Estructura | Función |
|---|---|
| **Roboport** | Conecta áreas logísticas y permite que los robots se recarguen y operen dentro de la red. |
| **Robot logístico** | Transporta objetos entre cofres logísticos y entidades con solicitudes. |
| **Cofre proveedor pasivo** | Pone objetos a disposición de la red sin solicitar su retirada inmediata. |
| **Cofre proveedor activo** | Prioriza la retirada de su contenido por parte de los robots logísticos. |
| **Cofre de almacenamiento** | Recibe objetos que la red debe guardar, incluyendo sobrantes devueltos. |
| **Cofre solicitador** | Pide a la red cantidades de objetos configuradas por el jugador. |
| **Cofre buffer** | Combina funciones de solicitud y provisión según su configuración. |

El personaje no necesita un robotport personal para recibir materiales logísticos. Necesita una red logística disponible y solicitudes configuradas. El robotport personal tiene otro propósito: controla robots de construcción para trabajos de montaje y reparación.

---

#### 7. *Spidertron*: inventario, equipamiento y control remoto

##### 7.1. ¿Qué es el Spidertron?


El **Spidertron** es un vehículo avanzado de *Factorio* capaz de desplazarse por terrenos accidentados y salvar muchos obstáculos que dificultan la circulación de vehículos convencionales. También tiene su propio inventario y una cuadrícula de equipamiento configurable.

En calidad normal, dispone de:

- **Inventario:** 80 espacios.
- **Cuadrícula de equipamiento:** 10 × 6 casillas.
- **Nombre y color propios:** pueden utilizarse para distinguir vehículos en una flota.
- **Solicitudes logísticas y eliminación automática:** permiten abastecerlo mediante una red logística disponible.
- **Control remoto:** permite enviar órdenes de movimiento desde el mapa o desde el mundo.

El Spidertron no utiliza combustible convencional para desplazarse. Su movilidad puede mejorarse mediante módulos compatibles, como exoesqueletos. La configuración de equipamiento permite adaptar el vehículo a transporte, construcción remota o desplazamiento.

Más información: [Factorio Wiki: Spidertron](https://wiki.factorio.com/Spidertron).

##### 7.2. El inventario del Spidertron

El inventario del Spidertron es independiente del inventario del personaje mientras el vehículo está desplegado. Puede almacenar objetos que necesita transportar o utilizar, y puede recibir materiales mediante solicitudes logísticas.

Esto permite preparar un vehículo para tareas concretas. Por ejemplo, puede llevar materiales para una construcción, recambios o artículos que se necesitan en una zona alejada de la base.

Hay una particularidad importante al recogerlo: **el equipamiento, el nombre, el color y las solicitudes logísticas configuradas se conservan**, pero el contenido del inventario se transfiere al inventario del personaje y no queda guardado dentro del objeto recogido. Antes de guardar o mover el Spidertron, comprueba que tengas espacio suficiente para su carga.

##### 7.3. Cuadrícula de equipamiento del Spidertron

El Spidertron acepta módulos de equipamiento compatibles con su cuadrícula. Los módulos se colocan en la interfaz de equipamiento del vehículo y sus efectos se suman a los módulos que pudiera llevar el personaje en su propia armadura; son configuraciones separadas.

Usos comunes de la cuadrícula:

- **Exoesqueletos:** aumentan la movilidad del vehículo.
- **Baterías y generadores portátiles:** proporcionan energía a los módulos que la consumen.
- **Robotports personales:** permiten que robots de construcción trabajen con el vehículo como base móvil, dentro de los límites de su sistema.
- **Escudos de energía:** ofrecen protección adicional si hay energía disponible.
- **Equipamiento de cinturón de herramientas (*Space Age*):** amplía la capacidad de inventario del vehículo.

La selección debe responder a la función prevista. Si se utiliza para transportar materiales, la capacidad y la logística pueden ser más importantes que maximizar la movilidad. Si se utiliza como plataforma de construcción remota, conviene reservar espacio para robotports, energía y los módulos que favorezcan el desplazamiento.

##### 7.4. Solicitudes logísticas del Spidertron

Las solicitudes logísticas del Spidertron funcionan de manera similar a las solicitudes personales del jugador. Cuando está dentro del alcance de una red logística operativa, los robots pueden entregar las cantidades solicitadas y retirar objetos definidos como sobrantes.

Una configuración práctica puede pedir:

- Materiales para una obra específica.
- Componentes de construcción que deben reponerse tras utilizarlos.
- Un número controlado de objetos para no llenar el inventario con recursos innecesarios.

Las solicitudes no crean objetos: dependen de que la red tenga existencias disponibles y de que el vehículo esté dentro del alcance logístico. Si el Spidertron se aleja de la red, las entregas pendientes no pueden completarse normalmente hasta que vuelva a una zona válida.

##### 7.5. Control remoto de movimiento


El **control remoto del Spidertron** permite vincular el control a uno o varios vehículos y enviarles órdenes de movimiento desde el mapa o desde el mundo.

Flujo básico:

1. Obtener el control remoto al desbloquear la tecnología correspondiente.
2. Vincularlo con el Spidertron haciendo clic y arrastrando sobre los vehículos que se quieren asociar.
3. Seleccionar el punto de destino con el control remoto.
4. Observar la ruta prevista y comprobar si el destino es accesible.
5. Utilizar varios puntos de paso cuando se necesite dividir un trayecto en etapas.

El movimiento remoto debe planificarse teniendo en cuenta que el Spidertron puede dirigirse en línea recta al destino y no necesariamente rodear obstáculos como lo haría una ruta ferroviaria. Utilizar puntos de paso ayuda a orientar el desplazamiento alrededor de áreas que conviene evitar.

Más información: [Factorio Wiki: Spidertron remote](https://wiki.factorio.com/Spidertron_remote).

##### 7.6. Recoger, transportar y volver a desplegar el Spidertron

Al recoger el vehículo, el juego conserva su cuadrícula de equipamiento y su configuración, pero transfiere sus artículos almacenados al inventario del personaje. Esto es importante cuando se trasladan vehículos entre bases o se empaquetan para reorganizar una zona.

Antes de recogerlo:

- Comprueba cuánto espacio ocupa su contenido en el inventario del personaje.
- Deposita los materiales que no necesites llevar contigo.
- Confirma que los módulos de equipamiento estén configurados como esperas.
- Recuerda que una vez desplegado, sus solicitudes logísticas pueden reabastecerlo si está dentro de una red apropiada.

---

#### 8. Objetos e insumos relacionados con la mecánica

| Objeto | Función | Requisito o consideración |
|---|---|---|
| **Armadura modular** | Permite instalar módulos de equipamiento. | Necesita investigación y fabricación de etapa intermedia. |
| **Armadura de potencia / MK2** | Amplía la cuadrícula y la capacidad de inventario. | Requiere materiales y tecnologías avanzadas. |
| **Armadura mecánica** | Cuadrícula amplia, inventario adicional y capacidad de vuelo. | Exclusiva de *Space Age*. |
| **Panel solar portátil** | Genera energía para la cuadrícula durante la iluminación. | Su producción varía con la ubicación y el ciclo de luz. |
| **Reactor de fisión/fusión portátil** | Proporciona energía para módulos compatibles. | Ocupa una superficie grande dentro de la cuadrícula. |
| **Batería personal** | Almacena energía. | No genera electricidad por sí sola. |
| **Exoesqueleto** | Aumenta la velocidad de movimiento. | Consume energía y ocupa 2 × 4 casillas. |
| **Escudo de energía** | Absorbe daño mientras dispone de energía. | Requiere potencia suficiente para recargarse. |
| **Visión nocturna** | Mejora la visibilidad en la oscuridad. | Necesita una fuente de energía. |
| **Robotport personal** | Permite desplegar robots de construcción. | No sustituye una red de logística personal. |
| **Equipamiento de cinturón de herramientas** | Añade espacios de inventario. | Exclusivo de *Space Age*; ocupa 3 × 1 casillas. |
| **Spidertron** | Vehículo con inventario y cuadrícula propia. | Admite solicitudes logísticas y control remoto. |
| **Control remoto del Spidertron** | Envía órdenes de movimiento. | Debe vincularse previamente al vehículo. |
| **Roboport y robots logísticos** | Forman la infraestructura de logística automatizada. | Necesarios para abastecer solicitudes personales cuando hay red disponible. |

---

#### 9. Estrategias y consejos de optimización

##### 9.1. Configurar el equipamiento según la tarea

Evita intentar mantener una única configuración ideal para todos los trabajos. Una armadura preparada para construir con planos puede necesitar varios robotports personales y materiales disponibles; una orientada al desplazamiento puede priorizar movilidad y almacenamiento de energía.

Antes de cambiar la configuración, pregunta qué limita actualmente el trabajo: la velocidad de desplazamiento, la autonomía eléctrica, el número de robots disponibles, el espacio de inventario o la protección. Modifica primero ese aspecto y comprueba el resultado antes de rediseñar toda la cuadrícula.

##### 9.2. Usar la logística personal para evitar cargar de más

Las solicitudes logísticas pueden convertir el inventario en una reserva de materiales adaptada al trabajo. Es más eficiente solicitar los objetos que se utilizan con frecuencia y dejar el resto en la red logística que llevar todos los posibles materiales a cada lugar.

Mantén un ojo en los límites máximos de cada solicitud: si se solicitan demasiados objetos, el inventario puede llenarse y dificultar la recogida de productos o la recuperación de materiales sobrantes.

##### 9.3. Diferenciar robots de construcción y robots logísticos

Los dos tipos de robot cumplen funciones diferentes:

- **Robots de construcción:** levantan estructuras, reparan y deconstruyen elementos según las órdenes del jugador o los planos. Un robotport personal puede desplegarlos desde el equipamiento del personaje o del Spidertron.
- **Robots logísticos:** entregan y trasladan objetos entre cofres logísticos y entidades que solicitan materiales. Dependen de la red logística y no se despliegan mediante el robotport personal.

Confundir ambos sistemas suele generar la impresión de que el equipamiento está incompleto cuando, en realidad, falta activar o ampliar la red logística correspondiente.

##### 9.4. Ampliar la cuadrícula con criterio

Una cuadrícula de mayor tamaño permite más combinaciones, pero cada módulo tiene un coste de oportunidad. Los generadores y las baterías ocupan espacio que podría dedicarse a movilidad o construcción. Por ello, conviene planificar filas y rectángulos según las dimensiones de los módulos que se van a instalar.

Las armaduras de calidad superior y el equipamiento de mayor calidad pueden ofrecer más espacio o mejores prestaciones, pero no hace falta perseguir la máxima calidad en todas las piezas. Conviene priorizar las mejoras que resuelven un problema real de capacidad, energía o movilidad.

##### 9.5. Mantener un Spidertron listo para la tarea

Un Spidertron preparado para una obra remota debería llevar los materiales necesarios, tener solicitudes logísticas razonables y contar con una cuadrícula configurada para el propósito previsto. Evita cargar el vehículo con objetos que no se usarán y revisa sus solicitudes antes de enviarlo a una zona sin red logística.

Si se va a recoger el vehículo, recuerda que su inventario se vacía hacia el inventario del personaje; planifica el espacio necesario antes de guardarlo.

---

#### 10. Errores comunes y cómo resolverlos

| Problema | Causa habitual | Solución |
|---|---|---|
| No queda espacio para recoger materiales. | El inventario está lleno o contiene muchos objetos innecesarios. | Deposita sobrantes, usa filtros o ajusta las solicitudes logísticas. |
| No se puede retirar la armadura. | Los objetos guardados ocupan espacios que desaparecerían al retirarla. | Libera espacio en el inventario antes de quitar la armadura. |
| Los módulos no funcionan. | La armadura no está equipada, falta energía o el módulo no es compatible. | Revisa el tipo de cuadrícula, la fuente de energía y el estado del equipo. |
| Los exoesqueletos dejan de proporcionar su bonificación. | No hay suficiente energía o están desactivados. | Verifica el suministro eléctrico y el estado de los módulos. |
| Los escudos no se recargan. | La cuadrícula carece de energía suficiente o las baterías están descargadas. | Añade generación o almacenamiento y revisa el consumo. |
| Los robots personales no construyen un plano. | No hay robotport personal activo, faltan materiales o el robot no puede alcanzar el objetivo. | Activa el sistema, comprueba los materiales y revisa el área de construcción. |
| El personaje no recibe objetos pedidos. | No está dentro del alcance de una red logística, la logística está desactivada o no hay existencias. | Entra en el área de la red, activa solicitudes y verifica los cofres disponibles. |
| El Spidertron no recibe sus materiales. | Está lejos de una red logística o las solicitudes no están configuradas. | Revisa el alcance, la disponibilidad de objetos y los límites solicitados. |
| El inventario del Spidertron aparece vacío después de recogerlo. | Su carga se transfirió al inventario del personaje cuando el vehículo se recogió. | Revisa el inventario del personaje y vuelve a cargar el vehículo después de desplegarlo. |
| El control remoto no actúa sobre el vehículo esperado. | El control remoto no está vinculado al Spidertron correcto. | Vuelve a asociarlo y confirma el destino marcado. |
| La cuadrícula no admite otro módulo. | La superficie libre no tiene la forma necesaria o el módulo es demasiado grande. | Reorganiza los módulos considerando dimensiones, no solo casillas totales. |

---

#### 11. Lista de comprobación

Antes de salir de la base o iniciar una construcción extensa, revisa lo siguiente:

- [ ] El inventario tiene espacio para los materiales que se recogerán durante la tarea.
- [ ] Los artículos de uso frecuente están accesibles o reservados mediante filtros.
- [ ] La armadura equipada corresponde al trabajo previsto.
- [ ] Los módulos están colocados en una cuadrícula compatible.
- [ ] Los equipos eléctricos disponen de generación y almacenamiento suficientes.
- [ ] La logística personal está activada si se necesitan entregas automáticas.
- [ ] El personaje está dentro del alcance de una red logística que dispone de los materiales solicitados.
- [ ] El robotport personal está activo si se espera que los robots de construcción trabajen.
- [ ] El Spidertron tiene sus solicitudes, equipamiento y carga preparados para el destino.
- [ ] Hay suficiente espacio en el inventario del personaje antes de recoger el Spidertron.
- [ ] Los controles remotos están vinculados al vehículo que se quiere mover.

---

#### 12. Fuentes de referencia

Las siguientes páginas de la *Factorio Wiki* reúnen datos de recetas, dimensiones, investigación y estadísticas para consultar los valores según la versión del juego:

- [Player — Personaje e inventario](https://wiki.factorio.com/Player)
- [Armor — Armaduras](https://wiki.factorio.com/Armor)
- [Modular armor — Armadura modular](https://wiki.factorio.com/Modular_armor)
- [Power armor — Armadura de potencia](https://wiki.factorio.com/Power_armor)
- [Power armor MK2 — Armadura de potencia MK2](https://wiki.factorio.com/Power_armor_MK2)
- [Mech armor — Armadura mecánica de *Space Age*](https://wiki.factorio.com/Mech_armor)
- [Equipment modules — Módulos de equipamiento](https://wiki.factorio.com/Equipment_modules)
- [Personal roboport — Robotport personal](https://wiki.factorio.com/Personal_roboport)
- [Exoskeleton — Exoesqueleto](https://wiki.factorio.com/Exoskeleton)
- [Toolbelt equipment — Equipamiento de cinturón de herramientas](https://wiki.factorio.com/Toolbelt_equipment)
- [Logistic network — Red logística](https://wiki.factorio.com/Logistic_network)
- [Logistic robotics — Tecnología de robótica logística](https://wiki.factorio.com/Logistic_robotics_%28research%29)
- [Spidertron — Vehículo y cuadrícula de equipamiento](https://wiki.factorio.com/Spidertron)
- [Spidertron remote — Control remoto](https://wiki.factorio.com/Spidertron_remote)

"""

print("Conectando con el servidor local de Ollama...")
# 2. Inicializamos los modelos con las clases vigentes
llm = OllamaLLM(model="qwen2.5:7b")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Ruta permanente dentro de tu Google Drive para almacenar los índices vectoriales
persist_directory = "/content/drive/MyDrive/engi_ai_rag/chroma_db"

if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)

# 3. Procesamos e Indexamos en ChromaDB de forma persistente
if len(os.listdir(persist_directory)) == 0:
    print("Procesando tus guías de Factorio y creando base de datos vectorial en Drive por única vez...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
    docs = text_splitter.create_documents([CONOCIMIENTO_FACTORIO])

    vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory=persist_directory)
    print("¡Base de datos vectorial guardada con éxito en Google Drive!")
else:
    print("¡Se detectó una base de datos existente! Cargando índices vectoriales desde Google Drive...")
    vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

# Configuramos a Chroma como nuestro recuperador RAG
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 4. Prompt de sistema avanzado para forzar flexibilidad y Markdown largo
system_instruction = (
    "Eres EngiAI, un asistente experto en las mecánicas de Factorio 2.x y la expansión Space Age.\n"
    "Usa el contexto técnico provisto únicamente como tu fuente de hechos verídicos y proporciones exactas.\n\n"
    "DIRECTRICES DE REDACCIÓN (OBLIGATORIAS):\n"
    "1. FLEXIBILIDAD: Prohibido copiar y pegar texto de forma literal. Debes reformular la información, elegir sinónimos y redactar respuestas fluidas con coherencia propia y sentido lógico.\n"
    "2. DETALLE Y EXTENSIÓN: Genera respuestas largas, profundas y explicativas paso a paso si la pregunta requiere mucho detalle.\n"
    "3. FORMATO MARKDOWN IMPECABLE: Estructura obligatoriamente tus respuestas usando títulos (###), subtítulos, listas limpias con viñetas, tablas comparativas cuando sea necesario y **negritas** en los términos clave.\n\n"
    "Contexto extraído de tu base de datos de Factorio:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", "{question}"),
])

# 5. Función auxiliar para formatear los documentos extraídos de Chroma
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 6. CONSTRUCCIÓN DE LA CADENA CON SINTAXIS MODERNA LCEL (Operador '|')
# Reemplaza por completo el antiguo e inexistente create_retrieval_chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 7. Función definitiva para chatear con EngiAI
def preguntar(texto_usuario):
    print("EngiAI está analizando los vectores...")
    respuesta_final = rag_chain.invoke(texto_usuario)
    return respuesta_final

print("¡SISTEMA ACTUALIZADO Y LISTO! Ya podés usar la función preguntar() abajo.")


Conectando con el servidor local de Ollama...
¡Se detectó una base de datos existente! Cargando índices vectoriales desde Google Drive...
¡SISTEMA ACTUALIZADO Y LISTO! Ya podés usar la función preguntar() abajo.


In [46]:
import subprocess
import time

print("Despertando el servidor de Ollama en segundo plano...")
# Ejecutamos el serve de forma directa e independiente en la máquina virtual
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Esperamos 5 segundos a que el puerto se estabilice
print("¡Servidor reconectado con éxito!")


Despertando el servidor de Ollama en segundo plano...
¡Servidor reconectado con éxito!


In [47]:
from flask import Flask, request, make_response
from flask_cors import CORS
from pyngrok import ngrok

NGROK_TOKEN = 'AUTH_TOKEN';

# ngrok.set_auth_token(NGROK_TOKEN)

ngrok.kill()
public_url = ngrok.connect(5000)
print("\n" + "="*50)
print(f" URL:\n {public_url.public_url}/api/chat")
print("="*50 + "\n")

app = Flask(__name__)
CORS(app, resources={r"/api/*": {"origins": "*", "allow_headers": "*", "methods": ["POST", "OPTIONS"]}})

# INTERCEPTOR DE SEGURIDAD (CORS): Responde con éxito a la verificación Preflight del navegador
@app.route("/api/chat", methods=["POST", "OPTIONS"])
def chat():
    # 1. Si el navegador pregunta por seguridad (Preflight OPTIONS)
    if request.method == "OPTIONS":
        respuesta = make_response()
        respuesta.headers.add("Access-Control-Allow-Origin", "*")
        respuesta.headers.add("Access-Control-Allow-Headers", "*")
        respuesta.headers.add("Access-Control-Allow-Methods", "POST, OPTIONS")
        return respuesta, 200

    # 2. Si es la petición real de tu ia (POST)
    datos = request.get_json(silent=True) or {}
    mensaje_usuario = datos.get("message", "")

    if not mensaje_usuario:
        return "Error: Mensaje vacío", 400, {'Content-Type': 'text/plain; charset=utf-8'}

    respuesta_ia = preguntar(mensaje_usuario)
    # respuesta_ia = respuesta_ia.replace("[[NL]]", "\n")

    # Creamos la respuesta POST y le aseguramos los encabezados de CORS por si acaso
    respuesta_ia = make_response(respuesta_ia, 200)
    respuesta_ia.headers["Content-Type"] = "text/plain; charset=utf-8"
    respuesta_ia.headers["Access-Control-Allow-Origin"] = "*"

    return respuesta_ia

if __name__ == "__main__":
    app.run(port=5000, debug=False)


 URL:
 https://swore-throwing-status.ngrok-free.dev/api/chat

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Oct/2026 20:23:32] "OPTIONS /api/chat HTTP/1.1" 200 -


EngiAI está analizando los vectores...


INFO:werkzeug:127.0.0.1 - - [25/Oct/2026 20:23:42] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Oct/2026 20:24:18] "OPTIONS /api/chat HTTP/1.1" 200 -


EngiAI está analizando los vectores...


INFO:werkzeug:127.0.0.1 - - [25/Oct/2026 20:24:22] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Oct/2026 20:25:10] "OPTIONS /api/chat HTTP/1.1" 200 -


EngiAI está analizando los vectores...


INFO:werkzeug:127.0.0.1 - - [25/Oct/2026 20:25:42] "POST /api/chat HTTP/1.1" 200 -
